# Detección de anomalías con TracIn, explicada desde cero
## Qué hizo el Equipo 6, por qué, cómo está programado y qué tan bien funciona

**Equipo 6 · Aprendizaje no supervisado · INAOE**

Este cuaderno es para alguien que **no ha leído el artículo** ni nuestro código. No da nada por sabido: cada
concepto se explica con palabras simples, con un ejemplo de juguete y con el código real que usamos. Al
terminarlo deberías poder:

1. Explicar con tus palabras cómo detecta anomalías el artículo (Verdone et al., 2026, *TracIn* para series de tiempo).
2. Explicar cada una de nuestras **tres mejoras**: qué problema ataca, cómo lo resuelve y en qué líneas de código está.
3. Explicar el **método final** (las tres juntas) y **en qué casos funciona y en cuáles no**.
4. Entender cómo lo evaluamos (pruebas estadísticas incluidas) y leer todas las tablas de resultados.
5. Correrlo tú mismo en un dataset real o en tu propio archivo CSV.

---

### Cómo usar este cuaderno

* Ve al menú **Entorno de ejecución → Ejecutar todo** (o `Ctrl + F9`). No necesitas GPU ni subir archivos.
* Todo el cuaderno tarda **unos 20–30 minutos** en la CPU gratuita de Colab. Mientras corre puedes ir leyendo:
  cada sección imprime lo que va haciendo.
* Las celdas con un título y ▶ (por ejemplo *"Código del método"*) están **plegadas** porque son largas. Puedes
  abrirlas si quieres ver el código completo, pero **no hace falta**: en el texto te explicamos las partes
  importantes línea por línea.
* Si algo falla, vuelve a ejecutar desde la sección 0.

### Mapa del cuaderno

| Parte | Sección | Qué vas a aprender |
|---|---|---|
| **A. Lo básico** | 0 | Preparar el entorno (sólo ejecutar) |
| | 1 | Conceptos básicos: serie de tiempo, anomalía, red neuronal, gradiente, época… con ejemplos de juguete |
| | 2 | Los datos: cómo se fabrican series con anomalías y qué tipos de anomalía existen |
| **B. El artículo** | 3 | El método del artículo paso a paso: ventanas → LSTM → TracIn → proyección → umbral → métricas |
| **C. Nuestras mejoras** | 4 | **Mejora 1 (M1):** un umbral que no necesita etiquetas |
| | 5 | **Mejora 2 (M2):** quitar lo sospechoso del entrenamiento y volver a entrenar |
| | 6 | **Mejora 3 (M3):** proyección al objetivo, el "halo" y por qué importa tanto |
| | 7 | **Método final = M1 + M2 + M3**, y un laboratorio interactivo |
| **D. ¿Funciona?** | 8 | Cómo se evalúa bien: protocolo, hipótesis, pruebas estadísticas (con un mini-experimento en vivo) |
| | 9 | **Todos los resultados** de la evaluación grande (1 755 corridas), también los negativos |
| **E. Úsalo tú** | 10 | Pruébalo en datos reales (SMD, descargados en vivo) y en tu propio CSV |
| | 11 | Cómo está organizado el código del proyecto y cómo reproducir los experimentos |
| | 12 | Resumen, glosario y preguntas que te pueden hacer (con respuestas) |

### La historia del proyecto en un minuto

1. **El artículo** propone detectar anomalías en series de tiempo así: se entrena una red neuronal (LSTM) a
   predecir la serie y se mide **cuánto le costó aprender cada pedazo** (eso es *TracIn*). Lo que más cuesta
   aprender es lo raro.
2. **Reprodujimos** el artículo y encontramos tres puntos débiles:
   * su mejor umbral **usa las respuestas** (las etiquetas), cosa que en un problema real no se tiene;
   * cuando hay **muchas anomalías** (datos "contaminados"), la red las aprende y deja de verlas como raras;
   * su fórmula para pasar de "score por ventana" a "score por instante" **reparte la culpa** de una anomalía
     entre sus vecinos normales.
3. Propusimos **una mejora para cada punto** (M1, M2, M3) y las **combinamos** en un método final que no usa
   ninguna etiqueta.
4. Lo evaluamos con un **protocolo escrito antes de correr** (para no hacer trampa sin querer), en 6 bloques de
   datos, con pruebas estadísticas.
5. **Resultado:** la combinación funciona muy bien con series contaminadas y anomalías cortas, y explicamos
   **por qué** (el "halo"). Pero **no** funciona en benchmarks donde las anomalías son tramos muy largos (SMD,
   SMAP/MSL). Lo reportamos tal cual.

# PARTE A · Lo básico

## 0. Preparación

Ejecuta las celdas de esta sección y sigue adelante. Qué hace cada una:

1. **Librerías.** Carga las herramientas que usamos:
   * `numpy` (cálculo con arreglos de números), `pandas` (tablas), `matplotlib` (gráficas);
   * `torch` (**PyTorch**, para la red neuronal);
   * `scipy` (pruebas estadísticas) y `sklearn` (métricas como AUPRC y AUROC).
2. **Generador de series sintéticas.** Es el mismo generador que usa el artículo (de Lai et al., 2021). Fabrica
   series con anomalías de tipos conocidos, así sabemos exactamente dónde están.
3. **Código del método.** La red LSTM y el cálculo de TracIn, **copiados literalmente** de nuestro archivo
   `src/replica/lab.py`. No es una versión simplificada: es exactamente lo que corrimos.
4. **Código de las mejoras.** Las funciones de proyección, umbral IQR, reponderación y recorte (de `src/common.py`,
   `src/forecaster.py` y `src/final/core.py`), y el inyector de anomalías sobre datos reales (`src/data_bench.py`).
5. **Funciones auxiliares** para dibujar y medir.

In [ ]:
# @title 1. Librerías
import time, warnings, math, io, urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from scipy.stats import wilcoxon, spearmanr
from sklearn.metrics import average_precision_score, roc_auc_score
from IPython.display import display, clear_output, Markdown
warnings.filterwarnings("ignore")
pd.set_option("display.precision", 3)
pd.set_option("display.max_colwidth", 120)
torch.set_num_threads(max(1, torch.get_num_threads()))
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False, "font.size": 10})
print("Listo. PyTorch", torch.__version__)

In [ ]:
# @title 2. Generador de series sintéticas (el mismo benchmark del artículo, Lai et al. 2021)
from __future__ import annotations
"""Generador de series sintéticas multivariadas con anomalías.

Copia fiel del generador oficial del benchmark de Lai et al. (2021),
``tods/benchmark/synthetic/Generator/multivariate_generator.py`` (rama
``benchmark`` de github.com/datamllab/tods), que es el que usa el artículo.
Sólo se cambió: (1) configuraciones copiadas por dataset para evitar efectos
colaterales entre llamadas y (2) una función ``make_dataset`` que arma los
15 datasets del artículo con los parámetros del ``__main__`` original.

Tipos de anomalía (Lai et al., 2021; Fig. 1 del artículo):
    0 = puntual global, 1 = puntual contextual, 2 = shapelet (colectiva global),
    3 = estacional, 4 = de tendencia.
El tipo t se inyecta en el canal t, como en el ``__main__`` original.
"""

import copy

import numpy as np


def sine(length, freq=0.04, coef=1.5, offset=0.0, noise_amp=0.05):
    timestamp = np.arange(length)
    value = np.sin(2 * np.pi * freq * timestamp)
    if noise_amp != 0:
        value = value + noise_amp * np.random.normal(0, 1, length)
    return coef * value + offset


def cosine(length, freq=0.04, coef=1.5, offset=0.0, noise_amp=0.05):
    timestamp = np.arange(length)
    value = np.cos(2 * np.pi * freq * timestamp)
    if noise_amp != 0:
        value = value + noise_amp * np.random.normal(0, 1, length)
    return coef * value + offset


def square_sine(level=5, length=500, freq=0.04, coef=1.5, offset=0.0, noise_amp=0.05):
    value = np.zeros(length)
    for i in range(level):
        value += 1 / (2 * i + 1) * sine(length=length, freq=freq * (2 * i + 1), coef=coef,
                                        offset=offset, noise_amp=noise_amp)
    return value


class MultivariateDataGenerator:
    def __init__(self, dim, stream_length, behavior, behavior_config):
        self.dim = dim
        self.STREAM_LENGTH = stream_length
        self.behavior = behavior
        self.behavior_config = copy.deepcopy(behavior_config)
        self.data = np.empty(shape=[0, stream_length], dtype=float)
        self.generate_timeseries()

    def generate_timeseries(self):
        for i in range(self.dim):
            self.behavior_config[i]["length"] = self.STREAM_LENGTH
            self.data = np.append(self.data, [self.behavior[i](**self.behavior_config[i])], axis=0)
        self.data_origin = self.data.copy()
        self.label = np.zeros(self.STREAM_LENGTH, dtype=int)

    def point_global_outliers(self, dim_no, ratio, factor, radius):
        position = (np.random.rand(round(self.STREAM_LENGTH * ratio)) * self.STREAM_LENGTH).astype(int)
        maximum, minimum = max(self.data[dim_no]), min(self.data[dim_no])
        for i in position:
            local_std = self.data_origin[dim_no][max(0, i - radius):min(i + radius, self.STREAM_LENGTH)].std()
            self.data[dim_no][i] = self.data_origin[dim_no][i] * factor * local_std
            if 0 <= self.data[dim_no][i] < maximum:
                self.data[dim_no][i] = maximum
            if 0 > self.data[dim_no][i] > minimum:
                self.data[dim_no][i] = minimum
            self.label[i] = 1

    def point_contextual_outliers(self, dim_no, ratio, factor, radius):
        position = (np.random.rand(round(self.STREAM_LENGTH * ratio)) * self.STREAM_LENGTH).astype(int)
        maximum, minimum = max(self.data[dim_no]), min(self.data[dim_no])
        for i in position:
            local_std = self.data_origin[dim_no][max(0, i - radius):min(i + radius, self.STREAM_LENGTH)].std()
            self.data[dim_no][i] = self.data_origin[dim_no][i] * factor * local_std
            if self.data[dim_no][i] > maximum:
                self.data[dim_no][i] = maximum * min(0.95, abs(np.random.normal(0, 1)))
            if self.data[dim_no][i] < minimum:
                self.data[dim_no][i] = minimum * min(0.95, abs(np.random.normal(0, 1)))
            self.label[i] = 1

    def collective_global_outliers(self, dim_no, ratio, radius, coef=3.0, noise_amp=0.0,
                                   level=5, freq=0.04, offset=0.0):
        position = (np.random.rand(round(self.STREAM_LENGTH * ratio / (2 * radius))) * self.STREAM_LENGTH).astype(int)
        sub_data = square_sine(level=level, length=self.STREAM_LENGTH, freq=freq,
                               coef=coef, offset=offset, noise_amp=noise_amp)
        for i in position:
            start, end = max(0, i - radius), min(self.STREAM_LENGTH, i + radius)
            self.data[dim_no][start:end] = sub_data[start:end]
            self.label[start:end] = 1

    def collective_trend_outliers(self, dim_no, ratio, factor, radius):
        position = (np.random.rand(round(self.STREAM_LENGTH * ratio / (2 * radius))) * self.STREAM_LENGTH).astype(int)
        for i in position:
            start, end = max(0, i - radius), min(self.STREAM_LENGTH, i + radius)
            slope = np.random.choice([-1, 1]) * factor * np.arange(end - start)
            self.data[dim_no][start:end] = self.data_origin[dim_no][start:end] + slope
            self.data[dim_no][end:] = self.data[dim_no][end:] + slope[-1]
            self.label[start:end] = 1

    def collective_seasonal_outliers(self, dim_no, ratio, factor, radius):
        position = (np.random.rand(round(self.STREAM_LENGTH * ratio / (2 * radius))) * self.STREAM_LENGTH).astype(int)
        seasonal_config = self.behavior_config[dim_no]
        seasonal_config["freq"] = factor * self.behavior_config[dim_no]["freq"]
        for i in position:
            start, end = max(0, i - radius), min(self.STREAM_LENGTH, i + radius)
            self.data[dim_no][start:end] = self.behavior[dim_no](**seasonal_config)[start:end]
            self.label[start:end] = 1


BEHAVIOR = [sine, cosine, sine, cosine, sine]
BEHAVIOR_CONFIG = [
    {"freq": 0.04, "coef": 1.5, "offset": 0.0, "noise_amp": 0.05},
    {"freq": 0.04, "coef": 2.5, "offset": 0.0, "noise_amp": 0.05},
    {"freq": 0.04, "coef": 1.5, "offset": 0.0, "noise_amp": 0.05},
    {"freq": 0.04, "coef": 2.5, "offset": 2.0, "noise_amp": 0.05},
    {"freq": 0.04, "coef": 1.5, "offset": -2.0, "noise_amp": 0.05},
]

# Los 15 datasets del artículo (Tabla 2).
DATASETS = ["0", "1", "2", "3", "4", "01", "12", "23", "34",
            "012", "123", "234", "0123", "1234", "01234"]


def make_dataset(types: str, seed: int, length: int = 400, ratio: float = 0.05):
    """Genera la serie ``types`` (p. ej. "012") con la semilla ``seed``.
    ``ratio`` es la proporción por tipo (0.05 en el benchmark original)."""
    np.random.seed(seed)
    g = MultivariateDataGenerator(dim=5, stream_length=length, behavior=BEHAVIOR,
                                  behavior_config=BEHAVIOR_CONFIG)
    # Mismo orden de inyección que en el __main__ original.
    if "0" in types:
        g.point_global_outliers(dim_no=0, ratio=ratio, factor=3.5, radius=5)
    if "1" in types:
        g.point_contextual_outliers(dim_no=1, ratio=ratio, factor=2.5, radius=5)
    if "2" in types:
        g.collective_global_outliers(dim_no=2, ratio=ratio, radius=5, coef=1.5, noise_amp=0.03,
                                     level=20, freq=0.04, offset=0.0)
    if "3" in types:
        g.collective_seasonal_outliers(dim_no=3, ratio=ratio, factor=3, radius=5)
    if "4" in types:
        g.collective_trend_outliers(dim_no=4, ratio=ratio, factor=0.5, radius=5)
    return g.data.T.astype(np.float32), g.label.astype(np.int8)

In [ ]:
# @title 3. Código del método: LSTM + TracIn (copiado de src/replica/lab.py)
from __future__ import annotations
import math
from dataclasses import dataclass, asdict
from torch import nn
from torch.func import functional_call, grad, vmap

@dataclass(frozen=True)
class Cfg:
    lr: float = 1e-3
    batch: int = 32
    max_epochs: int = 100
    patience: int = 10
    hidden: int = 64
    sched: bool = True
    val_frac: float = 0.0
    norm: str = "z"
    grad: str = "last"
    lr_weight: bool = True

    def tag(self) -> str:
        d = asdict(self)
        base = asdict(Cfg())
        diff = {k: v for k, v in d.items() if v != base[k]}
        return "base" if not diff else ",".join(f"{k}={v}" for k, v in diff.items())


class LSTM(nn.Module):
    """LSTM de una capa + capa lineal final (como en ``forecaster.py``)."""

    def __init__(self, C: int, P: int, hidden: int):
        super().__init__()
        self.P, self.C, self.H = P, C, hidden
        self.ih = nn.Linear(C, 4 * hidden)
        self.hh = nn.Linear(hidden, 4 * hidden)
        self.head = nn.Linear(hidden, P * C)
        k = 1 / math.sqrt(hidden)
        for p in list(self.ih.parameters()) + list(self.hh.parameters()):
            nn.init.uniform_(p, -k, k)

    def forward(self, x):  # x: (B, L, C)
        B = x.shape[0]
        h = x.new_zeros(B, self.H)
        c = x.new_zeros(B, self.H)
        gx = self.ih(x)
        for t in range(x.shape[1]):
            i, f, g, o = (gx[:, t] + self.hh(h)).chunk(4, dim=1)
            c = torch.sigmoid(f) * c + torch.sigmoid(i) * torch.tanh(g)
            h = torch.sigmoid(o) * torch.tanh(c)
        return self.head(h).view(B, self.P, self.C), h


def normalize(X: np.ndarray, how: str) -> np.ndarray:
    if how == "z":
        sd = X.std(0)
        return (X - X.mean(0)) / np.where(sd > 0, sd, 1)
    if how == "minmax":
        lo, hi = X.min(0), X.max(0)
        return (X - lo) / np.where(hi > lo, hi - lo, 1)
    return X


def windows(Xt, idx, L, P):
    ar = torch.arange(L + P, device=Xt.device)
    w = Xt[idx[:, None] + ar]
    return w[:, :L], w[:, L:]


def _per_sample_sqgrad(model, Xt, M, L, P, bs=4096):
    """||grad_theta l(x)||^2 de cada ventana sobre TODOS los parámetros."""
    params = {k: v.detach() for k, v in model.named_parameters()}

    def loss_fn(p, xb, yb):
        pred, _ = functional_call(model, p, (xb[None],))
        return ((pred[0] - yb) ** 2).mean()

    g_fn = vmap(grad(loss_fn), in_dims=(None, 0, 0))
    out = []
    for s in range(0, M, bs):
        idx = torch.arange(s, min(s + bs, M), device=Xt.device)
        xb, yb = windows(Xt, idx, L, P)
        g = g_fn(params, xb, yb)
        out.append(sum((v ** 2).flatten(1).sum(1) for v in g.values()))
    return torch.cat(out)


@torch.no_grad()
def _stats(model, Xt, M, L, P, bs=8192):
    loss, hsq = [], []
    for s in range(0, M, bs):
        idx = torch.arange(s, min(s + bs, M), device=Xt.device)
        xb, yb = windows(Xt, idx, L, P)
        pred, h = model(xb)
        loss.append(((pred - yb) ** 2).mean(dim=(1, 2)))
        hsq.append((h ** 2).sum(1))
    return torch.cat(loss), torch.cat(hsq)


def run(X: np.ndarray, L: int, P: int, seed: int, cfg: Cfg, device: str = "cpu",
        weights: np.ndarray | None = None, keep_g2: bool = False, callback=None) -> dict:
    """Entrena y devuelve scores por ventana: self-influence (según cfg),
    pérdida final y pérdida por época (para diagnósticos). Con keep_g2 devuelve
    también la contribución de cada época, η_k‖∇ℓ‖² (épocas × ventanas).
    ``callback(época, si_acumulada, pérdida_por_ventana)`` (opcional) se llama
    al final de cada época; sólo sirve para mostrar el avance (notebook)."""
    torch.manual_seed(seed)
    np.random.seed(seed)
    Xn = normalize(X.astype(np.float64), cfg.norm)
    n, C = Xn.shape
    M = n - L - P + 1
    Xt = torch.as_tensor(Xn, dtype=torch.float32, device=device)
    model = LSTM(C, P, cfg.hidden).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=cfg.lr)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, factor=0.9, patience=3) if cfg.sched else None
    n_val = int(M * cfg.val_frac)
    tr_idx = torch.arange(M - n_val, device=device)
    va_idx = torch.arange(M - n_val, M, device=device)
    w_all = torch.ones(M, device=device) if weights is None else torch.as_tensor(weights, dtype=torch.float32, device=device)
    gen = torch.Generator().manual_seed(seed)
    D = P * C
    si = torch.zeros(M, device=device)
    lrs, losses, g2s = [], [], []
    best, bad = math.inf, 0
    for _ in range(cfg.max_epochs):
        lr_now = opt.param_groups[0]["lr"]
        perm = tr_idx[torch.randperm(len(tr_idx), generator=gen).to(device)]
        model.train()
        for b in perm.split(cfg.batch):
            xb, yb = windows(Xt, b, L, P)
            pred, _ = model(xb)
            l = ((pred - yb) ** 2).mean(dim=(1, 2))
            wb = w_all[b]
            loss = (l * wb).sum() / wb.sum().clamp_min(1e-12)
            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()
        model.eval()
        l_all, h_all = _stats(model, Xt, M, L, P)
        if cfg.grad == "last":
            g2 = (4.0 / D) * l_all * (1 + h_all)
        else:
            g2 = _per_sample_sqgrad(model, Xt, M, L, P)
        si += (lr_now if cfg.lr_weight else 1.0) * g2
        if keep_g2:
            g2s.append(((lr_now if cfg.lr_weight else 1.0) * g2).cpu().numpy().astype(np.float32))
        lrs.append(lr_now)
        losses.append(l_all.cpu().numpy().astype(np.float32))
        if callback is not None:
            callback(len(lrs), si.cpu().numpy().astype(np.float64), losses[-1])
        ref = va_idx if n_val > 0 else tr_idx
        mon = float((l_all[ref] * w_all[ref]).sum() / w_all[ref].sum())
        if sched is not None:
            sched.step(mon)
        if mon < best * (1 - 1e-4):
            best, bad = mon, 0
        else:
            bad += 1
            if bad >= cfg.patience:
                break
    out = {"si": si.cpu().numpy().astype(np.float64), "loss_final": losses[-1].astype(np.float64),
           "loss": np.stack(losses), "epochs": len(lrs)}
    if keep_g2:
        out["g2"] = np.stack(g2s)
    return out

In [ ]:
# @title 4. Código de las mejoras (copiado de src/common.py, src/forecaster.py, src/final/core.py, src/data_bench.py)
from __future__ import annotations

EDGE = 19


def project_sum(sample_scores: np.ndarray, span: int, n: int) -> np.ndarray:
    """Ec. (4) del artículo: cada instante suma el score de todas las
    ventanas que lo contienen. La ventana i cubre [i, i + span)."""
    s = np.asarray(sample_scores, dtype=np.float64)
    acc = np.zeros(n + 1)
    m = len(s)
    acc[:m] += s
    acc[span:span + m] -= s
    return np.cumsum(acc)[:n]


def project_targets_mean(sample_scores: np.ndarray, L: int, P: int, n: int) -> np.ndarray:
    """Proyección clásica de *prediction deviation*: el error de la ventana i
    se asigna sólo a los instantes que predice, [i + L, i + L + P), y se
    promedia sobre las predicciones que cubren cada instante."""
    s = np.asarray(sample_scores, dtype=np.float64)
    m = len(s)
    acc = np.zeros(n + 1)
    cnt = np.zeros(n + 1)
    acc[L:L + m] += s
    acc[L + P:L + P + m] -= s
    cnt[L:L + m] += 1
    cnt[L + P:L + P + m] -= 1
    acc, cnt = np.cumsum(acc)[:n], np.cumsum(cnt)[:n]
    out = np.zeros(n)
    nz = cnt > 0
    out[nz] = acc[nz] / cnt[nz]
    return out


def valid_mask(n: int, edge: int = EDGE) -> np.ndarray:
    m = np.zeros(n, dtype=bool)
    m[edge:n - edge] = True
    return m


def thr_iqr(s: np.ndarray, k: float = 1.5) -> float:
    q1, q3 = np.percentile(s, [25, 75])
    return float(q3 + k * (q3 - q1))


def robust_z(v: np.ndarray) -> np.ndarray:
    med = np.median(v)
    mad = 1.4826 * np.median(np.abs(v - med))
    return (v - med) / max(mad, 1e-12)


def weights_from_influence(s: np.ndarray, mode: str, z0: float = 2.0, w_min: float = 0.05) -> np.ndarray:
    """Pesos por ventana a partir de su self-influence (escala log, z robusto).

    * ``soft``: w = 1 si z <= z0;  w = max(w_min, exp(-(z - z0))) si z > z0.
      Baja el peso gradualmente sin descartar ninguna ventana.
    * ``trim``: w = 0 si z > z0 (recorte duro al estilo ITLM, Shen &
      Sanghavi, 2019). Se incluye como contraste de la versión suave.
    z0 = 2 y w_min = 0.05 se fijan a priori (no se ajustan con etiquetas)."""
    z = robust_z(np.log(np.maximum(s, 1e-30)))
    if mode == "soft":
        return np.where(z <= z0, 1.0, np.maximum(w_min, np.exp(-(z - z0))))
    if mode == "trim":
        return np.where(z <= z0, 1.0, 0.0)
    raise ValueError(mode)


def adapt_weights(si: np.ndarray) -> np.ndarray:
    """Mejoras 1+2: se descartan las ventanas cuya self-influence supera el
    umbral IQR de la mejora 1 (idéntico a rep_contam_fix.py)."""
    return np.where(si > thr_iqr(si), 0.0, 1.0)


def standardize(X: np.ndarray) -> np.ndarray:
    X = X.astype(np.float64)
    sd = X.std(0)
    sd[sd == 0] = 1.0
    return ((X - X.mean(0)) / sd).astype(np.float32)


SEG = 10   # longitud de los eventos colectivos (2·radius, radius = 5 como en Lai et al.)


def inject(bg: np.ndarray, mix: str, ratio: float, seed: int) -> tuple[np.ndarray, np.ndarray]:
    """bg: fondo (n × C) ya estandarizado. ``ratio`` = proporción nominal total
    de instantes anómalos; cada tipo de ``mix`` recibe ratio / len(mix).
    Devuelve (X, y). El canal de cada evento se elige al azar."""
    rng = np.random.default_rng(seed)
    X = bg.astype(np.float64).copy()
    n, C = X.shape
    y = np.zeros(n, np.int8)
    lo, hi = X.min(0), X.max(0)
    sd = X.std(0)
    sd[sd == 0] = 1.0
    r = ratio / len(mix)
    for t in mix:
        if t in "01":
            for i in rng.integers(0, n, round(n * r)):
                c = rng.integers(C)
                if t == "0":   # puntual global: fuera del rango del canal
                    s = rng.choice([-1, 1])
                    X[i, c] = (hi[c] if s > 0 else lo[c]) + s * rng.uniform(0.5, 1.5) * sd[c]
                else:          # puntual contextual: lejos de su vecindad, dentro del rango global
                    w = bg[max(0, i - 5):i + 5, c]
                    ls = max(w.std(), 0.1 * sd[c])
                    X[i, c] = np.clip(w.mean() + rng.choice([-1, 1]) * 2.5 * ls * rng.uniform(1, 2), lo[c], hi[c])
                y[i] = 1
        else:
            for i in rng.integers(0, n - SEG, round(n * r / SEG)):
                c = rng.integers(C)
                seg = slice(i, i + SEG)
                base = bg[seg, c].astype(np.float64)
                if t == "2":   # shapelet: onda cuadrada de amplitud 1.5 d.e.
                    k = np.arange(SEG)
                    X[seg, c] = base.mean() + 1.5 * sd[c] * np.sign(np.sin(2 * np.pi * k / 5 + 0.1))
                elif t == "3":  # estacional: patrón local acelerado ×3
                    src = bg[i:min(n, i + 3 * SEG):3, c]
                    X[i:i + len(src), c] = src
                else:          # tendencia: rampa de 0.5 d.e. por paso
                    X[seg, c] = base + rng.choice([-1, 1]) * 0.5 * sd[c] * np.arange(SEG)
                y[seg] = 1
    return X.astype(np.float32), y

In [ ]:
# @title 5. Funciones auxiliares para dibujar y medir
TIPOS = {"0": "puntual global", "1": "puntual contextual", "2": "de forma (shapelet)",
         "3": "estacional", "4": "de tendencia"}
CFG = Cfg()           # configuración que reproduce al artículo en sintéticos (datos estandarizados, última capa)
L, P = 4, 1           # el LSTM ve 4 instantes y predice el siguiente


def intervalos(y):
    """Tramos [inicio, fin) donde y == 1."""
    d = np.diff(np.r_[0, np.asarray(y).astype(int), 0])
    return list(zip(np.where(d == 1)[0], np.where(d == -1)[0]))


def sombrear(ax, y, color="#f4a3a3", alpha=0.35, label="anomalía real"):
    for i, (a, b) in enumerate(intervalos(y)):
        ax.axvspan(a - 0.5, b - 0.5, color=color, alpha=alpha, lw=0, label=label if i == 0 else None)


def dibujar_serie(X, y, titulo="", ax=None, max_canales=6):
    if ax is None:
        _, ax = plt.subplots(figsize=(12, 3.2))
    Xn = normalize(X.astype(np.float64), "z")
    for c in range(min(Xn.shape[1], max_canales)):
        ax.plot(Xn[:, c] - 4 * c, lw=0.8, label=f"canal {c}")
    sombrear(ax, y)
    ax.set_yticks([]); ax.set_xlabel("tiempo"); ax.set_title(titulo)
    ax.legend(loc="upper right", fontsize=7, ncol=7)
    return ax


def proyectar(s_ventana, n, objetivo):
    """Score por ventana -> score por instante.
    objetivo=False: Ec. 4 del artículo (suma sobre los L+P instantes de la ventana).
    objetivo=True : mejora 3 (sólo el instante predicho)."""
    return project_targets_mean(s_ventana, L, P, n) if objetivo else project_sum(s_ventana, L + P, n)


def f1(y, pred):
    y = np.asarray(y).astype(bool); pred = np.asarray(pred).astype(bool)
    tp = np.sum(y & pred); fp = np.sum(~y & pred); fn = np.sum(y & ~pred)
    return 2 * tp / (2 * tp + fp + fn) if tp + fp + fn else 0.0


def umbral_dinamico(y, s):
    """Umbral del artículo: elige, CON ETIQUETAS, el percentil que maximiza el F1 en la 1.ª mitad."""
    k = len(s) // 2
    mejor, thr = -1, None
    for p in np.arange(50, 99.95, 0.1):
        t = np.percentile(s, p)
        f = f1(y[:k], s[:k] >= t)
        if f > mejor:
            mejor, thr = f, t
    return thr


def evaluar(y, s, thr):
    """Métricas en los instantes válidos. El F1 se mide en la 2.ª mitad (la 1.ª la usa el umbral dinámico)."""
    m = valid_mask(len(y)); y, s = np.asarray(y)[m].astype(bool), s[m]
    k = len(s) // 2
    pred = s[k:] >= thr
    tp = np.sum(y[k:] & pred)
    return {"AUPRC": average_precision_score(y, s), "AUROC": roc_auc_score(y, s),
            "% marcado": pred.mean(), "precisión": tp / max(pred.sum(), 1),
            "exhaustividad": tp / max(y[k:].sum(), 1), "F1": f1(y[k:], pred)}


def dibujar_deteccion(y, s, thr, titulo, ax):
    ax.plot(s, color="#3182bd", lw=0.9, label="score")
    sombrear(ax, y)
    ax.axhline(thr, color="k", ls="--", lw=1, label="umbral")
    y = np.asarray(y).astype(bool); pred = s >= thr
    ax.plot(np.where(pred & y)[0], s[pred & y], "o", ms=3, color="#31a354", label="acierto")
    ax.plot(np.where(pred & ~y)[0], s[pred & ~y], "x", ms=4, color="#e6550d", label="falsa alarma")
    ax.axvline(len(y) // 2, color="gray", lw=0.8); ax.set_title(titulo); ax.legend(loc="upper right", fontsize=7, ncol=5)


print("Funciones auxiliares listas.")

## 1. Conceptos básicos (si ya los sabes, sáltate esta sección)

### 1.1 Serie de tiempo
Una **serie de tiempo** es una lista de mediciones tomadas una tras otra en el tiempo: la temperatura cada hora,
el uso de CPU de un servidor cada minuto, las transacciones de una tarjeta… Si en cada instante se miden
**varias cosas a la vez** (varios sensores), es **multivariada**; cada sensor es un **canal**.

En el código, una serie es una matriz `X` de tamaño *(instantes × canales)*. Por ejemplo, `X[100, 2]` es lo que
midió el canal 2 en el instante 100.

### 1.2 Anomalía
Una **anomalía** es un instante o un tramo que **no se comporta como el resto**: un pico imposible, un sensor que
se queda trabado, un patrón que de pronto cambia de forma. Las **etiquetas** `y` dicen cuáles instantes son
anómalos (`y[t] = 1`) y cuáles normales (`y[t] = 0`).

### 1.3 No supervisado
En la vida real casi nunca tenemos etiquetas: nadie revisó la serie instante por instante. Un método **no
supervisado** tiene que encontrar las anomalías **sin ver nunca las etiquetas**. Las etiquetas sólo se usan **al
final**, para calificar qué tan bien lo hizo (como la hoja de respuestas de un examen que el alumno no ve).

> ⚠️ Esto es importante para todo el proyecto: si un método usa las etiquetas para tomar alguna decisión, ya no es
> totalmente no supervisado. El artículo lo hace en un paso (el umbral) y nosotros lo quitamos (Mejora 1).

### 1.4 Contaminación
Llamamos **contaminación** a la **proporción de anomalías** dentro de los datos. Con 1 % de contaminación hay 1
instante raro de cada 100; con 30 % hay casi uno de cada tres. Mucha contaminación es un problema porque el
modelo se entrena con esos mismos datos y puede acabar **aprendiendo las anomalías como si fueran normales**.

### 1.5 Red neuronal, entrenamiento, gradiente, época, tasa de aprendizaje
* Una **red neuronal** es una función con muchos números ajustables (**parámetros**). Aquí la usamos para
  **predecir** el siguiente valor de la serie a partir de los anteriores.
* **Entrenar** es ajustar esos parámetros para que el **error de predicción** (la **pérdida**) sea pequeño.
* El **gradiente** dice en qué dirección y cuánto mover cada parámetro para reducir el error **de un ejemplo**.
  Un gradiente **grande** significa "este ejemplo está muy mal predicho; me obliga a cambiar mucho".
* Una **época** es una pasada completa por todos los datos. Se entrenan muchas épocas.
* La **tasa de aprendizaje** (η, *learning rate*) es el tamaño del paso que se da en cada ajuste.

### 1.6 La idea de TracIn con un ejemplo de juguete
TracIn (Pruthi et al., 2020) mide **cuánto "empujó" cada ejemplo al modelo durante todo el entrenamiento**: suma,
época por época, el tamaño del gradiente de ese ejemplo (al cuadrado) multiplicado por la tasa de aprendizaje.
Los ejemplos que el modelo **no logra aprender** (los raros) empujan mucho, época tras época, y acumulan un
valor alto.

Vamos a verlo con el modelo más simple posible: **una recta** `ŷ = a·x + b` que se ajusta a 40 puntos. Tres
puntos están "fuera de lugar" (son las anomalías). Entrenamos con descenso de gradiente y vamos sumando, para
cada punto, `η · ‖gradiente‖²`. **No le decimos al modelo cuáles son los raros.**

In [ ]:
rng = np.random.default_rng(0)
x = np.linspace(0, 1, 40)
yv = 2 * x + 1 + rng.normal(0, 0.05, 40)          # datos normales: una recta con un poco de ruido
raros = [8, 25, 33]
yv[raros] += [1.2, -1.0, 0.9]                      # tres puntos anómalos

a, b, eta = 0.0, 0.0, 0.1                          # parámetros de la recta y tasa de aprendizaje
esfuerzo = np.zeros(40)                            # aquí acumulamos la "self-influence" de cada punto
for epoca in range(200):
    err = (a * x + b) - yv                         # error de cada punto
    grad_a, grad_b = 2 * err * x, 2 * err          # gradiente del error al cuadrado, punto por punto
    esfuerzo += eta * (grad_a ** 2 + grad_b ** 2)  # TracIn: suma de η·‖gradiente‖² a lo largo de las épocas
    a -= eta * grad_a.mean(); b -= eta * grad_b.mean()   # paso de entrenamiento normal

fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 3.4))
a1.scatter(x, yv, c=["#de2d26" if i in raros else "#3182bd" for i in range(40)])
a1.plot(x, a * x + b, "k--", label=f"recta aprendida: {a:.2f}·x + {b:.2f}")
a1.set_title("Datos (rojo = anómalos, el modelo NO lo sabe)"); a1.legend()
a2.bar(range(40), esfuerzo, color=["#de2d26" if i in raros else "#9ecae1" for i in range(40)])
a2.set_title("Esfuerzo acumulado (TracIn) de cada punto"); a2.set_xlabel("punto")
plt.tight_layout(); plt.show()
top3 = np.argsort(esfuerzo)[-3:][::-1]
print("Los 3 puntos con más esfuerzo son:", sorted(top3.tolist()), " | los anómalos eran:", raros)

**Qué acabas de ver.** Sin saber cuáles eran los raros, los tres puntos con mayor esfuerzo acumulado son
exactamente los anómalos. **Ésa es toda la idea del artículo**, sólo que en lugar de una recta usa una red LSTM
y en lugar de puntos sueltos usa *ventanas* de una serie de tiempo.

Fíjate también en algo que será importante en la Mejora 2: los puntos raros **jalan la recta** hacia ellos (la
recta aprendida no es exactamente `2x + 1`). Si hubiera muchos puntos raros, la recta se movería tanto que ya no
parecerían raros. Eso es la **contaminación**.

## 2. Los datos: cómo se fabrica una serie con anomalías

Para saber si un detector funciona necesitamos series donde **sepamos exactamente** dónde están las anomalías.
El artículo usa el generador de **Lai et al. (2021)**, que fabrica series de 5 canales con ondas (senos, cosenos,
ondas cuadradas) más un poco de ruido, y **les inyecta** anomalías de cinco tipos. En nuestro código el canal
`t` recibe la anomalía de tipo `t`:

| Tipo | Nombre | Qué es | Familia |
|---|---|---|---|
| 0 | Puntual global | Un solo instante con un valor fuera del rango normal (un pico enorme) | **Puntual** |
| 1 | Puntual contextual | Un solo instante con un valor posible en general, pero raro **para ese momento** | **Puntual** |
| 2 | De forma (*shapelet*) | Un tramo donde la forma de la onda cambia | **De patrón** |
| 3 | Estacional | Un tramo donde el ritmo (la frecuencia) cambia | **De patrón** |
| 4 | De tendencia | Un tramo donde se suma una pendiente | **De patrón** |

Las **puntuales** duran **un instante**; las **de patrón** duran **un tramo** (unos 10 instantes). Esta
diferencia va a ser **clave** para entender dónde funciona nuestra Mejora 3 y dónde no.

La función que usamos es `make_dataset(tipos, semilla, length, ratio)`:
* `tipos`: qué tipos de anomalía meter, como texto (`"01234"` = todos, `"01"` = sólo puntuales, `"234"` = sólo de patrón);
* `semilla`: número que fija el azar (misma semilla → misma serie, así los experimentos son repetibles);
* `length`: largo de la serie (400 en el benchmark del artículo);
* `ratio`: proporción de anomalías **por tipo**.

Veamos cada tipo por separado y luego todos juntos.

In [ ]:
fig, axes = plt.subplots(5, 1, figsize=(12, 9), sharex=True)
for t, ax in zip("01234", axes):
    Xt, yt = make_dataset(t, seed=3)
    Xn = normalize(Xt.astype(np.float64), "z")
    ax.plot(Xn[:, int(t)], lw=0.9, color="k")
    sombrear(ax, yt)
    ax.set_title(f"Tipo {t}: {TIPOS[t]} (canal {t})", fontsize=9); ax.set_yticks([])
axes[-1].set_xlabel("tiempo")
plt.tight_layout(); plt.show()

In [ ]:
X, y = make_dataset("01234", seed=3)
print(f"Serie de {len(y)} instantes y {X.shape[1]} canales; {y.mean():.1%} de los instantes son anómalos.")
print("Forma de X:", X.shape, "  · primeras 3 filas:")
print(np.round(X[:3], 3))
print("Etiquetas y (primeros 40 instantes):", y[:40])
dibujar_serie(X, y, "Serie sintética con los 5 tipos de anomalía (franjas rojas = anomalías reales)")
plt.show()

**Para llevarte:** las franjas rojas son las respuestas correctas. **El algoritmo nunca las ve.** Sólo las usamos
al final para calificar.

# PARTE B · El método del artículo

## 3. El método del artículo, paso a paso

El método tiene **cinco pasos**. Los vamos a recorrer uno por uno, con el código real:

```
 serie X ──► (3.1) cortar en ventanas ──► (3.2) entrenar LSTM a predecir ──► (3.3) TracIn: esfuerzo por ventana
                                                                                        │
                       anomalías ◄── (3.5) umbral ◄── (3.4) pasar de ventanas a instantes (Ec. 4)
```

### 3.1 Ventanas
La red recibe **L = 4** instantes seguidos y debe predecir el **siguiente** (P = 1). Cada pedazo de
L + P = 5 instantes es una **ventana**. Hay una ventana por cada posición: la ventana `i` usa los instantes
`i, i+1, i+2, i+3` como entrada y predice el instante `i+4` (su **objetivo**).

En el código (`windows` en `lab.py`):
```python
def windows(Xt, idx, L, P):
    ar = torch.arange(L + P)          # [0, 1, 2, 3, 4]
    w = Xt[idx[:, None] + ar]         # para cada inicio i, toma los instantes i..i+4
    return w[:, :L], w[:, L:]         # entrada (4 instantes) y objetivo (1 instante)
```

In [ ]:
fig, ax = plt.subplots(figsize=(12, 2.6))
Xn = normalize(X.astype(np.float64), "z")
t0 = 60
ax.plot(np.arange(40, 100), Xn[40:100, 0], "k.-", lw=0.8, ms=4)
ax.axvspan(t0 - 0.5, t0 + L - 0.5, color="#9ecae1", alpha=0.6, label="entrada (L = 4 instantes)")
ax.axvspan(t0 + L - 0.5, t0 + L + P - 0.5, color="#fdae6b", alpha=0.8, label="objetivo: lo que debe predecir (P = 1)")
ax.set_title(f"La ventana que empieza en t = {t0}: el LSTM ve los instantes azules y predice el naranja")
ax.legend(loc="upper right")
plt.show()
n = len(y)
print(f"Con n = {n} instantes hay M = n - L - P + 1 = {n - L - P + 1} ventanas.")
print("Cada instante (salvo los de las orillas) pertenece a L + P = 5 ventanas distintas. Esto importará en la Mejora 3.")

### 3.2 La red: un LSTM que predice el siguiente instante
Un **LSTM** (*Long Short-Term Memory*) es una red neuronal hecha para secuencias: lee los instantes de la ventana
**uno por uno** y va guardando una "memoria" (`h` y `c`) de lo que ha visto. Al final, una capa lineal (`head`)
convierte esa memoria en la predicción del siguiente instante.

Nuestro `LSTM` (en `lab.py`) está escrito a mano para poder calcular gradientes por ventana de forma eficiente,
pero hace exactamente lo mismo que el `nn.LSTM` de PyTorch (lo verificamos con una prueba automática). Lo esencial:

```python
for t in range(x.shape[1]):                          # recorre los L = 4 instantes de la ventana
    i, f, g, o = (gx[:, t] + self.hh(h)).chunk(4, dim=1)   # cuatro "compuertas"
    c = sigmoid(f) * c + sigmoid(i) * tanh(g)        # actualiza la memoria (olvida y agrega)
    h = sigmoid(o) * tanh(c)                         # salida de este paso
return self.head(h)                                  # predicción del instante siguiente
```

La pérdida de una ventana es el **error cuadrático medio** entre la predicción y el valor real del objetivo.

### 3.3 TracIn: el "esfuerzo" de cada ventana
Al final de **cada época** `k` calculamos, para **cada ventana** `x`, el tamaño de su gradiente al cuadrado y lo
multiplicamos por la tasa de aprendizaje de esa época. Sumando sobre las épocas obtenemos la **self-influence**:

$$\text{TracIn}(x) = \sum_{\text{épocas } k} \eta_k \, \lVert \nabla_\theta \ell_k(x) \rVert^2$$

Leído término a término:
* $\ell_k(x)$: el error de la ventana $x$ con el modelo como estaba al final de la época $k$;
* $\nabla_\theta \ell_k(x)$: el gradiente de ese error respecto a los parámetros $\theta$ (cuánto "jala" esa ventana al modelo);
* $\lVert\cdot\rVert^2$: su tamaño al cuadrado;
* $\eta_k$: la tasa de aprendizaje en esa época.

En el código (función `run` de `lab.py`), dentro del ciclo de épocas:
```python
l_all, h_all = _stats(model, Xt, M, L, P)       # error y memoria final de TODAS las ventanas
if cfg.grad == "last":                          # gradiente sólo de la última capa (fórmula cerrada, rápido)
    g2 = (4.0 / D) * l_all * (1 + h_all)
else:                                           # gradiente de TODOS los parámetros (vmap, más lento)
    g2 = _per_sample_sqgrad(model, Xt, M, L, P)
si += lr_now * g2                               # acumula η_k · ‖∇ℓ‖²  → self-influence
```
* En los **sintéticos** el artículo usa el gradiente de la **última capa** (hay una fórmula exacta y rápida).
* En los **datos reales** usamos el gradiente de **todos los parámetros**, como en el artículo.

Además, `run` usa **Adam** como optimizador, baja la tasa de aprendizaje si el error se estanca
(`ReduceLROnPlateau`) y para el entrenamiento si no mejora en 10 épocas (*early stopping*).

**Ejecuta la celda.** A la izquierda verás bajar el error de predicción. A la derecha se va acumulando la
self-influence, ya convertida a instantes. Los picos deberían ir apareciendo sobre las franjas rojas.

In [ ]:
historia = []
def en_vivo(epoca, si, perdida):
    historia.append(float(perdida.mean()))
    if epoca % 10 and epoca > 3:
        return
    clear_output(wait=True)
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 3.2), gridspec_kw={"width_ratios": [1, 3]})
    a1.plot(historia, "k-"); a1.set_title("error de predicción promedio"); a1.set_xlabel("época"); a1.set_yscale("log")
    a2.plot(proyectar(si, len(y), objetivo=False), color="#3182bd", lw=0.9)
    sombrear(a2, y); a2.set_title(f"self-influence acumulada tras {epoca} épocas"); a2.set_xlabel("tiempo"); a2.legend(loc="upper right")
    plt.show()

t = time.time()
r = run(X, L, P, seed=0, cfg=CFG, callback=en_vivo)
print(f"Entrenamiento terminado: {r['epochs']} épocas en {time.time() - t:.0f} s.")
print("r['si'] tiene un número por VENTANA:", r["si"].shape, "· primeros valores:", np.round(r["si"][:5], 4))

### 3.4 De "esfuerzo por ventana" a "score por instante": la Ecuación 4 del artículo
TracIn da **un número por ventana**, pero queremos saber **qué instantes** son anómalos. El artículo (Ec. 4) le da a
cada instante la **suma** de los scores de **todas las ventanas que lo contienen** (hasta 5 ventanas).

Veámoslo con un ejemplo de juguete: una serie de 12 instantes (8 ventanas). Todas las ventanas tienen score 1,
salvo la ventana 4, que tiene score 9 (porque su objetivo, el instante 8, es una anomalía).

In [ ]:
n_j = 12
s_j = np.array([1, 1, 1, 1, 9, 1, 1, 1], dtype=float)         # score de las 8 ventanas
tabla = pd.DataFrame({"instante": range(n_j),
                      "ventanas que lo contienen": [", ".join(str(i) for i in range(len(s_j)) if i <= t < i + L + P) for t in range(n_j)],
                      "score Ec. 4 (suma)": project_sum(s_j, L + P, n_j)})
display(tabla.set_index("instante"))
print("La ventana 4 cubre los instantes 4, 5, 6, 7 y 8 (entrada 4-7, objetivo 8).")
print("Con la Ec. 4, su score alto se reparte a LOS CINCO instantes: 4, 5, 6 y 7 son normales pero quedan altos.")

En el código, `project_sum(s, span, n)` hace esa suma de forma eficiente (con sumas acumuladas en lugar de dos
ciclos anidados):
```python
acc = np.zeros(n + 1)
acc[:m] += s              # cada ventana i "empieza" a sumar en el instante i
acc[span:span + m] -= s   # ...y "deja" de sumar en el instante i + span
return np.cumsum(acc)[:n] # la suma acumulada da, en cada instante, la suma de las ventanas que lo cubren
```
Guarda este ejemplo: **es el corazón de la Mejora 3.**

### 3.5 El umbral: ¿a partir de qué score decimos "anómalo"?
Hay que elegir un corte. El artículo usa dos:
* **Estático:** un percentil fijo (el 85 en sintéticos, el 87 en reales). Siempre marca el 13–15 % de los instantes.
* **Dinámico:** prueba muchos cortes y se queda con el que da **el mejor F1 en la primera mitad de la serie,
  mirando las etiquetas**. Después lo aplica a la segunda mitad. Es el que mejor le funciona, pero **usa las
  respuestas**.

Por eso, para comparar de forma justa, **todos los F1 se miden sólo en la segunda mitad** (la línea gris en las
gráficas).

### 3.6 Cómo se califica: AUPRC, AUROC y F1 (con ejemplo de juguete)
* **AUROC** (0.5 = azar, 1 = perfecto): la probabilidad de que un instante anómalo elegido al azar tenga **más score**
  que uno normal elegido al azar. **No depende del umbral.**
* **AUPRC** (de la proporción de anomalías hasta 1): qué tan "arriba" quedan las anomalías al ordenar por score,
  dando más peso a los primeros lugares. Es más exigente que el AUROC cuando hay pocas anomalías. **No depende del
  umbral.** Mide la **calidad del score**.
* **F1** (0 a 1): con un umbral dado, combina **precisión** (de lo que marqué, cuánto era anómalo de verdad) y
  **exhaustividad** (de lo anómalo, cuánto encontré). **Sí depende del umbral.**

In [ ]:
y_j  = np.array([0, 0, 0, 1, 0, 0, 1, 0, 0, 0])
sc_j = np.array([.1, .3, .2, .9, .4, .8, .7, .1, .2, .3])
pred = sc_j >= 0.6
print("etiquetas :", y_j)
print("scores    :", sc_j)
print("marcado   :", pred.astype(int), "(umbral 0.6)")
tp = int((pred & (y_j == 1)).sum()); fp = int((pred & (y_j == 0)).sum()); fn = int((~pred & (y_j == 1)).sum())
print(f"aciertos = {tp}, falsas alarmas = {fp}, anomalías perdidas = {fn}")
print(f"precisión = {tp}/{tp + fp} = {tp / (tp + fp):.2f}  · exhaustividad = {tp}/{tp + fn} = {tp / (tp + fn):.2f}  · F1 = {f1(y_j, pred):.2f}")
print(f"AUROC = {roc_auc_score(y_j, sc_j):.3f} (hay un normal, el 0.8, por encima de una anomalía, el 0.7)")
print(f"AUPRC = {average_precision_score(y_j, sc_j):.3f}")

### 3.7 Todo junto: el método del artículo sobre nuestra serie

In [ ]:
s_art = proyectar(r["si"], len(y), objetivo=False)
m = valid_mask(len(y))                       # se excluyen 19 instantes en cada orilla (ahí hay menos ventanas)
thr_art = umbral_dinamico(np.asarray(y)[m].astype(bool), s_art[m])   # usa las etiquetas de la 1.ª mitad
res_art = evaluar(y, s_art, thr_art)
fig, ax = plt.subplots(figsize=(12, 3.2))
dibujar_deteccion(y, s_art, thr_art, "Método del artículo: score y detecciones (el F1 se mide a la derecha de la línea gris)", ax)
plt.show()
display(pd.DataFrame([res_art], index=["Artículo (umbral dinámico, con etiquetas)"]).round(3))

# PARTE C · Nuestras tres mejoras

## 4. Mejora 1 (M1) · Un umbral que **no** necesita etiquetas

### El problema
El umbral dinámico del artículo se calibra **viendo las respuestas**. En un problema realmente no supervisado esas
etiquetas no existen. Es como fijar la calificación mínima para aprobar **después** de ver quién sacó qué.

### La idea
Usamos la regla clásica del **diagrama de caja** (*boxplot*): es atípico lo que queda por encima de

$$\text{umbral} = Q_3 + 1.5 \times (Q_3 - Q_1)$$

* $Q_1$ es el valor que deja por debajo al 25 % de los scores; $Q_3$, el que deja por debajo al 75 %.
* $Q_3 - Q_1$ se llama **rango intercuartil (IQR)** y mide qué tan dispersos están los scores "típicos".
* El 1.5 es el valor estándar en estadística (Tukey, 1977). **Lo fijamos antes de ver resultados** y nunca lo ajustamos.

Sólo usa el score, **nunca** las etiquetas. Y se adapta solo: si hay más anomalías, la cola de scores altos crece y
el umbral cambia.

### El código (`thr_iqr` en `src/common.py`)
```python
def thr_iqr(s, k=1.5):
    q1, q3 = np.percentile(s, [25, 75])   # cuartiles del score
    return float(q3 + k * (q3 - q1))      # Q3 + 1.5 · IQR
```
Son dos líneas: la mejora es conceptual (quitar las etiquetas del proceso), no de programación.

### Ejemplo de juguete

In [ ]:
sc_j = np.array([1, 2, 2, 3, 3, 3, 4, 4, 5, 15, 18], dtype=float)
q1, q3 = np.percentile(sc_j, [25, 75])
print(f"scores = {sc_j.tolist()}")
print(f"Q1 = {q1}, Q3 = {q3}, IQR = Q3 - Q1 = {q3 - q1}")
print(f"umbral = Q3 + 1.5·IQR = {q3} + 1.5·{q3 - q1} = {thr_iqr(sc_j)}")
print("Se marcan como anómalos:", sc_j[sc_j > thr_iqr(sc_j)].tolist())

### En nuestra serie

In [ ]:
m = valid_mask(len(y))
thr_m1 = thr_iqr(s_art[m])
res_m1 = evaluar(y, s_art, thr_m1)
fig, ax = plt.subplots(figsize=(11, 3))
ax.hist(s_art[m][np.asarray(y)[m] == 0], bins=80, alpha=0.6, label="instantes normales", color="#9ecae1")
ax.hist(s_art[m][np.asarray(y)[m] == 1], bins=80, alpha=0.7, label="instantes anómalos", color="#fb6a4a")
ax.axvline(thr_art, color="k", ls="--", label="umbral dinámico (usa etiquetas)")
ax.axvline(thr_m1, color="#31a354", lw=2, label="umbral IQR (mejora 1, sin etiquetas)")
ax.set_yscale("log"); ax.set_xlabel("score"); ax.legend(fontsize=8); ax.set_title("¿Dónde cortar?")
plt.show()
display(pd.DataFrame([res_art, res_m1], index=["Artículo · dinámico (CON etiquetas)", "Mejora 1 · IQR (SIN etiquetas)"]).round(3))
print("Nota: el AUPRC y el AUROC no cambian (el score es el mismo); sólo cambia dónde se corta.")
print(f"En esta serie hay {np.mean(y):.0%} de instantes anómalos y el IQR marca {res_m1['% marcado']:.0%}.")

**¿Por qué aquí M1 sola no le gana al umbral con etiquetas?** El umbral IQR se pone donde la distribución del score
dice que empiezan los valores atípicos. Si el score es malo (con el halo del artículo, muchos normales tienen score
alto y muchas anomalías no), ningún umbral sin etiquetas puede hacer milagros. Sobre un score mejor (con M3), el
IQR sí rinde. En la evaluación grande de sintéticos (300 corridas), el F1 fue:
* artículo con etiquetas: 0.469;
* M1 sobre el score del artículo: 0.481;
* M1 sobre el score de M3: **0.665**.

**M1 vale sobre todo en combinación.**

### ⚠️ Cuándo **no** funciona bien la Mejora 1
Cuando las anomalías son **muy raras** (menos del 2 % de los instantes) y el score tiene **colas pesadas** (algunos
normales con valores muy altos, como pasa en datos reales), la regla IQR marca **demasiado**. Muchas falsas
alarmas → precisión baja → F1 bajo. El umbral dinámico del artículo no tiene ese problema porque **mira las
respuestas**. Es el precio de no usar etiquetas, y lo reportamos (ver sección 9).

En las series sintéticas esto casi no se nota. Por eso aquí mostramos lo que pasó en los **4 datasets reales del
artículo**, en la evaluación grande (método final, promedio de 10 semillas):

In [ ]:
REAL_IQR = {"cicids": {"anom": 0.0128089874436669, "iqr": 0.16756740876534287}, "creditcard": {"anom": 0.0017277161488785647, "iqr": 0.12844059571090954}, "gecco": {"anom": 0.012463623693883001, "iqr": 0.16432919564134224}, "swan_sf": {"anom": 0.23791700705223323, "iqr": 0.13910655040762906}}
if REAL_IQR:
    t_ = pd.DataFrame(REAL_IQR).T.rename(columns={"anom": "% real de anomalías", "iqr": "% que marca el IQR"})
    display((100 * t_).round(1))
    print("En Credit Card, CICIDS y GECCO las anomalías son menos del 1.3 %, pero el IQR marca entre 12 y 17 %:")
    print("aunque encuentre las anomalías, la mayoría de lo marcado son falsas alarmas.")

## 5. Mejora 2 (M2) · Quitar lo sospechoso del entrenamiento y volver a entrenar

### El problema: el modelo "memoriza" las anomalías
¿Recuerdas la recta del ejemplo de juguete, que se movía hacia los puntos raros? Con una red neuronal pasa lo
mismo, y peor: la red es tan flexible que, si hay **muchas** anomalías (digamos 25 % de los datos), termina
**aprendiéndoselas**. Ya no le cuestan y su self-influence baja, así que dejan de destacar. El artículo menciona
esto para explicar por qué le va mal en SWAN-SF (24 % de anomalías), pero **no lo mide ni lo corrige**.

> **Analogía.** Si una cuarta parte del cuadernillo son ejercicios con trampa, el estudiante acaba memorizando las
> trampas y ya no le parecen raras.

Primero comprobemos que el problema existe: la misma clase de serie con 2 % y con 25 % de anomalías.

In [ ]:
def serie_contaminada(tipos="01234", contaminacion=0.25, semilla=0, largo=2000):
    """Serie larga con una proporción TOTAL `contaminacion` de anomalías (repartida entre los tipos)."""
    return make_dataset(tipos, 10_000 + semilla, length=largo, ratio=contaminacion / len(tipos))

filas = {}
for c in (0.02, 0.25):
    Xc, yc = serie_contaminada(contaminacion=c)
    rc = run(Xc, L, P, seed=0, cfg=CFG)
    sc = proyectar(rc["si"], len(yc), objetivo=False)
    mc = valid_mask(len(yc))
    filas[f"{yc.mean():.0%} de anomalías"] = {"AUROC del artículo": roc_auc_score(yc[mc], sc[mc]),
                                              "AUPRC del artículo": average_precision_score(yc[mc], sc[mc])}
    print(f"contaminación {yc.mean():.0%}: listo")
display(pd.DataFrame(filas).T.round(3))
print("Con muchas anomalías el método del artículo empeora. (En la sección 6 veremos que buena parte de")
print("esa caída NO es por memorización, sino por la proyección; pero la memorización también existe.)")

### La idea: entrenar sólo con lo que parece normal
1. Entrenamos normal y calculamos la self-influence de cada ventana.
2. Las ventanas con self-influence **muy alta** (sospechosas de ser anómalas) se **quitan del entrenamiento**
   (peso 0) o se les **baja el peso**.
3. **Volvemos a entrenar desde cero** sin ellas y medimos la self-influence de **todas** las ventanas, **también de
   las quitadas**.
4. Repetimos varias rondas.

**Punto clave:** quitar una ventana del entrenamiento **no** la quita del análisis. Al contrario: como el modelo ya
no la ve al entrenar, no puede memorizarla, y cuando le calculamos el score **le cuesta más que antes**. Lo
anómalo resalta más.

### Las tres variantes que probamos
| Variante | Qué ventanas | Qué se les hace | Rondas |
|---|---|---|---|
| **Suave** (`soft`) | $z > 2$ | se les baja el peso gradualmente: $w = \max(0.05,\; e^{-(z-2)})$ | 2 |
| **Recorte** (`trim`) | $z > 2$ | peso 0 (se quitan) | 2 |
| **Recorte adaptativo** (`adapt`, la del método final) | las que pasan el **umbral IQR de la Mejora 1** | peso 0 | 3 |

Aquí $z$ es un **z-score robusto** del logaritmo de la self-influence: cuántas "desviaciones típicas" está cada
ventana por encima de la mediana, usando mediana y MAD en vez de media y desviación estándar (así las propias
anomalías no deforman la escala). Los valores 2, 0.05, 2 rondas y 3 rondas **se fijaron antes de evaluar**.

### El código
```python
def robust_z(v):                                   # src/forecaster.py
    med = np.median(v)
    mad = 1.4826 * np.median(np.abs(v - med))      # MAD escalada: equivale a la desv. estándar si fuera normal
    return (v - med) / max(mad, 1e-12)

def weights_from_influence(s, mode, z0=2.0, w_min=0.05):
    z = robust_z(np.log(np.maximum(s, 1e-30)))     # z robusto del log de la self-influence
    if mode == "soft":
        return np.where(z <= z0, 1.0, np.maximum(w_min, np.exp(-(z - z0))))
    if mode == "trim":
        return np.where(z <= z0, 1.0, 0.0)

def adapt_weights(si):                             # src/final/core.py  ← la del método final
    return np.where(si > thr_iqr(si), 0.0, 1.0)    # quita lo que pasa el umbral IQR (¡la Mejora 1!)
```
¿Y dónde se usan los pesos? En `run`, la pérdida de cada lote se pondera:
```python
loss = (l * wb).sum() / wb.sum()    # una ventana con peso 0 no aporta nada al entrenamiento
```

**¿Por qué el recorte adaptativo es el elegido?** Porque reutiliza el umbral IQR de la Mejora 1, y así **la
cantidad que se quita se ajusta sola**: si hay más anomalías, el umbral deja pasar más ventanas sospechosas y se
quitan más. Con un corte fijo $z > 2$, la cantidad quitada es casi constante.

### En vivo: tres rondas de recorte adaptativo con 25 % de anomalías
Las marcas moradas son las ventanas quitadas. La tabla dice qué fracción se quitó y **qué fracción de las
ventanas anómalas** quedó fuera (esto último usa las etiquetas, **sólo para diagnosticar**; el método no las ve).

In [ ]:
Xc, yc = serie_contaminada(contaminacion=0.25)
n = len(yc); mc = valid_mask(n)
M = n - L - P + 1
cum = np.r_[0, np.cumsum(yc.astype(int))]
ventana_anomala = (cum[np.arange(M) + L + P] - cum[np.arange(M)]) > 0   # sólo para el diagnóstico

r0 = run(Xc, L, P, seed=0, cfg=CFG)
s = r0["si"]
tabla = [{"ronda": 0, "ventanas quitadas": "0 %", "de las anómalas": "—", "de las normales": "—",
          "AUROC": roc_auc_score(yc[mc], proyectar(s, n, False)[mc])}]
fig, axes = plt.subplots(4, 1, figsize=(12, 8), sharex=True)
axes[0].plot(proyectar(s, n, False), lw=0.7); sombrear(axes[0], yc); axes[0].set_title("ronda 0 (artículo)")
for k in range(1, 4):
    w = adapt_weights(s)                       # mejora 1 + 2: quitar lo que pasa el umbral IQR
    s = run(Xc, L, P, seed=0, cfg=CFG, weights=w)["si"]
    tabla.append({"ronda": k, "ventanas quitadas": f"{(w == 0).mean():.0%}",
                  "de las anómalas": f"{(w[ventana_anomala] == 0).mean():.0%}",
                  "de las normales": f"{(w[~ventana_anomala] == 0).mean():.0%}",
                  "AUROC": roc_auc_score(yc[mc], proyectar(s, n, False)[mc])})
    axes[k].plot(proyectar(s, n, False), lw=0.7); sombrear(axes[k], yc)
    quitadas = np.where(w == 0)[0] + L
    axes[k].plot(quitadas, np.full(len(quitadas), axes[k].get_ylim()[1] * 0.95), "|", color="purple", ms=4, label="ventanas quitadas")
    axes[k].set_title(f"ronda {k}"); axes[k].legend(loc="upper left", fontsize=7)
    print(f"ronda {k} lista")
plt.tight_layout(); plt.show()
display(pd.DataFrame(tabla).round(3))
s_recorte = s          # lo guardamos para la sección 6
print("Lo que se quita es, en su mayoría, anómalo; y el AUROC sube ronda a ronda.")

**Para llevarte:** la Mejora 2 sola ayuda **poco** cuando hay pocas anomalías (no hay mucho que memorizar), y
ayuda más cuando hay muchas. Pero su papel más importante aparece **combinada con la Mejora 3**, como veremos
en la sección 6.4.

## 6. Mejora 3 (M3) · Proyección al objetivo: culpar al instante correcto

Ésta es **la mejora más importante del proyecto**, así que le dedicamos más espacio.

### 6.1 El problema
Vuelve al ejemplo de juguete de la sección 3.4. El error de una ventana se mide **sólo en el instante que predice**
(su objetivo). Pero la Ec. 4 del artículo reparte ese score entre **los 5 instantes** de la ventana. Resultado: una
anomalía "contagia" su score alto a los **4 instantes normales anteriores**, que aparecen como falsas alarmas.

> **Analogía.** Un jugador mete un autogol y la mala calificación se la ponen a los cinco que estaban en la cancha.

A esos instantes normales contagiados los llamamos el **halo** de la anomalía: son los normales que están a
distancia 1 a L + P − 1 = 4 de una anomalía.

### 6.2 La idea y el código
Cada ventana pasa su score **sólo al instante que predijo** (su objetivo). Si varias ventanas predicen el mismo
instante (cuando P > 1), se promedian. Ésta es la forma estándar en los detectores basados en predicción: **no la
inventamos nosotros**. Lo nuestro es usarla con TracIn y medir cuánto cambia y por qué.

```python
def project_targets_mean(s, L, P, n):      # src/common.py
    acc[L:L + m] += s                     # la ventana i aporta a partir del instante i + L (su objetivo)...
    acc[L + P:L + P + m] -= s             # ...y sólo hasta i + L + P (con P = 1, sólo el instante i + 4)
    cnt[L:L + m] += 1; cnt[L + P:L + P + m] -= 1   # cuántas ventanas predicen cada instante
    out = cumsum(acc) / cumsum(cnt)       # promedio
```
Es el mismo truco de sumas acumuladas de `project_sum`, pero cubriendo **sólo el objetivo**.

Comparemos las dos proyecciones con el ejemplo de juguete:

In [ ]:
tabla = pd.DataFrame({"score Ec. 4 (artículo)": project_sum(s_j, L + P, n_j),
                      "score M3 (al objetivo)": project_targets_mean(s_j, L, P, n_j)}, index=pd.Index(range(n_j), name="instante"))
display(tabla)
print("Con M3, sólo el instante 8 (el objetivo de la ventana anómala) queda alto. Los instantes 4-7 ya no se contagian.")
print("(Los instantes 0-3 no son objetivo de ninguna ventana: por eso se excluyen las orillas al evaluar.)")

### 6.3 En una serie de verdad: acercamiento a una anomalía puntual

In [ ]:
X0, y0 = make_dataset("0", seed=1)                 # sólo anomalías puntuales
r_0 = run(X0, L, P, seed=0, cfg=CFG)
s_win = proyectar(r_0["si"], len(y0), objetivo=False)
s_tgt = proyectar(r_0["si"], len(y0), objetivo=True)
pico = [a for a, b in intervalos(y0) if 30 < a < len(y0) - 30][0]
fig, axes = plt.subplots(1, 2, figsize=(12, 3))
for ax, s_, tit in ((axes[0], s_win, "Artículo (Ec. 4): el score se reparte en 5 instantes"),
                    (axes[1], s_tgt, "Mejora 3: el score va al instante predicho")):
    rng_ = np.arange(pico - 12, pico + 8)
    ax.bar(rng_, s_[rng_], color=["#fb6a4a" if y0[i] else "#9ecae1" for i in rng_])
    ax.set_title(tit, fontsize=9); ax.set_xlabel("tiempo (rojo = anomalía real)")
plt.tight_layout(); plt.show()
m0 = valid_mask(len(y0))
display(pd.DataFrame({"AUROC": [roc_auc_score(y0[m0], s_win[m0]), roc_auc_score(y0[m0], s_tgt[m0])],
                      "AUPRC": [average_precision_score(y0[m0], s_win[m0]), average_precision_score(y0[m0], s_tgt[m0])]},
                     index=["Artículo (Ec. 4)", "Mejora 3 (al objetivo)"]).round(3))

### 6.4 El descubrimiento principal: el halo crece con la contaminación
Aquí está la explicación de **por qué el método del artículo empeora tanto cuando hay muchas anomalías**.

Piensa en el AUROC global como una mezcla de dos comparaciones:
* anomalías **contra normales lejanas** (fácil: las lejanas tienen score bajo);
* anomalías **contra normales del halo** (difícil con la Ec. 4: el halo tiene score contagiado).

El peso de cada parte depende de **cuántas normales caen en el halo**. Con 1 % de anomalías, casi ninguna normal
está cerca de una anomalía. Con 30 %, **casi todas** lo están. Entonces, con la Ec. 4, el AUROC global se va
pareciendo cada vez más al AUROC **contra el halo**, que es bajo.

Vamos a medirlo en vivo con anomalías puntuales a 1 %, 10 % y 30 %. Para cada serie calculamos:
* **% de normales en el halo:** qué fracción de los instantes normales está a distancia 1–4 de una anomalía;
* **AUROC anom. vs halo** y **AUROC anom. vs lejanas**, con cada proyección;
* el **AUROC global**.

(Esto usa las etiquetas **sólo para analizar**; ningún método las usa.)

In [ ]:
def distancia_a_anomalia(y):
    idx = np.flatnonzero(y); n = len(y)
    if len(idx) == 0:
        return np.full(n, np.inf)
    pos = np.searchsorted(idx, np.arange(n))
    izq = np.abs(np.arange(n) - idx[np.clip(pos - 1, 0, len(idx) - 1)])
    der = np.abs(idx[np.clip(pos, 0, len(idx) - 1)] - np.arange(n))
    return np.minimum(izq, der)

def medir_halo(y, sc):
    y = np.asarray(y).astype(bool); m = valid_mask(len(y)); d = distancia_a_anomalia(y)
    halo = ~y & (d >= 1) & (d <= L + P - 1) & m; lejos = ~y & (d >= L + P) & m; anom = y & m
    auc = lambda a, b: roc_auc_score(np.r_[np.ones(a.sum()), np.zeros(b.sum())], np.r_[sc[a], sc[b]])
    return {"% normales en el halo": halo.sum() / (~y & m).sum(), "AUROC anom. vs halo": auc(anom, halo),
            "AUROC anom. vs lejanas": auc(anom, lejos), "AUROC global": roc_auc_score(y[m], sc[m])}

filas = []
for c in (0.01, 0.10, 0.30):
    Xh, yh = serie_contaminada("01", c, semilla=0)
    rh = run(Xh, L, P, seed=0, cfg=CFG)
    for nombre, obj in (("Artículo (Ec. 4)", False), ("Mejora 3", True)):
        filas.append({"contaminación": f"{c:.0%}", "proyección": nombre, **medir_halo(yh, proyectar(rh["si"], len(yh), obj))})
    print(f"contaminación {c:.0%}: listo")
halo_df = pd.DataFrame(filas).set_index(["contaminación", "proyección"])
display(halo_df.round(3))

fig, ax = plt.subplots(figsize=(8, 3.4))
for nombre, col in (("Artículo (Ec. 4)", "#7f7f7f"), ("Mejora 3", "#1f77b4")):
    d_ = halo_df.xs(nombre, level="proyección")
    xs = 100 * d_["% normales en el halo"].values
    ax.plot(xs, d_["AUROC global"], "o-", color=col, label=f"{nombre}: AUROC global")
    ax.plot(xs, d_["AUROC anom. vs halo"], "s--", color=col, alpha=0.6, label=f"{nombre}: AUROC vs halo")
ax.set_xlabel("% de normales que caen en el halo"); ax.set_ylabel("AUROC"); ax.legend(fontsize=7)
ax.set_title("Con la Ec. 4, el AUROC global cae hacia el AUROC contra el halo")
plt.show()

**Qué deberías ver:** con la Ec. 4, el % de normales en el halo pasa de pocos a casi todas. El AUROC **contra el
halo** es bajo, y el AUROC global va cayendo hasta acercarse a él. Con la Mejora 3, el AUROC contra el halo es
alto, así que el AUROC global casi no cae.

En la evaluación grande (20 semillas, ver sección 9), con anomalías puntuales al 30 %, el **91 %** de las normales
caen en el halo. El AUROC del artículo pasa de 0.916 a 0.711, que es prácticamente su AUROC contra el halo
(0.686). Con M3 se queda en 0.931 (cae 0.022 en vez de 0.205). Es decir, **con anomalías puntuales casi toda la
degradación del artículo se debe a la proyección**, no a que el modelo aprenda las anomalías.

### 6.5 ⚠️ Dónde falla la Mejora 3: anomalías largas
La suma de la Ec. 4 tiene una **virtud**: en un evento **largo** (muchos instantes anómalos seguidos), cada instante
interior recibe el score de **5 ventanas anómalas**, así que **acumula evidencia**. La Mejora 3 sólo conserva la
ventana que predice ese instante, y **a mitad de un patrón el LSTM ya lo predice relativamente bien**: el
interior del evento queda con score bajo.

Ahí entra la Mejora 2: si quitamos las ventanas anómalas del entrenamiento, el modelo **no aprende el patrón**, y su
interior vuelve a ser difícil de predecir. Veámoslo con anomalías sólo de patrón (tipos 2, 3 y 4) al 20 %:

In [ ]:
Xp, yp = serie_contaminada("234", 0.20, semilla=1)
rp = run(Xp, L, P, seed=0, cfg=CFG)
sp = rp["si"]
for _ in range(3):
    sp = run(Xp, L, P, seed=0, cfg=CFG, weights=adapt_weights(sp))["si"]
mp = valid_mask(len(yp))
variantes = {"Artículo (Ec. 4)": proyectar(rp["si"], len(yp), False), "Mejora 3 sola": proyectar(rp["si"], len(yp), True),
             "M2 (recorte) + Ec. 4": proyectar(sp, len(yp), False), "M2 + M3 (método final)": proyectar(sp, len(yp), True)}
display(pd.DataFrame({k: {"AUROC": roc_auc_score(yp[mp], v[mp]), "AUPRC": average_precision_score(yp[mp], v[mp])}
                      for k, v in variantes.items()}).T.round(3))

# percentil del score a lo largo de cada evento largo
def perfil_eventos(y, sc, largo_min=8):
    pct = pd.Series(sc).rank(pct=True).values
    perf = []
    for a, b in intervalos(y):
        if b - a >= largo_min and a > 20 and b < len(y) - 20:
            perf.append(np.interp(np.linspace(0, 1, 10), np.linspace(0, 1, b - a), pct[a:b]))
    return np.mean(perf, axis=0)

fig, ax = plt.subplots(figsize=(8, 3.2))
for (k, v), c in zip(variantes.items(), ("#7f7f7f", "#1f77b4", "#8c564b", "#c0392b")):
    ax.plot(np.linspace(0, 100, 10), perfil_eventos(yp, v), "o-", color=c, label=k)
ax.set_xlabel("posición dentro del evento (% del largo)"); ax.set_ylabel("percentil del score")
ax.set_title("Anomalías de patrón: ¿qué tan alto queda el score DENTRO del evento?"); ax.legend(fontsize=8)
plt.show()
print("Con M3 sola, el interior del evento baja (el LSTM ya predice el patrón). El recorte (M2) lo recupera.")
print("Fíjate también en 'M2 (recorte) + Ec. 4': con anomalías largas, el recorte SIN M3 suele dar el mejor AUROC.")
print("Lo mismo pasó en la evaluación grande (mezcla 234) y en SMD.")

**Para llevarte (muy importante para entender los resultados):**
* La Mejora 3 **cambia integración de evidencia por localización**: ubica mejor el instante exacto, pero deja de
  sumar evidencia a lo largo de eventos largos.
* Gana claramente con anomalías **cortas** (puntuales) y cuando hay **mucha contaminación** (mucho halo).
* Con anomalías **largas**, sola puede **empeorar**; necesita la Mejora 2 para compensar, y aun así puede no
  alcanzar. Por eso en los benchmarks reales SMD y SMAP/MSL (anomalías de cientos de pasos) **no ayuda**.

## 7. El método final = Mejora 1 + Mejora 2 + Mejora 3

Las tres piezas encajan en un solo procedimiento, **sin ninguna etiqueta** y con L = 4, P = 1 fijos de antemano (el
artículo, además del umbral, elige L y P para cada dataset mirando las etiquetas):

```
serie ──► entrenar LSTM ──► self-influence ──► quitar ventanas sobre el umbral IQR (M1 + M2) ──┐
              ▲                                                                                │  ×3 rondas
              └────────────────────────── volver a entrenar sin ellas ◄────────────────────────┘
                                   │
                                   ▼
                 proyectar al instante predicho (M3) ──► umbral IQR (M1) ──► anomalías
```

**Lo que es propio de la combinación:**
* la self-influence hace **dos trabajos**: es el detector **y** decide qué limpiar del entrenamiento;
* **el mismo umbral IQR** sirve para limpiar (M2) y para decidir (M1);
* cada pieza **corrige un punto débil de otra**: M3 quita el halo, pero pierde evidencia en eventos largos → M2 la
  recupera; M2 y M3 mejoran el score, pero hace falta un umbral → M1 lo da sin etiquetas.

Cada pieza por separado tiene antecedentes en la literatura; **la combinación, el diagnóstico del halo y la
evaluación son nuestra contribución**.

La función siguiente implementa el método completo. Con los interruptores `M1`, `M2`, `M3` puedes apagar cada
mejora. Con las tres apagadas es el método del artículo (y entonces el umbral hay que calibrarlo con etiquetas).

In [ ]:
def metodo_final(X, semilla=0, M1=True, M2=True, M3=True, rondas=3, avisar=print, cfg=None):
    """Devuelve (score por instante, umbral, score del artículo)."""
    cfg = cfg or CFG
    n = len(X)
    r0 = run(X, L, P, seed=semilla, cfg=cfg)               # 1) entrenar y medir self-influence
    s = r0["si"]
    s_articulo = proyectar(s, n, objetivo=False)
    if M2:                                                  # 2) rondas de recorte adaptativo (M1 + M2)
        for k in range(1, rondas + 1):
            w = adapt_weights(s)                            #    peso 0 a lo que pasa el umbral IQR
            s = run(X, L, P, seed=semilla, cfg=cfg, weights=w)["si"]   # reentrenar sin eso, medir TODO
            avisar(f"  ronda {k}/{rondas}: se quitaron {(w == 0).mean():.0%} de las ventanas")
    score = proyectar(s, n, objetivo=M3)                    # 3) proyección (M3 o Ec. 4)
    thr = thr_iqr(score[valid_mask(n)]) if M1 else None     # 4) umbral IQR (M1) o ninguno
    return score, thr, s_articulo


def comparar(tipos="01234", contaminacion=0.15, semilla=0, M1=True, M2=True, M3=True, largo=2000):
    Xs, ys = serie_contaminada(tipos, contaminacion, semilla, largo)
    m = valid_mask(len(ys))
    print(f"Serie '{tipos}' con {ys.mean():.1%} de anomalías. Entrenando…")
    t = time.time()
    s, thr, s_art = metodo_final(Xs, semilla, M1, M2, M3)
    thr_art = umbral_dinamico(np.asarray(ys)[m].astype(bool), s_art[m])
    nombre = "Nuestro (" + " + ".join([f"M{i}" for i, on in ((1, M1), (2, M2), (3, M3)) if on]) + ")" if (M1 or M2 or M3) else "Artículo"
    if thr is None:
        thr = umbral_dinamico(np.asarray(ys)[m].astype(bool), s[m]); nombre += " · umbral con etiquetas"
    fig, axes = plt.subplots(3, 1, figsize=(12, 7.5), sharex=True)
    dibujar_serie(Xs, ys, f"Serie ({TIPOS.get(tipos, 'mezcla ' + tipos)}), {ys.mean():.0%} anómalo", axes[0])
    dibujar_deteccion(ys, s_art, thr_art, "Artículo (umbral dinámico, USA etiquetas)", axes[1])
    dibujar_deteccion(ys, s, thr, nombre, axes[2])
    plt.tight_layout(); plt.show()
    tabla = pd.DataFrame([evaluar(ys, s_art, thr_art), evaluar(ys, s, thr)],
                         index=["Artículo (con etiquetas)", nombre]).round(3)
    display(tabla)
    print(f"({time.time() - t:.0f} s)")
    return tabla

_ = comparar("01234", 0.20)

### Laboratorio interactivo

Mueve los controles y pulsa **Run Interact / Ejecutar**. Cosas para probar (y qué deberías ver):

| Prueba | Qué esperar |
|---|---|
| Contaminación 2 % vs 30 % con todas las mejoras | el artículo se cae mucho más que el método final |
| Sólo puntuales (`01`), prender/apagar **M3** | M3 marca la diferencia: sin ella aparecen falsas alarmas antes de cada pico |
| Sólo de patrón (`234`), **M3 sin M2** | M3 sola puede empeorar; al prender M2 se recupera |
| Apagar **M1** | el método vuelve a necesitar etiquetas para el umbral |

(Cada corrida tarda entre 20 s y 1 min en la CPU de Colab.)

In [ ]:
# @title Laboratorio (controles)
try:
    import ipywidgets as W
    W.interact_manual(comparar,
        tipos=W.Dropdown(options=[("todas (01234)", "01234"), ("puntuales (01)", "01"), ("de patrón (234)", "234"),
                                  ("sólo puntual global (0)", "0"), ("sólo tendencia (4)", "4")], value="01234", description="anomalías"),
        contaminacion=W.FloatSlider(value=0.15, min=0.01, max=0.30, step=0.01, description="contaminación", readout_format=".0%"),
        semilla=W.IntSlider(value=0, min=0, max=20, description="semilla"),
        M1=W.Checkbox(value=True, description="Mejora 1 (umbral IQR)"),
        M2=W.Checkbox(value=True, description="Mejora 2 (recorte)"),
        M3=W.Checkbox(value=True, description="Mejora 3 (proyección)"),
        largo=W.fixed(2000))
except ImportError:
    print("ipywidgets no está disponible; usa comparar(tipos, contaminacion, semilla, M1, M2, M3) directamente.")

# PARTE D · ¿Funciona de verdad?

## 8. Cómo se evalúa bien un método

Una sola serie no prueba nada: pudo salir bien por casualidad. Para afirmar que un método es mejor hay que
cuidar varias cosas. Así lo hicimos:

### 8.1 Protocolo escrito ANTES de correr (`PROTOCOLO_FINAL.md`)
Antes de lanzar los experimentos dejamos por escrito: qué métodos, qué datos, qué métricas, qué hipótesis y qué
prueba estadística. **Nada se ajustó mirando los resultados.** Esto evita el error más común: probar muchas cosas
y reportar sólo lo que salió bien. Cuando agregamos datos nuevos (SMD, SMAP/MSL, semisintético), también
escribimos su protocolo antes de descargar los datos. Todo cambio posterior quedó anotado como "desviación".

### 8.2 Las hipótesis (en simple)
| Hipótesis | Pregunta | Métrica |
|---|---|---|
| **H1** | ¿El método final da un mejor score que el artículo? | AUPRC |
| **H2** | ¿El final **sin** etiquetas detecta al menos tan bien como el artículo **con** etiquetas? | F1 |
| **H3a** | ¿El recorte (M2) aporta algo encima de M3? | AUPRC |
| **H3b** | ¿El recorte aporta encima de M1 + M3? | F1 |
| **H4** | Con más anomalías (1 % → 30 %), ¿el final se degrada menos que el artículo? | caída de AUROC |
| **H5a/H5b** | ¿M3 sola mejora al artículo? | AUROC / AUPRC |
| **H4r/H5c** | Lo mismo que H4, pero con anomalías inyectadas sobre datos reales (final / M3) | caída de AUROC |
| **H6** | En series reales, ¿la ventaja de M3 crece con la proporción de anomalías? | correlación |

### 8.3 Los 6 bloques de datos
| Bloque | Qué es | Tamaño |
|---|---|---|
| **Sintéticos** | las 15 series del benchmark del artículo (400 pasos) | 15 series × 20 semillas = 300 corridas |
| **Contaminación** | series de 4 000 pasos con 1, 5, 10, 15, 20, 25 y 30 % de anomalías; tres mezclas (todas, sólo puntuales, sólo de patrón) | 3 × 7 × 20 semillas = 420 |
| **Reales del artículo** | Credit Card (fraude con tarjeta), CICIDS (ataques de red), GECCO (calidad del agua), SWAN-SF (erupciones solares) | 4 × 10 semillas = 40 |
| **SMD** | *Server Machine Dataset*: 28 servidores de una empresa de internet, 38 métricas cada uno | 28 × 5 = 140 |
| **SMAP + MSL** | telemetría de un satélite y del rover Curiosity de la NASA; 81 canales | 81 × 5 = 405 |
| **Semisintético** | datos reales de 10 servidores SMD **sin** anomalías etiquetadas + anomalías inyectadas por nosotros (1–30 %) | 10 × 3 mezclas × 5 niveles × 3 semillas = 450 |

Total: **1 755 corridas** (cada una entrena varias veces el LSTM: el artículo, el recorte, etc.).

¿Por qué tantos bloques? Los **sintéticos** y la **contaminación** permiten controlar exactamente el tipo y la
cantidad de anomalías. Los **reales** prueban que sirve fuera del laboratorio. Pero el artículo sólo tiene 4
datasets reales, que son muy pocos para concluir algo general. Por eso agregamos **SMD y SMAP/MSL** (109 series
reales más). El **semisintético** es el punto intermedio: fondo real, pero anomalías controladas.

### 8.4 La prueba estadística, en simple
* **Wilcoxon pareada.** Para cada serie calculamos *(nuestro − artículo)*. Si nuestro método no fuera mejor, esas
  diferencias serían positivas y negativas más o menos por igual. El **p-valor** es la probabilidad de ver
  diferencias tan cargadas hacia un lado *si en realidad no hubiera mejora*. Si p < 0.05, decimos que la
  diferencia es **significativa**. "Pareada" quiere decir que se comparan los dos métodos **sobre la misma
  serie y la misma semilla**.
* **Corrección de Holm.** Si haces 20 pruebas, alguna saldrá "significativa" por puro azar. Holm ajusta los
  p-valores para que eso no pase.
* **Intervalo de confianza (IC 95 %) bootstrap.** Rango donde muy probablemente está la mejora real. Se obtiene
  remuestreando las series 10 000 veces.
* **Tamaño del efecto** (*rank-biserial*, de −1 a 1). No basta con que sea significativo: también importa qué
  tanto. 1 = el nuestro gana siempre, 0 = empate, −1 = pierde siempre.
* **Gana / total.** En cuántas series ganó. Es lo más fácil de entender.

### 8.5 Mini-experimento en vivo
Una versión pequeña de la evaluación (pocas semillas), para que veas el procedimiento completo. **Con tan pocas
series el p-valor no puede ser muy pequeño**: por eso corrimos la evaluación grande (sección 9).

In [ ]:
N_SEMILLAS = 3                      # súbelo si tienes tiempo (cada semilla ≈ 1–2 min)
NIVELES = [0.05, 0.15, 0.25]
filas = []
t = time.time()
for sem in range(N_SEMILLAS):
    for c in NIVELES:
        Xs, ys = serie_contaminada("01234", c, sem, largo=1500)
        m = valid_mask(len(ys)); yb = np.asarray(ys)[m].astype(bool)
        s, thr, s_art = metodo_final(Xs, sem, avisar=lambda *_: None)
        e_fin = evaluar(ys, s, thr); e_art = evaluar(ys, s_art, umbral_dinamico(yb, s_art[m]))
        filas.append({"semilla": sem, "contaminación": f"{c:.0%}", "AUPRC artículo": e_art["AUPRC"], "AUPRC final": e_fin["AUPRC"],
                      "F1 artículo (con etiquetas)": e_art["F1"], "F1 final (sin etiquetas)": e_fin["F1"]})
        print(f"semilla {sem}, contaminación {c:.0%}: listo ({time.time() - t:.0f} s)")
mini = pd.DataFrame(filas)
display(mini.round(3))
for met in ("AUPRC", "F1"):
    a = mini[[c_ for c_ in mini if c_.startswith(met) and "artículo" in c_][0]]
    b = mini[[c_ for c_ in mini if c_.startswith(met) and "final" in c_][0]]
    d = b - a
    p = wilcoxon(d).pvalue if (d != 0).sum() >= 2 else float("nan")
    print(f"{met}: el final gana en {(d > 0).sum()} de {len(d)} series; diferencia media {d.mean():+.3f}; p (Wilcoxon) = {p:.3f}")

fig, ax = plt.subplots(figsize=(6, 3.4))
ax.scatter(mini["AUPRC artículo"], mini["AUPRC final"], c="#3182bd")
ax.plot([0, 1], [0, 1], "k--", lw=0.8)
ax.set_xlabel("AUPRC artículo"); ax.set_ylabel("AUPRC método final"); ax.set_title("Cada punto es una serie: arriba de la diagonal = gana el final")
plt.show()

## 9. Todos los resultados de la evaluación grande

Los experimentos grandes se corrieron en nuestra máquina (GPU y CPU, muchas horas de cómputo). Aquí sólo se muestran
sus cifras, que vienen de `results/final/final_numbers.json`. Ejecuta las celdas en orden; cada una trae su
explicación.

**Nombres de los métodos en las tablas:**
* **Artículo (estático / dinámico):** el método original con su umbral fijo o con su umbral calibrado con etiquetas.
* **M1, M2, M3:** cada mejora sola. **M1+M3, M2+M3, M1+M2:** combinaciones de dos.
* **Método final:** M1 + M2 (recorte adaptativo) + M3.
* **Referencias:** "Error del LSTM" (usar el error de predicción como score, la forma clásica), **LOF** e
  **Isolation Forest** (dos detectores clásicos que no usan redes neuronales).

In [ ]:
# @title Resultados completos (datos incrustados)
RESULTADOS = {"synth": {"n_units": 300, "n_runs": 300, "methods": {"Artículo · umbral estático": {"auprc": 0.5755265687365038, "auroc": 0.8941274400995664, "f1_bin": 0.4633459763343623, "f1_mac": 0.6919128573955559}, "Artículo · umbral dinámico (usa etiquetas)": {"auprc": 0.5755265687365038, "auroc": 0.8941274400995664, "f1_bin": 0.4692818903558499, "f1_mac": 0.6970684397282919}, "M1 · umbral IQR": {"auprc": 0.5755265687365038, "auroc": 0.8941274400995664, "f1_bin": 0.4809691460128035, "f1_mac": 0.7078864674588147}, "M2 · reponderación suave": {"auprc": 0.601498383973394, "auroc": 0.8974098968496549, "f1_bin": 0.46843684276627034, "f1_mac": 0.6948666370925595}, "M2 · recorte z>2": {"auprc": 0.5964401343524467, "auroc": 0.8961134648909922, "f1_bin": 0.46410271912569373, "f1_mac": 0.6923074470614397}, "M3 · proyección al objetivo": {"auprc": 0.7819768044605112, "auroc": 0.9136052919158015, "f1_bin": 0.5676381463086577, "f1_mac": 0.7531461465028574}, "M1+M3": {"auprc": 0.7819768044605112, "auroc": 0.9136052919158015, "f1_bin": 0.6651344397404745, "f1_mac": 0.8153801458518449}, "M2+M3 (recorte z>2)": {"auprc": 0.7878357195721023, "auroc": 0.9184560790000618, "f1_bin": 0.5770850371735721, "f1_mac": 0.7584741697475258}, "M1+M2 (recorte adaptativo, sin M3)": {"auprc": 0.5991390552177092, "auroc": 0.8965966214305638, "f1_bin": 0.49253921260098404, "f1_mac": 0.7137610518081301}, "M1+M2+M3 con recorte z>2": {"auprc": 0.7878357195721023, "auroc": 0.9184560790000618, "f1_bin": 0.6843132161632747, "f1_mac": 0.8252658375676901}, "**Método final** (recorte adaptativo + proyección + IQR)": {"auprc": 0.7907138863456932, "auroc": 0.9189612757848283, "f1_bin": 0.686846941296598, "f1_mac": 0.8270381683517396}, "Error del LSTM (PD) + IQR": {"auprc": 0.6978246580832214, "auroc": 0.8652497339410684, "f1_bin": 0.6139030583273221, "f1_mac": 0.7876253539521211}}, "hyp": [{"h": "H1", "desc": "Final > Artículo", "metric": "auprc", "n": 300, "mean": 0.2151873176091895, "lo": 0.18504972373237585, "hi": 0.24528284336363232, "p": 5.729734213391088e-28, "rb": 0.731505016722408, "wins": 237, "losses": 62, "p_holm": 1.7189202640173265e-27}, {"h": "H2", "desc": "Final (sin etiquetas) ≥ Artículo dinámico (con etiquetas)", "metric": "f1_bin", "n": 300, "mean": 0.2175650509407481, "lo": 0.19040208484012341, "hi": 0.24407525916631478, "p": 2.510244235777085e-32, "rb": 0.8089314194577353, "wins": 245, "losses": 40, "p_holm": 1.004097694310834e-31}, {"h": "H3a", "desc": "Final > M3 (mismo score sin recorte)", "metric": "auprc", "n": 300, "mean": 0.008737081885182012, "lo": 0.0022777242322372243, "hi": 0.014622001791407206, "p": 6.7499098374874925e-18, "rb": 0.5779870779870779, "wins": 227, "losses": 69, "p_holm": 1.3499819674974985e-17}, {"h": "H3b", "desc": "Final > M1+M3 (todo menos el recorte)", "metric": "f1_bin", "n": 300, "mean": 0.021712501556123576, "lo": 0.0148210714108072, "hi": 0.02841613203185213, "p": 8.61399496805799e-14, "rb": 0.5566596931659693, "wins": 170, "losses": 69, "p_holm": 8.61399496805799e-14}], "hyp_h5": [{"h": "H5a", "desc": "M3 (sólo proyección al objetivo) > Artículo", "metric": "auroc", "n": 300, "mean": 0.019477851816235344, "lo": 0.010772847614936184, "hi": 0.028049218466396394, "p": 6.819372294468822e-06, "rb": 0.3002006688963211, "wins": 176, "losses": 123, "p_holm": 6.819372294468822e-06}, {"h": "H5b", "desc": "M3 (sólo proyección al objetivo) > Artículo", "metric": "auprc", "n": 300, "mean": 0.20645023572400745, "lo": 0.17716668610093764, "hi": 0.2360515481447077, "p": 3.215927229246637e-28, "rb": 0.7349832775919732, "wins": 237, "losses": 62, "p_holm": 6.431854458493274e-28}], "paper_protocol": {"f1_bin": {"final": 0.686846941296598, "paper": 0.4775435230724377, "mean": 0.2093034182241604, "lo": 0.18117774977280926, "hi": 0.23650365600374426, "p": 1.0262926672953335e-30, "wins": 238, "n": 300}, "auprc": {"final": 0.7907138863456932, "paper": 0.6062524518794435, "mean": 0.18446143446624974, "lo": 0.15126612843674372, "hi": 0.2170308381656854, "p": 1.5974627543736666e-21, "wins": 229, "n": 300}}}, "contam": {"n_units": 420, "n_runs": 420, "methods": {"Artículo · umbral estático": {"auprc": 0.4646124131849012, "auroc": 0.8565051280683944, "f1_bin": 0.4118881488847649, "f1_mac": 0.6570920228667809}, "Artículo · umbral dinámico (usa etiquetas)": {"auprc": 0.4646124131849012, "auroc": 0.8565051280683944, "f1_bin": 0.5195071814853226, "f1_mac": 0.7045268050145926}, "M1 · umbral IQR": {"auprc": 0.4646124131849012, "auroc": 0.8565051280683944, "f1_bin": 0.4159608840978719, "f1_mac": 0.6678940474774953}, "M2 · reponderación suave": {"auprc": 0.49997355494075435, "auroc": 0.8635936447148937, "f1_bin": 0.4199144506941894, "f1_mac": 0.6617340675809755}, "M2 · recorte z>2": {"auprc": 0.5211046529345424, "auroc": 0.8711350075197196, "f1_bin": 0.43369799020165944, "f1_mac": 0.6701568531622951}, "M3 · proyección al objetivo": {"auprc": 0.7475254584994989, "auroc": 0.9006400637075052, "f1_bin": 0.5545146091971151, "f1_mac": 0.7441824127401716}, "M1+M3": {"auprc": 0.7475254584994989, "auroc": 0.9006400637075052, "f1_bin": 0.6401821036894443, "f1_mac": 0.7986091082678405}, "M2+M3 (recorte z>2)": {"auprc": 0.7882256146830874, "auroc": 0.9205218987206074, "f1_bin": 0.5852036065869652, "f1_mac": 0.762278625757057}, "M1+M2 (recorte adaptativo, sin M3)": {"auprc": 0.5287555286055955, "auroc": 0.8748303504481516, "f1_bin": 0.4476231642659041, "f1_mac": 0.6855215700969672}, "M1+M2+M3 con recorte z>2": {"auprc": 0.7882256146830874, "auroc": 0.9205218987206074, "f1_bin": 0.6789246902165367, "f1_mac": 0.8202554395713705}, "**Método final** (recorte adaptativo + proyección + IQR)": {"auprc": 0.7929330749539261, "auroc": 0.9256872346148388, "f1_bin": 0.686916706585566, "f1_mac": 0.824648969657749}, "Error del LSTM (PD) + IQR": {"auprc": 0.673799607234026, "auroc": 0.8504811481918588, "f1_bin": 0.5791848985506696, "f1_mac": 0.7655123970955342}, "LOF + IQR": {"auprc": 0.6751877594076897, "auroc": 0.8246994424375078, "f1_bin": 0.5179562902324844, "f1_mac": 0.7269233014364241}, "IForest + IQR": {"auprc": 0.5191904333345135, "auroc": 0.8065255574181828, "f1_bin": 0.3483828797051497, "f1_mac": 0.6416862178364624}}, "hyp": [{"h": "H1", "desc": "Final > Artículo", "metric": "auprc", "n": 420, "mean": 0.3283206617690247, "lo": 0.30634608814312037, "hi": 0.35016879351988855, "p": 7.579968185970429e-62, "rb": 0.9344644271010066, "wins": 402, "losses": 18, "p_holm": 3.789984092985215e-61}, {"h": "H2", "desc": "Final (sin etiquetas) ≥ Artículo dinámico (con etiquetas)", "metric": "f1_bin", "n": 420, "mean": 0.1674095251002434, "lo": 0.14691438721246308, "hi": 0.18764028217339387, "p": 9.209694499075217e-37, "rb": 0.7157412825720285, "wins": 306, "losses": 111, "p_holm": 1.8419388998150435e-36}, {"h": "H3a", "desc": "Final > M3 (mismo score sin recorte)", "metric": "auprc", "n": 420, "mean": 0.045407616454427036, "lo": 0.034960730579850816, "hi": 0.05513776852360262, "p": 1.069199813742769e-53, "rb": 0.8687252573238321, "wins": 390, "losses": 30, "p_holm": 4.276799254971076e-53}, {"h": "H3b", "desc": "Final > M1+M3 (todo menos el recorte)", "metric": "f1_bin", "n": 420, "mean": 0.04673460289612156, "lo": 0.042098576523536275, "hi": 0.05147741932353754, "p": 6.70675550485478e-52, "rb": 0.8597054886211513, "wins": 351, "losses": 63, "p_holm": 2.012026651456434e-51}, {"h": "H4", "desc": "Caída de AUROC 1 %→30 % menor en Final que en Artículo", "metric": "auroc_drop", "n": 60, "mean": 0.12725650125286042, "lo": 0.10788615208001778, "hi": 0.1458975869887691, "p": 3.4620901010512977e-11, "rb": 0.9836065573770492, "wins": 57, "losses": 3, "drop_paper": 0.16986344192422925, "drop_final": 0.04260694067136882, "p_holm": 3.4620901010512977e-11}], "hyp_h5": [{"h": "H5a", "desc": "M3 (sólo proyección al objetivo) > Artículo", "metric": "auroc", "n": 420, "mean": 0.044134935639110785, "lo": 0.0349611561258318, "hi": 0.053528464312651944, "p": 6.341343602366334e-14, "rb": 0.42237303472457866, "wins": 259, "losses": 161, "p_holm": 6.341343602366334e-14}, {"h": "H5b", "desc": "M3 (sólo proyección al objetivo) > Artículo", "metric": "auprc", "n": 420, "mean": 0.28291304531459766, "lo": 0.2595899364100013, "hi": 0.3062504381661571, "p": 1.2885879970679965e-53, "rb": 0.8680466010632282, "wins": 345, "losses": 75, "p_holm": 2.577175994135993e-53}], "curves": {"01": [{"Método": "Artículo · umbral estático", "1%": "0.916", "5%": "0.870", "10%": "0.817", "15%": "0.780", "20%": "0.753", "25%": "0.728", "30%": "0.711", "Caída": "0.205"}, {"Método": "M3 · proyección al objetivo", "1%": "0.953", "5%": "0.958", "10%": "0.954", "15%": "0.952", "20%": "0.945", "25%": "0.938", "30%": "0.931", "Caída": "0.022"}, {"Método": "M2 · recorte z>2", "1%": "0.915", "5%": "0.870", "10%": "0.818", "15%": "0.782", "20%": "0.754", "25%": "0.731", "30%": "0.714", "Caída": "0.201"}, {"Método": "**Método final** (recorte adaptativo + proyección + IQR)", "1%": "0.953", "5%": "0.959", "10%": "0.956", "15%": "0.958", "20%": "0.956", "25%": "0.955", "30%": "0.954", "Caída": "-0.002"}, {"Método": "Error del LSTM (PD) + IQR", "1%": "0.955", "5%": "0.958", "10%": "0.955", "15%": "0.950", "20%": "0.944", "25%": "0.934", "30%": "0.928", "Caída": "0.027"}, {"Método": "LOF + IQR", "1%": "0.955", "5%": "0.954", "10%": "0.826", "15%": "0.731", "20%": "0.692", "25%": "0.672", "30%": "0.656", "Caída": "0.300"}, {"Método": "IForest + IQR", "1%": "0.922", "5%": "0.927", "10%": "0.924", "15%": "0.922", "20%": "0.922", "25%": "0.915", "30%": "0.910", "Caída": "0.012"}], "01234": [{"Método": "Artículo · umbral estático", "1%": "0.957", "5%": "0.931", "10%": "0.891", "15%": "0.857", "20%": "0.830", "25%": "0.809", "30%": "0.789", "Caída": "0.167"}, {"Método": "M3 · proyección al objetivo", "1%": "0.943", "5%": "0.939", "10%": "0.923", "15%": "0.902", "20%": "0.895", "25%": "0.880", "30%": "0.873", "Caída": "0.070"}, {"Método": "M2 · recorte z>2", "1%": "0.955", "5%": "0.937", "10%": "0.911", "15%": "0.879", "20%": "0.854", "25%": "0.833", "30%": "0.813", "Caída": "0.142"}, {"Método": "**Método final** (recorte adaptativo + proyección + IQR)", "1%": "0.935", "5%": "0.945", "10%": "0.946", "15%": "0.931", "20%": "0.927", "25%": "0.913", "30%": "0.904", "Caída": "0.031"}, {"Método": "Error del LSTM (PD) + IQR", "1%": "0.880", "5%": "0.881", "10%": "0.864", "15%": "0.853", "20%": "0.841", "25%": "0.831", "30%": "0.824", "Caída": "0.055"}, {"Método": "LOF + IQR", "1%": "0.895", "5%": "0.903", "10%": "0.902", "15%": "0.884", "20%": "0.873", "25%": "0.866", "30%": "0.853", "Caída": "0.042"}, {"Método": "IForest + IQR", "1%": "0.814", "5%": "0.782", "10%": "0.782", "15%": "0.787", "20%": "0.775", "25%": "0.776", "30%": "0.764", "Caída": "0.050"}], "234": [{"Método": "Artículo · umbral estático", "1%": "0.989", "5%": "0.952", "10%": "0.924", "15%": "0.892", "20%": "0.879", "25%": "0.862", "30%": "0.851", "Caída": "0.138"}, {"Método": "M3 · proyección al objetivo", "1%": "0.944", "5%": "0.897", "10%": "0.862", "15%": "0.826", "20%": "0.814", "25%": "0.798", "30%": "0.788", "Caída": "0.156"}, {"Método": "M2 · recorte z>2", "1%": "0.983", "5%": "0.976", "10%": "0.959", "15%": "0.928", "20%": "0.911", "25%": "0.893", "30%": "0.877", "Caída": "0.106"}, {"Método": "**Método final** (recorte adaptativo + proyección + IQR)", "1%": "0.939", "5%": "0.937", "10%": "0.921", "15%": "0.887", "20%": "0.872", "25%": "0.853", "30%": "0.841", "Caída": "0.098"}, {"Método": "Error del LSTM (PD) + IQR", "1%": "0.864", "5%": "0.784", "10%": "0.754", "15%": "0.733", "20%": "0.717", "25%": "0.710", "30%": "0.700", "Caída": "0.164"}, {"Método": "LOF + IQR", "1%": "0.888", "5%": "0.853", "10%": "0.849", "15%": "0.808", "20%": "0.768", "25%": "0.758", "30%": "0.732", "Caída": "0.156"}, {"Método": "IForest + IQR", "1%": "0.756", "5%": "0.732", "10%": "0.709", "15%": "0.697", "20%": "0.711", "25%": "0.702", "30%": "0.709", "Caída": "0.047"}]}, "anom_rate": [{"ds": "01", "ratio": 0.01, "anom_rate": 0.009931852599697075}, {"ds": "01", "ratio": 0.05, "anom_rate": 0.04876325088339218}, {"ds": "01", "ratio": 0.1, "anom_rate": 0.09522968197879854}, {"ds": "01", "ratio": 0.15, "anom_rate": 0.13985360928823823}, {"ds": "01", "ratio": 0.2, "anom_rate": 0.18214285714285708}, {"ds": "01", "ratio": 0.25, "anom_rate": 0.2223245835436648}, {"ds": "01", "ratio": 0.3, "anom_rate": 0.2604745078243311}, {"ds": "01234", "ratio": 0.01, "anom_rate": 0.011534578495709224}, {"ds": "01234", "ratio": 0.05, "anom_rate": 0.04873801110550222}, {"ds": "01234", "ratio": 0.1, "anom_rate": 0.09483846542150424}, {"ds": "01234", "ratio": 0.15, "anom_rate": 0.13821302372539118}, {"ds": "01234", "ratio": 0.2, "anom_rate": 0.1809187279151943}, {"ds": "01234", "ratio": 0.25, "anom_rate": 0.22204694598687527}, {"ds": "01234", "ratio": 0.3, "anom_rate": 0.25851842503785966}, {"ds": "234", "ratio": 0.01, "anom_rate": 0.00744573447753653}, {"ds": "234", "ratio": 0.05, "anom_rate": 0.05061837455830384}, {"ds": "234", "ratio": 0.1, "anom_rate": 0.09302120141342751}, {"ds": "234", "ratio": 0.15, "anom_rate": 0.13912165572942956}, {"ds": "234", "ratio": 0.2, "anom_rate": 0.1841367995961635}, {"ds": "234", "ratio": 0.25, "anom_rate": 0.21877839475012617}, {"ds": "234", "ratio": 0.3, "anom_rate": 0.2603735487127713}]}, "real": {"n_units": 40, "n_runs": 40, "methods": {"Artículo · umbral estático": {"auprc": 0.2199950213740812, "auroc": 0.7468610583991169, "f1_bin": 0.10534675187638891, "f1_mac": 0.5021193271689981}, "Artículo · umbral dinámico (usa etiquetas)": {"auprc": 0.2199950213740812, "auroc": 0.7468610583991169, "f1_bin": 0.23003052500425064, "f1_mac": 0.5764253644887276}, "M1 · umbral IQR": {"auprc": 0.2199950213740812, "auroc": 0.7468610583991169, "f1_bin": 0.11950613857221079, "f1_mac": 0.5114631237817869}, "M2 · reponderación suave": {"auprc": 0.2118413486623898, "auroc": 0.7416752796527957, "f1_bin": 0.10442743967618442, "f1_mac": 0.5020644663121734}, "M2 · recorte z>2": {"auprc": 0.23412283296666084, "auroc": 0.7414405639572346, "f1_bin": 0.10406689752986438, "f1_mac": 0.5017136073709634}, "M3 · proyección al objetivo": {"auprc": 0.24673327656905686, "auroc": 0.764742905402192, "f1_bin": 0.10300769901039879, "f1_mac": 0.500721804170866}, "M1+M3": {"auprc": 0.24673327656905686, "auroc": 0.764742905402192, "f1_bin": 0.09821814982211555, "f1_mac": 0.49694682799537865}, "M2+M3 (recorte z>2)": {"auprc": 0.26572696596157747, "auroc": 0.7623854468411667, "f1_bin": 0.0994193454456044, "f1_mac": 0.4984358038920552}, "M1+M2 (recorte adaptativo, sin M3)": {"auprc": 0.23881305270908199, "auroc": 0.7394911245970321, "f1_bin": 0.1166509794783194, "f1_mac": 0.5102267432876113}, "M1+M2+M3 con recorte z>2": {"auprc": 0.26572696596157747, "auroc": 0.7623854468411667, "f1_bin": 0.09370857678172224, "f1_mac": 0.4941097179354541}, "**Método final** (recorte adaptativo + proyección + IQR)": {"auprc": 0.2663358958453562, "auroc": 0.7760627029888756, "f1_bin": 0.10046147847916856, "f1_mac": 0.49332270755475827}, "Error del LSTM (PD) + IQR": {"auprc": 0.1252673098160799, "auroc": 0.7502024307862054, "f1_bin": 0.06891195531129346, "f1_mac": 0.48658558376574235}}, "hyp": [{"h": "H1", "desc": "Final > Artículo", "metric": "auprc", "n": 40, "mean": 0.04634087447127505, "lo": 0.03640227184241504, "hi": 0.0562867893861205, "p": 1.8189894035458565e-12, "rb": 1.0, "wins": 40, "losses": 0, "p_holm": 7.275957614183426e-12}, {"h": "H2", "desc": "Final (sin etiquetas) ≥ Artículo dinámico (con etiquetas)", "metric": "f1_bin", "n": 40, "mean": -0.12956904652508208, "lo": -0.16014381929821936, "hi": -0.0978458879194404, "p": 1.03669663076289e-07, "rb": -0.8658536585365854, "wins": 10, "losses": 30, "p_holm": 3.11008989228867e-07}, {"h": "H3a", "desc": "Final > M3 (mismo score sin recorte)", "metric": "auprc", "n": 40, "mean": 0.019602619276299398, "lo": 0.004958633136700444, "hi": 0.03523609227932115, "p": 0.42797189528209856, "rb": 0.14634146341463414, "wins": 23, "losses": 17, "p_holm": 0.8559437905641971}, {"h": "H3b", "desc": "Final > M1+M3 (todo menos el recorte)", "metric": "f1_bin", "n": 40, "mean": 0.002243328657053014, "lo": -0.0011901542251696615, "hi": 0.006031970476593111, "p": 0.599208943085614, "rb": -0.0975609756097561, "wins": 20, "losses": 20, "p_holm": 0.8559437905641971}], "hyp_h5": [{"h": "H5a", "desc": "M3 (sólo proyección al objetivo) > Artículo", "metric": "auroc", "n": 40, "mean": 0.017881847003075123, "lo": 0.006937038773776931, "hi": 0.028991218153847587, "p": 0.006371744650095934, "rb": 0.4878048780487805, "wins": 20, "losses": 20, "p_holm": 0.012743489300191868}, {"h": "H5b", "desc": "M3 (sólo proyección al objetivo) > Artículo", "metric": "auprc", "n": 40, "mean": 0.026738255194975653, "lo": 0.013301781645390974, "hi": 0.040934176397792094, "p": 0.006371744650095934, "rb": 0.4878048780487805, "wins": 20, "losses": 20, "p_holm": 0.012743489300191868}], "per_ds": {"cicids": {"Artículo · umbral estático": {"auprc": 0.01720055277485129, "auroc": 0.6462332837722201, "f1_bin": 0.00199881524837076}, "Artículo · umbral dinámico (usa etiquetas)": {"auprc": 0.01720055277485129, "auroc": 0.6462332837722201, "f1_bin": 0.00262501507728405}, "M1 · umbral IQR": {"auprc": 0.01720055277485129, "auroc": 0.6462332837722201, "f1_bin": 0.00217296493990402}, "M2 · reponderación suave": {"auprc": 0.01753971336401395, "auroc": 0.6574111372825726, "f1_bin": 0.0023214764459369196}, "M2 · recorte z>2": {"auprc": 0.017741957467237977, "auroc": 0.6601483947534204, "f1_bin": 0.0024315371290077902}, "M3 · proyección al objetivo": {"auprc": 0.01513155684431658, "auroc": 0.6227157534282106, "f1_bin": 0.00152799388588838}, "M1+M3": {"auprc": 0.01513155684431658, "auroc": 0.6227157534282106, "f1_bin": 0.00191605180365064}, "M2+M3 (recorte z>2)": {"auprc": 0.016066050767814012, "auroc": 0.6460691322651584, "f1_bin": 0.0020036224656490398}, "M1+M2 (recorte adaptativo, sin M3)": {"auprc": 0.01756414234701246, "auroc": 0.6608163445139703, "f1_bin": 0.00209926054390169}, "M1+M2+M3 con recorte z>2": {"auprc": 0.016066050767814012, "auroc": 0.6460691322651584, "f1_bin": 0.00203605880262106}, "**Método final** (recorte adaptativo + proyección + IQR)": {"auprc": 0.0199246388124137, "auroc": 0.7144601564525069, "f1_bin": 0.0029625721889109296}, "Error del LSTM (PD) + IQR": {"auprc": 0.02194519547729161, "auroc": 0.7377832153060241, "f1_bin": 0.00189720525999011}}, "creditcard": {"Artículo · umbral estático": {"auprc": 0.11352951092911605, "auroc": 0.8935036642378265, "f1_bin": 0.01825961994029065}, "Artículo · umbral dinámico (usa etiquetas)": {"auprc": 0.11352951092911605, "auroc": 0.8935036642378265, "f1_bin": 0.21071187609510628}, "M1 · umbral IQR": {"auprc": 0.11352951092911605, "auroc": 0.8935036642378265, "f1_bin": 0.017891686619161063}, "M2 · reponderación suave": {"auprc": 0.0753276181075421, "auroc": 0.8721231315208392, "f1_bin": 0.01596007418226043}, "M2 · recorte z>2": {"auprc": 0.07779729075824879, "auroc": 0.8673173538714145, "f1_bin": 0.01521926647325498}, "M3 · proyección al objetivo": {"auprc": 0.2086743043425574, "auroc": 0.9461096722877442, "f1_bin": 0.02148198001478303}, "M1+M3": {"auprc": 0.2086743043425574, "auroc": 0.9461096722877442, "f1_bin": 0.026538453193415423}, "M2+M3 (recorte z>2)": {"auprc": 0.1903721956265701, "auroc": 0.9330813315427976, "f1_bin": 0.020196696877626572}, "M1+M2 (recorte adaptativo, sin M3)": {"auprc": 0.07257867518586905, "auroc": 0.8608019328222494, "f1_bin": 0.0145565894796785}, "M1+M2+M3 con recorte z>2": {"auprc": 0.1903721956265701, "auroc": 0.9330813315427976, "f1_bin": 0.02182876435168092}, "**Método final** (recorte adaptativo + proyección + IQR)": {"auprc": 0.1859096926150518, "auroc": 0.923375088024617, "f1_bin": 0.020039285332060657}, "Error del LSTM (PD) + IQR": {"auprc": 0.06712123022048694, "auroc": 0.9436994361619868, "f1_bin": 0.034897154362716294}}, "gecco": {"Artículo · umbral estático": {"auprc": 0.41110005687851603, "auroc": 0.8849147290720621, "f1_bin": 0.10950115171252273}, "Artículo · umbral dinámico (usa etiquetas)": {"auprc": 0.41110005687851603, "auroc": 0.8849147290720621, "f1_bin": 0.32585497737793817}, "M1 · umbral IQR": {"auprc": 0.41110005687851603, "auroc": 0.8849147290720621, "f1_bin": 0.18397046586603324}, "M2 · reponderación suave": {"auprc": 0.42339905193463767, "auroc": 0.8870675160636905, "f1_bin": 0.11529063045103465}, "M2 · recorte z>2": {"auprc": 0.4948316246623139, "auroc": 0.8875789708146812, "f1_bin": 0.11671913866800732}, "M3 · proyección al objetivo": {"auprc": 0.38833512937365267, "auroc": 0.872971020754213, "f1_bin": 0.08156218276021422}, "M1+M3": {"auprc": 0.38833512937365267, "auroc": 0.872971020754213, "f1_bin": 0.0741836558951691}, "M2+M3 (recorte z>2)": {"auprc": 0.4882067364776838, "auroc": 0.8707103934746824, "f1_bin": 0.0818962057039845}, "M1+M2 (recorte adaptativo, sin M3)": {"auprc": 0.49729310765204654, "auroc": 0.8832257768219126, "f1_bin": 0.1595742588197858}, "M1+M2+M3 con recorte z>2": {"auprc": 0.4882067364776838, "auroc": 0.8707103934746824, "f1_bin": 0.07389436004697342}, "**Método final** (recorte adaptativo + proyección + IQR)": {"auprc": 0.4895499657616137, "auroc": 0.8714072145677513, "f1_bin": 0.06933569732600917}, "Error del LSTM (PD) + IQR": {"auprc": 0.10192622501209563, "auroc": 0.7938561729510313, "f1_bin": 0.05664982996906737}}, "swan_sf": {"Artículo · umbral estático": {"auprc": 0.3381499649138416, "auroc": 0.5627925565143587, "f1_bin": 0.2916274206043716}, "Artículo · umbral dinámico (usa etiquetas)": {"auprc": 0.3381499649138416, "auroc": 0.5627925565143587, "f1_bin": 0.38093023146667404}, "M1 · umbral IQR": {"auprc": 0.3381499649138416, "auroc": 0.5627925565143587, "f1_bin": 0.2739894368637449}, "M2 · reponderación suave": {"auprc": 0.3310990112433655, "auroc": 0.5500993337440805, "f1_bin": 0.2841375776255057}, "M2 · recorte z>2": {"auprc": 0.34612045897884264, "auroc": 0.5507175363894224, "f1_bin": 0.28189764784918736}, "M3 · proyección al objetivo": {"auprc": 0.37479211571570076, "auroc": 0.6171751751386001, "f1_bin": 0.30745863938070955}, "M1+M3": {"auprc": 0.37479211571570076, "auroc": 0.6171751751386001, "f1_bin": 0.29023443839622703}, "M2+M3 (recorte z>2)": {"auprc": 0.36826288097424187, "auroc": 0.5996809300820283, "f1_bin": 0.2935808567351575}, "M1+M2 (recorte adaptativo, sin M3)": {"auprc": 0.36781628565139984, "auroc": 0.5531204442299967, "f1_bin": 0.29037380906991167}, "M1+M2+M3 con recorte z>2": {"auprc": 0.36826288097424187, "auroc": 0.5996809300820283, "f1_bin": 0.27707512392561356}, "**Método final** (recorte adaptativo + proyección + IQR)": {"auprc": 0.36995928619234586, "auroc": 0.5950083529106276, "f1_bin": 0.3095083590696935}, "Error del LSTM (PD) + IQR": {"auprc": 0.3100765885544454, "auroc": 0.5254708987257797, "f1_bin": 0.1822036316534001}}}, "seeds": {"cicids": 10, "creditcard": 10, "gecco": 10, "swan_sf": 10}}, "smd": {"n_units": 28, "n_runs": 140, "methods": {"Artículo · umbral estático": {"auprc": 0.4537402366815204, "auroc": 0.8731862394461335, "f1_bin": 0.3069143883072156, "f1_mac": 0.6153898058645862}, "Artículo · umbral dinámico (usa etiquetas)": {"auprc": 0.4537402366815204, "auroc": 0.8731862394461335, "f1_bin": 0.3397464091283937, "f1_mac": 0.6670403413072451}, "M1 · umbral IQR": {"auprc": 0.4537402366815204, "auroc": 0.8731862394461335, "f1_bin": 0.3125053750804941, "f1_mac": 0.6202668865008257}, "M2 · reponderación suave": {"auprc": 0.4344687749021345, "auroc": 0.8597774886587858, "f1_bin": 0.3000060385693046, "f1_mac": 0.6115976669710577}, "M2 · recorte z>2": {"auprc": 0.44893235657668296, "auroc": 0.8560647001591387, "f1_bin": 0.2974517653453607, "f1_mac": 0.6104620268444783}, "M3 · proyección al objetivo": {"auprc": 0.3999721944580748, "auroc": 0.85058308160647, "f1_bin": 0.28783978522405684, "f1_mac": 0.6056547310443042}, "M1+M3": {"auprc": 0.3999721944580748, "auroc": 0.85058308160647, "f1_bin": 0.288228287730895, "f1_mac": 0.6053595561019849}, "M2+M3 (recorte z>2)": {"auprc": 0.3741066351196283, "auroc": 0.8216070225719077, "f1_bin": 0.263503656844234, "f1_mac": 0.5921623859091197}, "M1+M2 (recorte adaptativo, sin M3)": {"auprc": 0.4722304962587111, "auroc": 0.8593929992992455, "f1_bin": 0.30917341878603455, "f1_mac": 0.6186805092849678}, "M1+M2+M3 con recorte z>2": {"auprc": 0.3741066351196283, "auroc": 0.8216070225719077, "f1_bin": 0.2671211937223116, "f1_mac": 0.5936235931468371}, "**Método final** (recorte adaptativo + proyección + IQR)": {"auprc": 0.39209625035654716, "auroc": 0.8246503176097137, "f1_bin": 0.2690995909097069, "f1_mac": 0.5920129485942153}, "Error del LSTM (PD) + IQR": {"auprc": 0.22506193140537012, "auroc": 0.776688111996916, "f1_bin": 0.23143796921536922, "f1_mac": 0.5795365833050562}, "LOF + IQR": {"auprc": 0.10383765162567121, "auroc": 0.6197906677031311, "f1_bin": 0.14344824575650747, "f1_mac": 0.5415183425002756}, "IForest + IQR": {"auprc": 0.3713527801237591, "auroc": 0.8261083215790747, "f1_bin": 0.32672023141164325, "f1_mac": 0.6410223265152462}}, "hyp": [{"h": "H1", "desc": "Final > Artículo", "metric": "auprc", "n": 28, "mean": -0.06164398632497332, "lo": -0.11584894721876907, "hi": -0.0024086851310608126, "p": 0.020409412682056427, "rb": -0.4975369458128079, "wins": 9, "losses": 19, "p_holm": 0.06122823804616928}, {"h": "H2", "desc": "Final (sin etiquetas) ≥ Artículo dinámico (con etiquetas)", "metric": "f1_bin", "n": 28, "mean": -0.07064681821868679, "lo": -0.16637530329581812, "hi": 0.02882419732093993, "p": 0.18578492105007172, "rb": -0.29064039408866993, "wins": 14, "losses": 14, "p_holm": 0.37156984210014343}, {"h": "H3a", "desc": "Final > M3 (mismo score sin recorte)", "metric": "auprc", "n": 28, "mean": -0.007875944101527593, "lo": -0.05481547319151897, "hi": 0.04316801254463048, "p": 0.3989470452070236, "rb": -0.18719211822660098, "wins": 12, "losses": 16, "p_holm": 0.3989470452070236}, {"h": "H3b", "desc": "Final > M1+M3 (todo menos el recorte)", "metric": "f1_bin", "n": 28, "mean": -0.019128696821187997, "lo": -0.03286752616129025, "hi": -0.005855190352702325, "p": 0.003438994288444519, "rb": -0.6157635467980296, "wins": 6, "losses": 22, "p_holm": 0.013755977153778076}, {"h": "H5a", "desc": "M3 (sólo proyección al objetivo) > Artículo", "metric": "auroc", "n": 28, "mean": -0.02260315783966355, "lo": -0.028067076037043042, "hi": -0.017524867698480173, "p": 1.4901161193847656e-08, "rb": -0.9950738916256158, "wins": 1, "losses": 27, "p_holm": 8.940696716308594e-08}, {"h": "H5b", "desc": "M3 (sólo proyección al objetivo) > Artículo", "metric": "auprc", "n": 28, "mean": -0.053768042223445715, "lo": -0.07154813077270351, "hi": -0.035871227782097205, "p": 9.395182132720947e-06, "rb": -0.8669950738916257, "wins": 2, "losses": 26, "p_holm": 4.6975910663604736e-05}], "per_ds": {"machine-1-1": {"Artículo · umbral estático": {"auprc": 0.6805392303169626, "auroc": 0.9330019060177517, "f1_bin": 0.7225342446342876}, "Artículo · umbral dinámico (usa etiquetas)": {"auprc": 0.6805392303169626, "auroc": 0.9330019060177517, "f1_bin": 0.48418439142337444}, "M1 · umbral IQR": {"auprc": 0.6805392303169626, "auroc": 0.9330019060177517, "f1_bin": 0.7166486239415372}, "M2 · reponderación suave": {"auprc": 0.5743586289854525, "auroc": 0.9069738961522361, "f1_bin": 0.6455394519221807}, "M2 · recorte z>2": {"auprc": 0.5464140616196594, "auroc": 0.9000696227170166, "f1_bin": 0.6366079733147247}, "M3 · proyección al objetivo": {"auprc": 0.632359301058153, "auroc": 0.9163208165551554, "f1_bin": 0.6769252045978019}, "M1+M3": {"auprc": 0.632359301058153, "auroc": 0.9163208165551554, "f1_bin": 0.6786230075770985}, "M2+M3 (recorte z>2)": {"auprc": 0.45969925291050745, "auroc": 0.8616491786661763, "f1_bin": 0.5626791819752875}, "M1+M2 (recorte adaptativo, sin M3)": {"auprc": 0.5507660545262271, "auroc": 0.8989300171167619, "f1_bin": 0.6412133677670202}, "M1+M2+M3 con recorte z>2": {"auprc": 0.45969925291050745, "auroc": 0.8616491786661763, "f1_bin": 0.5677731328597797}, "**Método final** (recorte adaptativo + proyección + IQR)": {"auprc": 0.4673244352715497, "auroc": 0.860066340247827, "f1_bin": 0.5759518362545288}, "Error del LSTM (PD) + IQR": {"auprc": 0.3494412850950298, "auroc": 0.819962751298549, "f1_bin": 0.4892189736026287}, "LOF + IQR": {"auprc": 0.1496851421193837, "auroc": 0.5842653077636365, "f1_bin": 0.1804798255179934}, "IForest + IQR": {"auprc": 0.4932071435611869, "auroc": 0.8903652349605229, "f1_bin": 0.45720410316080995}}, "machine-1-2": {"Artículo · umbral estático": {"auprc": 0.1810328751888654, "auroc": 0.899308720741443, "f1_bin": 0.3202604163909174}, "Artículo · umbral dinámico (usa etiquetas)": {"auprc": 0.1810328751888654, "auroc": 0.899308720741443, "f1_bin": 0.2649189404067516}, "M1 · umbral IQR": {"auprc": 0.1810328751888654, "auroc": 0.899308720741443, "f1_bin": 0.304566891231313}, "M2 · reponderación suave": {"auprc": 0.22630980782728752, "auroc": 0.8876221245123241, "f1_bin": 0.27992975944760196}, "M2 · recorte z>2": {"auprc": 0.2940652946064663, "auroc": 0.8962385059517292, "f1_bin": 0.30338958644301106}, "M3 · proyección al objetivo": {"auprc": 0.155802240926936, "auroc": 0.8810028234832836, "f1_bin": 0.29298948688326076}, "M1+M3": {"auprc": 0.155802240926936, "auroc": 0.8810028234832836, "f1_bin": 0.29005160098723587}, "M2+M3 (recorte z>2)": {"auprc": 0.22692879188338666, "auroc": 0.8609911821624057, "f1_bin": 0.28348378117556494}, "M1+M2 (recorte adaptativo, sin M3)": {"auprc": 0.3199820922225283, "auroc": 0.9087580185743883, "f1_bin": 0.3305818776054588}, "M1+M2+M3 con recorte z>2": {"auprc": 0.22692879188338666, "auroc": 0.8609911821624057, "f1_bin": 0.2790718371125832}, "**Método final** (recorte adaptativo + proyección + IQR)": {"auprc": 0.24681759194024008, "auroc": 0.8629704541615808, "f1_bin": 0.2988157517980587}, "Error del LSTM (PD) + IQR": {"auprc": 0.09492248666326811, "auroc": 0.7886971985796694, "f1_bin": 0.18513033066808726}, "LOF + IQR": {"auprc": 0.0751533923068203, "auroc": 0.6580134497805997, "f1_bin": 0.1204379562043795}, "IForest + IQR": {"auprc": 0.2377358005353154, "auroc": 0.8601943774910623, "f1_bin": 0.3212179288119003}}, "machine-1-3": {"Artículo · umbral estático": {"auprc": 0.529153703288016, "auroc": 0.8563240029794048, "f1_bin": 0.3515880100905262}, "Artículo · umbral dinámico (usa etiquetas)": {"auprc": 0.529153703288016, "auroc": 0.8563240029794048, "f1_bin": 0.5045071603725495}, "M1 · umbral IQR": {"auprc": 0.529153703288016, "auroc": 0.8563240029794048, "f1_bin": 0.36090800949804497}, "M2 · reponderación suave": {"auprc": 0.49583990403028, "auroc": 0.8428433215391419, "f1_bin": 0.3557397677333448}, "M2 · recorte z>2": {"auprc": 0.46993200686037967, "auroc": 0.847734010985055, "f1_bin": 0.33956734205342554}, "M3 · proyección al objetivo": {"auprc": 0.4680121211989041, "auroc": 0.8259079641648581, "f1_bin": 0.2973047206216676}, "M1+M3": {"auprc": 0.4680121211989041, "auroc": 0.8259079641648581, "f1_bin": 0.33177339710346004}, "M2+M3 (recorte z>2)": {"auprc": 0.33955445052897854, "auroc": 0.7877630603955168, "f1_bin": 0.25380278549664714}, "M1+M2 (recorte adaptativo, sin M3)": {"auprc": 0.4641323869437774, "auroc": 0.8490840858987415, "f1_bin": 0.33639887643671784}, "M1+M2+M3 con recorte z>2": {"auprc": 0.33955445052897854, "auroc": 0.7877630603955168, "f1_bin": 0.2883471273021634}, "**Método final** (recorte adaptativo + proyección + IQR)": {"auprc": 0.3390695041905515, "auroc": 0.7859146519684985, "f1_bin": 0.26610672408450853}, "Error del LSTM (PD) + IQR": {"auprc": 0.3691445082727724, "auroc": 0.8424442872314165, "f1_bin": 0.39446861687191126}, "LOF + IQR": {"auprc": 0.0663027253659371, "auroc": 0.6142486485108118, "f1_bin": 0.1742367833209233}, "IForest + IQR": {"auprc": 0.20048084346674103, "auroc": 0.6810090376419845, "f1_bin": 0.1739909143565273}}, "machine-1-4": {"Artículo · umbral estático": {"auprc": 0.5395196291201626, "auroc": 0.869169535346493, "f1_bin": 0.3084113773516991}, "Artículo · umbral dinámico (usa etiquetas)": {"auprc": 0.5395196291201626, "auroc": 0.869169535346493, "f1_bin": 0.5666285042727155}, "M1 · umbral IQR": {"auprc": 0.5395196291201626, "auroc": 0.869169535346493, "f1_bin": 0.36518861318648793}, "M2 · reponderación suave": {"auprc": 0.5221869479408559, "auroc": 0.8626515437612872, "f1_bin": 0.34198989793288426}, "M2 · recorte z>2": {"auprc": 0.489990300856107, "auroc": 0.8634511912888966, "f1_bin": 0.3389244963635655}, "M3 · proyección al objetivo": {"auprc": 0.4277960441721059, "auroc": 0.835802728029786, "f1_bin": 0.2604847488675204}, "M1+M3": {"auprc": 0.4277960441721059, "auroc": 0.835802728029786, "f1_bin": 0.2850612802124643}, "M2+M3 (recorte z>2)": {"auprc": 0.3400842329086519, "auroc": 0.7973131968955316, "f1_bin": 0.2456363325240889}, "M1+M2 (recorte adaptativo, sin M3)": {"auprc": 0.4834368415906158, "auroc": 0.8668966815305434, "f1_bin": 0.3142859446167211}, "M1+M2+M3 con recorte z>2": {"auprc": 0.3400842329086519, "auroc": 0.7973131968955316, "f1_bin": 0.29789594900141786}, "**Método final** (recorte adaptativo + proyección + IQR)": {"auprc": 0.34227430075182425, "auroc": 0.7954879055490193, "f1_bin": 0.26105971850438714}, "Error del LSTM (PD) + IQR": {"auprc": 0.3276283351758851, "auroc": 0.8218228523634533, "f1_bin": 0.3581469735108128}, "LOF + IQR": {"auprc": 0.0706798110694135, "auroc": 0.5766575098890777, "f1_bin": 0.1489698890649762}, "IForest + IQR": {"auprc": 0.19453433860452934, "auroc": 0.7147492870664905, "f1_bin": 0.17498044936823626}}, "machine-1-5": {"Artículo · umbral estático": {"auprc": 0.619814510929788, "auroc": 0.9290953835709438, "f1_bin": 0.0648904624496561}, "Artículo · umbral dinámico (usa etiquetas)": {"auprc": 0.619814510929788, "auroc": 0.9290953835709438, "f1_bin": 0.5971620498203735}, "M1 · umbral IQR": {"auprc": 0.619814510929788, "auroc": 0.9290953835709438, "f1_bin": 0.0758291364678298}, "M2 · reponderación suave": {"auprc": 0.5878420829390041, "auroc": 0.9281134589273593, "f1_bin": 0.06277361360872807}, "M2 · recorte z>2": {"auprc": 0.6044788095173382, "auroc": 0.9352466904276986, "f1_bin": 0.05955007754107962}, "M3 · proyección al objetivo": {"auprc": 0.4790536545412521, "auroc": 0.8744213340122199, "f1_bin": 0.05628150790799208}, "M1+M3": {"auprc": 0.4790536545412521, "auroc": 0.8744213340122199, "f1_bin": 0.05193896315594058}, "M2+M3 (recorte z>2)": {"auprc": 0.4712565625194872, "auroc": 0.87719450101833, "f1_bin": 0.05279323294989412}, "M1+M2 (recorte adaptativo, sin M3)": {"auprc": 0.5871513882726942, "auroc": 0.9255967413441955, "f1_bin": 0.05998642071889434}, "M1+M2+M3 con recorte z>2": {"auprc": 0.4712565625194872, "auroc": 0.87719450101833, "f1_bin": 0.039339337691833405}, "**Método final** (recorte adaptativo + proyección + IQR)": {"auprc": 0.4063818735282195, "auroc": 0.8792292939579089, "f1_bin": 0.037261403143231675}, "Error del LSTM (PD) + IQR": {"auprc": 0.13845691496631946, "auroc": 0.8476410386965376, "f1_bin": 0.04228207247759892}, "LOF + IQR": {"auprc": 0.0342202119363623, "auroc": 0.7496571622539037, "f1_bin": 0.0495283018867924}, "IForest + IQR": {"auprc": 0.22453025513526978, "auroc": 0.8235294042769858, "f1_bin": 0.14551895979655466}}, "machine-1-6": {"Artículo · umbral estático": {"auprc": 0.6654951100074187, "auroc": 0.9179987695244284, "f1_bin": 0.7733171346777474}, "Artículo · umbral dinámico (usa etiquetas)": {"auprc": 0.6654951100074187, "auroc": 0.9179987695244284, "f1_bin": 0.14659514589987538}, "M1 · umbral IQR": {"auprc": 0.6654951100074187, "auroc": 0.9179987695244284, "f1_bin": 0.8534999696017719}, "M2 · reponderación suave": {"auprc": 0.6813025238230443, "auroc": 0.9133457200378144, "f1_bin": 0.7923692197940402}, "M2 · recorte z>2": {"auprc": 0.70518512439708, "auroc": 0.9133654134347615, "f1_bin": 0.8015335537462949}, "M3 · proyección al objetivo": {"auprc": 0.6975739011531983, "auroc": 0.9074081087950713, "f1_bin": 0.7801929623361901}, "M1+M3": {"auprc": 0.6975739011531983, "auroc": 0.9074081087950713, "f1_bin": 0.7810133779844917}, "M2+M3 (recorte z>2)": {"auprc": 0.6856989901335495, "auroc": 0.8865453638338521, "f1_bin": 0.7453716389480008}, "M1+M2 (recorte adaptativo, sin M3)": {"auprc": 0.7206880261327008, "auroc": 0.9153331547228912, "f1_bin": 0.8548653267798935}, "M1+M2+M3 con recorte z>2": {"auprc": 0.6856989901335495, "auroc": 0.8865453638338521, "f1_bin": 0.7303942010468571}, "**Método final** (recorte adaptativo + proyección + IQR)": {"auprc": 0.6991810931272096, "auroc": 0.8886033407184586, "f1_bin": 0.7527325528415924}, "Error del LSTM (PD) + IQR": {"auprc": 0.4721979882360733, "auroc": 0.8106993929462722, "f1_bin": 0.5163520648937767}, "LOF + IQR": {"auprc": 0.1989482378695912, "auroc": 0.524894215774937, "f1_bin": 0.1898454746136865}, "IForest + IQR": {"auprc": 0.7002623404253334, "auroc": 0.888578461289973, "f1_bin": 0.3588652875341082}}, "machine-1-7": {"Artículo · umbral estático": {"auprc": 0.3569222695348484, "auroc": 0.8318049854897268, "f1_bin": 0.14641323563521524}, "Artículo · umbral dinámico (usa etiquetas)": {"auprc": 0.3569222695348484, "auroc": 0.8318049854897268, "f1_bin": 0.08188500249734806}, "M1 · umbral IQR": {"auprc": 0.3569222695348484, "auroc": 0.8318049854897268, "f1_bin": 0.15839744198912217}, "M2 · reponderación suave": {"auprc": 0.3118242189926076, "auroc": 0.7705478504400941, "f1_bin": 0.13753190651879993}, "M2 · recorte z>2": {"auprc": 0.3053951109252384, "auroc": 0.7682972644803521, "f1_bin": 0.13294634144889345}, "M3 · proyección al objetivo": {"auprc": 0.3414625274368329, "auroc": 0.8224229667268543, "f1_bin": 0.1485470269994065}, "M1+M3": {"auprc": 0.3414625274368329, "auroc": 0.8224229667268543, "f1_bin": 0.17447794432476618}, "M2+M3 (recorte z>2)": {"auprc": 0.2723130882374041, "auroc": 0.7386825517666584, "f1_bin": 0.12264618802219796}, "M1+M2 (recorte adaptativo, sin M3)": {"auprc": 0.31061477399154, "auroc": 0.7716481041320552, "f1_bin": 0.14823926679498656}, "M1+M2+M3 con recorte z>2": {"auprc": 0.2723130882374041, "auroc": 0.7386825517666584, "f1_bin": 0.14718292478101216}, "**Método final** (recorte adaptativo + proyección + IQR)": {"auprc": 0.27769587860782596, "auroc": 0.7415240343231637, "f1_bin": 0.1441761882802924}, "Error del LSTM (PD) + IQR": {"auprc": 0.2101529005873392, "auroc": 0.6623275675498832, "f1_bin": 0.17592266789563588}, "LOF + IQR": {"auprc": 0.1546229554663906, "auroc": 0.534522256231666, "f1_bin": 0.12187812187812179}, "IForest + IQR": {"auprc": 0.3703746918556596, "auroc": 0.8563722830185652, "f1_bin": 0.35544231181331853}}, "machine-2-1": {"Artículo · umbral estático": {"auprc": 0.2696493260060859, "auroc": 0.7873764872501864, "f1_bin": 0.3205146439112133}, "Artículo · umbral dinámico (usa etiquetas)": {"auprc": 0.2696493260060859, "auroc": 0.7873764872501864, "f1_bin": 0.27328852875462084}, "M1 · umbral IQR": {"auprc": 0.2696493260060859, "auroc": 0.7873764872501864, "f1_bin": 0.32398941647030666}, "M2 · reponderación suave": {"auprc": 0.26600106363213916, "auroc": 0.7753309067522356, "f1_bin": 0.30816436026004596}, "M2 · recorte z>2": {"auprc": 0.2937257050581817, "auroc": 0.7743995504135147, "f1_bin": 0.31264210879276405}, "M3 · proyección al objetivo": {"auprc": 0.20329704636926832, "auroc": 0.7381615455314645, "f1_bin": 0.2917165753835763}, "M1+M3": {"auprc": 0.20329704636926832, "auroc": 0.7381615455314645, "f1_bin": 0.2922670664298665}, "M2+M3 (recorte z>2)": {"auprc": 0.2102970599349608, "auroc": 0.7141951573286626, "f1_bin": 0.273606659967382}, "M1+M2 (recorte adaptativo, sin M3)": {"auprc": 0.30813474439439104, "auroc": 0.7789350258584448, "f1_bin": 0.3192414317140847}, "M1+M2+M3 con recorte z>2": {"auprc": 0.2102970599349608, "auroc": 0.7141951573286626, "f1_bin": 0.27253359534139543}, "**Método final** (recorte adaptativo + proyección + IQR)": {"auprc": 0.23418151903039738, "auroc": 0.7166157213871347, "f1_bin": 0.2786110410978343}, "Error del LSTM (PD) + IQR": {"auprc": 0.09543743088383567, "auroc": 0.6264188619547509, "f1_bin": 0.19615928795079685}, "LOF + IQR": {"auprc": 0.107029195456214, "auroc": 0.5956268135690888, "f1_bin": 0.1861495844875346}, "IForest + IQR": {"auprc": 0.16946209822665495, "auroc": 0.7477686742976256, "f1_bin": 0.27557911974215854}}, "machine-2-2": {"Artículo · umbral estático": {"auprc": 0.22691538186798618, "auroc": 0.6390311594977657, "f1_bin": 0.33789290748218803}, "Artículo · umbral dinámico (usa etiquetas)": {"auprc": 0.22691538186798618, "auroc": 0.6390311594977657, "f1_bin": 0.4321801705282078}, "M1 · umbral IQR": {"auprc": 0.22691538186798618, "auroc": 0.6390311594977657, "f1_bin": 0.2900158101793402}, "M2 · reponderación suave": {"auprc": 0.20771672200211794, "auroc": 0.6184961581080521, "f1_bin": 0.3211734101644482}, "M2 · recorte z>2": {"auprc": 0.20087949860206916, "auroc": 0.6033859394324792, "f1_bin": 0.2961754938290051}, "M3 · proyección al objetivo": {"auprc": 0.17132062232744263, "auroc": 0.5848951703733409, "f1_bin": 0.25051085476534496}, "M1+M3": {"auprc": 0.17132062232744263, "auroc": 0.5848951703733409, "f1_bin": 0.24759462894018253}, "M2+M3 (recorte z>2)": {"auprc": 0.16079000411198718, "auroc": 0.5671945884052036, "f1_bin": 0.22475435328143428}, "M1+M2 (recorte adaptativo, sin M3)": {"auprc": 0.1980825434332904, "auroc": 0.6029697760614617, "f1_bin": 0.27685334600134925}, "M1+M2+M3 con recorte z>2": {"auprc": 0.16079000411198718, "auroc": 0.5671945884052036, "f1_bin": 0.2278505150876761}, "**Método final** (recorte adaptativo + proyección + IQR)": {"auprc": 0.16768192689061673, "auroc": 0.5955741588461484, "f1_bin": 0.25926588763101016}, "Error del LSTM (PD) + IQR": {"auprc": 0.15673026473401203, "auroc": 0.5675661531597866, "f1_bin": 0.20327715390865322}, "LOF + IQR": {"auprc": 0.12049535818942052, "auroc": 0.4981939737994271, "f1_bin": 0.10289892922434049}, "IForest + IQR": {"auprc": 0.1781885947955222, "auroc": 0.6062496393531535, "f1_bin": 0.15166582900299058}}, "machine-1-8": {"Artículo · umbral estático": {"auprc": 0.40307220130526317, "auroc": 0.898550698424731, "f1_bin": 0.22635401424056384}, "Artículo · umbral dinámico (usa etiquetas)": {"auprc": 0.40307220130526317, "auroc": 0.898550698424731, "f1_bin": 0.35415653718386847}, "M1 · umbral IQR": {"auprc": 0.40307220130526317, "auroc": 0.898550698424731, "f1_bin": 0.2653929605423241}, "M2 · reponderación suave": {"auprc": 0.3484261875933738, "auroc": 0.8658420321381953, "f1_bin": 0.19823582029584827}, "M2 · recorte z>2": {"auprc": 0.4044096661038261, "auroc": 0.8725210795794262, "f1_bin": 0.21040782512030526}, "M3 · proyección al objetivo": {"auprc": 0.34609736622722803, "auroc": 0.8743918639926545, "f1_bin": 0.20404106284532703}, "M1+M3": {"auprc": 0.34609736622722803, "auroc": 0.8743918639926545, "f1_bin": 0.2265258982529991}, "M2+M3 (recorte z>2)": {"auprc": 0.3294620557180947, "auroc": 0.8299751636610109, "f1_bin": 0.1823364606981638}, "M1+M2 (recorte adaptativo, sin M3)": {"auprc": 0.43473090699489203, "auroc": 0.8797049585792003, "f1_bin": 0.2807377171517552}, "M1+M2+M3 con recorte z>2": {"auprc": 0.3294620557180947, "auroc": 0.8299751636610109, "f1_bin": 0.19980204454303746}, "**Método final** (recorte adaptativo + proyección + IQR)": {"auprc": 0.3659141984370283, "auroc": 0.8345664063559781, "f1_bin": 0.2160883911928207}, "Error del LSTM (PD) + IQR": {"auprc": 0.18783746738406476, "auroc": 0.7901795265733145, "f1_bin": 0.11209263990806048}, "LOF + IQR": {"auprc": 0.11970786029313973, "auroc": 0.5740841857564923, "f1_bin": 0.1540312876052948}, "IForest + IQR": {"auprc": 0.21896627282707543, "auroc": 0.8314713653472857, "f1_bin": 0.2600604390697919}}, "machine-2-3": {"Artículo · umbral estático": {"auprc": 0.6899222225822899, "auroc": 0.9841667358267202, "f1_bin": 0.15483207080876396}, "Artículo · umbral dinámico (usa etiquetas)": {"auprc": 0.6899222225822899, "auroc": 0.9841667358267202, "f1_bin": 0.5669749586892292}, "M1 · umbral IQR": {"auprc": 0.6899222225822899, "auroc": 0.9841667358267202, "f1_bin": 0.16869239890418913}, "M2 · reponderación suave": {"auprc": 0.6411137653196123, "auroc": 0.9761123718909378, "f1_bin": 0.14678794789814348}, "M2 · recorte z>2": {"auprc": 0.6578278235576644, "auroc": 0.9749456497372394, "f1_bin": 0.14804884498453635}, "M3 · proyección al objetivo": {"auprc": 0.6376495790947059, "auroc": 0.960114681677737, "f1_bin": 0.1506860218094077}, "M1+M3": {"auprc": 0.6376495790947059, "auroc": 0.960114681677737, "f1_bin": 0.15606664214033242}, "M2+M3 (recorte z>2)": {"auprc": 0.5642621524278899, "auroc": 0.9520350385499728, "f1_bin": 0.14665108988671965}, "M1+M2 (recorte adaptativo, sin M3)": {"auprc": 0.5945927960385985, "auroc": 0.9738033164392016, "f1_bin": 0.17913917876655824}, "M1+M2+M3 con recorte z>2": {"auprc": 0.5642621524278899, "auroc": 0.9520350385499728, "f1_bin": 0.16028045388606615}, "**Método final** (recorte adaptativo + proyección + IQR)": {"auprc": 0.5133658926876267, "auroc": 0.950343955999579, "f1_bin": 0.1561847436653046}, "Error del LSTM (PD) + IQR": {"auprc": 0.20939580652442674, "auroc": 0.8955954426227528, "f1_bin": 0.1398096553963615}, "LOF + IQR": {"auprc": 0.1071169601109879, "auroc": 0.7477858448608039, "f1_bin": 0.1198257080610021}, "IForest + IQR": {"auprc": 0.5491566420567894, "auroc": 0.9401046908323023, "f1_bin": 0.3049390939314912}}, "machine-2-4": {"Artículo · umbral estático": {"auprc": 0.6514731600739913, "auroc": 0.9442182393740606, "f1_bin": 0.6463056206835937}, "Artículo · umbral dinámico (usa etiquetas)": {"auprc": 0.6514731600739913, "auroc": 0.9442182393740606, "f1_bin": 0.6416032447335152}, "M1 · umbral IQR": {"auprc": 0.6514731600739913, "auroc": 0.9442182393740606, "f1_bin": 0.6505271859495633}, "M2 · reponderación suave": {"auprc": 0.6432452239533736, "auroc": 0.933601900548453, "f1_bin": 0.628440664727033}, "M2 · recorte z>2": {"auprc": 0.6152730800605783, "auroc": 0.9199331106484344, "f1_bin": 0.5989654843837242}, "M3 · proyección al objetivo": {"auprc": 0.610616980401551, "auroc": 0.9182880255542939, "f1_bin": 0.5839097276326555}, "M1+M3": {"auprc": 0.610616980401551, "auroc": 0.9182880255542939, "f1_bin": 0.5883092088650443}, "M2+M3 (recorte z>2)": {"auprc": 0.48253042286924075, "auroc": 0.8564168970595581, "f1_bin": 0.4816374309826193}, "M1+M2 (recorte adaptativo, sin M3)": {"auprc": 0.6109528157903089, "auroc": 0.9183440812376709, "f1_bin": 0.6016822925497983}, "M1+M2+M3 con recorte z>2": {"auprc": 0.48253042286924075, "auroc": 0.8564168970595581, "f1_bin": 0.48415086190689394}, "**Método final** (recorte adaptativo + proyección + IQR)": {"auprc": 0.4756916267349801, "auroc": 0.8495214242671049, "f1_bin": 0.48267487979000767}, "Error del LSTM (PD) + IQR": {"auprc": 0.33064488353871624, "auroc": 0.8027890135592379, "f1_bin": 0.40550794907116183}, "LOF + IQR": {"auprc": 0.1510224011124211, "auroc": 0.6269675477652226, "f1_bin": 0.21198156682027652}, "IForest + IQR": {"auprc": 0.5227381539761862, "auroc": 0.9074956826369714, "f1_bin": 0.497031906714541}}, "machine-2-5": {"Artículo · umbral estático": {"auprc": 0.5761663898079618, "auroc": 0.9657463684163622, "f1_bin": 0.47975452282460845}, "Artículo · umbral dinámico (usa etiquetas)": {"auprc": 0.5761663898079618, "auroc": 0.9657463684163622, "f1_bin": 0.5898366563631274}, "M1 · umbral IQR": {"auprc": 0.5761663898079618, "auroc": 0.9657463684163622, "f1_bin": 0.451176602805042}, "M2 · reponderación suave": {"auprc": 0.5432125142226286, "auroc": 0.9515023328373298, "f1_bin": 0.4616501526832392}, "M2 · recorte z>2": {"auprc": 0.526544826647912, "auroc": 0.9490815831427184, "f1_bin": 0.45586170613413135}, "M3 · proyección al objetivo": {"auprc": 0.5721966874516535, "auroc": 0.9507278695519494, "f1_bin": 0.46358430202246154}, "M1+M3": {"auprc": 0.5721966874516535, "auroc": 0.9507278695519494, "f1_bin": 0.45899237521827574}, "M2+M3 (recorte z>2)": {"auprc": 0.4666635981533142, "auroc": 0.912804540368483, "f1_bin": 0.4201155246283104}, "M1+M2 (recorte adaptativo, sin M3)": {"auprc": 0.5387009215107302, "auroc": 0.9544056553414008, "f1_bin": 0.4639541481662823}, "M1+M2+M3 con recorte z>2": {"auprc": 0.4666635981533142, "auroc": 0.912804540368483, "f1_bin": 0.4265423059596789}, "**Método final** (recorte adaptativo + proyección + IQR)": {"auprc": 0.48358058737930626, "auroc": 0.9179842809162834, "f1_bin": 0.42782452194754733}, "Error del LSTM (PD) + IQR": {"auprc": 0.3983882376715101, "auroc": 0.9103392403673126, "f1_bin": 0.45314159949242117}, "LOF + IQR": {"auprc": 0.1378406926314, "auroc": 0.6575137796285644, "f1_bin": 0.23055555555555554}, "IForest + IQR": {"auprc": 0.26433802422781555, "auroc": 0.837641885389858, "f1_bin": 0.29144886841190726}}, "machine-2-7": {"Artículo · umbral estático": {"auprc": 0.4451496422017671, "auroc": 0.9713444888854632, "f1_bin": 0.33447286822689654}, "Artículo · umbral dinámico (usa etiquetas)": {"auprc": 0.4451496422017671, "auroc": 0.9713444888854632, "f1_bin": 0.3319891368009723}, "M1 · umbral IQR": {"auprc": 0.4451496422017671, "auroc": 0.9713444888854632, "f1_bin": 0.3162184597760393}, "M2 · reponderación suave": {"auprc": 0.5473737651791857, "auroc": 0.9744516456023256, "f1_bin": 0.3274250341868789}, "M2 · recorte z>2": {"auprc": 0.7204704554903089, "auroc": 0.9744888741130499, "f1_bin": 0.32825340998312813}, "M3 · proyección al objetivo": {"auprc": 0.5079382556464073, "auroc": 0.9733678914619691, "f1_bin": 0.3393806797142389}, "M1+M3": {"auprc": 0.5079382556464073, "auroc": 0.9733678914619691, "f1_bin": 0.312152927322527}, "M2+M3 (recorte z>2)": {"auprc": 0.8066748005162735, "auroc": 0.9741925731391137, "f1_bin": 0.34466865986529815}, "M1+M2 (recorte adaptativo, sin M3)": {"auprc": 0.7329579514132836, "auroc": 0.9743119974138155, "f1_bin": 0.3173074932567232}, "M1+M2+M3 con recorte z>2": {"auprc": 0.8066748005162735, "auroc": 0.9741925731391137, "f1_bin": 0.35454334769414386}, "**Método final** (recorte adaptativo + proyección + IQR)": {"auprc": 0.8123922268250844, "auroc": 0.9740198444058745, "f1_bin": 0.346082087481566}, "Error del LSTM (PD) + IQR": {"auprc": 0.14547240394900013, "auroc": 0.740154962644058, "f1_bin": 0.1592576122870784}, "LOF + IQR": {"auprc": 0.1034105769374769, "auroc": 0.5979671871125791, "f1_bin": 0.1581092094539527}, "IForest + IQR": {"auprc": 0.7465110170667313, "auroc": 0.9222150716241257, "f1_bin": 0.3641938467364196}}, "machine-2-6": {"Artículo · umbral estático": {"auprc": 0.42816792203544934, "auroc": 0.8700401229440662, "f1_bin": 0.13859489562748403}, "Artículo · umbral dinámico (usa etiquetas)": {"auprc": 0.42816792203544934, "auroc": 0.8700401229440662, "f1_bin": 0.5271129941086713}, "M1 · umbral IQR": {"auprc": 0.42816792203544934, "auroc": 0.8700401229440662, "f1_bin": 0.14074639782126802}, "M2 · reponderación suave": {"auprc": 0.42857455176627984, "auroc": 0.8554213509570061, "f1_bin": 0.12996458742165376}, "M2 · recorte z>2": {"auprc": 0.4188727724071355, "auroc": 0.8508669564805492, "f1_bin": 0.12608063047049517}, "M3 · proyección al objetivo": {"auprc": 0.34643361937411454, "auroc": 0.8588003696728185, "f1_bin": 0.13765324738030255}, "M1+M3": {"auprc": 0.34643361937411454, "auroc": 0.8588003696728185, "f1_bin": 0.1324654704634959}, "M2+M3 (recorte z>2)": {"auprc": 0.3261917593915383, "auroc": 0.8305209578001899, "f1_bin": 0.12672835486801076}, "M1+M2 (recorte adaptativo, sin M3)": {"auprc": 0.42269069922721075, "auroc": 0.8527156875107164, "f1_bin": 0.13034478519284343}, "M1+M2+M3 con recorte z>2": {"auprc": 0.3261917593915383, "auroc": 0.8305209578001899, "f1_bin": 0.12063003531510816}, "**Método final** (recorte adaptativo + proyección + IQR)": {"auprc": 0.3321746803887268, "auroc": 0.8264448162744105, "f1_bin": 0.11811630474663677}, "Error del LSTM (PD) + IQR": {"auprc": 0.09896908416755475, "auroc": 0.7978438254098192, "f1_bin": 0.12124740331145634}, "LOF + IQR": {"auprc": 0.0839267168760111, "auroc": 0.6525131380291989, "f1_bin": 0.0975206611570247}, "IForest + IQR": {"auprc": 0.320211589348205, "auroc": 0.8625738294861609, "f1_bin": 0.3373755313872055}}, "machine-2-8": {"Artículo · umbral estático": {"auprc": 0.9866501160721448, "auroc": 0.9990488205522835, "f1_bin": 0.2763040205962893}, "Artículo · umbral dinámico (usa etiquetas)": {"auprc": 0.9866501160721448, "auroc": 0.9990488205522835, "f1_bin": 0.061154254966780244}, "M1 · umbral IQR": {"auprc": 0.9866501160721448, "auroc": 0.9990488205522835, "f1_bin": 0.2483782420101627}, "M2 · reponderación suave": {"auprc": 0.9727830200226213, "auroc": 0.9986758432025843, "f1_bin": 0.3041340384511315}, "M2 · recorte z>2": {"auprc": 0.9840675707523199, "auroc": 0.9989086567530199, "f1_bin": 0.31651978422377225}, "M3 · proyección al objetivo": {"auprc": 0.9613650365179295, "auroc": 0.9955769125065007, "f1_bin": 0.26571247778964513}, "M1+M3": {"auprc": 0.9613650365179295, "auroc": 0.9955769125065007, "f1_bin": 0.2613178590047661}, "M2+M3 (recorte z>2)": {"auprc": 0.9627369466981553, "auroc": 0.9949334909030945, "f1_bin": 0.28143927590542916}, "M1+M2 (recorte adaptativo, sin M3)": {"auprc": 0.9882129635727885, "auroc": 0.998992480201599, "f1_bin": 0.30827168758746876}, "M1+M2+M3 con recorte z>2": {"auprc": 0.9627369466981553, "auroc": 0.9949334909030945, "f1_bin": 0.2857065480313764}, "**Método final** (recorte adaptativo + proyección + IQR)": {"auprc": 0.9649080886852242, "auroc": 0.9948530499896411, "f1_bin": 0.2809317023959418}, "Error del LSTM (PD) + IQR": {"auprc": 0.8721754750064484, "auroc": 0.9905041140083464, "f1_bin": 0.3732452795937183}, "LOF + IQR": {"auprc": 0.0482068849597312, "auroc": 0.7315202592713175, "f1_bin": 0.0890538033395176}, "IForest + IQR": {"auprc": 0.9402359765874173, "auroc": 0.9951011378002528, "f1_bin": 0.5586571183992303}}, "machine-2-9": {"Artículo · umbral estático": {"auprc": 0.7375262938976003, "auroc": 0.9647504935157653, "f1_bin": 0.6809833311734877}, "Artículo · umbral dinámico (usa etiquetas)": {"auprc": 0.7375262938976003, "auroc": 0.9647504935157653, "f1_bin": 0.326309119012085}, "M1 · umbral IQR": {"auprc": 0.7375262938976003, "auroc": 0.9647504935157653, "f1_bin": 0.6682706478524739}, "M2 · reponderación suave": {"auprc": 0.6656196250006745, "auroc": 0.9530213067410884, "f1_bin": 0.652943928484157}, "M2 · recorte z>2": {"auprc": 0.5814428164129704, "auroc": 0.9396522098471667, "f1_bin": 0.6262235490929442}, "M3 · proyección al objetivo": {"auprc": 0.6538260989158008, "auroc": 0.9481072767165826, "f1_bin": 0.6530764568459644}, "M1+M3": {"auprc": 0.6538260989158008, "auroc": 0.9481072767165826, "f1_bin": 0.6235986013235794}, "M2+M3 (recorte z>2)": {"auprc": 0.4672619850868637, "auroc": 0.9089100948055979, "f1_bin": 0.5679126618721114}, "M1+M2 (recorte adaptativo, sin M3)": {"auprc": 0.5886660939456994, "auroc": 0.9388484501663601, "f1_bin": 0.6428680233053512}, "M1+M2+M3 con recorte z>2": {"auprc": 0.4672619850868637, "auroc": 0.9089100948055979, "f1_bin": 0.557863133314304}, "**Método final** (recorte adaptativo + proyección + IQR)": {"auprc": 0.47203363520808894, "auroc": 0.9063410388338904, "f1_bin": 0.5569810250942115}, "Error del LSTM (PD) + IQR": {"auprc": 0.32699106382002596, "auroc": 0.8921153409741921, "f1_bin": 0.5528072057164751}, "LOF + IQR": {"auprc": 0.1452066716873213, "auroc": 0.6623811544528987, "f1_bin": 0.2509748316199929}, "IForest + IQR": {"auprc": 0.5273293721569792, "auroc": 0.8742601431917782, "f1_bin": 0.5638734476839098}}, "machine-3-10": {"Artículo · umbral estático": {"auprc": 0.25491690767738395, "auroc": 0.9030397992849857, "f1_bin": 0.5086837381491403}, "Artículo · umbral dinámico (usa etiquetas)": {"auprc": 0.25491690767738395, "auroc": 0.9030397992849857, "f1_bin": 0.06439699241994275}, "M1 · umbral IQR": {"auprc": 0.25491690767738395, "auroc": 0.9030397992849857, "f1_bin": 0.49693398459085925}, "M2 · reponderación suave": {"auprc": 0.27099084041043475, "auroc": 0.9084415774250697, "f1_bin": 0.5459486696342711}, "M2 · recorte z>2": {"auprc": 0.27685744398699763, "auroc": 0.9070673227385763, "f1_bin": 0.5359234858802083}, "M3 · proyección al objetivo": {"auprc": 0.2292179646439823, "auroc": 0.8875064214744924, "f1_bin": 0.48733974884401376}, "M1+M3": {"auprc": 0.2292179646439823, "auroc": 0.8875064214744924, "f1_bin": 0.4665528657406771}, "M2+M3 (recorte z>2)": {"auprc": 0.24330419313100743, "auroc": 0.8880136461402544, "f1_bin": 0.5099038322653959}, "M1+M2 (recorte adaptativo, sin M3)": {"auprc": 0.72173259057888, "auroc": 0.9512486895122452, "f1_bin": 0.622601934968826}, "M1+M2+M3 con recorte z>2": {"auprc": 0.24330419313100743, "auroc": 0.8880136461402544, "f1_bin": 0.48850379560782126}, "**Método final** (recorte adaptativo + proyección + IQR)": {"auprc": 0.5179680307742796, "auroc": 0.9004622912429339, "f1_bin": 0.5433147229236274}, "Error del LSTM (PD) + IQR": {"auprc": 0.07556276189803268, "auroc": 0.632186643873812, "f1_bin": 0.05937329571736464}, "LOF + IQR": {"auprc": 0.0939770513610506, "auroc": 0.5682047407718342, "f1_bin": 0.1452178267401101}, "IForest + IQR": {"auprc": 0.7942183045718734, "auroc": 0.9081607857789351, "f1_bin": 0.8606233807764075}}, "machine-3-11": {"Artículo · umbral estático": {"auprc": 0.49558358658521345, "auroc": 0.8175667426194483, "f1_bin": 0.08990820403019142}, "Artículo · umbral dinámico (usa etiquetas)": {"auprc": 0.49558358658521345, "auroc": 0.8175667426194483, "f1_bin": 0.03484870505638159}, "M1 · umbral IQR": {"auprc": 0.49558358658521345, "auroc": 0.8175667426194483, "f1_bin": 0.11094120283613773}, "M2 · reponderación suave": {"auprc": 0.4144544526650044, "auroc": 0.7795642652810608, "f1_bin": 0.08939991036710053}, "M2 · recorte z>2": {"auprc": 0.4097443044028076, "auroc": 0.7584069081539215, "f1_bin": 0.09357045747438646}, "M3 · proyección al objetivo": {"auprc": 0.4017397904767737, "auroc": 0.80060400207273, "f1_bin": 0.09344017130662263}, "M1+M3": {"auprc": 0.4017397904767737, "auroc": 0.80060400207273, "f1_bin": 0.10093815511360563}, "M2+M3 (recorte z>2)": {"auprc": 0.22257223323752923, "auroc": 0.7416414851253221, "f1_bin": 0.08413249408232286}, "M1+M2 (recorte adaptativo, sin M3)": {"auprc": 0.40333859856576637, "auroc": 0.7592160537206215, "f1_bin": 0.12502088370330772}, "M1+M2+M3 con recorte z>2": {"auprc": 0.22257223323752923, "auroc": 0.7416414851253221, "f1_bin": 0.09141126674054707}, "**Método final** (recorte adaptativo + proyección + IQR)": {"auprc": 0.21216434837443363, "auroc": 0.7473489817358405, "f1_bin": 0.09343619716507226}, "Error del LSTM (PD) + IQR": {"auprc": 0.13262137643491426, "auroc": 0.7995756049603555, "f1_bin": 0.11114870237323864}, "LOF + IQR": {"auprc": 0.1574626626025673, "auroc": 0.6582319683127834, "f1_bin": 0.1241379310344827}, "IForest + IQR": {"auprc": 0.11891355644867527, "auroc": 0.8359794004699135, "f1_bin": 0.16797101010423027}}, "machine-3-1": {"Artículo · umbral estático": {"auprc": 0.34480241255817734, "auroc": 0.9124851025394157, "f1_bin": 0.09829923392840219}, "Artículo · umbral dinámico (usa etiquetas)": {"auprc": 0.34480241255817734, "auroc": 0.9124851025394157, "f1_bin": 0.3514486245591879}, "M1 · umbral IQR": {"auprc": 0.34480241255817734, "auroc": 0.9124851025394157, "f1_bin": 0.08988759602369793}, "M2 · reponderación suave": {"auprc": 0.36527549241036894, "auroc": 0.909707441814023, "f1_bin": 0.09280268754054825}, "M2 · recorte z>2": {"auprc": 0.4811754490464789, "auroc": 0.9132632973290378, "f1_bin": 0.09465659966102861}, "M3 · proyección al objetivo": {"auprc": 0.2168867888382929, "auroc": 0.9013762001559137, "f1_bin": 0.0994158302603744}, "M1+M3": {"auprc": 0.2168867888382929, "auroc": 0.9013762001559137, "f1_bin": 0.0864126406357699}, "M2+M3 (recorte z>2)": {"auprc": 0.3653443469917742, "auroc": 0.8722220988082949, "f1_bin": 0.09255605222986339}, "M1+M2 (recorte adaptativo, sin M3)": {"auprc": 0.5357401094876051, "auroc": 0.9254049910729745, "f1_bin": 0.0929697706987413}, "M1+M2+M3 con recorte z>2": {"auprc": 0.3653443469917742, "auroc": 0.8722220988082949, "f1_bin": 0.08350757698120771}, "**Método final** (recorte adaptativo + proyección + IQR)": {"auprc": 0.4042756026520097, "auroc": 0.8822911790544223, "f1_bin": 0.08077307686401179}, "Error del LSTM (PD) + IQR": {"auprc": 0.1596902348467043, "auroc": 0.8783265537100975, "f1_bin": 0.09279295779541204}, "LOF + IQR": {"auprc": 0.0455783553504876, "auroc": 0.566921717451625, "f1_bin": 0.08125}, "IForest + IQR": {"auprc": 0.33630599875348877, "auroc": 0.848308937835107, "f1_bin": 0.28667113639143577}}, "machine-3-2": {"Artículo · umbral estático": {"auprc": 0.08083495543181285, "auroc": 0.6246512397317983, "f1_bin": 0.047270157764096724}, "Artículo · umbral dinámico (usa etiquetas)": {"auprc": 0.08083495543181285, "auroc": 0.6246512397317983, "f1_bin": 0.02525046433187434}, "M1 · umbral IQR": {"auprc": 0.08083495543181285, "auroc": 0.6246512397317983, "f1_bin": 0.04088189968548378}, "M2 · reponderación suave": {"auprc": 0.08755682082232616, "auroc": 0.6402608092456711, "f1_bin": 0.05095818069010738}, "M2 · recorte z>2": {"auprc": 0.09157689686394847, "auroc": 0.6294804586952486, "f1_bin": 0.0487292746398416}, "M3 · proyección al objetivo": {"auprc": 0.0757560854578267, "auroc": 0.6081302786164434, "f1_bin": 0.0445756296888018}, "M1+M3": {"auprc": 0.0757560854578267, "auroc": 0.6081302786164434, "f1_bin": 0.03917661143427438}, "M2+M3 (recorte z>2)": {"auprc": 0.08299536043472353, "auroc": 0.6080347424248651, "f1_bin": 0.03801970561946766}, "M1+M2 (recorte adaptativo, sin M3)": {"auprc": 0.09152621265644303, "auroc": 0.627260299623372, "f1_bin": 0.040299184911029164}, "M1+M2+M3 con recorte z>2": {"auprc": 0.08299536043472353, "auroc": 0.6080347424248651, "f1_bin": 0.03636277814887884}, "**Método final** (recorte adaptativo + proyección + IQR)": {"auprc": 0.0820330040529612, "auroc": 0.6034606824077646, "f1_bin": 0.0350254352216732}, "Error del LSTM (PD) + IQR": {"auprc": 0.07452041122487975, "auroc": 0.536613703738824, "f1_bin": 0.03701350408199586}, "LOF + IQR": {"auprc": 0.0552633229563251, "auroc": 0.539339243587466, "f1_bin": 0.0238500851788756}, "IForest + IQR": {"auprc": 0.06374094261641164, "auroc": 0.5909579619969199, "f1_bin": 0.050904514197685824}}, "machine-3-3": {"Artículo · umbral estático": {"auprc": 0.18278311499107286, "auroc": 0.8309594187096444, "f1_bin": 0.2500264270346623}, "Artículo · umbral dinámico (usa etiquetas)": {"auprc": 0.18278311499107286, "auroc": 0.8309594187096444, "f1_bin": 0.17893384681665744}, "M1 · umbral IQR": {"auprc": 0.18278311499107286, "auroc": 0.8309594187096444, "f1_bin": 0.2582242791134594}, "M2 · reponderación suave": {"auprc": 0.21412058341010526, "auroc": 0.8039683431642107, "f1_bin": 0.24574224314573687}, "M2 · recorte z>2": {"auprc": 0.26758995667123664, "auroc": 0.7991213487307974, "f1_bin": 0.24566885014223866}, "M3 · proyección al objetivo": {"auprc": 0.16310895572699538, "auroc": 0.795747337199736, "f1_bin": 0.22917505169744365}, "M1+M3": {"auprc": 0.16310895572699538, "auroc": 0.795747337199736, "f1_bin": 0.22931379149557518}, "M2+M3 (recorte z>2)": {"auprc": 0.21713756408238058, "auroc": 0.7474472234938643, "f1_bin": 0.20115024115212154}, "M1+M2 (recorte adaptativo, sin M3)": {"auprc": 0.26535946219972256, "auroc": 0.8110296893779811, "f1_bin": 0.2756182899077023}, "M1+M2+M3 con recorte z>2": {"auprc": 0.21713756408238058, "auroc": 0.7474472234938643, "f1_bin": 0.19881370223245645}, "**Método final** (recorte adaptativo + proyección + IQR)": {"auprc": 0.22560440853491287, "auroc": 0.7494403874023348, "f1_bin": 0.1988775620139001}, "Error del LSTM (PD) + IQR": {"auprc": 0.1174772287713497, "auroc": 0.7663624136970236, "f1_bin": 0.17978226832255723}, "LOF + IQR": {"auprc": 0.10884877437977233, "auroc": 0.6575034952602403, "f1_bin": 0.1946647440519106}, "IForest + IQR": {"auprc": 0.09888903204078361, "auroc": 0.7518171506264814, "f1_bin": 0.2545510726153667}}, "machine-3-4": {"Artículo · umbral estático": {"auprc": 0.1741888765484334, "auroc": 0.7974799806271123, "f1_bin": 0.0927084011010669}, "Artículo · umbral dinámico (usa etiquetas)": {"auprc": 0.1741888765484334, "auroc": 0.7974799806271123, "f1_bin": 0.06371663347598387}, "M1 · umbral IQR": {"auprc": 0.1741888765484334, "auroc": 0.7974799806271123, "f1_bin": 0.09545014024632761}, "M2 · reponderación suave": {"auprc": 0.16244849641020348, "auroc": 0.7705821310754264, "f1_bin": 0.08418470303063938}, "M2 · recorte z>2": {"auprc": 0.18873576598289254, "auroc": 0.7604692507777688, "f1_bin": 0.07991630363069378}, "M3 · proyección al objetivo": {"auprc": 0.1541945481591353, "auroc": 0.7708534697838572, "f1_bin": 0.07696810461902051}, "M1+M3": {"auprc": 0.1541945481591353, "auroc": 0.7708534697838572, "f1_bin": 0.07944330199756351}, "M2+M3 (recorte z>2)": {"auprc": 0.17741677657753505, "auroc": 0.7282910568697546, "f1_bin": 0.06614754201139991}, "M1+M2 (recorte adaptativo, sin M3)": {"auprc": 0.19560513522297449, "auroc": 0.7640057011692354, "f1_bin": 0.07569103178511753}, "M1+M2+M3 con recorte z>2": {"auprc": 0.17741677657753505, "auroc": 0.7282910568697546, "f1_bin": 0.06922036776770985}, "**Método final** (recorte adaptativo + proyección + IQR)": {"auprc": 0.17999836997280494, "auroc": 0.7273183358386142, "f1_bin": 0.07308686175902221}, "Error del LSTM (PD) + IQR": {"auprc": 0.05933666721103002, "auroc": 0.5528508825787755, "f1_bin": 0.03845804301784008}, "LOF + IQR": {"auprc": 0.0957052926836217, "auroc": 0.6128465964537937, "f1_bin": 0.1082621082621082}, "IForest + IQR": {"auprc": 0.19383268262208161, "auroc": 0.7947674603386716, "f1_bin": 0.33640901770992293}}, "machine-3-5": {"Artículo · umbral estático": {"auprc": 0.4432200698815675, "auroc": 0.8981743159116868, "f1_bin": 0.16032473852495607}, "Artículo · umbral dinámico (usa etiquetas)": {"auprc": 0.4432200698815675, "auroc": 0.8981743159116868, "f1_bin": 0.2618346825321819}, "M1 · umbral IQR": {"auprc": 0.4432200698815675, "auroc": 0.8981743159116868, "f1_bin": 0.18498865925189728}, "M2 · reponderación suave": {"auprc": 0.43120128069353375, "auroc": 0.8812936862575548, "f1_bin": 0.15401941160738425}, "M2 · recorte z>2": {"auprc": 0.4853737499342602, "auroc": 0.875426142192054, "f1_bin": 0.15827453775866585}, "M3 · proyección al objetivo": {"auprc": 0.35712739973960084, "auroc": 0.8881950360910313, "f1_bin": 0.1611582299327195}, "M1+M3": {"auprc": 0.35712739973960084, "auroc": 0.8881950360910313, "f1_bin": 0.1591359737604458}, "M2+M3 (recorte z>2)": {"auprc": 0.412172626504543, "auroc": 0.857732693718315, "f1_bin": 0.15019128444820312}, "M1+M2 (recorte adaptativo, sin M3)": {"auprc": 0.6743728386397503, "auroc": 0.8859211929778178, "f1_bin": 0.1498827700186582}, "M1+M2+M3 con recorte z>2": {"auprc": 0.412172626504543, "auroc": 0.857732693718315, "f1_bin": 0.15576349574812495}, "**Método final** (recorte adaptativo + proyección + IQR)": {"auprc": 0.614380536978046, "auroc": 0.8681705927070871, "f1_bin": 0.156001497517156}, "Error del LSTM (PD) + IQR": {"auprc": 0.1400035559794684, "auroc": 0.8071537475307494, "f1_bin": 0.19514063894431344}, "LOF + IQR": {"auprc": 0.1433824062547337, "auroc": 0.6505027640044136, "f1_bin": 0.2478551000953288}, "IForest + IQR": {"auprc": 0.46750746421375045, "auroc": 0.8209358301038273, "f1_bin": 0.23942759017229748}}, "machine-3-6": {"Artículo · umbral estático": {"auprc": 0.6341888284478879, "auroc": 0.9765561275498026, "f1_bin": 0.524034892811511}, "Artículo · umbral dinámico (usa etiquetas)": {"auprc": 0.6341888284478879, "auroc": 0.9765561275498026, "f1_bin": 0.6255929584893282}, "M1 · umbral IQR": {"auprc": 0.6341888284478879, "auroc": 0.9765561275498026, "f1_bin": 0.5357467516401024}, "M2 · reponderación suave": {"auprc": 0.5521633837854607, "auroc": 0.9598674734454017, "f1_bin": 0.5086912937080742}, "M2 · recorte z>2": {"auprc": 0.5866756245046366, "auroc": 0.9634690632669178, "f1_bin": 0.519916460679859}, "M3 · proyección al objetivo": {"auprc": 0.5747674016979568, "auroc": 0.9580100497638651, "f1_bin": 0.4971206291670427}, "M1+M3": {"auprc": 0.5747674016979568, "auroc": 0.9580100497638651, "f1_bin": 0.499489769311777}, "M2+M3 (recorte z>2)": {"auprc": 0.46275319719711705, "auroc": 0.9136652351985675, "f1_bin": 0.4454771268601303}, "M1+M2 (recorte adaptativo, sin M3)": {"auprc": 0.5712722785144144, "auroc": 0.9634079687738175, "f1_bin": 0.535289244980067}, "M1+M2+M3 con recorte z>2": {"auprc": 0.46275319719711705, "auroc": 0.9136652351985675, "f1_bin": 0.45705346807002234}, "**Método final** (recorte adaptativo + proyección + IQR)": {"auprc": 0.45559133341737057, "auroc": 0.9153681802236371, "f1_bin": 0.45363231539269816}, "Error del LSTM (PD) + IQR": {"auprc": 0.36072258640123056, "auroc": 0.9025294661518352, "f1_bin": 0.4532249050757744}, "LOF + IQR": {"auprc": 0.10342853131320391, "auroc": 0.626129163676826, "f1_bin": 0.1656174334140435}, "IForest + IQR": {"auprc": 0.46001313811557465, "auroc": 0.9402536006333161, "f1_bin": 0.34250712416596074}}, "machine-3-7": {"Artículo · umbral estático": {"auprc": 0.21362490217352142, "auroc": 0.7239926118421085, "f1_bin": 0.12147749381398203}, "Artículo · umbral dinámico (usa etiquetas)": {"auprc": 0.21362490217352142, "auroc": 0.7239926118421085, "f1_bin": 0.32200293435957267}, "M1 · umbral IQR": {"auprc": 0.21362490217352142, "auroc": 0.7239926118421085, "f1_bin": 0.14948916117927716}, "M2 · reponderación suave": {"auprc": 0.18795915198460383, "auroc": 0.7187357148651585, "f1_bin": 0.11826107780595763}, "M2 · recorte z>2": {"auprc": 0.15892801246757415, "auroc": 0.7017589802827393, "f1_bin": 0.11453869640044792}, "M3 · proyección al objetivo": {"auprc": 0.1513985379697316, "auroc": 0.6885728551466312, "f1_bin": 0.10909465552034017}, "M1+M3": {"auprc": 0.1513985379697316, "auroc": 0.6885728551466312, "f1_bin": 0.10855671325708213}, "M2+M3 (recorte z>2)": {"auprc": 0.12516565228323923, "auroc": 0.6712233910671909, "f1_bin": 0.09909765726006622}, "M1+M2 (recorte adaptativo, sin M3)": {"auprc": 0.1326503439829184, "auroc": 0.6572363190377113, "f1_bin": 0.10541678535066823}, "M1+M2+M3 con recorte z>2": {"auprc": 0.12516565228323923, "auroc": 0.6712233910671909, "f1_bin": 0.09694503858992913}, "**Método final** (recorte adaptativo + proyección + IQR)": {"auprc": 0.08949675850492986, "auroc": 0.6695341399522505, "f1_bin": 0.09557107140249096}, "Error del LSTM (PD) + IQR": {"auprc": 0.09435140404873552, "auroc": 0.6724737173105761, "f1_bin": 0.09795984948559362}, "LOF + IQR": {"auprc": 0.074077892459262, "auroc": 0.6898609187111661, "f1_bin": 0.1639097744360902}, "IForest + IQR": {"auprc": 0.22581675258453826, "auroc": 0.7180605644830762, "f1_bin": 0.3176618045911632}}, "machine-3-8": {"Artículo · umbral estático": {"auprc": 0.35025285841748854, "auroc": 0.8349775429569863, "f1_bin": 0.32833057555915407}, "Artículo · umbral dinámico (usa etiquetas)": {"auprc": 0.35025285841748854, "auroc": 0.8349775429569863, "f1_bin": 0.2178805336161349}, "M1 · umbral IQR": {"auprc": 0.35025285841748854, "auroc": 0.8349775429569863, "f1_bin": 0.3387105386886392}, "M2 · reponderación suave": {"auprc": 0.34633104579327156, "auroc": 0.8299558875933306, "f1_bin": 0.3307668374009726}, "M2 · recorte z>2": {"auprc": 0.33710657212969053, "auroc": 0.8209481435043463, "f1_bin": 0.3301530365267923}, "M3 · proyección al objetivo": {"auprc": 0.27555916158184834, "auroc": 0.7955920916469152, "f1_bin": 0.3144039066849744}, "M1+M3": {"auprc": 0.27555916158184834, "auroc": 0.7955920916469152, "f1_bin": 0.31070436256076217}, "M2+M3 (recorte z>2)": {"auprc": 0.2552987274942645, "auroc": 0.7735357146149754, "f1_bin": 0.29755594385917095}, "M1+M2 (recorte adaptativo, sin M3)": {"auprc": 0.32988941314301556, "auroc": 0.8130083057989876, "f1_bin": 0.3486298803724149}, "M1+M2+M3 con recorte z>2": {"auprc": 0.2552987274942645, "auroc": 0.7735357146149754, "f1_bin": 0.2917549839716981}, "**Método final** (recorte adaptativo + proyección + IQR)": {"auprc": 0.2569424409554575, "auroc": 0.7701524673886858, "f1_bin": 0.28734564217993724}, "Error del LSTM (PD) + IQR": {"auprc": 0.1233507129600278, "auroc": 0.7478668047158521, "f1_bin": 0.251665037450276}, "LOF + IQR": {"auprc": 0.074585737562948, "auroc": 0.5827073486873636, "f1_bin": 0.1421052631578947}, "IForest + IQR": {"auprc": 0.4496345523809161, "auroc": 0.8457459058569224, "f1_bin": 0.49325829787682895}}, "machine-3-9": {"Artículo · umbral estático": {"auprc": 0.5431601301334121, "auroc": 0.8683549043611538, "f1_bin": 0.08911523307973489}, "Artículo · umbral dinámico (usa etiquetas)": {"auprc": 0.5431601301334121, "auroc": 0.8683549043611538, "f1_bin": 0.6165062841037133}, "M1 · umbral IQR": {"auprc": 0.5431601301334121, "auroc": 0.8683549043611538, "f1_bin": 0.09044948077113484}, "M2 · reponderación suave": {"auprc": 0.4688935956439127, "auroc": 0.8568385881306304, "f1_bin": 0.0846005034795764}, "M2 · recorte z>2": {"auprc": 0.4673672842813629, "auroc": 0.8578143793513708, "f1_bin": 0.07560351895013707}, "M3 · proyección al objetivo": {"auprc": 0.3866637277204651, "auroc": 0.8560201942230034, "f1_bin": 0.09382496414947619}, "M1+M3": {"auprc": 0.3866637277204651, "auroc": 0.8560201942230034, "f1_bin": 0.09843762185099907}, "M2+M3 (recorte z>2)": {"auprc": 0.33841895138519246, "auroc": 0.8518718077926503, "f1_bin": 0.07760689880325025}, "M1+M2 (recorte adaptativo, sin M3)": {"auprc": 0.446472912251141, "auroc": 0.8959865371846645, "f1_bin": 0.07946476490052826}, "M1+M2+M3 con recorte z>2": {"auprc": 0.33841895138519246, "auroc": 0.8518718077926503, "f1_bin": 0.07014959949100258}, "**Método final** (recorte adaptativo + proyección + IQR)": {"auprc": 0.3395711160816143, "auroc": 0.8766009369159107, "f1_bin": 0.05885940308272418}, "Error del LSTM (PD) + IQR": {"auprc": 0.1801106028977081, "auroc": 0.8442260277063939, "f1_bin": 0.08563644920933702}, "LOF + IQR": {"auprc": 0.0815684242067992, "auroc": 0.6150783043199286, "f1_bin": 0.033203125}, "IForest + IQR": {"auprc": 0.33074226426374975, "auroc": 0.8363652003858217, "f1_bin": 0.20613637500360946}}}, "seeds": {"machine-1-1": 5, "machine-1-2": 5, "machine-1-3": 5, "machine-1-4": 5, "machine-1-5": 5, "machine-1-6": 5, "machine-1-7": 5, "machine-1-8": 5, "machine-2-1": 5, "machine-2-2": 5, "machine-2-3": 5, "machine-2-4": 5, "machine-2-5": 5, "machine-2-6": 5, "machine-2-7": 5, "machine-2-8": 5, "machine-2-9": 5, "machine-3-1": 5, "machine-3-10": 5, "machine-3-11": 5, "machine-3-2": 5, "machine-3-3": 5, "machine-3-4": 5, "machine-3-5": 5, "machine-3-6": 5, "machine-3-7": 5, "machine-3-8": 5, "machine-3-9": 5}}, "nasa": {"n_units": 81, "n_runs": 405, "methods": {"Artículo · umbral estático": {"auprc": 0.20911816748132045, "auroc": 0.5796667502714266, "f1_bin": 0.13264504613407987, "f1_mac": 0.4369358622527343}, "Artículo · umbral dinámico (usa etiquetas)": {"auprc": 0.20911816748132045, "auroc": 0.5796667502714266, "f1_bin": 0.25358580488823296, "f1_mac": 0.43664832391773395}, "M1 · umbral IQR": {"auprc": 0.20911816748132045, "auroc": 0.5796667502714266, "f1_bin": 0.15676449107286852, "f1_mac": 0.43595104644996957}, "M2 · reponderación suave": {"auprc": 0.21572982585744643, "auroc": 0.6039194080749946, "f1_bin": 0.12937030454374193, "f1_mac": 0.42919746434889744}, "M2 · recorte z>2": {"auprc": 0.22030067911991683, "auroc": 0.6102967599550676, "f1_bin": 0.13667176755773344, "f1_mac": 0.43346239046284035}, "M3 · proyección al objetivo": {"auprc": 0.18773664867693393, "auroc": 0.5756707634651738, "f1_bin": 0.1295726211613373, "f1_mac": 0.43506712784513485}, "M1+M3": {"auprc": 0.18773664867693393, "auroc": 0.5756707634651738, "f1_bin": 0.17174591744893553, "f1_mac": 0.43516299499762784}, "M2+M3 (recorte z>2)": {"auprc": 0.20145336744038972, "auroc": 0.6077365855212317, "f1_bin": 0.1371589328950601, "f1_mac": 0.4322075024798976}, "M1+M2 (recorte adaptativo, sin M3)": {"auprc": 0.2197577315382385, "auroc": 0.606382308070627, "f1_bin": 0.16299158314168213, "f1_mac": 0.4469058526181886}, "M1+M2+M3 con recorte z>2": {"auprc": 0.20145336744038972, "auroc": 0.6077365855212317, "f1_bin": 0.17639465460659076, "f1_mac": 0.44582291402536506}, "**Método final** (recorte adaptativo + proyección + IQR)": {"auprc": 0.2025518747224406, "auroc": 0.6045320314842287, "f1_bin": 0.18279147469437174, "f1_mac": 0.44592185128148093}, "Error del LSTM (PD) + IQR": {"auprc": 0.15803963571514354, "auroc": 0.5138245876481492, "f1_bin": 0.15230193636716818, "f1_mac": 0.42343890238608367}, "LOF + IQR": {"auprc": 0.16226107043936475, "auroc": 0.5284605215283379, "f1_bin": 0.2443564520325392, "f1_mac": 0.3130105848082306}, "IForest + IQR": {"auprc": 0.2307196173093862, "auroc": 0.6502000986902444, "f1_bin": 0.11840355345020002, "f1_mac": 0.3810421341130583}}, "hyp": [{"h": "H1", "desc": "Final > Artículo", "metric": "auprc", "n": 81, "mean": -0.006566292758879891, "lo": -0.024896932701238408, "hi": 0.010461508718578684, "p": 0.9455706445611634, "rb": -0.008732309545317675, "wins": 43, "losses": 38, "p_holm": 0.9455706445611634}, {"h": "H2", "desc": "Final (sin etiquetas) ≥ Artículo dinámico (con etiquetas)", "metric": "f1_bin", "n": 81, "mean": -0.0707943301938612, "lo": -0.11486109503833461, "hi": -0.028589405305615836, "p": 0.0020307211562018077, "rb": -0.4073820915926179, "wins": 29, "losses": 47, "p_holm": 0.010153605781009038}, {"h": "H3a", "desc": "Final > M3 (mismo score sin recorte)", "metric": "auprc", "n": 81, "mean": 0.014815226045506657, "lo": 0.005223082549567621, "hi": 0.02549097877702495, "p": 0.0006468957598689835, "rb": 0.4363143631436314, "wins": 55, "losses": 26, "p_holm": 0.003881374559213901}, {"h": "H3b", "desc": "Final > M1+M3 (todo menos el recorte)", "metric": "f1_bin", "n": 81, "mean": 0.011045557245436233, "lo": 0.00026868790697364174, "hi": 0.02260451530281839, "p": 0.08090238804712598, "rb": 0.23509811181044057, "wins": 41, "losses": 32, "p_holm": 0.24270716414137794}, {"h": "H5a", "desc": "M3 (sólo proyección al objetivo) > Artículo", "metric": "auroc", "n": 81, "mean": -0.003995986806252733, "lo": -0.0123491208882565, "hi": 0.004128224653443022, "p": 0.3573277498577224, "rb": -0.11773562180066245, "wins": 37, "losses": 44, "p_holm": 0.7146554997154448}, {"h": "H5b", "desc": "M3 (sólo proyección al objetivo) > Artículo", "metric": "auprc", "n": 81, "mean": -0.021381518804386546, "lo": -0.034601026993792194, "hi": -0.010345469684364726, "p": 0.0034836322832986714, "rb": -0.37368262571514604, "wins": 31, "losses": 50, "p_holm": 0.013934529133194686}], "per_ds": {"A-1": {"Artículo · umbral estático": {"auprc": 0.01361959238593364, "auroc": 0.5536308697483925, "f1_bin": 0.038092969990860384}, "Artículo · umbral dinámico (usa etiquetas)": {"auprc": 0.01361959238593364, "auroc": 0.5536308697483925, "f1_bin": 0.03917690138536554}, "M1 · umbral IQR": {"auprc": 0.01361959238593364, "auroc": 0.5536308697483925, "f1_bin": 0.05456900837804364}, "M2 · reponderación suave": {"auprc": 0.012548841812810898, "auroc": 0.5559810482840548, "f1_bin": 0.04112664942034242}, "M2 · recorte z>2": {"auprc": 0.0203299792952949, "auroc": 0.5404762792753594, "f1_bin": 0.03416195288544706}, "M3 · proyección al objetivo": {"auprc": 0.01212716628027182, "auroc": 0.5319984252947393, "f1_bin": 0.036880858901874355}, "M1+M3": {"auprc": 0.01212716628027182, "auroc": 0.5319984252947393, "f1_bin": 0.048492873595778876}, "M2+M3 (recorte z>2)": {"auprc": 0.01423185284594462, "auroc": 0.5234783029097515, "f1_bin": 0.01902639531926056}, "M1+M2 (recorte adaptativo, sin M3)": {"auprc": 0.018366399191912, "auroc": 0.5353664988362375, "f1_bin": 0.026475227824952964}, "M1+M2+M3 con recorte z>2": {"auprc": 0.01423185284594462, "auroc": 0.5234783029097515, "f1_bin": 0.03718659807027308}, "**Método final** (recorte adaptativo + proyección + IQR)": {"auprc": 0.013379764437657019, "auroc": 0.5180941922383607, "f1_bin": 0.04675394517247288}, "Error del LSTM (PD) + IQR": {"auprc": 0.01740716825339276, "auroc": 0.5352651099185712, "f1_bin": 0.0440142714283685}, "LOF + IQR": {"auprc": 0.0225718448982344, "auroc": 0.5077581860500453, "f1_bin": 0.0387596899224806}, "IForest + IQR": {"auprc": 0.0106625363938308, "auroc": 0.4950407834849333, "f1_bin": 0.0387596899224806}}, "A-2": {"Artículo · umbral estático": {"auprc": 0.013267139245806438, "auroc": 0.2646453536601637, "f1_bin": 0.03621995343410526}, "Artículo · umbral dinámico (usa etiquetas)": {"auprc": 0.013267139245806438, "auroc": 0.2646453536601637, "f1_bin": 0.03384875849554986}, "M1 · umbral IQR": {"auprc": 0.013267139245806438, "auroc": 0.2646453536601637, "f1_bin": 0.04046343112503436}, "M2 · reponderación suave": {"auprc": 0.012872314733018319, "auroc": 0.2565164778429427, "f1_bin": 0.03037587201477212}, "M2 · recorte z>2": {"auprc": 0.01376105054702092, "auroc": 0.27784062233514895, "f1_bin": 0.03921260795884632}, "M3 · proyección al objetivo": {"auprc": 0.013993951226583898, "auroc": 0.26662350695834275, "f1_bin": 0.03914293144243784}, "M1+M3": {"auprc": 0.013993951226583898, "auroc": 0.26662350695834275, "f1_bin": 0.053322782817291456}, "M2+M3 (recorte z>2)": {"auprc": 0.01443769598592622, "auroc": 0.2844906980386697, "f1_bin": 0.04903812591440344}, "M1+M2 (recorte adaptativo, sin M3)": {"auprc": 0.01349660198573584, "auroc": 0.27315384927748093, "f1_bin": 0.04771519597691977}, "M1+M2+M3 con recorte z>2": {"auprc": 0.01443769598592622, "auroc": 0.2844906980386697, "f1_bin": 0.049885929072597}, "**Método final** (recorte adaptativo + proyección + IQR)": {"auprc": 0.01444795982650458, "auroc": 0.2835395601654455, "f1_bin": 0.05202326066982081}, "Error del LSTM (PD) + IQR": {"auprc": 0.01463997474375984, "auroc": 0.30216807921894845, "f1_bin": 0.05011278225714124}, "LOF + IQR": {"auprc": 0.0165523943063646, "auroc": 0.544970211679806, "f1_bin": 0.05580693815987929}, "IForest + IQR": {"auprc": 0.01552808244200266, "auroc": 0.3377851644303672, "f1_bin": 0.04362866006157497}}, "A-3": {"Artículo · umbral estático": {"auprc": 0.020010163062771404, "auroc": 0.39173150479701113, "f1_bin": 0.03800868951244764}, "Artículo · umbral dinámico (usa etiquetas)": {"auprc": 0.020010163062771404, "auroc": 0.39173150479701113, "f1_bin": 0.05503392928560582}, "M1 · umbral IQR": {"auprc": 0.020010163062771404, "auroc": 0.39173150479701113, "f1_bin": 0.05251602758809356}, "M2 · reponderación suave": {"auprc": 0.019252190762195417, "auroc": 0.3652875175315568, "f1_bin": 0.038768684911057526}, "M2 · recorte z>2": {"auprc": 0.01988904415311696, "auroc": 0.3738555817378101, "f1_bin": 0.04518632515000397}, "M3 · proyección al objetivo": {"auprc": 0.02212568492053656, "auroc": 0.4349952103988909, "f1_bin": 0.050179353984437035}, "M1+M3": {"auprc": 0.02212568492053656, "auroc": 0.4349952103988909, "f1_bin": 0.05814203703707558}, "M2+M3 (recorte z>2)": {"auprc": 0.022114844229679638, "auroc": 0.43055617306155886, "f1_bin": 0.05496092769691406}, "M1+M2 (recorte adaptativo, sin M3)": {"auprc": 0.01972648330879636, "auroc": 0.37258536066167897, "f1_bin": 0.04081710748223581}, "M1+M2+M3 con recorte z>2": {"auprc": 0.022114844229679638, "auroc": 0.43055617306155886, "f1_bin": 0.06278374987646147}, "**Método final** (recorte adaptativo + proyección + IQR)": {"auprc": 0.022025466088977878, "auroc": 0.4295346609487856, "f1_bin": 0.06532991192180351}, "Error del LSTM (PD) + IQR": {"auprc": 0.02208866350203994, "auroc": 0.43102880092908824, "f1_bin": 0.06126459636435346}, "LOF + IQR": {"auprc": 0.0923880501353087, "auroc": 0.8086291636184326, "f1_bin": 0.0122484689413823}, "IForest + IQR": {"auprc": 0.032307730030871416, "auroc": 0.5744725039172336, "f1_bin": 0.048648727617535706}}, "A-4": {"Artículo · umbral estático": {"auprc": 0.013832142911313041, "auroc": 0.3343611168853887, "f1_bin": 0.046374237820094415}, "Artículo · umbral dinámico (usa etiquetas)": {"auprc": 0.013832142911313041, "auroc": 0.3343611168853887, "f1_bin": 0.0391181858613718}, "M1 · umbral IQR": {"auprc": 0.013832142911313041, "auroc": 0.3343611168853887, "f1_bin": 0.053953030291242855}, "M2 · reponderación suave": {"auprc": 0.01334018674771854, "auroc": 0.29812515831933306, "f1_bin": 0.0443183243325979}, "M2 · recorte z>2": {"auprc": 0.013118503158008638, "auroc": 0.29382114430658113, "f1_bin": 0.03603882702757324}, "M3 · proyección al objetivo": {"auprc": 0.01611582826828706, "auroc": 0.42148905935313696, "f1_bin": 0.055840645176838254}, "M1+M3": {"auprc": 0.01611582826828706, "auroc": 0.42148905935313696, "f1_bin": 0.055778797131587995}, "M2+M3 (recorte z>2)": {"auprc": 0.014663818253851619, "auroc": 0.3528090819352955, "f1_bin": 0.054014051400172555}, "M1+M2 (recorte adaptativo, sin M3)": {"auprc": 0.013543034690834718, "auroc": 0.2995536956702005, "f1_bin": 0.04812256218805008}, "M1+M2+M3 con recorte z>2": {"auprc": 0.014663818253851619, "auroc": 0.3528090819352955, "f1_bin": 0.05941329599988586}, "**Método final** (recorte adaptativo + proyección + IQR)": {"auprc": 0.01498187059474256, "auroc": 0.3576833295279897, "f1_bin": 0.06298764873947012}, "Error del LSTM (PD) + IQR": {"auprc": 0.01596164493216786, "auroc": 0.40977041850828255, "f1_bin": 0.05947018223739033}, "LOF + IQR": {"auprc": 0.0150185860494254, "auroc": 0.2324434508900528, "f1_bin": 0.0222575516693163}, "IForest + IQR": {"auprc": 0.023402517717885603, "auroc": 0.5982417040669468, "f1_bin": 0.04301068149354518}}, "A-5": {"Artículo · umbral estático": {"auprc": 0.7508001564686486, "auroc": 0.887394592937088, "f1_bin": 0.09500929762176613}, "Artículo · umbral dinámico (usa etiquetas)": {"auprc": 0.7508001564686486, "auroc": 0.887394592937088, "f1_bin": 0.04845730026319782}, "M1 · umbral IQR": {"auprc": 0.7508001564686486, "auroc": 0.887394592937088, "f1_bin": 0.8408492127786975}, "M2 · reponderación suave": {"auprc": 0.7506771130068736, "auroc": 0.8836638217406858, "f1_bin": 0.07863977116995352}, "M2 · recorte z>2": {"auprc": 0.5811622008363352, "auroc": 0.8124018330181769, "f1_bin": 0.061003744959730974}, "M3 · proyección al objetivo": {"auprc": 0.6901414215841111, "auroc": 0.8529450094546942, "f1_bin": 0.07224102715432187}, "M1+M3": {"auprc": 0.6901414215841111, "auroc": 0.8529450094546942, "f1_bin": 0.7914285714285715}, "M2+M3 (recorte z>2)": {"auprc": 0.49535865156601017, "auroc": 0.7573256844006065, "f1_bin": 0.03479105533957216}, "M1+M2 (recorte adaptativo, sin M3)": {"auprc": 0.7506410260455969, "auroc": 0.8817950290455018, "f1_bin": 0.8460586101229263}, "M1+M2+M3 con recorte z>2": {"auprc": 0.49535865156601017, "auroc": 0.7573256844006065, "f1_bin": 0.05297532656023218}, "**Método final** (recorte adaptativo + proyección + IQR)": {"auprc": 0.6720411127718847, "auroc": 0.8269548218940053, "f1_bin": 0.7857142857142857}, "Error del LSTM (PD) + IQR": {"auprc": 0.7226997946147249, "auroc": 0.851983356331238, "f1_bin": 0.8}, "LOF + IQR": {"auprc": 0.4354803860741784, "auroc": 0.7992176453552751, "f1_bin": 0.6126126126126126}, "IForest + IQR": {"auprc": 0.5405324358306531, "auroc": 0.9069538849423349, "f1_bin": 0.17549813724311977}}, "A-6": {"Artículo · umbral estático": {"auprc": 0.9213518538912803, "auroc": 0.9834911394381434, "f1_bin": 0.0}, "Artículo · umbral dinámico (usa etiquetas)": {"auprc": 0.9213518538912803, "auroc": 0.9834911394381434, "f1_bin": 0.0}, "M1 · umbral IQR": {"auprc": 0.9213518538912803, "auroc": 0.9834911394381434, "f1_bin": 0.0}, "M2 · reponderación suave": {"auprc": 0.9269272050945183, "auroc": 0.978577403057981, "f1_bin": 0.0}, "M2 · recorte z>2": {"auprc": 0.8901989664928471, "auroc": 0.9618611083230173, "f1_bin": 0.0}, "M3 · proyección al objetivo": {"auprc": 0.7840318577310218, "auroc": 0.9604988457291979, "f1_bin": 0.0}, "M1+M3": {"auprc": 0.7840318577310218, "auroc": 0.9604988457291979, "f1_bin": 0.0}, "M2+M3 (recorte z>2)": {"auprc": 0.8238905829111326, "auroc": 0.9540806539752641, "f1_bin": 0.0}, "M1+M2 (recorte adaptativo, sin M3)": {"auprc": 0.9283091926354382, "auroc": 0.9781246166371128, "f1_bin": 0.0}, "M1+M2+M3 con recorte z>2": {"auprc": 0.8238905829111326, "auroc": 0.9540806539752641, "f1_bin": 0.0}, "**Método final** (recorte adaptativo + proyección + IQR)": {"auprc": 0.829973317319066, "auroc": 0.9543349281229437, "f1_bin": 0.0}, "Error del LSTM (PD) + IQR": {"auprc": 0.7942821708533211, "auroc": 0.9266636555254439, "f1_bin": 0.0}, "LOF + IQR": {"auprc": 0.3200390673454483, "auroc": 0.48949446284586295, "f1_bin": 0.0}, "IForest + IQR": {"auprc": 0.787082114122154, "auroc": 0.929355281207133, "f1_bin": 0.0}}, "A-7": {"Artículo · umbral estático": {"auprc": 0.2868794564596887, "auroc": 0.5665147462243876, "f1_bin": 0.09462132544410479}, "Artículo · umbral dinámico (usa etiquetas)": {"auprc": 0.2868794564596887, "auroc": 0.5665147462243876, "f1_bin": 0.6378249897110668}, "M1 · umbral IQR": {"auprc": 0.2868794564596887, "auroc": 0.5665147462243876, "f1_bin": 0.05943611712128686}, "M2 · reponderación suave": {"auprc": 0.27045840452714914, "auroc": 0.550118302343877, "f1_bin": 0.0341881161569753}, "M2 · recorte z>2": {"auprc": 0.26178375702787793, "auroc": 0.5259002897156332, "f1_bin": 0.011419156379578481}, "M3 · proyección al objetivo": {"auprc": 0.31598603280310933, "auroc": 0.5516062496031477, "f1_bin": 0.26420758747882755}, "M1+M3": {"auprc": 0.31598603280310933, "auroc": 0.5516062496031477, "f1_bin": 0.20516819612121928}, "M2+M3 (recorte z>2)": {"auprc": 0.2899572716949567, "auroc": 0.5495763635306995, "f1_bin": 0.18056729519777553}, "M1+M2 (recorte adaptativo, sin M3)": {"auprc": 0.24484292987155723, "auroc": 0.4751170579764891, "f1_bin": 0.0050796307394225205}, "M1+M2+M3 con recorte z>2": {"auprc": 0.2899572716949567, "auroc": 0.5495763635306995, "f1_bin": 0.15414481449388343}, "**Método final** (recorte adaptativo + proyección + IQR)": {"auprc": 0.26061711284774286, "auroc": 0.4948975757839918, "f1_bin": 0.1018994063713367}, "Error del LSTM (PD) + IQR": {"auprc": 0.3786573849064806, "auroc": 0.6514365246177573, "f1_bin": 0.2828064145451823}, "LOF + IQR": {"auprc": 0.2797716150081566, "auroc": 0.4911175710594315, "f1_bin": 0.7169304269931323}, "IForest + IQR": {"auprc": 0.18619863534679218, "auroc": 0.1777916407031092, "f1_bin": 0.0}}, "A-8": {"Artículo · umbral estático": {"auprc": 0.3607294809084748, "auroc": 0.3107026641169762, "f1_bin": 0.07355780913535698}, "Artículo · umbral dinámico (usa etiquetas)": {"auprc": 0.3607294809084748, "auroc": 0.3107026641169762, "f1_bin": 0.5535130917307215}, "M1 · umbral IQR": {"auprc": 0.3607294809084748, "auroc": 0.3107026641169762, "f1_bin": 0.08698641996587556}, "M2 · reponderación suave": {"auprc": 0.3740685184559568, "auroc": 0.3548953475887725, "f1_bin": 0.07375371518012155}, "M2 · recorte z>2": {"auprc": 0.3882807663027503, "auroc": 0.3898661412524629, "f1_bin": 0.07844448053918926}, "M3 · proyección al objetivo": {"auprc": 0.37498154406722756, "auroc": 0.32592879631591004, "f1_bin": 0.07267533438782824}, "M1+M3": {"auprc": 0.37498154406722756, "auroc": 0.32592879631591004, "f1_bin": 0.08359830209681941}, "M2+M3 (recorte z>2)": {"auprc": 0.4063433812960068, "auroc": 0.4111837082906531, "f1_bin": 0.11530628588153632}, "M1+M2 (recorte adaptativo, sin M3)": {"auprc": 0.3911704419037773, "auroc": 0.39829869101060017, "f1_bin": 0.08039174038638489}, "M1+M2+M3 con recorte z>2": {"auprc": 0.4063433812960068, "auroc": 0.4111837082906531, "f1_bin": 0.14754974526385217}, "**Método final** (recorte adaptativo + proyección + IQR)": {"auprc": 0.4109266294354758, "auroc": 0.4237951987278631, "f1_bin": 0.14497520298070085}, "Error del LSTM (PD) + IQR": {"auprc": 0.3564095030072769, "auroc": 0.2863914432543954, "f1_bin": 0.06810434064131746}, "LOF + IQR": {"auprc": 0.47653202466339495, "auroc": 0.4948351648351649, "f1_bin": 0.9519859225741578}, "IForest + IQR": {"auprc": 0.5055380636847926, "auroc": 0.598675764689496, "f1_bin": 0.012258551752914461}}, "A-9": {"Artículo · umbral estático": {"auprc": 0.34078638267019545, "auroc": 0.28216011497602755, "f1_bin": 0.0}, "Artículo · umbral dinámico (usa etiquetas)": {"auprc": 0.34078638267019545, "auroc": 0.28216011497602755, "f1_bin": 0.56995977747391}, "M1 · umbral IQR": {"auprc": 0.34078638267019545, "auroc": 0.28216011497602755, "f1_bin": 0.0}, "M2 · reponderación suave": {"auprc": 0.31643018200556516, "auroc": 0.2136512260490419, "f1_bin": 0.0}, "M2 · recorte z>2": {"auprc": 0.3637449119098107, "auroc": 0.3456411170732543, "f1_bin": 0.0}, "M3 · proyección al objetivo": {"auprc": 0.3413755298615094, "auroc": 0.2818471767442126, "f1_bin": 0.0}, "M1+M3": {"auprc": 0.3413755298615094, "auroc": 0.2818471767442126, "f1_bin": 0.0}, "M2+M3 (recorte z>2)": {"auprc": 0.36441977558590705, "auroc": 0.3454054676472773, "f1_bin": 0.0}, "M1+M2 (recorte adaptativo, sin M3)": {"auprc": 0.3678274013458733, "auroc": 0.35592139114135996, "f1_bin": 0.0}, "M1+M2+M3 con recorte z>2": {"auprc": 0.36441977558590705, "auroc": 0.3454054676472773, "f1_bin": 0.0}, "**Método final** (recorte adaptativo + proyección + IQR)": {"auprc": 0.36849501325100925, "auroc": 0.35566844388061236, "f1_bin": 0.0}, "Error del LSTM (PD) + IQR": {"auprc": 0.3416409685958003, "auroc": 0.27572682907316287, "f1_bin": 0.0021722265321954997}, "LOF + IQR": {"auprc": 0.44566638640552175, "auroc": 0.43758476053327844, "f1_bin": 0.9397194490079614}, "IForest + IQR": {"auprc": 0.48632318884845116, "auroc": 0.6378088037807227, "f1_bin": 0.0}}, "B-1": {"Artículo · umbral estático": {"auprc": 0.0094302044466821, "auroc": 0.36001508737364324, "f1_bin": 0.03720782288287716}, "Artículo · umbral dinámico (usa etiquetas)": {"auprc": 0.0094302044466821, "auroc": 0.36001508737364324, "f1_bin": 0.03720782288287716}, "M1 · umbral IQR": {"auprc": 0.0094302044466821, "auroc": 0.36001508737364324, "f1_bin": 0.20253164556962017}, "M2 · reponderación suave": {"auprc": 0.016924131432824337, "auroc": 0.7216107990095583, "f1_bin": 0.0383724776711875}, "M2 · recorte z>2": {"auprc": 0.01635785140238648, "auroc": 0.7156003443471161, "f1_bin": 0.037333634774973555}, "M3 · proyección al objetivo": {"auprc": 0.0080682741794788, "auroc": 0.343929462090755, "f1_bin": 0.03718132178497452}, "M1+M3": {"auprc": 0.0080682741794788, "auroc": 0.343929462090755, "f1_bin": 0.1066666666666666}, "M2+M3 (recorte z>2)": {"auprc": 0.015232456079487899, "auroc": 0.6897175111158443, "f1_bin": 0.0357734349122225}, "M1+M2 (recorte adaptativo, sin M3)": {"auprc": 0.017010536103229563, "auroc": 0.7205135031994108, "f1_bin": 0.20253164556962017}, "M1+M2+M3 con recorte z>2": {"auprc": 0.015232456079487899, "auroc": 0.6897175111158443, "f1_bin": 0.02222222222222216}, "**Método final** (recorte adaptativo + proyección + IQR)": {"auprc": 0.015189861649868899, "auroc": 0.6873184412080547, "f1_bin": 0.1066666666666666}, "Error del LSTM (PD) + IQR": {"auprc": 0.008300583506547322, "auroc": 0.344585496596466, "f1_bin": 0.1066666666666666}, "LOF + IQR": {"auprc": 0.0088683487384461, "auroc": 0.49760554505356025, "f1_bin": 0.034855179185076}, "IForest + IQR": {"auprc": 0.015483987319241398, "auroc": 0.6839319470699432, "f1_bin": 0.0}}, "C-1": {"Artículo · umbral estático": {"auprc": 0.217191256633811, "auroc": 0.5918662754869651, "f1_bin": 0.08782204477844342}, "Artículo · umbral dinámico (usa etiquetas)": {"auprc": 0.217191256633811, "auroc": 0.5918662754869651, "f1_bin": 0.11418168340808241}, "M1 · umbral IQR": {"auprc": 0.217191256633811, "auroc": 0.5918662754869651, "f1_bin": 0.01767615631161522}, "M2 · reponderación suave": {"auprc": 0.2143261853945583, "auroc": 0.5978515258687673, "f1_bin": 0.09941867033986378}, "M2 · recorte z>2": {"auprc": 0.20618404949691954, "auroc": 0.5980782627334351, "f1_bin": 0.06011491422800498}, "M3 · proyección al objetivo": {"auprc": 0.18373893863427876, "auroc": 0.588090453607695, "f1_bin": 0.10878006551036952}, "M1+M3": {"auprc": 0.18373893863427876, "auroc": 0.588090453607695, "f1_bin": 0.1219503812590023}, "M2+M3 (recorte z>2)": {"auprc": 0.18180250329117983, "auroc": 0.5980852959301235, "f1_bin": 0.10274928575904316}, "M1+M2 (recorte adaptativo, sin M3)": {"auprc": 0.20207007044968264, "auroc": 0.5964067063204994, "f1_bin": 0.02177736623195246}, "M1+M2+M3 con recorte z>2": {"auprc": 0.18180250329117983, "auroc": 0.5980852959301235, "f1_bin": 0.12266037508205267}, "**Método final** (recorte adaptativo + proyección + IQR)": {"auprc": 0.18345838002978518, "auroc": 0.6070003751038234, "f1_bin": 0.13573000545695385}, "Error del LSTM (PD) + IQR": {"auprc": 0.17727755335672915, "auroc": 0.5835727969348659, "f1_bin": 0.12105865652167172}, "LOF + IQR": {"auprc": 0.154448914158771, "auroc": 0.5075841304289581, "f1_bin": 0.1370967741935483}, "IForest + IQR": {"auprc": 0.15545226068589918, "auroc": 0.53220283069421, "f1_bin": 0.0399647661126306}}, "C-2": {"Artículo · umbral estático": {"auprc": 0.12850934772187023, "auroc": 0.6494241514014909, "f1_bin": 0.03521497814678702}, "Artículo · umbral dinámico (usa etiquetas)": {"auprc": 0.12850934772187023, "auroc": 0.6494241514014909, "f1_bin": 0.07542174188393666}, "M1 · umbral IQR": {"auprc": 0.12850934772187023, "auroc": 0.6494241514014909, "f1_bin": 0.0}, "M2 · reponderación suave": {"auprc": 0.1272245356986777, "auroc": 0.6507665011750424, "f1_bin": 0.01244359965332152}, "M2 · recorte z>2": {"auprc": 0.12443078375642624, "auroc": 0.6524458001960999, "f1_bin": 0.0}, "M3 · proyección al objetivo": {"auprc": 0.1060379366428579, "auroc": 0.6659323300079374, "f1_bin": 0.07425923808959282}, "M1+M3": {"auprc": 0.1060379366428579, "auroc": 0.6659323300079374, "f1_bin": 0.05586904132134244}, "M2+M3 (recorte z>2)": {"auprc": 0.10801631124822308, "auroc": 0.6795986179633636, "f1_bin": 0.06462372679702871}, "M1+M2 (recorte adaptativo, sin M3)": {"auprc": 0.12351362474443266, "auroc": 0.6534403062892006, "f1_bin": 0.0}, "M1+M2+M3 con recorte z>2": {"auprc": 0.10801631124822308, "auroc": 0.6795986179633636, "f1_bin": 0.07471555802643678}, "**Método final** (recorte adaptativo + proyección + IQR)": {"auprc": 0.10946263601197423, "auroc": 0.6886301028745739, "f1_bin": 0.07357831948138152}, "Error del LSTM (PD) + IQR": {"auprc": 0.10570431009481049, "auroc": 0.6783356419155526, "f1_bin": 0.060457871224136664}, "LOF + IQR": {"auprc": 0.1166125549857362, "auroc": 0.6139985681602416, "f1_bin": 0.0683760683760683}, "IForest + IQR": {"auprc": 0.08738946150072029, "auroc": 0.5687808351361026, "f1_bin": 0.06078042897751741}}, "D-11": {"Artículo · umbral estático": {"auprc": 0.010967191521729801, "auroc": 0.4260725497035228, "f1_bin": 0.2318840579710145}, "Artículo · umbral dinámico (usa etiquetas)": {"auprc": 0.010967191521729801, "auroc": 0.4260725497035228, "f1_bin": 0.029649512044232364}, "M1 · umbral IQR": {"auprc": 0.010967191521729801, "auroc": 0.4260725497035228, "f1_bin": 0.2318840579710145}, "M2 · reponderación suave": {"auprc": 0.0098949743636999, "auroc": 0.2975289098763113, "f1_bin": 0.061910792176727125}, "M2 · recorte z>2": {"auprc": 0.010198007085042201, "auroc": 0.3201358518240276, "f1_bin": 0.10440410862529899}, "M3 · proyección al objetivo": {"auprc": 0.00963075242444386, "auroc": 0.3850062157352007, "f1_bin": 0.123076923076923}, "M1+M3": {"auprc": 0.00963075242444386, "auroc": 0.3850062157352007, "f1_bin": 0.123076923076923}, "M2+M3 (recorte z>2)": {"auprc": 0.00855291742654218, "auroc": 0.2657696332269056, "f1_bin": 0.05448432772573971}, "M1+M2 (recorte adaptativo, sin M3)": {"auprc": 0.011267538119716342, "auroc": 0.3839531181526298, "f1_bin": 0.22995169082125605}, "M1+M2+M3 con recorte z>2": {"auprc": 0.00855291742654218, "auroc": 0.2657696332269056, "f1_bin": 0.10538794538794531}, "**Método final** (recorte adaptativo + proyección + IQR)": {"auprc": 0.00943534180909566, "auroc": 0.32393415792439156, "f1_bin": 0.1114696685428392}, "Error del LSTM (PD) + IQR": {"auprc": 0.014140834326386739, "auroc": 0.5825700052766674, "f1_bin": 0.123076923076923}, "LOF + IQR": {"auprc": 0.0084675180455302, "auroc": 0.4999318057828696, "f1_bin": 0.0326203208556149}, "IForest + IQR": {"auprc": 0.036252651041458715, "auroc": 0.6372653448167923, "f1_bin": 0.05518997080853564}}, "D-12": {"Artículo · umbral estático": {"auprc": 0.5772832301268653, "auroc": 0.5942443455056793, "f1_bin": 0.5333117467483935}, "Artículo · umbral dinámico (usa etiquetas)": {"auprc": 0.5772832301268653, "auroc": 0.5942443455056793, "f1_bin": 0.616912967623868}, "M1 · umbral IQR": {"auprc": 0.5772832301268653, "auroc": 0.5942443455056793, "f1_bin": 0.616912967623868}, "M2 · reponderación suave": {"auprc": 0.7943881240265622, "auroc": 0.9216059481227576, "f1_bin": 0.5289506148484251}, "M2 · recorte z>2": {"auprc": 0.8088390187100511, "auroc": 0.9247279118660907, "f1_bin": 0.5267196984364259}, "M3 · proyección al objetivo": {"auprc": 0.5909690662168289, "auroc": 0.5942821723795575, "f1_bin": 0.5295865552767708}, "M1+M3": {"auprc": 0.5909690662168289, "auroc": 0.5942821723795575, "f1_bin": 0.6149681761831253}, "M2+M3 (recorte z>2)": {"auprc": 0.832466150800672, "auroc": 0.9297374152448284, "f1_bin": 0.5263209542581166}, "M1+M2 (recorte adaptativo, sin M3)": {"auprc": 0.8039894899127515, "auroc": 0.9233952233705398, "f1_bin": 0.7396359148847964}, "M1+M2+M3 con recorte z>2": {"auprc": 0.832466150800672, "auroc": 0.9297374152448284, "f1_bin": 0.7343597267076714}, "**Método final** (recorte adaptativo + proyección + IQR)": {"auprc": 0.8276250060380856, "auroc": 0.9284430095402796, "f1_bin": 0.7367101427811045}, "Error del LSTM (PD) + IQR": {"auprc": 0.23680030890456688, "auroc": 0.07521304686635688, "f1_bin": 0.07987373811102078}, "LOF + IQR": {"auprc": 0.3451739153379756, "auroc": 0.4743107441358195, "f1_bin": 0.8169944452784869}, "IForest + IQR": {"auprc": 0.8513979042078657, "auroc": 0.9507269277974736, "f1_bin": 0.0}}, "D-13": {"Artículo · umbral estático": {"auprc": 0.05698773741405514, "auroc": 0.5308133284069954, "f1_bin": 0.1125941334231139}, "Artículo · umbral dinámico (usa etiquetas)": {"auprc": 0.05698773741405514, "auroc": 0.5308133284069954, "f1_bin": 0.1125941334231139}, "M1 · umbral IQR": {"auprc": 0.05698773741405514, "auroc": 0.5308133284069954, "f1_bin": 0.2090464594375209}, "M2 · reponderación suave": {"auprc": 0.03328437730424466, "auroc": 0.6557731354809504, "f1_bin": 0.048599479173758245}, "M2 · recorte z>2": {"auprc": 0.03158594212121964, "auroc": 0.6319618641529028, "f1_bin": 0.03766098211878036}, "M3 · proyección al objetivo": {"auprc": 0.06419790372664738, "auroc": 0.5234860664523044, "f1_bin": 0.11142877927330541}, "M1+M3": {"auprc": 0.06419790372664738, "auroc": 0.5234860664523044, "f1_bin": 0.1493763869510331}, "M2+M3 (recorte z>2)": {"auprc": 0.032246599476196555, "auroc": 0.632246127166091, "f1_bin": 0.03736179946626}, "M1+M2 (recorte adaptativo, sin M3)": {"auprc": 0.02578257754908288, "auroc": 0.49386238208410715, "f1_bin": 0.2209944751381215}, "M1+M2+M3 con recorte z>2": {"auprc": 0.032246599476196555, "auroc": 0.632246127166091, "f1_bin": 0.0}, "**Método final** (recorte adaptativo + proyección + IQR)": {"auprc": 0.02493466450563282, "auroc": 0.47061039989881037, "f1_bin": 0.16}, "Error del LSTM (PD) + IQR": {"auprc": 0.05133367189135246, "auroc": 0.4972543987537697, "f1_bin": 0.14726144442229747}, "LOF + IQR": {"auprc": 0.025409743499551003, "auroc": 0.5159931230985335, "f1_bin": 0.0810266733769501}, "IForest + IQR": {"auprc": 0.32262233606890106, "auroc": 0.8383229980094932, "f1_bin": 0.6983571703967539}}, "D-14": {"Artículo · umbral estático": {"auprc": 0.09723792507102365, "auroc": 0.38819838866350487, "f1_bin": 0.13030592438740937}, "Artículo · umbral dinámico (usa etiquetas)": {"auprc": 0.09723792507102365, "auroc": 0.38819838866350487, "f1_bin": 0.16413511337651482}, "M1 · umbral IQR": {"auprc": 0.09723792507102365, "auroc": 0.38819838866350487, "f1_bin": 0.13971076241041971}, "M2 · reponderación suave": {"auprc": 0.10100351499695222, "auroc": 0.3788549225758528, "f1_bin": 0.13230437128980024}, "M2 · recorte z>2": {"auprc": 0.108473379912492, "auroc": 0.3752684608498561, "f1_bin": 0.1396507401481849}, "M3 · proyección al objetivo": {"auprc": 0.10803554666058286, "auroc": 0.4662781174409081, "f1_bin": 0.11755616459209146}, "M1+M3": {"auprc": 0.10803554666058286, "auroc": 0.4662781174409081, "f1_bin": 0.11579239451510684}, "M2+M3 (recorte z>2)": {"auprc": 0.10809011951155209, "auroc": 0.4301910367026647, "f1_bin": 0.1119198229517862}, "M1+M2 (recorte adaptativo, sin M3)": {"auprc": 0.11295076022803918, "auroc": 0.37984686589337746, "f1_bin": 0.15124014182759635}, "M1+M2+M3 con recorte z>2": {"auprc": 0.10809011951155209, "auroc": 0.4301910367026647, "f1_bin": 0.11754668717640462}, "**Método final** (recorte adaptativo + proyección + IQR)": {"auprc": 0.11500558471777142, "auroc": 0.44966954269279846, "f1_bin": 0.11935071101437092}, "Error del LSTM (PD) + IQR": {"auprc": 0.07398049603068566, "auroc": 0.19520522636801702, "f1_bin": 0.10129646740572482}, "LOF + IQR": {"auprc": 0.0879869428590649, "auroc": 0.5219396986838847, "f1_bin": 0.2929765886287625}, "IForest + IQR": {"auprc": 0.3834288124864237, "auroc": 0.8182159114717255, "f1_bin": 0.07394075339819935}}, "D-15": {"Artículo · umbral estático": {"auprc": 0.37238070037221005, "auroc": 0.6375576287645597, "f1_bin": 0.24954214171071545}, "Artículo · umbral dinámico (usa etiquetas)": {"auprc": 0.37238070037221005, "auroc": 0.6375576287645597, "f1_bin": 0.6289241729736503}, "M1 · umbral IQR": {"auprc": 0.37238070037221005, "auroc": 0.6375576287645597, "f1_bin": 0.13058576904466124}, "M2 · reponderación suave": {"auprc": 0.37044155931074896, "auroc": 0.6405125327703335, "f1_bin": 0.253774935790764}, "M2 · recorte z>2": {"auprc": 0.36149568295760526, "auroc": 0.6387711217413266, "f1_bin": 0.24011905493564564}, "M3 · proyección al objetivo": {"auprc": 0.37782866164424567, "auroc": 0.6533093677980555, "f1_bin": 0.2615221668714306}, "M1+M3": {"auprc": 0.37782866164424567, "auroc": 0.6533093677980555, "f1_bin": 0.32145972519152577}, "M2+M3 (recorte z>2)": {"auprc": 0.3773698626629466, "auroc": 0.6554447096714883, "f1_bin": 0.27013580175678664}, "M1+M2 (recorte adaptativo, sin M3)": {"auprc": 0.3652009902504133, "auroc": 0.6433480317722978, "f1_bin": 0.14930655709745716}, "M1+M2+M3 con recorte z>2": {"auprc": 0.3773698626629466, "auroc": 0.6554447096714883, "f1_bin": 0.3146593234975444}, "**Método final** (recorte adaptativo + proyección + IQR)": {"auprc": 0.38478277002049, "auroc": 0.6691164769395124, "f1_bin": 0.37186913421824475}, "Error del LSTM (PD) + IQR": {"auprc": 0.36858969598790475, "auroc": 0.6367017167903513, "f1_bin": 0.3386952957345787}, "LOF + IQR": {"auprc": 0.3271627732974974, "auroc": 0.5365801984236426, "f1_bin": 0.1704834605597964}, "IForest + IQR": {"auprc": 0.44146701201290134, "auroc": 0.740185172857235, "f1_bin": 0.0}}, "D-16": {"Artículo · umbral estático": {"auprc": 0.3200068441414018, "auroc": 0.4427376912708299, "f1_bin": 0.06815871733912904}, "Artículo · umbral dinámico (usa etiquetas)": {"auprc": 0.3200068441414018, "auroc": 0.4427376912708299, "f1_bin": 0.13723419353979976}, "M1 · umbral IQR": {"auprc": 0.3200068441414018, "auroc": 0.4427376912708299, "f1_bin": 0.034068103557199705}, "M2 · reponderación suave": {"auprc": 0.3140688681921964, "auroc": 0.44174587493173456, "f1_bin": 0.05642647064557631}, "M2 · recorte z>2": {"auprc": 0.31330820453453645, "auroc": 0.44273625948811723, "f1_bin": 0.057741868598212044}, "M3 · proyección al objetivo": {"auprc": 0.2926473412335783, "auroc": 0.431285372703267, "f1_bin": 0.10200256505079448}, "M1+M3": {"auprc": 0.2926473412335783, "auroc": 0.431285372703267, "f1_bin": 0.08581919727372986}, "M2+M3 (recorte z>2)": {"auprc": 0.2903492073609363, "auroc": 0.4334918521336631, "f1_bin": 0.10375740764090276}, "M1+M2 (recorte adaptativo, sin M3)": {"auprc": 0.3182822644222182, "auroc": 0.44831714396166084, "f1_bin": 0.03885108723418458}, "M1+M2+M3 con recorte z>2": {"auprc": 0.2903492073609363, "auroc": 0.4334918521336631, "f1_bin": 0.08811885007990161}, "**Método final** (recorte adaptativo + proyección + IQR)": {"auprc": 0.2948959516735115, "auroc": 0.4471296847419007, "f1_bin": 0.08781158126858951}, "Error del LSTM (PD) + IQR": {"auprc": 0.2859476919246299, "auroc": 0.4475238340686561, "f1_bin": 0.10467581199288511}, "LOF + IQR": {"auprc": 0.5126755517174857, "auroc": 0.6458020130864939, "f1_bin": 0.301994301994302}, "IForest + IQR": {"auprc": 0.3768285153149944, "auroc": 0.5224739773491975, "f1_bin": 0.0}}, "D-1": {"Artículo · umbral estático": {"auprc": 0.2356797233074393, "auroc": 0.07477448661941744, "f1_bin": 0.052960826445256036}, "Artículo · umbral dinámico (usa etiquetas)": {"auprc": 0.2356797233074393, "auroc": 0.07477448661941744, "f1_bin": 0.15173128992486187}, "M1 · umbral IQR": {"auprc": 0.2356797233074393, "auroc": 0.07477448661941744, "f1_bin": 0.07658796750450247}, "M2 · reponderación suave": {"auprc": 0.368417750936037, "auroc": 0.5311464948986455, "f1_bin": 0.05327502425941646}, "M2 · recorte z>2": {"auprc": 0.4172944428085583, "auroc": 0.6251749600553207, "f1_bin": 0.0538719056895637}, "M3 · proyección al objetivo": {"auprc": 0.23873111214614395, "auroc": 0.07733532997727223, "f1_bin": 0.05124092001576828}, "M1+M3": {"auprc": 0.23873111214614395, "auroc": 0.07733532997727223, "f1_bin": 0.06712306851832296}, "M2+M3 (recorte z>2)": {"auprc": 0.449772134995551, "auroc": 0.6684856187354116, "f1_bin": 0.05273064262428023}, "M1+M2 (recorte adaptativo, sin M3)": {"auprc": 0.41083074595543606, "auroc": 0.6157169391401215, "f1_bin": 0.1203764036038945}, "M1+M2+M3 con recorte z>2": {"auprc": 0.449772134995551, "auroc": 0.6684856187354116, "f1_bin": 0.14157146630823975}, "**Método final** (recorte adaptativo + proyección + IQR)": {"auprc": 0.4405519280505693, "auroc": 0.6568119189730737, "f1_bin": 0.13557685073556963}, "Error del LSTM (PD) + IQR": {"auprc": 0.2417199237213994, "auroc": 0.09742953333758148, "f1_bin": 0.07221565474079716}, "LOF + IQR": {"auprc": 0.3952677220599636, "auroc": 0.3823671972169711, "f1_bin": 0.8838859325965343}, "IForest + IQR": {"auprc": 0.5796372536813106, "auroc": 0.7950087382673567, "f1_bin": 0.04693593942043388}}, "D-2": {"Artículo · umbral estático": {"auprc": 0.6184702310520703, "auroc": 0.6939200253612985, "f1_bin": 0.23266084620392496}, "Artículo · umbral dinámico (usa etiquetas)": {"auprc": 0.6184702310520703, "auroc": 0.6939200253612985, "f1_bin": 0.9602844300188978}, "M1 · umbral IQR": {"auprc": 0.6184702310520703, "auroc": 0.6939200253612985, "f1_bin": 0.3050393161637384}, "M2 · reponderación suave": {"auprc": 0.6186292093640541, "auroc": 0.6955881525319062, "f1_bin": 0.22603811460737142}, "M2 · recorte z>2": {"auprc": 0.6280712664048629, "auroc": 0.6982073174689711, "f1_bin": 0.23085650670846208}, "M3 · proyección al objetivo": {"auprc": 0.6619111638316877, "auroc": 0.7344502445702092, "f1_bin": 0.22297853821188332}, "M1+M3": {"auprc": 0.6619111638316877, "auroc": 0.7344502445702092, "f1_bin": 0.38182768286682267}, "M2+M3 (recorte z>2)": {"auprc": 0.6683033899512186, "auroc": 0.7373270931075907, "f1_bin": 0.22437876784724087}, "M1+M2 (recorte adaptativo, sin M3)": {"auprc": 0.6371432226878356, "auroc": 0.712583118410316, "f1_bin": 0.30974534269635656}, "M1+M2+M3 con recorte z>2": {"auprc": 0.6683033899512186, "auroc": 0.7373270931075907, "f1_bin": 0.3590286390216216}, "**Método final** (recorte adaptativo + proyección + IQR)": {"auprc": 0.6745652283891908, "auroc": 0.7450588304975079, "f1_bin": 0.3888431794178643}, "Error del LSTM (PD) + IQR": {"auprc": 0.5105140417355978, "auroc": 0.3515607831360915, "f1_bin": 0.38474369382856555}, "LOF + IQR": {"auprc": 0.4996744435619841, "auroc": 0.5142489835209477, "f1_bin": 0.9927024482109228}, "IForest + IQR": {"auprc": 0.7698569534974838, "auroc": 0.8076030786308438, "f1_bin": 0.2960634610131951}}, "D-3": {"Artículo · umbral estático": {"auprc": 0.38654359263162064, "auroc": 0.49783633356671275, "f1_bin": 0.22080425520779734}, "Artículo · umbral dinámico (usa etiquetas)": {"auprc": 0.38654359263162064, "auroc": 0.49783633356671275, "f1_bin": 0.7332478977980706}, "M1 · umbral IQR": {"auprc": 0.38654359263162064, "auroc": 0.49783633356671275, "f1_bin": 0.31559054460085506}, "M2 · reponderación suave": {"auprc": 0.45518035572726073, "auroc": 0.6674787207410189, "f1_bin": 0.22064032349207285}, "M2 · recorte z>2": {"auprc": 0.46933420870120823, "auroc": 0.6705588888934739, "f1_bin": 0.24463475483094016}, "M3 · proyección al objetivo": {"auprc": 0.39574915146857076, "auroc": 0.49846563865058036, "f1_bin": 0.20378876758802064}, "M1+M3": {"auprc": 0.39574915146857076, "auroc": 0.49846563865058036, "f1_bin": 0.34947113941345165}, "M2+M3 (recorte z>2)": {"auprc": 0.4915843595631098, "auroc": 0.7004617269074648, "f1_bin": 0.23017550479006993}, "M1+M2 (recorte adaptativo, sin M3)": {"auprc": 0.4692022668310112, "auroc": 0.6608792561383623, "f1_bin": 0.26961830449211815}, "M1+M2+M3 con recorte z>2": {"auprc": 0.4915843595631098, "auroc": 0.7004617269074648, "f1_bin": 0.33275269277263564}, "**Método final** (recorte adaptativo + proyección + IQR)": {"auprc": 0.4992662013328726, "auroc": 0.6942563767854792, "f1_bin": 0.3337357258133111}, "Error del LSTM (PD) + IQR": {"auprc": 0.3390911419898738, "auroc": 0.340433675516289, "f1_bin": 0.367642408209685}, "LOF + IQR": {"auprc": 0.4331479235858201, "auroc": 0.519271604912799, "f1_bin": 0.2894082545997016}, "IForest + IQR": {"auprc": 0.6083384861299885, "auroc": 0.8280984568066806, "f1_bin": 0.15189638045955642}}, "D-5": {"Artículo · umbral estático": {"auprc": 0.07617108836301942, "auroc": 0.723635786719516, "f1_bin": 0.0652645106009294}, "Artículo · umbral dinámico (usa etiquetas)": {"auprc": 0.07617108836301942, "auroc": 0.723635786719516, "f1_bin": 0.0475256617990659}, "M1 · umbral IQR": {"auprc": 0.07617108836301942, "auroc": 0.723635786719516, "f1_bin": 0.059039384593701436}, "M2 · reponderación suave": {"auprc": 0.08646518645805246, "auroc": 0.7652203314008983, "f1_bin": 0.05956322601075122}, "M2 · recorte z>2": {"auprc": 0.11971936375546685, "auroc": 0.7837919940492446, "f1_bin": 0.07457474351401769}, "M3 · proyección al objetivo": {"auprc": 0.0372408015986588, "auroc": 0.678102624522418, "f1_bin": 0.05751410852066909}, "M1+M3": {"auprc": 0.0372408015986588, "auroc": 0.678102624522418, "f1_bin": 0.05236648649617578}, "M2+M3 (recorte z>2)": {"auprc": 0.07151731993732317, "auroc": 0.7680178106525803, "f1_bin": 0.06048626971762715}, "M1+M2 (recorte adaptativo, sin M3)": {"auprc": 0.1231815046045519, "auroc": 0.7913698441307814, "f1_bin": 0.06922882861539731}, "M1+M2+M3 con recorte z>2": {"auprc": 0.07151731993732317, "auroc": 0.7680178106525803, "f1_bin": 0.0493295790467485}, "**Método final** (recorte adaptativo + proyección + IQR)": {"auprc": 0.07830096049899551, "auroc": 0.7714504186075544, "f1_bin": 0.04911278039380966}, "Error del LSTM (PD) + IQR": {"auprc": 0.02527884167997794, "auroc": 0.6834421791000522, "f1_bin": 0.04931352120060738}, "LOF + IQR": {"auprc": 0.0129901378120446, "auroc": 0.5919103017251468, "f1_bin": 0.026017160254636}, "IForest + IQR": {"auprc": 0.02535291980184314, "auroc": 0.8213576981396086, "f1_bin": 0.06780476628727418}}, "D-6": {"Artículo · umbral estático": {"auprc": 0.05243395535229754, "auroc": 0.6473101046958097, "f1_bin": 0.06059546165759684}, "Artículo · umbral dinámico (usa etiquetas)": {"auprc": 0.05243395535229754, "auroc": 0.6473101046958097, "f1_bin": 0.06559528184352278}, "M1 · umbral IQR": {"auprc": 0.05243395535229754, "auroc": 0.6473101046958097, "f1_bin": 0.05366038139963471}, "M2 · reponderación suave": {"auprc": 0.055490883630056095, "auroc": 0.7098684346505768, "f1_bin": 0.05975850266495941}, "M2 · recorte z>2": {"auprc": 0.07716526177926918, "auroc": 0.7188873784709802, "f1_bin": 0.07180584168463663}, "M3 · proyección al objetivo": {"auprc": 0.03216957745402203, "auroc": 0.6583951412240744, "f1_bin": 0.05614771206825327}, "M1+M3": {"auprc": 0.03216957745402203, "auroc": 0.6583951412240744, "f1_bin": 0.05605341945716674}, "M2+M3 (recorte z>2)": {"auprc": 0.05044488057477999, "auroc": 0.7359066084758293, "f1_bin": 0.06810079676775208}, "M1+M2 (recorte adaptativo, sin M3)": {"auprc": 0.07904616506379096, "auroc": 0.7188857885573918, "f1_bin": 0.06725201068495709}, "M1+M2+M3 con recorte z>2": {"auprc": 0.05044488057477999, "auroc": 0.7359066084758293, "f1_bin": 0.06007224490180918}, "**Método final** (recorte adaptativo + proyección + IQR)": {"auprc": 0.051835389680699895, "auroc": 0.727889946181425, "f1_bin": 0.05855114474853336}, "Error del LSTM (PD) + IQR": {"auprc": 0.02267401003970112, "auroc": 0.5972322784256676, "f1_bin": 0.05470473823463882}, "LOF + IQR": {"auprc": 0.0222130071170339, "auroc": 0.5996176257820387, "f1_bin": 0.0385852090032154}, "IForest + IQR": {"auprc": 0.029159924896163897, "auroc": 0.7958528693965483, "f1_bin": 0.09082171333897095}}, "D-4": {"Artículo · umbral estático": {"auprc": 0.43773895557580067, "auroc": 0.6176393058156329, "f1_bin": 0.19769598791771092}, "Artículo · umbral dinámico (usa etiquetas)": {"auprc": 0.43773895557580067, "auroc": 0.6176393058156329, "f1_bin": 0.7889306516099879}, "M1 · umbral IQR": {"auprc": 0.43773895557580067, "auroc": 0.6176393058156329, "f1_bin": 0.3011682513580498}, "M2 · reponderación suave": {"auprc": 0.4632225755087519, "auroc": 0.6736389284251312, "f1_bin": 0.2119879405784623}, "M2 · recorte z>2": {"auprc": 0.4809891936419451, "auroc": 0.6923924404351792, "f1_bin": 0.23589265138323556}, "M3 · proyección al objetivo": {"auprc": 0.46180274150908973, "auroc": 0.6409264889227202, "f1_bin": 0.2125115401619711}, "M1+M3": {"auprc": 0.46180274150908973, "auroc": 0.6409264889227202, "f1_bin": 0.3429874400845364}, "M2+M3 (recorte z>2)": {"auprc": 0.5114592888511786, "auroc": 0.7275870672129866, "f1_bin": 0.22555183814225802}, "M1+M2 (recorte adaptativo, sin M3)": {"auprc": 0.49475921272181866, "auroc": 0.7001816162045676, "f1_bin": 0.27437873154398884}, "M1+M2+M3 con recorte z>2": {"auprc": 0.5114592888511786, "auroc": 0.7275870672129866, "f1_bin": 0.34140567340223765}, "**Método final** (recorte adaptativo + proyección + IQR)": {"auprc": 0.5293375471029194, "auroc": 0.7359922627808613, "f1_bin": 0.3332469991464561}, "Error del LSTM (PD) + IQR": {"auprc": 0.4105748438990866, "auroc": 0.5032062606847495, "f1_bin": 0.3607772139577769}, "LOF + IQR": {"auprc": 0.5447884642245696, "auroc": 0.5693151659227322, "f1_bin": 0.6580721935739786}, "IForest + IQR": {"auprc": 0.6408769022652214, "auroc": 0.8484546501422294, "f1_bin": 0.13073676257913736}}, "D-7": {"Artículo · umbral estático": {"auprc": 0.45959514040673116, "auroc": 0.6959388150140841, "f1_bin": 0.2430025021007361}, "Artículo · umbral dinámico (usa etiquetas)": {"auprc": 0.45959514040673116, "auroc": 0.6959388150140841, "f1_bin": 0.8352195057864387}, "M1 · umbral IQR": {"auprc": 0.45959514040673116, "auroc": 0.6959388150140841, "f1_bin": 0.31471239070975043}, "M2 · reponderación suave": {"auprc": 0.46756597385307347, "auroc": 0.7001196618082666, "f1_bin": 0.2459719313124651}, "M2 · recorte z>2": {"auprc": 0.48891808633922496, "auroc": 0.734422284317335, "f1_bin": 0.2620815923830414}, "M3 · proyección al objetivo": {"auprc": 0.48653373833342445, "auroc": 0.7260677481698726, "f1_bin": 0.22559897635098203}, "M1+M3": {"auprc": 0.48653373833342445, "auroc": 0.7260677481698726, "f1_bin": 0.3659019502491448}, "M2+M3 (recorte z>2)": {"auprc": 0.5219976471336449, "auroc": 0.7729970659036709, "f1_bin": 0.25204192324584124}, "M1+M2 (recorte adaptativo, sin M3)": {"auprc": 0.49534662073556934, "auroc": 0.7314365256554872, "f1_bin": 0.2852514765331093}, "M1+M2+M3 con recorte z>2": {"auprc": 0.5219976471336449, "auroc": 0.7729970659036709, "f1_bin": 0.3586674306376727}, "**Método final** (recorte adaptativo + proyección + IQR)": {"auprc": 0.528595009049153, "auroc": 0.7687834312135469, "f1_bin": 0.3597112124548286}, "Error del LSTM (PD) + IQR": {"auprc": 0.4579182008456783, "auroc": 0.6599943588762074, "f1_bin": 0.3534265744236903}, "LOF + IQR": {"auprc": 0.4823406134472929, "auroc": 0.6564769576225723, "f1_bin": 0.8100156494522691}, "IForest + IQR": {"auprc": 0.6581374071559707, "auroc": 0.857290853328282, "f1_bin": 0.1587400297769931}}, "D-8": {"Artículo · umbral estático": {"auprc": 0.011522256578573462, "auroc": 0.5772685027768333, "f1_bin": 0.025731748638152564}, "Artículo · umbral dinámico (usa etiquetas)": {"auprc": 0.011522256578573462, "auroc": 0.5772685027768333, "f1_bin": 0.025731748638152564}, "M1 · umbral IQR": {"auprc": 0.011522256578573462, "auroc": 0.5772685027768333, "f1_bin": 0.025731748638152564}, "M2 · reponderación suave": {"auprc": 0.01145236561444836, "auroc": 0.5772931857392927, "f1_bin": 0.025738231778555266}, "M2 · recorte z>2": {"auprc": 0.01243749711051562, "auroc": 0.5784044227838856, "f1_bin": 0.0257056516967141}, "M3 · proyección al objetivo": {"auprc": 0.00835693873952868, "auroc": 0.5405206090143186, "f1_bin": 0.025710849996564263}, "M1+M3": {"auprc": 0.00835693873952868, "auroc": 0.5405206090143186, "f1_bin": 0.025710849996564263}, "M2+M3 (recorte z>2)": {"auprc": 0.00903810808266214, "auroc": 0.540552847985694, "f1_bin": 0.025700463877307178}, "M1+M2 (recorte adaptativo, sin M3)": {"auprc": 0.01243681266695944, "auroc": 0.5780447567594796, "f1_bin": 0.0256991685563114}, "M1+M2+M3 con recorte z>2": {"auprc": 0.00903810808266214, "auroc": 0.540552847985694, "f1_bin": 0.025700463877307178}, "**Método final** (recorte adaptativo + proyección + IQR)": {"auprc": 0.00903795823347232, "auroc": 0.540436989182314, "f1_bin": 0.0256991685563114}, "Error del LSTM (PD) + IQR": {"auprc": 0.0178624379351164, "auroc": 0.5405533517196218, "f1_bin": 0.0256991685563114}, "LOF + IQR": {"auprc": 0.0161847281025733, "auroc": 0.5072827332602919, "f1_bin": 0.0256991685563114}, "IForest + IQR": {"auprc": 0.0066277660772568, "auroc": 0.5046320853325273, "f1_bin": 0.0256991685563114}}, "D-9": {"Artículo · umbral estático": {"auprc": 0.6993954375852824, "auroc": 0.8663020189996764, "f1_bin": 0.8934306569343067}, "Artículo · umbral dinámico (usa etiquetas)": {"auprc": 0.6993954375852824, "auroc": 0.8663020189996764, "f1_bin": 0.5276041693630301}, "M1 · umbral IQR": {"auprc": 0.6993954375852824, "auroc": 0.8663020189996764, "f1_bin": 0.5562903007498914}, "M2 · reponderación suave": {"auprc": 0.7012566946204448, "auroc": 0.8667232397041096, "f1_bin": 0.8934306569343067}, "M2 · recorte z>2": {"auprc": 0.7270673558310701, "auroc": 0.88250104768805, "f1_bin": 0.8869309838472834}, "M3 · proyección al objetivo": {"auprc": 0.7224101925434177, "auroc": 0.8688594364687473, "f1_bin": 0.8912725499756217}, "M1+M3": {"auprc": 0.7224101925434177, "auroc": 0.8688594364687473, "f1_bin": 0.555701549215148}, "M2+M3 (recorte z>2)": {"auprc": 0.7521305512219734, "auroc": 0.8863884114480228, "f1_bin": 0.8892144945346849}, "M1+M2 (recorte adaptativo, sin M3)": {"auprc": 0.7270714003379317, "auroc": 0.8825535979421417, "f1_bin": 0.4720589980234663}, "M1+M2+M3 con recorte z>2": {"auprc": 0.7521305512219734, "auroc": 0.8863884114480228, "f1_bin": 0.47184305416398387}, "**Método final** (recorte adaptativo + proyección + IQR)": {"auprc": 0.7521313735561265, "auroc": 0.8864041497056947, "f1_bin": 0.47182346586455576}, "Error del LSTM (PD) + IQR": {"auprc": 0.24874888980606785, "auroc": 0.2556811087411977, "f1_bin": 0.18724270125460615}, "LOF + IQR": {"auprc": 0.1546938775510204, "auroc": 0.4932595089070775, "f1_bin": 0.47168637212196635}, "IForest + IQR": {"auprc": 0.7648562418607523, "auroc": 0.889350478577126, "f1_bin": 0.47168637212196635}}, "E-10": {"Artículo · umbral estático": {"auprc": 0.07544358252865838, "auroc": 0.7062069859571661, "f1_bin": 0.12281259707598083}, "Artículo · umbral dinámico (usa etiquetas)": {"auprc": 0.07544358252865838, "auroc": 0.7062069859571661, "f1_bin": 0.14374512033477888}, "M1 · umbral IQR": {"auprc": 0.07544358252865838, "auroc": 0.7062069859571661, "f1_bin": 0.11897022688999903}, "M2 · reponderación suave": {"auprc": 0.0756807086733911, "auroc": 0.7069922865454934, "f1_bin": 0.12490143102905085}, "M2 · recorte z>2": {"auprc": 0.08564332857231863, "auroc": 0.7425390724790197, "f1_bin": 0.10618165854465696}, "M3 · proyección al objetivo": {"auprc": 0.07716944042790386, "auroc": 0.7424355146814912, "f1_bin": 0.10974243954353033}, "M1+M3": {"auprc": 0.07716944042790386, "auroc": 0.7424355146814912, "f1_bin": 0.1143153872456171}, "M2+M3 (recorte z>2)": {"auprc": 0.08474359195530845, "auroc": 0.7619107099962252, "f1_bin": 0.11592731635037437}, "M1+M2 (recorte adaptativo, sin M3)": {"auprc": 0.07519031965558862, "auroc": 0.7045961970343426, "f1_bin": 0.11039335349386126}, "M1+M2+M3 con recorte z>2": {"auprc": 0.08474359195530845, "auroc": 0.7619107099962252, "f1_bin": 0.3869304797885329}, "**Método final** (recorte adaptativo + proyección + IQR)": {"auprc": 0.07578819144117703, "auroc": 0.7404778681430134, "f1_bin": 0.108167770419426}, "Error del LSTM (PD) + IQR": {"auprc": 0.0507996906832584, "auroc": 0.44649386698389815, "f1_bin": 0.07638736563088308}, "LOF + IQR": {"auprc": 0.0393940990907793, "auroc": 0.5092584331354449, "f1_bin": 0.146098003629764}, "IForest + IQR": {"auprc": 0.12201834949696935, "auroc": 0.7886919155523527, "f1_bin": 0.141352063213345}}, "E-11": {"Artículo · umbral estático": {"auprc": 0.06757189107687846, "auroc": 0.7382178632175835, "f1_bin": 0.10976447070957743}, "Artículo · umbral dinámico (usa etiquetas)": {"auprc": 0.06757189107687846, "auroc": 0.7382178632175835, "f1_bin": 0.13138375586433743}, "M1 · umbral IQR": {"auprc": 0.06757189107687846, "auroc": 0.7382178632175835, "f1_bin": 0.1047200840987266}, "M2 · reponderación suave": {"auprc": 0.06636550245198965, "auroc": 0.7335115055761697, "f1_bin": 0.1057918953951597}, "M2 · recorte z>2": {"auprc": 0.08452837291590558, "auroc": 0.7852585258525853, "f1_bin": 0.09895393237397802}, "M3 · proyección al objetivo": {"auprc": 0.08777249019485144, "auroc": 0.776337856007823, "f1_bin": 0.10857173320378329}, "M1+M3": {"auprc": 0.08777249019485144, "auroc": 0.776337856007823, "f1_bin": 0.10626818034071914}, "M2+M3 (recorte z>2)": {"auprc": 0.10785218913207913, "auroc": 0.8111143845081308, "f1_bin": 0.11481359127112431}, "M1+M2 (recorte adaptativo, sin M3)": {"auprc": 0.06738240703753096, "auroc": 0.7333279881660483, "f1_bin": 0.09509336485105088}, "M1+M2+M3 con recorte z>2": {"auprc": 0.10785218913207913, "auroc": 0.8111143845081308, "f1_bin": 0.3872476546156906}, "**Método final** (recorte adaptativo + proyección + IQR)": {"auprc": 0.09068717539544482, "auroc": 0.7771095075609257, "f1_bin": 0.0977272727272727}, "Error del LSTM (PD) + IQR": {"auprc": 0.08436707083328673, "auroc": 0.7586280737301603, "f1_bin": 0.10893538232860736}, "LOF + IQR": {"auprc": 0.0438869320497719, "auroc": 0.6016460214759706, "f1_bin": 0.1302140807768704}, "IForest + IQR": {"auprc": 0.12047771373515477, "auroc": 0.809760606945817, "f1_bin": 0.1301566291639091}}, "E-12": {"Artículo · umbral estático": {"auprc": 0.07183914665552923, "auroc": 0.4186912680120848, "f1_bin": 0.08611332645044406}, "Artículo · umbral dinámico (usa etiquetas)": {"auprc": 0.07183914665552923, "auroc": 0.4186912680120848, "f1_bin": 0.19491474191732203}, "M1 · umbral IQR": {"auprc": 0.07183914665552923, "auroc": 0.4186912680120848, "f1_bin": 0.11356513630093805}, "M2 · reponderación suave": {"auprc": 0.08862935781557603, "auroc": 0.5654794430026783, "f1_bin": 0.08388231848149161}, "M2 · recorte z>2": {"auprc": 0.09123601141731304, "auroc": 0.5711660134338944, "f1_bin": 0.06981048071420046}, "M3 · proyección al objetivo": {"auprc": 0.06984767888306116, "auroc": 0.4295973461743876, "f1_bin": 0.08939829891265569}, "M1+M3": {"auprc": 0.06984767888306116, "auroc": 0.4295973461743876, "f1_bin": 0.13362005896415}, "M2+M3 (recorte z>2)": {"auprc": 0.09302826885982142, "auroc": 0.5982028291561723, "f1_bin": 0.08259595555089949}, "M1+M2 (recorte adaptativo, sin M3)": {"auprc": 0.09395912034104323, "auroc": 0.575386054373341, "f1_bin": 0.08349010712404299}, "M1+M2+M3 con recorte z>2": {"auprc": 0.09302826885982142, "auroc": 0.5982028291561723, "f1_bin": 0.13403144488347957}, "**Método final** (recorte adaptativo + proyección + IQR)": {"auprc": 0.09696112318380343, "auroc": 0.6020926634833571, "f1_bin": 0.1361671867819026}, "Error del LSTM (PD) + IQR": {"auprc": 0.07218036739770844, "auroc": 0.4377365446402779, "f1_bin": 0.1338701261599881}, "LOF + IQR": {"auprc": 0.0899569254651784, "auroc": 0.6217615960862984, "f1_bin": 0.2553571428571428}, "IForest + IQR": {"auprc": 0.1866585174056321, "auroc": 0.6744072594590141, "f1_bin": 0.241908713692946}}, "E-1": {"Artículo · umbral estático": {"auprc": 0.08575908291575467, "auroc": 0.5754793813414478, "f1_bin": 0.07526478347312789}, "Artículo · umbral dinámico (usa etiquetas)": {"auprc": 0.08575908291575467, "auroc": 0.5754793813414478, "f1_bin": 0.2158734584800855}, "M1 · umbral IQR": {"auprc": 0.08575908291575467, "auroc": 0.5754793813414478, "f1_bin": 0.09501873275919324}, "M2 · reponderación suave": {"auprc": 0.08641518646949552, "auroc": 0.5790498819391616, "f1_bin": 0.07281887786700819}, "M2 · recorte z>2": {"auprc": 0.09246479599339848, "auroc": 0.5932145644592418, "f1_bin": 0.06549757027856803}, "M3 · proyección al objetivo": {"auprc": 0.08533428697618169, "auroc": 0.5978591963959335, "f1_bin": 0.07083961583423647}, "M1+M3": {"auprc": 0.08533428697618169, "auroc": 0.5978591963959335, "f1_bin": 0.09964961924528486}, "M2+M3 (recorte z>2)": {"auprc": 0.09345771987892766, "auroc": 0.6162529515209595, "f1_bin": 0.07388956726002818}, "M1+M2 (recorte adaptativo, sin M3)": {"auprc": 0.08762716825384724, "auroc": 0.5828980230984302, "f1_bin": 0.10559828612952124}, "M1+M2+M3 con recorte z>2": {"auprc": 0.09345771987892766, "auroc": 0.6162529515209595, "f1_bin": 0.15579324598450547}, "**Método final** (recorte adaptativo + proyección + IQR)": {"auprc": 0.09002903296039237, "auroc": 0.6119530177140654, "f1_bin": 0.10800727087390596}, "Error del LSTM (PD) + IQR": {"auprc": 0.0490481548224021, "auroc": 0.2929503354113358, "f1_bin": 0.11147131465219337}, "LOF + IQR": {"auprc": 0.0590903918898974, "auroc": 0.2959283830110947, "f1_bin": 0.0261194029850746}, "IForest + IQR": {"auprc": 0.19304186241003923, "auroc": 0.6938014355012398, "f1_bin": 0.21402991362966078}}, "E-13": {"Artículo · umbral estático": {"auprc": 0.029282079140465656, "auroc": 0.47222996031313375, "f1_bin": 0.07087157841471022}, "Artículo · umbral dinámico (usa etiquetas)": {"auprc": 0.029282079140465656, "auroc": 0.47222996031313375, "f1_bin": 0.10243247081291684}, "M1 · umbral IQR": {"auprc": 0.029282079140465656, "auroc": 0.47222996031313375, "f1_bin": 0.08906001057329857}, "M2 · reponderación suave": {"auprc": 0.028726866437812935, "auroc": 0.46394882502162416, "f1_bin": 0.07218875443413401}, "M2 · recorte z>2": {"auprc": 0.030413891141331716, "auroc": 0.47539350690885823, "f1_bin": 0.10178431699176985}, "M3 · proyección al objetivo": {"auprc": 0.02888131889026862, "auroc": 0.46790247461421597, "f1_bin": 0.0748621345990865}, "M1+M3": {"auprc": 0.02888131889026862, "auroc": 0.46790247461421597, "f1_bin": 0.08653836723607343}, "M2+M3 (recorte z>2)": {"auprc": 0.029706332855471564, "auroc": 0.47420108375673264, "f1_bin": 0.07728986112805111}, "M1+M2 (recorte adaptativo, sin M3)": {"auprc": 0.03031405711575722, "auroc": 0.473859184311331, "f1_bin": 0.09255853868244428}, "M1+M2+M3 con recorte z>2": {"auprc": 0.029706332855471564, "auroc": 0.47420108375673264, "f1_bin": 0.09301374198491186}, "**Método final** (recorte adaptativo + proyección + IQR)": {"auprc": 0.02964023546862992, "auroc": 0.47077304891079175, "f1_bin": 0.09657191322965081}, "Error del LSTM (PD) + IQR": {"auprc": 0.03047907045529924, "auroc": 0.4949017641030114, "f1_bin": 0.09884057381764352}, "LOF + IQR": {"auprc": 0.0319293776997249, "auroc": 0.5147471961156297, "f1_bin": 0.11507750117426022}, "IForest + IQR": {"auprc": 0.0307234343390373, "auroc": 0.4886657562673994, "f1_bin": 0.09961826480404565}}, "E-2": {"Artículo · umbral estático": {"auprc": 0.20686806389258314, "auroc": 0.6483950134916527, "f1_bin": 0.0082116943869523}, "Artículo · umbral dinámico (usa etiquetas)": {"auprc": 0.20686806389258314, "auroc": 0.6483950134916527, "f1_bin": 0.6147051827396095}, "M1 · umbral IQR": {"auprc": 0.20686806389258314, "auroc": 0.6483950134916527, "f1_bin": 0.06006447537067925}, "M2 · reponderación suave": {"auprc": 0.21441677886322794, "auroc": 0.6713877773530555, "f1_bin": 0.00844088635855214}, "M2 · recorte z>2": {"auprc": 0.21706282261627657, "auroc": 0.6750977197252316, "f1_bin": 0.009149957401695781}, "M3 · proyección al objetivo": {"auprc": 0.2069220006172622, "auroc": 0.6411215672090748, "f1_bin": 0.00602047046950188}, "M1+M3": {"auprc": 0.2069220006172622, "auroc": 0.6411215672090748, "f1_bin": 0.2917198572440699}, "M2+M3 (recorte z>2)": {"auprc": 0.2233190903673164, "auroc": 0.6879263418670254, "f1_bin": 0.0325451045066526}, "M1+M2 (recorte adaptativo, sin M3)": {"auprc": 0.21604375468013287, "auroc": 0.6735385588689269, "f1_bin": 0.039948922577836554}, "M1+M2+M3 con recorte z>2": {"auprc": 0.2233190903673164, "auroc": 0.6879263418670254, "f1_bin": 0.31602064163828025}, "**Método final** (recorte adaptativo + proyección + IQR)": {"auprc": 0.2227761178109861, "auroc": 0.6853038666124743, "f1_bin": 0.32293882551300745}, "Error del LSTM (PD) + IQR": {"auprc": 0.11560954701649787, "auroc": 0.29558276399043243, "f1_bin": 0.13089025384180106}, "LOF + IQR": {"auprc": 0.1646187398194787, "auroc": 0.380912728846008, "f1_bin": 0.4941385435168738}, "IForest + IQR": {"auprc": 0.2824359806185379, "auroc": 0.745792265646043, "f1_bin": 0.2182786588228373}}, "E-3": {"Artículo · umbral estático": {"auprc": 0.2984182725419787, "auroc": 0.31402715066118425, "f1_bin": 0.07281726138900754}, "Artículo · umbral dinámico (usa etiquetas)": {"auprc": 0.2984182725419787, "auroc": 0.31402715066118425, "f1_bin": 0.5351197104830689}, "M1 · umbral IQR": {"auprc": 0.2984182725419787, "auroc": 0.31402715066118425, "f1_bin": 0.11391106054832004}, "M2 · reponderación suave": {"auprc": 0.354823350498146, "auroc": 0.46141670805173496, "f1_bin": 0.07188537671475846}, "M2 · recorte z>2": {"auprc": 0.3844739206979618, "auroc": 0.5344926293450466, "f1_bin": 0.07209265148121999}, "M3 · proyección al objetivo": {"auprc": 0.3066308316756735, "auroc": 0.331926993654975, "f1_bin": 0.06976070446079523}, "M1+M3": {"auprc": 0.3066308316756735, "auroc": 0.331926993654975, "f1_bin": 0.1323242202701739}, "M2+M3 (recorte z>2)": {"auprc": 0.4050573152694417, "auroc": 0.5641937931651403, "f1_bin": 0.0705018459264175}, "M1+M2 (recorte adaptativo, sin M3)": {"auprc": 0.3558516555503367, "auroc": 0.45280010240876517, "f1_bin": 0.0965058236272878}, "M1+M2+M3 con recorte z>2": {"auprc": 0.4050573152694417, "auroc": 0.5641937931651403, "f1_bin": 0.103968508013582}, "**Método final** (recorte adaptativo + proyección + IQR)": {"auprc": 0.37725650790621973, "auroc": 0.48017750030074857, "f1_bin": 0.08246612765749099}, "Error del LSTM (PD) + IQR": {"auprc": 0.23468003219072914, "auroc": 0.08021416387253187, "f1_bin": 0.08533689567421228}, "LOF + IQR": {"auprc": 0.41608954789777186, "auroc": 0.545442347258252, "f1_bin": 0.8713305898491084}, "IForest + IQR": {"auprc": 0.5252779391561034, "auroc": 0.7434535258535864, "f1_bin": 0.02943839849713048}}, "E-5": {"Artículo · umbral estático": {"auprc": 0.07030585857531886, "auroc": 0.4742728595068576, "f1_bin": 0.0626063224776964}, "Artículo · umbral dinámico (usa etiquetas)": {"auprc": 0.07030585857531886, "auroc": 0.4742728595068576, "f1_bin": 0.14457713858951476}, "M1 · umbral IQR": {"auprc": 0.07030585857531886, "auroc": 0.4742728595068576, "f1_bin": 0.09989458703231574}, "M2 · reponderación suave": {"auprc": 0.07044750063689384, "auroc": 0.47393137780290406, "f1_bin": 0.06380553223545857}, "M2 · recorte z>2": {"auprc": 0.06838112471175464, "auroc": 0.4738971040011621, "f1_bin": 0.06291302038117702}, "M3 · proyección al objetivo": {"auprc": 0.04966374215076964, "auroc": 0.48034686029597956, "f1_bin": 0.07126226755735605}, "M1+M3": {"auprc": 0.04966374215076964, "auroc": 0.48034686029597956, "f1_bin": 0.10553823157744693}, "M2+M3 (recorte z>2)": {"auprc": 0.050068229931306926, "auroc": 0.48141562971730983, "f1_bin": 0.05737612624624451}, "M1+M2 (recorte adaptativo, sin M3)": {"auprc": 0.070521308067973, "auroc": 0.4742835774311137, "f1_bin": 0.06094704439106542}, "M1+M2+M3 con recorte z>2": {"auprc": 0.050068229931306926, "auroc": 0.48141562971730983, "f1_bin": 0.1093225860792072}, "**Método final** (recorte adaptativo + proyección + IQR)": {"auprc": 0.051027228727014054, "auroc": 0.48273825297834616, "f1_bin": 0.11381134593724}, "Error del LSTM (PD) + IQR": {"auprc": 0.050495641934614956, "auroc": 0.49642021329847064, "f1_bin": 0.11570494984514297}, "LOF + IQR": {"auprc": 0.0404716110443647, "auroc": 0.5124793149950827, "f1_bin": 0.1443020903573836}, "IForest + IQR": {"auprc": 0.05025403768505994, "auroc": 0.4861306919342713, "f1_bin": 0.1443020903573836}}, "E-4": {"Artículo · umbral estático": {"auprc": 0.353951967584993, "auroc": 0.5852537479531245, "f1_bin": 0.04168788256163338}, "Artículo · umbral dinámico (usa etiquetas)": {"auprc": 0.353951967584993, "auroc": 0.5852537479531245, "f1_bin": 0.7114255722377979}, "M1 · umbral IQR": {"auprc": 0.353951967584993, "auroc": 0.5852537479531245, "f1_bin": 0.12655733854221163}, "M2 · reponderación suave": {"auprc": 0.35352068957042204, "auroc": 0.5840339639191671, "f1_bin": 0.04275040374187582}, "M2 · recorte z>2": {"auprc": 0.35649255734598073, "auroc": 0.5825171567968673, "f1_bin": 0.039733355877296324}, "M3 · proyección al objetivo": {"auprc": 0.3701789820527379, "auroc": 0.6137616713255483, "f1_bin": 0.04106321135609834}, "M1+M3": {"auprc": 0.3701789820527379, "auroc": 0.6137616713255483, "f1_bin": 0.1720113145850294}, "M2+M3 (recorte z>2)": {"auprc": 0.37622125658929323, "auroc": 0.6119450369988256, "f1_bin": 0.03649920469892118}, "M1+M2 (recorte adaptativo, sin M3)": {"auprc": 0.3577274785526172, "auroc": 0.5833749966402295, "f1_bin": 0.22215140887400892}, "M1+M2+M3 con recorte z>2": {"auprc": 0.37622125658929323, "auroc": 0.6119450369988256, "f1_bin": 0.2716943442186962}, "**Método final** (recorte adaptativo + proyección + IQR)": {"auprc": 0.3777748969345198, "auroc": 0.613175139404628, "f1_bin": 0.2740085975301948}, "Error del LSTM (PD) + IQR": {"auprc": 0.281500194631193, "auroc": 0.4226380167843792, "f1_bin": 0.20735941205372557}, "LOF + IQR": {"auprc": 0.3928473997101477, "auroc": 0.5248157488979953, "f1_bin": 0.5033540372670807}, "IForest + IQR": {"auprc": 0.38420227123754747, "auroc": 0.6135858648772701, "f1_bin": 0.04350837560695988}}, "E-6": {"Artículo · umbral estático": {"auprc": 0.2092409060691643, "auroc": 0.6027875756096839, "f1_bin": 0.12084539091988748}, "Artículo · umbral dinámico (usa etiquetas)": {"auprc": 0.2092409060691643, "auroc": 0.6027875756096839, "f1_bin": 0.0316691749045094}, "M1 · umbral IQR": {"auprc": 0.2092409060691643, "auroc": 0.6027875756096839, "f1_bin": 0.08887326931418452}, "M2 · reponderación suave": {"auprc": 0.21085726100262475, "auroc": 0.6104524379963618, "f1_bin": 0.12484104032658898}, "M2 · recorte z>2": {"auprc": 0.28382158823190384, "auroc": 0.6078003682505878, "f1_bin": 0.11409971085789257}, "M3 · proyección al objetivo": {"auprc": 0.1128122260868839, "auroc": 0.6271932354289602, "f1_bin": 0.13067336738310786}, "M1+M3": {"auprc": 0.1128122260868839, "auroc": 0.6271932354289602, "f1_bin": 0.03208582323593094}, "M2+M3 (recorte z>2)": {"auprc": 0.19670629237798862, "auroc": 0.6313116523951078, "f1_bin": 0.12193792269452441}, "M1+M2 (recorte adaptativo, sin M3)": {"auprc": 0.28382770990746475, "auroc": 0.6089010529896327, "f1_bin": 0.08859362530778721}, "M1+M2+M3 con recorte z>2": {"auprc": 0.19670629237798862, "auroc": 0.6313116523951078, "f1_bin": 0.0437803380895054}, "**Método final** (recorte adaptativo + proyección + IQR)": {"auprc": 0.19679696021599408, "auroc": 0.632487761953355, "f1_bin": 0.031809781964677444}, "Error del LSTM (PD) + IQR": {"auprc": 0.07782181088747799, "auroc": 0.6199276809086472, "f1_bin": 0.03284111653587248}, "LOF + IQR": {"auprc": 0.1335291474652159, "auroc": 0.6225838176789861, "f1_bin": 0.0314510364546104}, "IForest + IQR": {"auprc": 0.06722369123285105, "auroc": 0.6128277652069747, "f1_bin": 0.0314510364546104}}, "E-7": {"Artículo · umbral estático": {"auprc": 0.06751425390305898, "auroc": 0.6465176348302873, "f1_bin": 0.1607419829918859}, "Artículo · umbral dinámico (usa etiquetas)": {"auprc": 0.06751425390305898, "auroc": 0.6465176348302873, "f1_bin": 0.2047826433280357}, "M1 · umbral IQR": {"auprc": 0.06751425390305898, "auroc": 0.6465176348302873, "f1_bin": 0.1999470665769204}, "M2 · reponderación suave": {"auprc": 0.06920428676895164, "auroc": 0.6654952123630189, "f1_bin": 0.18126975418892988}, "M2 · recorte z>2": {"auprc": 0.116745361113295, "auroc": 0.6613171579592878, "f1_bin": 0.19917011392109826}, "M3 · proyección al objetivo": {"auprc": 0.050115165372713245, "auroc": 0.6229977140653343, "f1_bin": 0.18137571443952785}, "M1+M3": {"auprc": 0.050115165372713245, "auroc": 0.6229977140653343, "f1_bin": 0.1796246968540099}, "M2+M3 (recorte z>2)": {"auprc": 0.09925286873368001, "auroc": 0.6160442953839083, "f1_bin": 0.19851473065970868}, "M1+M2 (recorte adaptativo, sin M3)": {"auprc": 0.1253653046860986, "auroc": 0.6852552537975329, "f1_bin": 0.1662723863875344}, "M1+M2+M3 con recorte z>2": {"auprc": 0.09925286873368001, "auroc": 0.6160442953839083, "f1_bin": 0.17067582928589015}, "**Método final** (recorte adaptativo + proyección + IQR)": {"auprc": 0.10960793148089858, "auroc": 0.6583335968266792, "f1_bin": 0.1787223234247349}, "Error del LSTM (PD) + IQR": {"auprc": 0.050870006902676256, "auroc": 0.6280339403180891, "f1_bin": 0.15532274345385874}, "LOF + IQR": {"auprc": 0.0342991667851374, "auroc": 0.2442552141621958, "f1_bin": 0.1588785046728972}, "IForest + IQR": {"auprc": 0.04634032945074246, "auroc": 0.5706525267972733, "f1_bin": 0.11791273743739125}}, "E-8": {"Artículo · umbral estático": {"auprc": 0.10584742437273978, "auroc": 0.5889499683194073, "f1_bin": 0.14008200966753406}, "Artículo · umbral dinámico (usa etiquetas)": {"auprc": 0.10584742437273978, "auroc": 0.5889499683194073, "f1_bin": 0.282350152827609}, "M1 · umbral IQR": {"auprc": 0.10584742437273978, "auroc": 0.5889499683194073, "f1_bin": 0.38812682582400443}, "M2 · reponderación suave": {"auprc": 0.1162752193720121, "auroc": 0.6662614025152371, "f1_bin": 0.13746382759065903}, "M2 · recorte z>2": {"auprc": 0.11804115341476835, "auroc": 0.6677154468941702, "f1_bin": 0.1458933783519681}, "M3 · proyección al objetivo": {"auprc": 0.10476868195370123, "auroc": 0.5942715737495037, "f1_bin": 0.1331541648341266}, "M1+M3": {"auprc": 0.10476868195370123, "auroc": 0.5942715737495037, "f1_bin": 0.4563325106195393}, "M2+M3 (recorte z>2)": {"auprc": 0.1289902512867365, "auroc": 0.6912881530897601, "f1_bin": 0.3190127072046118}, "M1+M2 (recorte adaptativo, sin M3)": {"auprc": 0.11839786619099135, "auroc": 0.6707331482596679, "f1_bin": 0.5001608920499965}, "M1+M2+M3 con recorte z>2": {"auprc": 0.1289902512867365, "auroc": 0.6912881530897601, "f1_bin": 0.46285409898385954}, "**Método final** (recorte adaptativo + proyección + IQR)": {"auprc": 0.12976659805233767, "auroc": 0.6941124060467005, "f1_bin": 0.44977343083216537}, "Error del LSTM (PD) + IQR": {"auprc": 0.061242062496092896, "auroc": 0.36587089612946155, "f1_bin": 0.14776592596810215}, "LOF + IQR": {"auprc": 0.0849156490055089, "auroc": 0.40973386466727824, "f1_bin": 0.2205513784461152}, "IForest + IQR": {"auprc": 0.16609880054066975, "auroc": 0.7166334022142359, "f1_bin": 0.4718302892582916}}, "E-9": {"Artículo · umbral estático": {"auprc": 0.07259246683264212, "auroc": 0.48788156673914285, "f1_bin": 0.10559199488113338}, "Artículo · umbral dinámico (usa etiquetas)": {"auprc": 0.07259246683264212, "auroc": 0.48788156673914285, "f1_bin": 0.1518523135494863}, "M1 · umbral IQR": {"auprc": 0.07259246683264212, "auroc": 0.48788156673914285, "f1_bin": 0.1460129681070169}, "M2 · reponderación suave": {"auprc": 0.08414398079828708, "auroc": 0.6230741867668445, "f1_bin": 0.10609600625910191}, "M2 · recorte z>2": {"auprc": 0.08732996005326822, "auroc": 0.6273385820081132, "f1_bin": 0.10317074875236407}, "M3 · proyección al objetivo": {"auprc": 0.051135233969481, "auroc": 0.47240722918721145, "f1_bin": 0.09850243848048637}, "M1+M3": {"auprc": 0.051135233969481, "auroc": 0.47240722918721145, "f1_bin": 0.1421219082813669}, "M2+M3 (recorte z>2)": {"auprc": 0.06936741790915182, "auroc": 0.635713815089526, "f1_bin": 0.15423368064520265}, "M1+M2 (recorte adaptativo, sin M3)": {"auprc": 0.0855887356379608, "auroc": 0.6216053283158047, "f1_bin": 0.14309571155461492}, "M1+M2+M3 con recorte z>2": {"auprc": 0.06936741790915182, "auroc": 0.635713815089526, "f1_bin": 0.16370546863563654}, "**Método final** (recorte adaptativo + proyección + IQR)": {"auprc": 0.06819840858299983, "auroc": 0.6277287942269618, "f1_bin": 0.15342408854002895}, "Error del LSTM (PD) + IQR": {"auprc": 0.044369831842216365, "auroc": 0.36741792779957827, "f1_bin": 0.12913916134100079}, "LOF + IQR": {"auprc": 0.0616619958674518, "auroc": 0.4918135723140146, "f1_bin": 0.1659574468085106}, "IForest + IQR": {"auprc": 0.09339721866724757, "auroc": 0.7092597813184189, "f1_bin": 0.1362772845643663}}, "F-1": {"Artículo · umbral estático": {"auprc": 0.010715203782740199, "auroc": 0.4769432964610848, "f1_bin": 0.030142967250054737}, "Artículo · umbral dinámico (usa etiquetas)": {"auprc": 0.010715203782740199, "auroc": 0.4769432964610848, "f1_bin": 0.04700042587031936}, "M1 · umbral IQR": {"auprc": 0.010715203782740199, "auroc": 0.4769432964610848, "f1_bin": 0.026737185964714406}, "M2 · reponderación suave": {"auprc": 0.01049529321203514, "auroc": 0.4654232101718164, "f1_bin": 0.02936686676711704}, "M2 · recorte z>2": {"auprc": 0.01073349600726344, "auroc": 0.4660230143795907, "f1_bin": 0.02879278578889142}, "M3 · proyección al objetivo": {"auprc": 0.01150523746685896, "auroc": 0.5126589639425754, "f1_bin": 0.03601679754004894}, "M1+M3": {"auprc": 0.01150523746685896, "auroc": 0.5126589639425754, "f1_bin": 0.038391266086104145}, "M2+M3 (recorte z>2)": {"auprc": 0.01073146584850586, "auroc": 0.4605352044973592, "f1_bin": 0.04649896876343358}, "M1+M2 (recorte adaptativo, sin M3)": {"auprc": 0.010745918435837061, "auroc": 0.46674451459355526, "f1_bin": 0.034328315518267136}, "M1+M2+M3 con recorte z>2": {"auprc": 0.01073146584850586, "auroc": 0.4605352044973592, "f1_bin": 0.04120548488989108}, "**Método final** (recorte adaptativo + proyección + IQR)": {"auprc": 0.01073060179924322, "auroc": 0.4598366834907292, "f1_bin": 0.04113687931658348}, "Error del LSTM (PD) + IQR": {"auprc": 0.01161532505389214, "auroc": 0.5023543135841114, "f1_bin": 0.04255623916225088}, "LOF + IQR": {"auprc": 0.014190751576174698, "auroc": 0.5464238608585548, "f1_bin": 0.05221932114882501}, "IForest + IQR": {"auprc": 0.00998448295976768, "auroc": 0.44136784904067666, "f1_bin": 0.0114692140081317}}, "F-3": {"Artículo · umbral estático": {"auprc": 0.2908217804480868, "auroc": 0.7411171243205742, "f1_bin": 0.12491434191231347}, "Artículo · umbral dinámico (usa etiquetas)": {"auprc": 0.2908217804480868, "auroc": 0.7411171243205742, "f1_bin": 0.11702351995311852}, "M1 · umbral IQR": {"auprc": 0.2908217804480868, "auroc": 0.7411171243205742, "f1_bin": 0.12505414795834607}, "M2 · reponderación suave": {"auprc": 0.2886142891671224, "auroc": 0.7239631133204185, "f1_bin": 0.1233326973629447}, "M2 · recorte z>2": {"auprc": 0.29990021196898364, "auroc": 0.7104472083650569, "f1_bin": 0.13362914542974272}, "M3 · proyección al objetivo": {"auprc": 0.11133252997010248, "auroc": 0.6319721791890693, "f1_bin": 0.11696663913348866}, "M1+M3": {"auprc": 0.11133252997010248, "auroc": 0.6319721791890693, "f1_bin": 0.10988339101491296}, "M2+M3 (recorte z>2)": {"auprc": 0.12144955507689514, "auroc": 0.5982268054571591, "f1_bin": 0.09611581957439908}, "M1+M2 (recorte adaptativo, sin M3)": {"auprc": 0.30047511274016603, "auroc": 0.7126390085161547, "f1_bin": 0.12436066309040283}, "M1+M2+M3 con recorte z>2": {"auprc": 0.12144955507689514, "auroc": 0.5982268054571591, "f1_bin": 0.11220871505755221}, "**Método final** (recorte adaptativo + proyección + IQR)": {"auprc": 0.12028507328001006, "auroc": 0.5982661967152395, "f1_bin": 0.11179546016217053}, "Error del LSTM (PD) + IQR": {"auprc": 0.12155836412630941, "auroc": 0.6721994726274853, "f1_bin": 0.12001697594810672}, "LOF + IQR": {"auprc": 0.063404500830109, "auroc": 0.8499986771592436, "f1_bin": 0.1551724137931034}, "IForest + IQR": {"auprc": 0.08030389105539695, "auroc": 0.7240927517145485, "f1_bin": 0.0672198447065897}}, "F-2": {"Artículo · umbral estático": {"auprc": 0.3024453173919981, "auroc": 0.42670519346735186, "f1_bin": 0.21695606733969058}, "Artículo · umbral dinámico (usa etiquetas)": {"auprc": 0.3024453173919981, "auroc": 0.42670519346735186, "f1_bin": 0.4581360566664193}, "M1 · umbral IQR": {"auprc": 0.3024453173919981, "auroc": 0.42670519346735186, "f1_bin": 0.14446500267869405}, "M2 · reponderación suave": {"auprc": 0.3217367602242269, "auroc": 0.4892499262034856, "f1_bin": 0.21678024593634496}, "M2 · recorte z>2": {"auprc": 0.3140748034193213, "auroc": 0.473436327162539, "f1_bin": 0.20993109789624972}, "M3 · proyección al objetivo": {"auprc": 0.3191320815374286, "auroc": 0.48472424802857883, "f1_bin": 0.17357347174405313}, "M1+M3": {"auprc": 0.3191320815374286, "auroc": 0.48472424802857883, "f1_bin": 0.21131611332900951}, "M2+M3 (recorte z>2)": {"auprc": 0.3670381905536945, "auroc": 0.5915432809026668, "f1_bin": 0.17445192277016958}, "M1+M2 (recorte adaptativo, sin M3)": {"auprc": 0.31857850950218847, "auroc": 0.48356652228654723, "f1_bin": 0.1413121502786002}, "M1+M2+M3 con recorte z>2": {"auprc": 0.3670381905536945, "auroc": 0.5915432809026668, "f1_bin": 0.26136931695316856}, "**Método final** (recorte adaptativo + proyección + IQR)": {"auprc": 0.38328073381162725, "auroc": 0.6167685018403947, "f1_bin": 0.2743436232728417}, "Error del LSTM (PD) + IQR": {"auprc": 0.2620634173599753, "auroc": 0.29790049820177467, "f1_bin": 0.21979040383364995}, "LOF + IQR": {"auprc": 0.3935679193706161, "auroc": 0.535486424453454, "f1_bin": 0.1910213919855378}, "IForest + IQR": {"auprc": 0.4715397782380567, "auroc": 0.7183825370338017, "f1_bin": 0.04947491264812992}}, "F-4": {"Artículo · umbral estático": {"auprc": 0.05808208791922624, "auroc": 0.7151596570063301, "f1_bin": 0.09928797318495228}, "Artículo · umbral dinámico (usa etiquetas)": {"auprc": 0.05808208791922624, "auroc": 0.7151596570063301, "f1_bin": 0.1281610092896846}, "M1 · umbral IQR": {"auprc": 0.05808208791922624, "auroc": 0.7151596570063301, "f1_bin": 0.07575174813600002}, "M2 · reponderación suave": {"auprc": 0.057319332091347666, "auroc": 0.7679317073585492, "f1_bin": 0.13549537301276823}, "M2 · recorte z>2": {"auprc": 0.0675633430862014, "auroc": 0.7923026234679431, "f1_bin": 0.18758226620701085}, "M3 · proyección al objetivo": {"auprc": 0.04204393069731346, "auroc": 0.7041811387491869, "f1_bin": 0.09412975395631291}, "M1+M3": {"auprc": 0.04204393069731346, "auroc": 0.7041811387491869, "f1_bin": 0.10049691875612765}, "M2+M3 (recorte z>2)": {"auprc": 0.06450977273798811, "auroc": 0.8161718879531339, "f1_bin": 0.11968044653765555}, "M1+M2 (recorte adaptativo, sin M3)": {"auprc": 0.06831968640143342, "auroc": 0.788853130858802, "f1_bin": 0.17871419198235156}, "M1+M2+M3 con recorte z>2": {"auprc": 0.06450977273798811, "auroc": 0.8161718879531339, "f1_bin": 0.14424567736310773}, "**Método final** (recorte adaptativo + proyección + IQR)": {"auprc": 0.062116493589750334, "auroc": 0.8036416506889207, "f1_bin": 0.14938194044974534}, "Error del LSTM (PD) + IQR": {"auprc": 0.037180263716138064, "auroc": 0.6774448076931252, "f1_bin": 0.10008717186646865}, "LOF + IQR": {"auprc": 0.0198280084407019, "auroc": 0.44132801639295477, "f1_bin": 0.0564516129032258}, "IForest + IQR": {"auprc": 0.05998224090509454, "auroc": 0.7999196507144284, "f1_bin": 0.18271694795124022}}, "F-5": {"Artículo · umbral estático": {"auprc": 0.058893702686843796, "auroc": 0.669221885350454, "f1_bin": 0.16064477866981458}, "Artículo · umbral dinámico (usa etiquetas)": {"auprc": 0.058893702686843796, "auroc": 0.669221885350454, "f1_bin": 0.22760133053061166}, "M1 · umbral IQR": {"auprc": 0.058893702686843796, "auroc": 0.669221885350454, "f1_bin": 0.17257159731386246}, "M2 · reponderación suave": {"auprc": 0.0742272649116287, "auroc": 0.6789752396293662, "f1_bin": 0.1808967573399863}, "M2 · recorte z>2": {"auprc": 0.17598730211714614, "auroc": 0.7667635887546724, "f1_bin": 0.42322410144338346}, "M3 · proyección al objetivo": {"auprc": 0.07274392868523546, "auroc": 0.7170782159476159, "f1_bin": 0.11579103569908164}, "M1+M3": {"auprc": 0.07274392868523546, "auroc": 0.7170782159476159, "f1_bin": 0.31302199118837076}, "M2+M3 (recorte z>2)": {"auprc": 0.1575566882346073, "auroc": 0.7555310342870017, "f1_bin": 0.43146414665575283}, "M1+M2 (recorte adaptativo, sin M3)": {"auprc": 0.177155921384654, "auroc": 0.765301774224165, "f1_bin": 0.41094938340737935}, "M1+M2+M3 con recorte z>2": {"auprc": 0.1575566882346073, "auroc": 0.7555310342870017, "f1_bin": 0.37909740875947606}, "**Método final** (recorte adaptativo + proyección + IQR)": {"auprc": 0.16035985262530134, "auroc": 0.7534940737967972, "f1_bin": 0.36843084806017357}, "Error del LSTM (PD) + IQR": {"auprc": 0.058222101797589465, "auroc": 0.6879710759416198, "f1_bin": 0.1211574000538036}, "LOF + IQR": {"auprc": 0.068105937042709, "auroc": 0.5375512477757889, "f1_bin": 0.1451104100946372}, "IForest + IQR": {"auprc": 0.20026446374079088, "auroc": 0.65557627247939, "f1_bin": 0.4038412171089242}}, "F-7": {"Artículo · umbral estático": {"auprc": 0.07698677394623488, "auroc": 0.49489031257865423, "f1_bin": 0.0}, "Artículo · umbral dinámico (usa etiquetas)": {"auprc": 0.07698677394623488, "auroc": 0.49489031257865423, "f1_bin": 0.21963292063737133}, "M1 · umbral IQR": {"auprc": 0.07698677394623488, "auroc": 0.49489031257865423, "f1_bin": 0.0}, "M2 · reponderación suave": {"auprc": 0.07849478790632111, "auroc": 0.5045602852320753, "f1_bin": 0.0}, "M2 · recorte z>2": {"auprc": 0.08041811165570148, "auroc": 0.514507892831058, "f1_bin": 0.0}, "M3 · proyección al objetivo": {"auprc": 0.09906379110322015, "auroc": 0.5982050494147997, "f1_bin": 0.00774783435596454}, "M1+M3": {"auprc": 0.09906379110322015, "auroc": 0.5982050494147997, "f1_bin": 0.04798348986358208}, "M2+M3 (recorte z>2)": {"auprc": 0.11665535439478722, "auroc": 0.6369175212150878, "f1_bin": 0.0583379490573553}, "M1+M2 (recorte adaptativo, sin M3)": {"auprc": 0.08456956811449526, "auroc": 0.5395285970685167, "f1_bin": 0.0}, "M1+M2+M3 con recorte z>2": {"auprc": 0.11665535439478722, "auroc": 0.6369175212150878, "f1_bin": 0.1527484480034194}, "**Método final** (recorte adaptativo + proyección + IQR)": {"auprc": 0.12036932290133895, "auroc": 0.6473942514022004, "f1_bin": 0.2454856977953769}, "Error del LSTM (PD) + IQR": {"auprc": 0.095441582647002, "auroc": 0.5804195818593305, "f1_bin": 0.07293034791365818}, "LOF + IQR": {"auprc": 0.067375006910207, "auroc": 0.3192799300405232, "f1_bin": 0.1617371770872332}, "IForest + IQR": {"auprc": 0.12351010221581524, "auroc": 0.6471069398956887, "f1_bin": 0.0}}, "F-8": {"Artículo · umbral estático": {"auprc": 0.20705233087485203, "auroc": 0.4861995605133876, "f1_bin": 0.15333491360147483}, "Artículo · umbral dinámico (usa etiquetas)": {"auprc": 0.20705233087485203, "auroc": 0.4861995605133876, "f1_bin": 0.42847682721628677}, "M1 · umbral IQR": {"auprc": 0.20705233087485203, "auroc": 0.4861995605133876, "f1_bin": 0.08583163037943141}, "M2 · reponderación suave": {"auprc": 0.20160682638862243, "auroc": 0.4768802648916579, "f1_bin": 0.14254925631189191}, "M2 · recorte z>2": {"auprc": 0.22957458978161357, "auroc": 0.49655198958668656, "f1_bin": 0.300162504828683}, "M3 · proyección al objetivo": {"auprc": 0.21188184178203198, "auroc": 0.49784605571762486, "f1_bin": 0.16841795615731783}, "M1+M3": {"auprc": 0.21188184178203198, "auroc": 0.49784605571762486, "f1_bin": 0.22626559659487405}, "M2+M3 (recorte z>2)": {"auprc": 0.21923644674108492, "auroc": 0.4920995383191136, "f1_bin": 0.1745989810230708}, "M1+M2 (recorte adaptativo, sin M3)": {"auprc": 0.2305111602760627, "auroc": 0.4976348102189635, "f1_bin": 0.1284261467156157}, "M1+M2+M3 con recorte z>2": {"auprc": 0.21923644674108492, "auroc": 0.4920995383191136, "f1_bin": 0.3320884325305702}, "**Método final** (recorte adaptativo + proyección + IQR)": {"auprc": 0.22000400417944954, "auroc": 0.492994307468673, "f1_bin": 0.34249374267639227}, "Error del LSTM (PD) + IQR": {"auprc": 0.20680427507304225, "auroc": 0.4702933643120074, "f1_bin": 0.23516824667700006}, "LOF + IQR": {"auprc": 0.2291196250644297, "auroc": 0.5311464642122333, "f1_bin": 0.5887096774193549}, "IForest + IQR": {"auprc": 0.21481182978866, "auroc": 0.5128956729163876, "f1_bin": 0.3312794295207584}}, "G-2": {"Artículo · umbral estático": {"auprc": 0.007485952723512941, "auroc": 0.578150266946229, "f1_bin": 0.01909522247695498}, "Artículo · umbral dinámico (usa etiquetas)": {"auprc": 0.007485952723512941, "auroc": 0.578150266946229, "f1_bin": 0.022145403736920637}, "M1 · umbral IQR": {"auprc": 0.007485952723512941, "auroc": 0.578150266946229, "f1_bin": 0.01691926599437588}, "M2 · reponderación suave": {"auprc": 0.00751735194299306, "auroc": 0.5820198149798033, "f1_bin": 0.01927834632678248}, "M2 · recorte z>2": {"auprc": 0.01816681036165104, "auroc": 0.5934663486980929, "f1_bin": 0.027079831928656078}, "M3 · proyección al objetivo": {"auprc": 0.0071409732630461605, "auroc": 0.5658422706171582, "f1_bin": 0.0175274807934736}, "M1+M3": {"auprc": 0.0071409732630461605, "auroc": 0.5658422706171582, "f1_bin": 0.03509845667153216}, "M2+M3 (recorte z>2)": {"auprc": 0.01099761879577916, "auroc": 0.5803652172748038, "f1_bin": 0.02028788062627476}, "M1+M2 (recorte adaptativo, sin M3)": {"auprc": 0.01816789392783556, "auroc": 0.5936820492895949, "f1_bin": 0.02421743302205458}, "M1+M2+M3 con recorte z>2": {"auprc": 0.01099761879577916, "auroc": 0.5803652172748038, "f1_bin": 0.026551478744749103}, "**Método final** (recorte adaptativo + proyección + IQR)": {"auprc": 0.011154600725703002, "auroc": 0.5806773802426296, "f1_bin": 0.03342434373657623}, "Error del LSTM (PD) + IQR": {"auprc": 0.018649379388430677, "auroc": 0.5661611323611175, "f1_bin": 0.0354666181394165}, "LOF + IQR": {"auprc": 0.0188476667232874, "auroc": 0.5271601878336828, "f1_bin": 0.0221442073994058}, "IForest + IQR": {"auprc": 0.008803679782903518, "auroc": 0.5616907041083594, "f1_bin": 0.0221442073994058}}, "G-1": {"Artículo · umbral estático": {"auprc": 0.013476647660375662, "auroc": 0.3392284512337022, "f1_bin": 0.03141177800029036}, "Artículo · umbral dinámico (usa etiquetas)": {"auprc": 0.013476647660375662, "auroc": 0.3392284512337022, "f1_bin": 0.03149700090304296}, "M1 · umbral IQR": {"auprc": 0.013476647660375662, "auroc": 0.3392284512337022, "f1_bin": 0.04927439364912734}, "M2 · reponderación suave": {"auprc": 0.01289132223023996, "auroc": 0.31025489552565366, "f1_bin": 0.03021348535743344}, "M2 · recorte z>2": {"auprc": 0.0127411182465386, "auroc": 0.299570765084385, "f1_bin": 0.037085094010258515}, "M3 · proyección al objetivo": {"auprc": 0.01453767513027634, "auroc": 0.35971437380036003, "f1_bin": 0.05377649591165482}, "M1+M3": {"auprc": 0.01453767513027634, "auroc": 0.35971437380036003, "f1_bin": 0.054846112176897145}, "M2+M3 (recorte z>2)": {"auprc": 0.012960786155315939, "auroc": 0.275485077224493, "f1_bin": 0.04923306753973198}, "M1+M2 (recorte adaptativo, sin M3)": {"auprc": 0.0127732033999356, "auroc": 0.3001772234985231, "f1_bin": 0.04719180846560776}, "M1+M2+M3 con recorte z>2": {"auprc": 0.012960786155315939, "auroc": 0.275485077224493, "f1_bin": 0.05376776644897115}, "**Método final** (recorte adaptativo + proyección + IQR)": {"auprc": 0.012976530664111039, "auroc": 0.2754995972193215, "f1_bin": 0.052350873200687244}, "Error del LSTM (PD) + IQR": {"auprc": 0.015889432784037576, "auroc": 0.42332667004803526, "f1_bin": 0.05292435311974231}, "LOF + IQR": {"auprc": 0.0169475734699789, "auroc": 0.44852562381279154, "f1_bin": 0.0088105726872246}, "IForest + IQR": {"auprc": 0.01337103861786738, "auroc": 0.2835821622858052, "f1_bin": 0.05282717677868744}}, "G-3": {"Artículo · umbral estático": {"auprc": 0.01768624359800406, "auroc": 0.5289119126801399, "f1_bin": 0.03166738896612878}, "Artículo · umbral dinámico (usa etiquetas)": {"auprc": 0.01768624359800406, "auroc": 0.5289119126801399, "f1_bin": 0.0255895634721525}, "M1 · umbral IQR": {"auprc": 0.01768624359800406, "auroc": 0.5289119126801399, "f1_bin": 0.03166836722625646}, "M2 · reponderación suave": {"auprc": 0.020281087623949517, "auroc": 0.5288632567378448, "f1_bin": 0.0300464128657133}, "M2 · recorte z>2": {"auprc": 0.02032901643117466, "auroc": 0.547172437662709, "f1_bin": 0.05133455545951626}, "M3 · proyección al objetivo": {"auprc": 0.01175558992313996, "auroc": 0.5184589108091433, "f1_bin": 0.02954176041469664}, "M1+M3": {"auprc": 0.01175558992313996, "auroc": 0.5184589108091433, "f1_bin": 0.028348526697604143}, "M2+M3 (recorte z>2)": {"auprc": 0.01240109304582946, "auroc": 0.5274707938944316, "f1_bin": 0.035060888950835575}, "M1+M2 (recorte adaptativo, sin M3)": {"auprc": 0.020915226558010043, "auroc": 0.5475732221770775, "f1_bin": 0.04706541740113756}, "M1+M2+M3 con recorte z>2": {"auprc": 0.01240109304582946, "auroc": 0.5274707938944316, "f1_bin": 0.0324028098815401}, "**Método final** (recorte adaptativo + proyección + IQR)": {"auprc": 0.012633383981821419, "auroc": 0.5292509994532477, "f1_bin": 0.031146372722459643}, "Error del LSTM (PD) + IQR": {"auprc": 0.02658816375127774, "auroc": 0.5187884670368532, "f1_bin": 0.02451068610598054}, "LOF + IQR": {"auprc": 0.026127464090147802, "auroc": 0.510798358739761, "f1_bin": 0.0255895634721525}, "IForest + IQR": {"auprc": 0.00763053875546302, "auroc": 0.5324530119031495, "f1_bin": 0.0255895634721525}}, "G-4": {"Artículo · umbral estático": {"auprc": 0.27938816425744556, "auroc": 0.7203746593133805, "f1_bin": 0.055905511811023566}, "Artículo · umbral dinámico (usa etiquetas)": {"auprc": 0.27938816425744556, "auroc": 0.7203746593133805, "f1_bin": 0.018100737720807898}, "M1 · umbral IQR": {"auprc": 0.27938816425744556, "auroc": 0.7203746593133805, "f1_bin": 0.023090019703193337}, "M2 · reponderación suave": {"auprc": 0.3000709215169838, "auroc": 0.6843469693286074, "f1_bin": 0.05967146081199032}, "M2 · recorte z>2": {"auprc": 0.26758735645383014, "auroc": 0.6569832759657587, "f1_bin": 0.03246281733487898}, "M3 · proyección al objetivo": {"auprc": 0.1688713737383682, "auroc": 0.6384268915304987, "f1_bin": 0.04132452493990454}, "M1+M3": {"auprc": 0.1688713737383682, "auroc": 0.6384268915304987, "f1_bin": 0.023309697558627263}, "M2+M3 (recorte z>2)": {"auprc": 0.16617047231372115, "auroc": 0.5255194004768546, "f1_bin": 0.01591112649106376}, "M1+M2 (recorte adaptativo, sin M3)": {"auprc": 0.32979694840603774, "auroc": 0.6837745731553871, "f1_bin": 0.043331738506371995}, "M1+M2+M3 con recorte z>2": {"auprc": 0.16617047231372115, "auroc": 0.5255194004768546, "f1_bin": 0.24825396825396823}, "**Método final** (recorte adaptativo + proyección + IQR)": {"auprc": 0.1994137023511491, "auroc": 0.5469428840748465, "f1_bin": 0.0382409177820267}, "Error del LSTM (PD) + IQR": {"auprc": 0.11703449181558248, "auroc": 0.5923353507952551, "f1_bin": 0.03474433562250194}, "LOF + IQR": {"auprc": 0.0683349333514574, "auroc": 0.532258064516129, "f1_bin": 0.0161964472309299}, "IForest + IQR": {"auprc": 0.0682080950829932, "auroc": 0.47741125086904396, "f1_bin": 0.0}}, "G-7": {"Artículo · umbral estático": {"auprc": 0.17351419669984963, "auroc": 0.788741373412621, "f1_bin": 0.08010750237739031}, "Artículo · umbral dinámico (usa etiquetas)": {"auprc": 0.17351419669984963, "auroc": 0.788741373412621, "f1_bin": 0.6185280384882639}, "M1 · umbral IQR": {"auprc": 0.17351419669984963, "auroc": 0.788741373412621, "f1_bin": 0.08010750237739031}, "M2 · reponderación suave": {"auprc": 0.17342556434881037, "auroc": 0.7960505159042286, "f1_bin": 0.08028086788663177}, "M2 · recorte z>2": {"auprc": 0.17688437375896476, "auroc": 0.7962831028078563, "f1_bin": 0.08030789455749385}, "M3 · proyección al objetivo": {"auprc": 0.14760884636678698, "auroc": 0.7204729621348135, "f1_bin": 0.07911832586759163}, "M1+M3": {"auprc": 0.14760884636678698, "auroc": 0.7204729621348135, "f1_bin": 0.07911832586759163}, "M2+M3 (recorte z>2)": {"auprc": 0.1740984720790716, "auroc": 0.7411772549913903, "f1_bin": 0.08026147013145571}, "M1+M2 (recorte adaptativo, sin M3)": {"auprc": 0.17685585003060206, "auroc": 0.7957335232422634, "f1_bin": 0.0802306029305789}, "M1+M2+M3 con recorte z>2": {"auprc": 0.1740984720790716, "auroc": 0.7411772549913903, "f1_bin": 0.08026147013145571}, "**Método final** (recorte adaptativo + proyección + IQR)": {"auprc": 0.17409452283809157, "auroc": 0.7410555019799359, "f1_bin": 0.0802306029305789}, "Error del LSTM (PD) + IQR": {"auprc": 0.1389683065896437, "auroc": 0.7194906279169994, "f1_bin": 0.07912546992002216}, "LOF + IQR": {"auprc": 0.0551272101848132, "auroc": 0.5143533272392408, "f1_bin": 0.0802306029305789}, "IForest + IQR": {"auprc": 0.1552032791647805, "auroc": 0.6693640917515234, "f1_bin": 0.0802306029305789}}, "G-6": {"Artículo · umbral estático": {"auprc": 0.022398328974326778, "auroc": 0.747680936779715, "f1_bin": 0.146788990825688}, "Artículo · umbral dinámico (usa etiquetas)": {"auprc": 0.022398328974326778, "auroc": 0.747680936779715, "f1_bin": 0.11780444549290754}, "M1 · umbral IQR": {"auprc": 0.022398328974326778, "auroc": 0.747680936779715, "f1_bin": 0.146788990825688}, "M2 · reponderación suave": {"auprc": 0.06417513303311134, "auroc": 0.8066995030287643, "f1_bin": 0.1049056278907663}, "M2 · recorte z>2": {"auprc": 0.06443086310587351, "auroc": 0.8069503762515999, "f1_bin": 0.10469209031572228}, "M3 · proyección al objetivo": {"auprc": 0.021468856667261737, "auroc": 0.7391575365041503, "f1_bin": 0.0761904761904762}, "M1+M3": {"auprc": 0.021468856667261737, "auroc": 0.7391575365041503, "f1_bin": 0.0761904761904762}, "M2+M3 (recorte z>2)": {"auprc": 0.06472145578987643, "auroc": 0.7974162620355671, "f1_bin": 0.09336716927248763}, "M1+M2 (recorte adaptativo, sin M3)": {"auprc": 0.06451318945712228, "auroc": 0.8069252190481958, "f1_bin": 0.146788990825688}, "M1+M2+M3 con recorte z>2": {"auprc": 0.06472145578987643, "auroc": 0.7974162620355671, "f1_bin": 0.0761904761904762}, "**Método final** (recorte adaptativo + proyección + IQR)": {"auprc": 0.06496898672100473, "auroc": 0.7975925953964648, "f1_bin": 0.0761904761904762}, "Error del LSTM (PD) + IQR": {"auprc": 0.018326966429142, "auroc": 0.6434280882505378, "f1_bin": 0.0761904761904762}, "LOF + IQR": {"auprc": 0.039524858623207, "auroc": 0.40658815910999396, "f1_bin": 0.1467391304347826}, "IForest + IQR": {"auprc": 0.2142481332265042, "auroc": 0.9788741219728372, "f1_bin": 0.6217821890503374}}, "M-1": {"Artículo · umbral estático": {"auprc": 0.5183398548588343, "auroc": 0.4674779576921787, "f1_bin": 0.24249378383199094}, "Artículo · umbral dinámico (usa etiquetas)": {"auprc": 0.5183398548588343, "auroc": 0.4674779576921787, "f1_bin": 0.13054233280535626}, "M1 · umbral IQR": {"auprc": 0.5183398548588343, "auroc": 0.4674779576921787, "f1_bin": 0.19320704563645436}, "M2 · reponderación suave": {"auprc": 0.5179088209324128, "auroc": 0.4677402463885415, "f1_bin": 0.2488294047957434}, "M2 · recorte z>2": {"auprc": 0.5188144556493928, "auroc": 0.46888263099668104, "f1_bin": 0.253803206625577}, "M3 · proyección al objetivo": {"auprc": 0.49141742173681663, "auroc": 0.42376410619898486, "f1_bin": 0.215771716656647}, "M1+M3": {"auprc": 0.49141742173681663, "auroc": 0.42376410619898486, "f1_bin": 0.21973808393107425}, "M2+M3 (recorte z>2)": {"auprc": 0.48875224954605717, "auroc": 0.41928931417013493, "f1_bin": 0.20743830188445153}, "M1+M2 (recorte adaptativo, sin M3)": {"auprc": 0.5071841983514925, "auroc": 0.4542019670854026, "f1_bin": 0.1715117778167094}, "M1+M2+M3 con recorte z>2": {"auprc": 0.48875224954605717, "auroc": 0.41928931417013493, "f1_bin": 0.23472678876599523}, "**Método final** (recorte adaptativo + proyección + IQR)": {"auprc": 0.4809004829529674, "auroc": 0.4002557434519619, "f1_bin": 0.35036977541241177}, "Error del LSTM (PD) + IQR": {"auprc": 0.5637031590050017, "auroc": 0.6017828607187955, "f1_bin": 0.26875763959910504}, "LOF + IQR": {"auprc": 0.4948080062931292, "auroc": 0.4754892570189764, "f1_bin": 0.2863741339491917}, "IForest + IQR": {"auprc": 0.45082502039398814, "auroc": 0.3147485109568987, "f1_bin": 0.04456619842469186}}, "M-2": {"Artículo · umbral estático": {"auprc": 0.5838015674968267, "auroc": 0.6093029474353019, "f1_bin": 0.27359716043035853}, "Artículo · umbral dinámico (usa etiquetas)": {"auprc": 0.5838015674968267, "auroc": 0.6093029474353019, "f1_bin": 0.2387112474064618}, "M1 · umbral IQR": {"auprc": 0.5838015674968267, "auroc": 0.6093029474353019, "f1_bin": 0.25783204237509105}, "M2 · reponderación suave": {"auprc": 0.5808516651444948, "auroc": 0.5926903189449704, "f1_bin": 0.27785360965128064}, "M2 · recorte z>2": {"auprc": 0.5611338614935791, "auroc": 0.5382225510808434, "f1_bin": 0.2879345068509292}, "M3 · proyección al objetivo": {"auprc": 0.5742468452538907, "auroc": 0.6184322862538693, "f1_bin": 0.2502225034280682}, "M1+M3": {"auprc": 0.5742468452538907, "auroc": 0.6184322862538693, "f1_bin": 0.22609676830964723}, "M2+M3 (recorte z>2)": {"auprc": 0.5272922106798171, "auroc": 0.5033473337707471, "f1_bin": 0.24473477221016973}, "M1+M2 (recorte adaptativo, sin M3)": {"auprc": 0.545231252841645, "auroc": 0.5304424904495306, "f1_bin": 0.17030250628008584}, "M1+M2+M3 con recorte z>2": {"auprc": 0.5272922106798171, "auroc": 0.5033473337707471, "f1_bin": 0.3064793901524554}, "**Método final** (recorte adaptativo + proyección + IQR)": {"auprc": 0.5249508296564974, "auroc": 0.5066909958190255, "f1_bin": 0.37161075326276066}, "Error del LSTM (PD) + IQR": {"auprc": 0.5556952444572407, "auroc": 0.5823812397331456, "f1_bin": 0.2765493498926812}, "LOF + IQR": {"auprc": 0.5355498946245578, "auroc": 0.5376367517069518, "f1_bin": 0.25}, "IForest + IQR": {"auprc": 0.5141844592262342, "auroc": 0.43158982390099754, "f1_bin": 0.10095129916164483}}, "M-3": {"Artículo · umbral estático": {"auprc": 0.12989723729884148, "auroc": 0.39444333655584407, "f1_bin": 0.1809415211540478}, "Artículo · umbral dinámico (usa etiquetas)": {"auprc": 0.12989723729884148, "auroc": 0.39444333655584407, "f1_bin": 0.23850616544574083}, "M1 · umbral IQR": {"auprc": 0.12989723729884148, "auroc": 0.39444333655584407, "f1_bin": 0.15941763221836494}, "M2 · reponderación suave": {"auprc": 0.1284287834442722, "auroc": 0.3890806740394244, "f1_bin": 0.18581133428131374}, "M2 · recorte z>2": {"auprc": 0.1284279508955876, "auroc": 0.386974625979217, "f1_bin": 0.18186447637979625}, "M3 · proyección al objetivo": {"auprc": 0.11033995230930246, "auroc": 0.3886534384767784, "f1_bin": 0.15376559658113598}, "M1+M3": {"auprc": 0.11033995230930246, "auroc": 0.3886534384767784, "f1_bin": 0.1441525168415454}, "M2+M3 (recorte z>2)": {"auprc": 0.1080979340923263, "auroc": 0.3857434245607342, "f1_bin": 0.1523972954418707}, "M1+M2 (recorte adaptativo, sin M3)": {"auprc": 0.12110544678468597, "auroc": 0.3837813056804339, "f1_bin": 0.14431284474506617}, "M1+M2+M3 con recorte z>2": {"auprc": 0.1080979340923263, "auroc": 0.3857434245607342, "f1_bin": 0.14007631498328835}, "**Método final** (recorte adaptativo + proyección + IQR)": {"auprc": 0.10476929908352443, "auroc": 0.3916863557738578, "f1_bin": 0.1700594546663282}, "Error del LSTM (PD) + IQR": {"auprc": 0.09837214304110398, "auroc": 0.3677798056956071, "f1_bin": 0.1345698551698761}, "LOF + IQR": {"auprc": 0.1416950374459724, "auroc": 0.562094386328462, "f1_bin": 0.2845691382765531}, "IForest + IQR": {"auprc": 0.15350474321478164, "auroc": 0.5269197421413366, "f1_bin": 0.05483131839756378}}, "M-4": {"Artículo · umbral estático": {"auprc": 0.13658234752278298, "auroc": 0.3994758530201663, "f1_bin": 0.19270805397348126}, "Artículo · umbral dinámico (usa etiquetas)": {"auprc": 0.13658234752278298, "auroc": 0.3994758530201663, "f1_bin": 0.270325544817125}, "M1 · umbral IQR": {"auprc": 0.13658234752278298, "auroc": 0.3994758530201663, "f1_bin": 0.16038217124235826}, "M2 · reponderación suave": {"auprc": 0.13466659547792645, "auroc": 0.38828266123613037, "f1_bin": 0.19653865445538338}, "M2 · recorte z>2": {"auprc": 0.13293742976379028, "auroc": 0.38766944799418673, "f1_bin": 0.1915922072703266}, "M3 · proyección al objetivo": {"auprc": 0.11803440599778767, "auroc": 0.3967362112442169, "f1_bin": 0.19142559120860117}, "M1+M3": {"auprc": 0.11803440599778767, "auroc": 0.3967362112442169, "f1_bin": 0.18623903713187934}, "M2+M3 (recorte z>2)": {"auprc": 0.11529553962105035, "auroc": 0.38326123749712415, "f1_bin": 0.17631907685206116}, "M1+M2 (recorte adaptativo, sin M3)": {"auprc": 0.12477060708391548, "auroc": 0.3717254481217497, "f1_bin": 0.162587971784297}, "M1+M2+M3 con recorte z>2": {"auprc": 0.11529553962105035, "auroc": 0.38326123749712415, "f1_bin": 0.17510969969533774}, "**Método final** (recorte adaptativo + proyección + IQR)": {"auprc": 0.10832569073172311, "auroc": 0.37297009788177193, "f1_bin": 0.1905157331286107}, "Error del LSTM (PD) + IQR": {"auprc": 0.09807166730969702, "auroc": 0.3390663760054123, "f1_bin": 0.1448313337088753}, "LOF + IQR": {"auprc": 0.1474048394409612, "auroc": 0.5521185697461725, "f1_bin": 0.2521739130434782}, "IForest + IQR": {"auprc": 0.17564784750518708, "auroc": 0.5910266310401618, "f1_bin": 0.14750025635002556}}, "M-5": {"Artículo · umbral estático": {"auprc": 0.17871840853214965, "auroc": 0.496451407731188, "f1_bin": 0.1961536240505463}, "Artículo · umbral dinámico (usa etiquetas)": {"auprc": 0.17871840853214965, "auroc": 0.496451407731188, "f1_bin": 0.31760148937621135}, "M1 · umbral IQR": {"auprc": 0.17871840853214965, "auroc": 0.496451407731188, "f1_bin": 0.1522114354661597}, "M2 · reponderación suave": {"auprc": 0.177620059633242, "auroc": 0.49542326664005243, "f1_bin": 0.18140880240647633}, "M2 · recorte z>2": {"auprc": 0.1809241470487429, "auroc": 0.4980408143932986, "f1_bin": 0.19725821133671528}, "M3 · proyección al objetivo": {"auprc": 0.1491464272288111, "auroc": 0.4927531108118897, "f1_bin": 0.16829977327428325}, "M1+M3": {"auprc": 0.1491464272288111, "auroc": 0.4927531108118897, "f1_bin": 0.14233940631559214}, "M2+M3 (recorte z>2)": {"auprc": 0.14837472916345182, "auroc": 0.4935723420235331, "f1_bin": 0.1724559478034739}, "M1+M2 (recorte adaptativo, sin M3)": {"auprc": 0.17277505161763793, "auroc": 0.5031155483080837, "f1_bin": 0.15812355322432226}, "M1+M2+M3 con recorte z>2": {"auprc": 0.14837472916345182, "auroc": 0.4935723420235331, "f1_bin": 0.1656122221288558}, "**Método final** (recorte adaptativo + proyección + IQR)": {"auprc": 0.1476149754016785, "auroc": 0.4994531128417833, "f1_bin": 0.20044748416659516}, "Error del LSTM (PD) + IQR": {"auprc": 0.15195354347127493, "auroc": 0.5408527921185999, "f1_bin": 0.1766084739727703}, "LOF + IQR": {"auprc": 0.230062847620326, "auroc": 0.5664536406141105, "f1_bin": 0.2809364548494983}, "IForest + IQR": {"auprc": 0.16315981789019324, "auroc": 0.5464698459310784, "f1_bin": 0.1453576030176806}}, "M-6": {"Artículo · umbral estático": {"auprc": 0.39374753694757897, "auroc": 0.8224294556708538, "f1_bin": 0.4849734406938869}, "Artículo · umbral dinámico (usa etiquetas)": {"auprc": 0.39374753694757897, "auroc": 0.8224294556708538, "f1_bin": 0.446788726369018}, "M1 · umbral IQR": {"auprc": 0.39374753694757897, "auroc": 0.8224294556708538, "f1_bin": 0.46370723989333573}, "M2 · reponderación suave": {"auprc": 0.3761667461199942, "auroc": 0.7977550215425694, "f1_bin": 0.44930068750716473}, "M2 · recorte z>2": {"auprc": 0.37508770902511324, "auroc": 0.7927929486012502, "f1_bin": 0.4428867859198323}, "M3 · proyección al objetivo": {"auprc": 0.2564657525986809, "auroc": 0.7849107955579828, "f1_bin": 0.3297209065484469}, "M1+M3": {"auprc": 0.2564657525986809, "auroc": 0.7849107955579828, "f1_bin": 0.35935220067575563}, "M2+M3 (recorte z>2)": {"auprc": 0.230020155517373, "auroc": 0.7559830693610048, "f1_bin": 0.2969580568010047}, "M1+M2 (recorte adaptativo, sin M3)": {"auprc": 0.36308180983053856, "auroc": 0.7737547788093937, "f1_bin": 0.4263806076889975}, "M1+M2+M3 con recorte z>2": {"auprc": 0.230020155517373, "auroc": 0.7559830693610048, "f1_bin": 0.3138136551488536}, "**Método final** (recorte adaptativo + proyección + IQR)": {"auprc": 0.21396124620660265, "auroc": 0.7076822015899022, "f1_bin": 0.3259669754333169}, "Error del LSTM (PD) + IQR": {"auprc": 0.09473610134499895, "auroc": 0.5015337702530493, "f1_bin": 0.15186189066171496}, "LOF + IQR": {"auprc": 0.1414371799665048, "auroc": 0.5753337581163905, "f1_bin": 0.3333333333333333}, "IForest + IQR": {"auprc": 0.5256895297867483, "auroc": 0.7606244310941198, "f1_bin": 0.11360818179565713}}, "M-7": {"Artículo · umbral estático": {"auprc": 0.30226296486612103, "auroc": 0.7661648266958576, "f1_bin": 0.0}, "Artículo · umbral dinámico (usa etiquetas)": {"auprc": 0.30226296486612103, "auroc": 0.7661648266958576, "f1_bin": 0.0}, "M1 · umbral IQR": {"auprc": 0.30226296486612103, "auroc": 0.7661648266958576, "f1_bin": 0.0}, "M2 · reponderación suave": {"auprc": 0.3087867771030381, "auroc": 0.764349563364864, "f1_bin": 0.0}, "M2 · recorte z>2": {"auprc": 0.32575461069491557, "auroc": 0.7648257141033884, "f1_bin": 0.0}, "M3 · proyección al objetivo": {"auprc": 0.19310309137642884, "auroc": 0.7476405994590534, "f1_bin": 0.0}, "M1+M3": {"auprc": 0.19310309137642884, "auroc": 0.7476405994590534, "f1_bin": 0.0}, "M2+M3 (recorte z>2)": {"auprc": 0.18982078577001632, "auroc": 0.7444695337158902, "f1_bin": 0.0}, "M1+M2 (recorte adaptativo, sin M3)": {"auprc": 0.21774213211002164, "auroc": 0.7688469788971956, "f1_bin": 0.0}, "M1+M2+M3 con recorte z>2": {"auprc": 0.18982078577001632, "auroc": 0.7444695337158902, "f1_bin": 0.0}, "**Método final** (recorte adaptativo + proyección + IQR)": {"auprc": 0.15278749791974652, "auroc": 0.753864920453374, "f1_bin": 0.0}, "Error del LSTM (PD) + IQR": {"auprc": 0.06161521793391274, "auroc": 0.6043496615402739, "f1_bin": 0.0}, "LOF + IQR": {"auprc": 0.1586179851323182, "auroc": 0.6959973885340939, "f1_bin": 0.0}, "IForest + IQR": {"auprc": 0.4175238797560173, "auroc": 0.7575970586647163, "f1_bin": 0.0}}, "P-11": {"Artículo · umbral estático": {"auprc": 0.09675171315492548, "auroc": 0.6913026678044147, "f1_bin": 0.05748113571122415}, "Artículo · umbral dinámico (usa etiquetas)": {"auprc": 0.09675171315492548, "auroc": 0.6913026678044147, "f1_bin": 0.20555449679379773}, "M1 · umbral IQR": {"auprc": 0.09675171315492548, "auroc": 0.6913026678044147, "f1_bin": 0.0713959367374076}, "M2 · reponderación suave": {"auprc": 0.11321044565351154, "auroc": 0.7269284023763907, "f1_bin": 0.05914166460910868}, "M2 · recorte z>2": {"auprc": 0.11656521347459765, "auroc": 0.7317249225848347, "f1_bin": 0.06451986664158194}, "M3 · proyección al objetivo": {"auprc": 0.11250597888153979, "auroc": 0.6931880021252274, "f1_bin": 0.1016279069767441}, "M1+M3": {"auprc": 0.11250597888153979, "auroc": 0.6931880021252274, "f1_bin": 0.12553944386959898}, "M2+M3 (recorte z>2)": {"auprc": 0.15279027120825783, "auroc": 0.7850380769911932, "f1_bin": 0.13326161631394623}, "M1+M2 (recorte adaptativo, sin M3)": {"auprc": 0.11480468583353669, "auroc": 0.7295503211991433, "f1_bin": 0.07759723900107532}, "M1+M2+M3 con recorte z>2": {"auprc": 0.15279027120825783, "auroc": 0.7850380769911932, "f1_bin": 0.17742016507264627}, "**Método final** (recorte adaptativo + proyección + IQR)": {"auprc": 0.15290254919886787, "auroc": 0.7884379041822973, "f1_bin": 0.1778079549035088}, "Error del LSTM (PD) + IQR": {"auprc": 0.09808451455306293, "auroc": 0.6514176232873404, "f1_bin": 0.11742974206495002}, "LOF + IQR": {"auprc": 0.0871419711202376, "auroc": 0.5833138789157046, "f1_bin": 0.0528301886792452}, "IForest + IQR": {"auprc": 0.19118976413572678, "auroc": 0.7940960538390945, "f1_bin": 0.2572247555028851}}, "P-10": {"Artículo · umbral estático": {"auprc": 0.20048208476110588, "auroc": 0.7984784307063032, "f1_bin": 0.20359500362193614}, "Artículo · umbral dinámico (usa etiquetas)": {"auprc": 0.20048208476110588, "auroc": 0.7984784307063032, "f1_bin": 0.14169426432261883}, "M1 · umbral IQR": {"auprc": 0.20048208476110588, "auroc": 0.7984784307063032, "f1_bin": 0.19925645799776198}, "M2 · reponderación suave": {"auprc": 0.1304236020017136, "auroc": 0.7456363961640288, "f1_bin": 0.19625694318341044}, "M2 · recorte z>2": {"auprc": 0.1516141774403125, "auroc": 0.7680596580780759, "f1_bin": 0.18242574628480504}, "M3 · proyección al objetivo": {"auprc": 0.15563513196832895, "auroc": 0.8100293322316048, "f1_bin": 0.24407277391600507}, "M1+M3": {"auprc": 0.15563513196832895, "auroc": 0.8100293322316048, "f1_bin": 0.29022962175403205}, "M2+M3 (recorte z>2)": {"auprc": 0.12254458907060768, "auroc": 0.7872478798807148, "f1_bin": 0.18057937138741229}, "M1+M2 (recorte adaptativo, sin M3)": {"auprc": 0.12367203818080126, "auroc": 0.7295334000033464, "f1_bin": 0.1936054958076397}, "M1+M2+M3 con recorte z>2": {"auprc": 0.12254458907060768, "auroc": 0.7872478798807148, "f1_bin": 0.18209634473799738}, "**Método final** (recorte adaptativo + proyección + IQR)": {"auprc": 0.10807367084488528, "auroc": 0.7953984820344908, "f1_bin": 0.18996417537348603}, "Error del LSTM (PD) + IQR": {"auprc": 0.09111007141424998, "auroc": 0.7519404448871951, "f1_bin": 0.29838946472109074}, "LOF + IQR": {"auprc": 0.1115690883040471, "auroc": 0.7219287711995841, "f1_bin": 0.0972727272727272}, "IForest + IQR": {"auprc": 0.1077137779329413, "auroc": 0.8363413350219637, "f1_bin": 0.11804075079964012}}, "P-14": {"Artículo · umbral estático": {"auprc": 0.18516143120501508, "auroc": 0.7554942830221116, "f1_bin": 0.23119992353225483}, "Artículo · umbral dinámico (usa etiquetas)": {"auprc": 0.18516143120501508, "auroc": 0.7554942830221116, "f1_bin": 0.17408726161071253}, "M1 · umbral IQR": {"auprc": 0.18516143120501508, "auroc": 0.7554942830221116, "f1_bin": 0.22701181370992446}, "M2 · reponderación suave": {"auprc": 0.13518685937967154, "auroc": 0.7265928953714603, "f1_bin": 0.22731638132405235}, "M2 · recorte z>2": {"auprc": 0.16434938269538318, "auroc": 0.7875281480486369, "f1_bin": 0.2120160362405877}, "M3 · proyección al objetivo": {"auprc": 0.14716478048882203, "auroc": 0.7667213735402236, "f1_bin": 0.24713717101028237}, "M1+M3": {"auprc": 0.14716478048882203, "auroc": 0.7667213735402236, "f1_bin": 0.2924709579925596}, "M2+M3 (recorte z>2)": {"auprc": 0.12995762549537743, "auroc": 0.7951247626733154, "f1_bin": 0.19544305579291266}, "M1+M2 (recorte adaptativo, sin M3)": {"auprc": 0.12850825262949617, "auroc": 0.7127314199392932, "f1_bin": 0.22137402342950593}, "M1+M2+M3 con recorte z>2": {"auprc": 0.12995762549537743, "auroc": 0.7951247626733154, "f1_bin": 0.1900740564491981}, "**Método final** (recorte adaptativo + proyección + IQR)": {"auprc": 0.11657336312615256, "auroc": 0.7904488750644693, "f1_bin": 0.20271525529938028}, "Error del LSTM (PD) + IQR": {"auprc": 0.08930227304918721, "auroc": 0.6802975402574637, "f1_bin": 0.2994714400927223}, "LOF + IQR": {"auprc": 0.11000955625812621, "auroc": 0.6424829091906609, "f1_bin": 0.1191111111111111}, "IForest + IQR": {"auprc": 0.1237714176697329, "auroc": 0.8189503420040752, "f1_bin": 0.1172862570335568}}, "P-15": {"Artículo · umbral estático": {"auprc": 0.00875411488353004, "auroc": 0.5821645640737524, "f1_bin": 0.0}, "Artículo · umbral dinámico (usa etiquetas)": {"auprc": 0.00875411488353004, "auroc": 0.5821645640737524, "f1_bin": 0.0}, "M1 · umbral IQR": {"auprc": 0.00875411488353004, "auroc": 0.5821645640737524, "f1_bin": 0.0}, "M2 · reponderación suave": {"auprc": 0.007995352584371519, "auroc": 0.5427669101247936, "f1_bin": 0.0}, "M2 · recorte z>2": {"auprc": 0.00776964297447486, "auroc": 0.5257299487546181, "f1_bin": 0.0}, "M3 · proyección al objetivo": {"auprc": 0.00896545734493742, "auroc": 0.5851030866404481, "f1_bin": 0.0}, "M1+M3": {"auprc": 0.00896545734493742, "auroc": 0.5851030866404481, "f1_bin": 0.0}, "M2+M3 (recorte z>2)": {"auprc": 0.00899222462012382, "auroc": 0.5690450652910432, "f1_bin": 0.0}, "M1+M2 (recorte adaptativo, sin M3)": {"auprc": 0.00838752336271098, "auroc": 0.5661474028295623, "f1_bin": 0.0}, "M1+M2+M3 con recorte z>2": {"auprc": 0.00899222462012382, "auroc": 0.5690450652910432, "f1_bin": 0.0}, "**Método final** (recorte adaptativo + proyección + IQR)": {"auprc": 0.009496659667787821, "auroc": 0.6010044775865298, "f1_bin": 0.0}, "Error del LSTM (PD) + IQR": {"auprc": 0.00936766291231528, "auroc": 0.5779457582103275, "f1_bin": 0.0}, "LOF + IQR": {"auprc": 0.0101192939891713, "auroc": 0.6044401314333384, "f1_bin": 0.0}, "IForest + IQR": {"auprc": 0.009908381525681279, "auroc": 0.6251153446720126, "f1_bin": 0.0}}, "P-1": {"Artículo · umbral estático": {"auprc": 0.08366724396109895, "auroc": 0.41785210871421474, "f1_bin": 0.1265520257717222}, "Artículo · umbral dinámico (usa etiquetas)": {"auprc": 0.08366724396109895, "auroc": 0.41785210871421474, "f1_bin": 0.13910518656856816}, "M1 · umbral IQR": {"auprc": 0.08366724396109895, "auroc": 0.41785210871421474, "f1_bin": 0.14366168207447255}, "M2 · reponderación suave": {"auprc": 0.0835988881826615, "auroc": 0.4122157324017259, "f1_bin": 0.12272640904470733}, "M2 · recorte z>2": {"auprc": 0.0836396686015169, "auroc": 0.40569779433539105, "f1_bin": 0.1398492031614635}, "M3 · proyección al objetivo": {"auprc": 0.08690779863268838, "auroc": 0.4497789020203923, "f1_bin": 0.131619182544588}, "M1+M3": {"auprc": 0.08690779863268838, "auroc": 0.4497789020203923, "f1_bin": 0.1537377204930337}, "M2+M3 (recorte z>2)": {"auprc": 0.08763797794047966, "auroc": 0.45181448064063884, "f1_bin": 0.13596089955540328}, "M1+M2 (recorte adaptativo, sin M3)": {"auprc": 0.08513894288004323, "auroc": 0.3962760038628296, "f1_bin": 0.15164154648008915}, "M1+M2+M3 con recorte z>2": {"auprc": 0.08763797794047966, "auroc": 0.45181448064063884, "f1_bin": 0.14981222593905968}, "**Método final** (recorte adaptativo + proyección + IQR)": {"auprc": 0.08608455291581463, "auroc": 0.4341905108032904, "f1_bin": 0.14593560324166263}, "Error del LSTM (PD) + IQR": {"auprc": 0.0900666017118528, "auroc": 0.478120618853452, "f1_bin": 0.14332096194149285}, "LOF + IQR": {"auprc": 0.0900348684546395, "auroc": 0.4626338029335691, "f1_bin": 0.0666666666666666}, "IForest + IQR": {"auprc": 0.09458816635156783, "auroc": 0.5122486762077727, "f1_bin": 0.14870547982390964}}, "P-2": {"Artículo · umbral estático": {"auprc": 0.20490298605872997, "auroc": 0.6061608043624046, "f1_bin": 0.06745087760705674}, "Artículo · umbral dinámico (usa etiquetas)": {"auprc": 0.20490298605872997, "auroc": 0.6061608043624046, "f1_bin": 0.6250798913376623}, "M1 · umbral IQR": {"auprc": 0.20490298605872997, "auroc": 0.6061608043624046, "f1_bin": 0.20529149692301535}, "M2 · reponderación suave": {"auprc": 0.22613980424907082, "auroc": 0.685137711005958, "f1_bin": 0.07198507274968662}, "M2 · recorte z>2": {"auprc": 0.2227345633139052, "auroc": 0.6794180563434207, "f1_bin": 0.06764515757373213}, "M3 · proyección al objetivo": {"auprc": 0.1978290765967421, "auroc": 0.5743249277415066, "f1_bin": 0.13720786674820645}, "M1+M3": {"auprc": 0.1978290765967421, "auroc": 0.5743249277415066, "f1_bin": 0.5882237312196135}, "M2+M3 (recorte z>2)": {"auprc": 0.2448050945370054, "auroc": 0.7183989561422361, "f1_bin": 0.24731718974372505}, "M1+M2 (recorte adaptativo, sin M3)": {"auprc": 0.21156072563371345, "auroc": 0.6530892106119276, "f1_bin": 0.2898662945794334}, "M1+M2+M3 con recorte z>2": {"auprc": 0.2448050945370054, "auroc": 0.7183989561422361, "f1_bin": 0.5814422600237852}, "**Método final** (recorte adaptativo + proyección + IQR)": {"auprc": 0.22586251266713497, "auroc": 0.6856226562233912, "f1_bin": 0.5178297258950874}, "Error del LSTM (PD) + IQR": {"auprc": 0.22484727203029697, "auroc": 0.619345592319988, "f1_bin": 0.6355009597552064}, "LOF + IQR": {"auprc": 0.2086659589933833, "auroc": 0.5544203641522709, "f1_bin": 0.41345589673381583}, "IForest + IQR": {"auprc": 0.2997833019935428, "auroc": 0.7700201536894993, "f1_bin": 0.5173520335779763}}, "P-3": {"Artículo · umbral estático": {"auprc": 0.20578301283183467, "auroc": 0.6241235817450645, "f1_bin": 0.06012145339882036}, "Artículo · umbral dinámico (usa etiquetas)": {"auprc": 0.20578301283183467, "auroc": 0.6241235817450645, "f1_bin": 0.5813193287336212}, "M1 · umbral IQR": {"auprc": 0.20578301283183467, "auroc": 0.6241235817450645, "f1_bin": 0.3876843553724059}, "M2 · reponderación suave": {"auprc": 0.21145128050615075, "auroc": 0.6436204077306827, "f1_bin": 0.06011429952855955}, "M2 · recorte z>2": {"auprc": 0.21963900702944578, "auroc": 0.6590412306444844, "f1_bin": 0.15520467071217575}, "M3 · proyección al objetivo": {"auprc": 0.2044637094183695, "auroc": 0.6116651652447318, "f1_bin": 0.15501738303600532}, "M1+M3": {"auprc": 0.2044637094183695, "auroc": 0.6116651652447318, "f1_bin": 0.48994999868198763}, "M2+M3 (recorte z>2)": {"auprc": 0.2361589578410143, "auroc": 0.6622317312278099, "f1_bin": 0.28597226637421935}, "M1+M2 (recorte adaptativo, sin M3)": {"auprc": 0.22433029666777132, "auroc": 0.6642493983798101, "f1_bin": 0.4488005033221129}, "M1+M2+M3 con recorte z>2": {"auprc": 0.2361589578410143, "auroc": 0.6622317312278099, "f1_bin": 0.5315061857488421}, "**Método final** (recorte adaptativo + proyección + IQR)": {"auprc": 0.2448955916433564, "auroc": 0.6651419979257668, "f1_bin": 0.5347319446680479}, "Error del LSTM (PD) + IQR": {"auprc": 0.252568102677141, "auroc": 0.6490680144136505, "f1_bin": 0.5298464198849282}, "LOF + IQR": {"auprc": 0.1918892252352368, "auroc": 0.3977064833670206, "f1_bin": 0.3543717429067747}, "IForest + IQR": {"auprc": 0.23510037121903996, "auroc": 0.6340962302113009, "f1_bin": 0.4417503831817564}}, "R-1": {"Artículo · umbral estático": {"auprc": 0.01432802124909928, "auroc": 0.5419352826510722, "f1_bin": 0.0439882884941362}, "Artículo · umbral dinámico (usa etiquetas)": {"auprc": 0.01432802124909928, "auroc": 0.5419352826510722, "f1_bin": 0.0439882884941362}, "M1 · umbral IQR": {"auprc": 0.01432802124909928, "auroc": 0.5419352826510722, "f1_bin": 0.0439882884941362}, "M2 · reponderación suave": {"auprc": 0.01431196764802534, "auroc": 0.5416337881741391, "f1_bin": 0.0439739413680781}, "M2 · recorte z>2": {"auprc": 0.01452951798309032, "auroc": 0.5417554256010397, "f1_bin": 0.0439739413680781}, "M3 · proyección al objetivo": {"auprc": 0.01247839143420826, "auroc": 0.5204141217240633, "f1_bin": 0.043978718548020324}, "M1+M3": {"auprc": 0.01247839143420826, "auroc": 0.5204141217240633, "f1_bin": 0.043978718548020324}, "M2+M3 (recorte z>2)": {"auprc": 0.013189306875940063, "auroc": 0.5204889755252328, "f1_bin": 0.0439739413680781}, "M1+M2 (recorte adaptativo, sin M3)": {"auprc": 0.01452951798309032, "auroc": 0.5417554256010397, "f1_bin": 0.0439739413680781}, "M1+M2+M3 con recorte z>2": {"auprc": 0.013189306875940063, "auroc": 0.5204889755252328, "f1_bin": 0.0439739413680781}, "**Método final** (recorte adaptativo + proyección + IQR)": {"auprc": 0.013189306875940063, "auroc": 0.5204889755252328, "f1_bin": 0.0439739413680781}, "Error del LSTM (PD) + IQR": {"auprc": 0.023811980678074697, "auroc": 0.5111592809183452, "f1_bin": 0.04354889154233486}, "LOF + IQR": {"auprc": 0.0234475385738222, "auroc": 0.5043014944769331, "f1_bin": 0.0439739413680781}, "IForest + IQR": {"auprc": 0.0112488319306711, "auroc": 0.5002763699371887, "f1_bin": 0.0439739413680781}}, "P-4": {"Artículo · umbral estático": {"auprc": 0.06243528668457036, "auroc": 0.5449174072102452, "f1_bin": 0.05575651020371072}, "Artículo · umbral dinámico (usa etiquetas)": {"auprc": 0.06243528668457036, "auroc": 0.5449174072102452, "f1_bin": 0.05575651020371072}, "M1 · umbral IQR": {"auprc": 0.06243528668457036, "auroc": 0.5449174072102452, "f1_bin": 0.05575651020371072}, "M2 · reponderación suave": {"auprc": 0.0623877414499879, "auroc": 0.5446253322476355, "f1_bin": 0.05573970088498785}, "M2 · recorte z>2": {"auprc": 0.06501561190825185, "auroc": 0.5451985386359406, "f1_bin": 0.055722891566265}, "M3 · proyección al objetivo": {"auprc": 0.06023351528457956, "auroc": 0.5277589305753149, "f1_bin": 0.05573408652187852}, "M1+M3": {"auprc": 0.06023351528457956, "auroc": 0.5277589305753149, "f1_bin": 0.05573408652187852}, "M2+M3 (recorte z>2)": {"auprc": 0.06222763572669374, "auroc": 0.5278091348237566, "f1_bin": 0.055722891566265}, "M1+M2 (recorte adaptativo, sin M3)": {"auprc": 0.06501561190825185, "auroc": 0.5451985386359406, "f1_bin": 0.055722891566265}, "M1+M2+M3 con recorte z>2": {"auprc": 0.06222763572669374, "auroc": 0.5278091348237566, "f1_bin": 0.055722891566265}, "**Método final** (recorte adaptativo + proyección + IQR)": {"auprc": 0.06222763572669374, "auroc": 0.5278091348237566, "f1_bin": 0.055722891566265}, "Error del LSTM (PD) + IQR": {"auprc": 0.0667974856021237, "auroc": 0.5286655129581989, "f1_bin": 0.05572848904407176}, "LOF + IQR": {"auprc": 0.0891348546516682, "auroc": 0.5646506445866899, "f1_bin": 0.055722891566265}, "IForest + IQR": {"auprc": 0.05789306873223969, "auroc": 0.5038657889579093, "f1_bin": 0.055722891566265}}, "P-7": {"Artículo · umbral estático": {"auprc": 0.5032538059739704, "auroc": 0.8321903802354479, "f1_bin": 0.54806395669205}, "Artículo · umbral dinámico (usa etiquetas)": {"auprc": 0.5032538059739704, "auroc": 0.8321903802354479, "f1_bin": 0.709114100395022}, "M1 · umbral IQR": {"auprc": 0.5032538059739704, "auroc": 0.8321903802354479, "f1_bin": 0.4993357866362132}, "M2 · reponderación suave": {"auprc": 0.5317517289377428, "auroc": 0.8486024822614937, "f1_bin": 0.5782332694145327}, "M2 · recorte z>2": {"auprc": 0.5367668339244991, "auroc": 0.8436715751694888, "f1_bin": 0.5931410127024817}, "M3 · proyección al objetivo": {"auprc": 0.40553102287495185, "auroc": 0.7577146392004618, "f1_bin": 0.4685040300116901}, "M1+M3": {"auprc": 0.40553102287495185, "auroc": 0.7577146392004618, "f1_bin": 0.41065577210404447}, "M2+M3 (recorte z>2)": {"auprc": 0.4467991986019447, "auroc": 0.771636498093043, "f1_bin": 0.49217311458992763}, "M1+M2 (recorte adaptativo, sin M3)": {"auprc": 0.5299462211202235, "auroc": 0.8406612537039649, "f1_bin": 0.7098622856410053}, "M1+M2+M3 con recorte z>2": {"auprc": 0.4467991986019447, "auroc": 0.771636498093043, "f1_bin": 0.5013270353988661}, "**Método final** (recorte adaptativo + proyección + IQR)": {"auprc": 0.4524217211642075, "auroc": 0.77436168235883, "f1_bin": 0.5551263682035794}, "Error del LSTM (PD) + IQR": {"auprc": 0.4414041549386953, "auroc": 0.7717235463687715, "f1_bin": 0.5006937242111595}, "LOF + IQR": {"auprc": 0.2113692308297197, "auroc": 0.5136558643413552, "f1_bin": 0.5810667608618862}, "IForest + IQR": {"auprc": 0.4196163380069054, "auroc": 0.7262362098429087, "f1_bin": 0.24351262580189342}}, "S-2": {"Artículo · umbral estático": {"auprc": 0.7009630679466785, "auroc": 0.8880049084773493, "f1_bin": 0.0}, "Artículo · umbral dinámico (usa etiquetas)": {"auprc": 0.7009630679466785, "auroc": 0.8880049084773493, "f1_bin": 0.0}, "M1 · umbral IQR": {"auprc": 0.7009630679466785, "auroc": 0.8880049084773493, "f1_bin": 0.0}, "M2 · reponderación suave": {"auprc": 0.6092458890488966, "auroc": 0.8468657326925044, "f1_bin": 0.0}, "M2 · recorte z>2": {"auprc": 0.5962621286272666, "auroc": 0.8153901216893343, "f1_bin": 0.0}, "M3 · proyección al objetivo": {"auprc": 0.3642562021942351, "auroc": 0.7772625012782494, "f1_bin": 0.0}, "M1+M3": {"auprc": 0.3642562021942351, "auroc": 0.7772625012782494, "f1_bin": 0.0}, "M2+M3 (recorte z>2)": {"auprc": 0.2596483011879305, "auroc": 0.6888894569996932, "f1_bin": 0.0}, "M1+M2 (recorte adaptativo, sin M3)": {"auprc": 0.5993463530434587, "auroc": 0.8317414868595971, "f1_bin": 0.0}, "M1+M2+M3 con recorte z>2": {"auprc": 0.2596483011879305, "auroc": 0.6888894569996932, "f1_bin": 0.0}, "**Método final** (recorte adaptativo + proyección + IQR)": {"auprc": 0.27404389685127944, "auroc": 0.7108139891604458, "f1_bin": 0.0}, "Error del LSTM (PD) + IQR": {"auprc": 0.08353641813468714, "auroc": 0.6890632989058185, "f1_bin": 0.0}, "LOF + IQR": {"auprc": 0.10275187021972929, "auroc": 0.6274670211678086, "f1_bin": 0.0}, "IForest + IQR": {"auprc": 0.06778351913715747, "auroc": 0.61546170365068, "f1_bin": 0.0}}, "T-12": {"Artículo · umbral estático": {"auprc": 0.19102991360538638, "auroc": 0.6691070668253328, "f1_bin": 0.0}, "Artículo · umbral dinámico (usa etiquetas)": {"auprc": 0.19102991360538638, "auroc": 0.6691070668253328, "f1_bin": 0.0}, "M1 · umbral IQR": {"auprc": 0.19102991360538638, "auroc": 0.6691070668253328, "f1_bin": 0.0}, "M2 · reponderación suave": {"auprc": 0.1891886681903828, "auroc": 0.6726377501446554, "f1_bin": 0.0}, "M2 · recorte z>2": {"auprc": 0.16993572874604007, "auroc": 0.6699222318052629, "f1_bin": 0.0}, "M3 · proyección al objetivo": {"auprc": 0.10500164628963908, "auroc": 0.6577034910168091, "f1_bin": 0.0}, "M1+M3": {"auprc": 0.10500164628963908, "auroc": 0.6577034910168091, "f1_bin": 0.0}, "M2+M3 (recorte z>2)": {"auprc": 0.10434717817836241, "auroc": 0.6545505493265791, "f1_bin": 0.0}, "M1+M2 (recorte adaptativo, sin M3)": {"auprc": 0.17117048623565026, "auroc": 0.6670793439377564, "f1_bin": 0.0}, "M1+M2+M3 con recorte z>2": {"auprc": 0.10434717817836241, "auroc": 0.6545505493265791, "f1_bin": 0.0}, "**Método final** (recorte adaptativo + proyección + IQR)": {"auprc": 0.10359335238622232, "auroc": 0.6515402615078368, "f1_bin": 0.0}, "Error del LSTM (PD) + IQR": {"auprc": 0.09241285085416791, "auroc": 0.6423776615682464, "f1_bin": 0.0}, "LOF + IQR": {"auprc": 0.0732983061771831, "auroc": 0.427783297123996, "f1_bin": 0.0}, "IForest + IQR": {"auprc": 0.07185438803001405, "auroc": 0.5850722913050281, "f1_bin": 0.0}}, "T-13": {"Artículo · umbral estático": {"auprc": 0.19202287044051336, "auroc": 0.6707142857142857, "f1_bin": 0.2364502909307628}, "Artículo · umbral dinámico (usa etiquetas)": {"auprc": 0.19202287044051336, "auroc": 0.6707142857142857, "f1_bin": 0.2872101817268586}, "M1 · umbral IQR": {"auprc": 0.19202287044051336, "auroc": 0.6707142857142857, "f1_bin": 0.37293151440315864}, "M2 · reponderación suave": {"auprc": 0.2096252624318075, "auroc": 0.6397648716807596, "f1_bin": 0.2914308181228267}, "M2 · recorte z>2": {"auprc": 0.1976042661785892, "auroc": 0.6398865153538051, "f1_bin": 0.3086747574676506}, "M3 · proyección al objetivo": {"auprc": 0.15716871686942233, "auroc": 0.6048460910844089, "f1_bin": 0.3150579647122422}, "M1+M3": {"auprc": 0.15716871686942233, "auroc": 0.6048460910844089, "f1_bin": 0.3249736516598087}, "M2+M3 (recorte z>2)": {"auprc": 0.14951492373868314, "auroc": 0.5720219551995254, "f1_bin": 0.27571468887772016}, "M1+M2 (recorte adaptativo, sin M3)": {"auprc": 0.20112997890447545, "auroc": 0.6494785640112742, "f1_bin": 0.37589075832817165}, "M1+M2+M3 con recorte z>2": {"auprc": 0.14951492373868314, "auroc": 0.5720219551995254, "f1_bin": 0.26644537939502466}, "**Método final** (recorte adaptativo + proyección + IQR)": {"auprc": 0.15255046904115754, "auroc": 0.5788317757009346, "f1_bin": 0.2970594719139245}, "Error del LSTM (PD) + IQR": {"auprc": 0.18692611476590887, "auroc": 0.5812709538644117, "f1_bin": 0.31838538091969804}, "LOF + IQR": {"auprc": 0.2004849242010452, "auroc": 0.58226246105919, "f1_bin": 0.2990033222591362}, "IForest + IQR": {"auprc": 0.20080467144927255, "auroc": 0.7266787197745141, "f1_bin": 0.1566465896699932}}, "S-1": {"Artículo · umbral estático": {"auprc": 0.06954509940166306, "auroc": 0.5504273192111029, "f1_bin": 0.0365725998306321}, "Artículo · umbral dinámico (usa etiquetas)": {"auprc": 0.06954509940166306, "auroc": 0.5504273192111029, "f1_bin": 0.3195873485213351}, "M1 · umbral IQR": {"auprc": 0.06954509940166306, "auroc": 0.5504273192111029, "f1_bin": 0.16219741426327094}, "M2 · reponderación suave": {"auprc": 0.053888936033045044, "auroc": 0.41526850933945525, "f1_bin": 0.03858040807038688}, "M2 · recorte z>2": {"auprc": 0.0520393139693923, "auroc": 0.38614943128456636, "f1_bin": 0.048743337361740385}, "M3 · proyección al objetivo": {"auprc": 0.07808131382020556, "auroc": 0.5885764504852341, "f1_bin": 0.06227399134416496}, "M1+M3": {"auprc": 0.07808131382020556, "auroc": 0.5885764504852341, "f1_bin": 0.10077927744092958}, "M2+M3 (recorte z>2)": {"auprc": 0.0612452961160526, "auroc": 0.48266269044140664, "f1_bin": 0.1063754833486025}, "M1+M2 (recorte adaptativo, sin M3)": {"auprc": 0.051117563670371305, "auroc": 0.3672421866847543, "f1_bin": 0.16852588941768915}, "M1+M2+M3 con recorte z>2": {"auprc": 0.0612452961160526, "auroc": 0.48266269044140664, "f1_bin": 0.12085580013983524}, "**Método final** (recorte adaptativo + proyección + IQR)": {"auprc": 0.06144903117217766, "auroc": 0.4839345716372743, "f1_bin": 0.1039960421820201}, "Error del LSTM (PD) + IQR": {"auprc": 0.04769355725207132, "auroc": 0.3283906396744235, "f1_bin": 0.1200246806147304}, "LOF + IQR": {"auprc": 0.05912018932759659, "auroc": 0.4064205494104143, "f1_bin": 0.1339187705817782}, "IForest + IQR": {"auprc": 0.15450509292391656, "auroc": 0.7195574519983304, "f1_bin": 0.020962607353054403}}, "T-1": {"Artículo · umbral estático": {"auprc": 0.17724047360567421, "auroc": 0.4618132703715544, "f1_bin": 0.042972952301846515}, "Artículo · umbral dinámico (usa etiquetas)": {"auprc": 0.17724047360567421, "auroc": 0.4618132703715544, "f1_bin": 0.01576779128562086}, "M1 · umbral IQR": {"auprc": 0.17724047360567421, "auroc": 0.4618132703715544, "f1_bin": 0.047993054388801276}, "M2 · reponderación suave": {"auprc": 0.22538655376337088, "auroc": 0.656376785693142, "f1_bin": 0.04323195865235676}, "M2 · recorte z>2": {"auprc": 0.24079345103742272, "auroc": 0.6941755729314674, "f1_bin": 0.02933633015662156}, "M3 · proyección al objetivo": {"auprc": 0.17351218336205435, "auroc": 0.45168363371163683, "f1_bin": 0.040498340528165776}, "M1+M3": {"auprc": 0.17351218336205435, "auroc": 0.45168363371163683, "f1_bin": 0.04005230792017718}, "M2+M3 (recorte z>2)": {"auprc": 0.23235960315275928, "auroc": 0.6725695461986123, "f1_bin": 0.0325263965256759}, "M1+M2 (recorte adaptativo, sin M3)": {"auprc": 0.2301775075283159, "auroc": 0.6672880146170788, "f1_bin": 0.02394631551000752}, "M1+M2+M3 con recorte z>2": {"auprc": 0.23235960315275928, "auroc": 0.6725695461986123, "f1_bin": 0.02428993888392062}, "**Método final** (recorte adaptativo + proyección + IQR)": {"auprc": 0.2183302656439441, "auroc": 0.6425536022455478, "f1_bin": 0.018975324645624}, "Error del LSTM (PD) + IQR": {"auprc": 0.19265846067056486, "auroc": 0.48832935197025673, "f1_bin": 0.025292225304293342}, "LOF + IQR": {"auprc": 0.1807443060473036, "auroc": 0.5109313577484135, "f1_bin": 0.0180722891566265}, "IForest + IQR": {"auprc": 0.25486480347039114, "auroc": 0.7038605901297718, "f1_bin": 0.0}}, "T-2": {"Artículo · umbral estático": {"auprc": 0.23833008048885418, "auroc": 0.6040049606977851, "f1_bin": 0.1999459598386037}, "Artículo · umbral dinámico (usa etiquetas)": {"auprc": 0.23833008048885418, "auroc": 0.6040049606977851, "f1_bin": 0.6148821756436016}, "M1 · umbral IQR": {"auprc": 0.23833008048885418, "auroc": 0.6040049606977851, "f1_bin": 0.18259264663747352}, "M2 · reponderación suave": {"auprc": 0.2515979562162471, "auroc": 0.6487187077812292, "f1_bin": 0.19195500591764367}, "M2 · recorte z>2": {"auprc": 0.2557182864181538, "auroc": 0.6596796117778302, "f1_bin": 0.17565543051091928}, "M3 · proyección al objetivo": {"auprc": 0.23683876953358624, "auroc": 0.6017841111894966, "f1_bin": 0.18277761686072}, "M1+M3": {"auprc": 0.23683876953358624, "auroc": 0.6017841111894966, "f1_bin": 0.19431434456865593}, "M2+M3 (recorte z>2)": {"auprc": 0.24962414544086328, "auroc": 0.641862640904953, "f1_bin": 0.18388583438862963}, "M1+M2 (recorte adaptativo, sin M3)": {"auprc": 0.24722294110807397, "auroc": 0.6401925603479893, "f1_bin": 0.23922813583994906}, "M1+M2+M3 con recorte z>2": {"auprc": 0.24962414544086328, "auroc": 0.641862640904953, "f1_bin": 0.28500287683539416}, "**Método final** (recorte adaptativo + proyección + IQR)": {"auprc": 0.23843417777092007, "auroc": 0.6203417083641669, "f1_bin": 0.1997695229616197}, "Error del LSTM (PD) + IQR": {"auprc": 0.20179245185563563, "auroc": 0.4790710621036925, "f1_bin": 0.15888158362848898}, "LOF + IQR": {"auprc": 0.20731487017014957, "auroc": 0.5034542083496389, "f1_bin": 0.5701181866290351}, "IForest + IQR": {"auprc": 0.2746632353626948, "auroc": 0.6695943660765178, "f1_bin": 0.0}}, "T-8": {"Artículo · umbral estático": {"auprc": 0.10332623455517839, "auroc": 0.5437771047505298, "f1_bin": 0.2725124339239637}, "Artículo · umbral dinámico (usa etiquetas)": {"auprc": 0.10332623455517839, "auroc": 0.5437771047505298, "f1_bin": 0.2136704441329143}, "M1 · umbral IQR": {"auprc": 0.10332623455517839, "auroc": 0.5437771047505298, "f1_bin": 0.21849236768736485}, "M2 · reponderación suave": {"auprc": 0.10565396599403742, "auroc": 0.5442235777559755, "f1_bin": 0.2767948350071736}, "M2 · recorte z>2": {"auprc": 0.10818420565639994, "auroc": 0.5419272277438895, "f1_bin": 0.26677563942291604}, "M3 · proyección al objetivo": {"auprc": 0.08622818517150196, "auroc": 0.5147080151857697, "f1_bin": 0.15477246199908096}, "M1+M3": {"auprc": 0.08622818517150196, "auroc": 0.5147080151857697, "f1_bin": 0.15865217135890514}, "M2+M3 (recorte z>2)": {"auprc": 0.08712083445580987, "auroc": 0.5139160232620967, "f1_bin": 0.15188021265180704}, "M1+M2 (recorte adaptativo, sin M3)": {"auprc": 0.10198010007595344, "auroc": 0.5409774061909027, "f1_bin": 0.22831012895989944}, "M1+M2+M3 con recorte z>2": {"auprc": 0.08712083445580987, "auroc": 0.5139160232620967, "f1_bin": 0.15696870070764038}, "**Método final** (recorte adaptativo + proyección + IQR)": {"auprc": 0.08548032199764088, "auroc": 0.5107551650101666, "f1_bin": 0.17641617738189438}, "Error del LSTM (PD) + IQR": {"auprc": 0.07887677206580115, "auroc": 0.5075331655504842, "f1_bin": 0.15793484448959236}, "LOF + IQR": {"auprc": 0.068799227121979, "auroc": 0.49627109727139584, "f1_bin": 0.2365591397849462}, "IForest + IQR": {"auprc": 0.09754027319327752, "auroc": 0.5210055595842398, "f1_bin": 0.0592592592592592}}, "T-5": {"Artículo · umbral estático": {"auprc": 0.32318436063193046, "auroc": 0.8002142704092565, "f1_bin": 0.19326406153715392}, "Artículo · umbral dinámico (usa etiquetas)": {"auprc": 0.32318436063193046, "auroc": 0.8002142704092565, "f1_bin": 0.06988210405975401}, "M1 · umbral IQR": {"auprc": 0.32318436063193046, "auroc": 0.8002142704092565, "f1_bin": 0.2450691404590545}, "M2 · reponderación suave": {"auprc": 0.3640800014373732, "auroc": 0.7962359831440612, "f1_bin": 0.18074650947056087}, "M2 · recorte z>2": {"auprc": 0.31864677596036456, "auroc": 0.7816227412327692, "f1_bin": 0.16733056981553754}, "M3 · proyección al objetivo": {"auprc": 0.15601285706359141, "auroc": 0.7609474323262624, "f1_bin": 0.11616156484825076}, "M1+M3": {"auprc": 0.15601285706359141, "auroc": 0.7609474323262624, "f1_bin": 0.1220899945147795}, "M2+M3 (recorte z>2)": {"auprc": 0.16776349467833745, "auroc": 0.7294246839511462, "f1_bin": 0.09543075703854902}, "M1+M2 (recorte adaptativo, sin M3)": {"auprc": 0.3203227440126596, "auroc": 0.7971073494750375, "f1_bin": 0.22970630518876947}, "M1+M2+M3 con recorte z>2": {"auprc": 0.16776349467833745, "auroc": 0.7294246839511462, "f1_bin": 0.0879114091503022}, "**Método final** (recorte adaptativo + proyección + IQR)": {"auprc": 0.1758714581572745, "auroc": 0.7025730304978215, "f1_bin": 0.09992503361060967}, "Error del LSTM (PD) + IQR": {"auprc": 0.051664085580448126, "auroc": 0.7509802871223483, "f1_bin": 0.08714139178759103}, "LOF + IQR": {"auprc": 0.0607240442455686, "auroc": 0.6075369616455968, "f1_bin": 0.046594982078853}, "IForest + IQR": {"auprc": 0.0155817769861343, "auroc": 0.615906006713806, "f1_bin": 0.0}}, "T-4": {"Artículo · umbral estático": {"auprc": 0.031100099536606002, "auroc": 0.46749570712274197, "f1_bin": 0.09152002467620923}, "Artículo · umbral dinámico (usa etiquetas)": {"auprc": 0.031100099536606002, "auroc": 0.46749570712274197, "f1_bin": 0.1120107110745742}, "M1 · umbral IQR": {"auprc": 0.031100099536606002, "auroc": 0.46749570712274197, "f1_bin": 0.09769330979707994}, "M2 · reponderación suave": {"auprc": 0.03133775871767767, "auroc": 0.47099594752386836, "f1_bin": 0.09183771509794185}, "M2 · recorte z>2": {"auprc": 0.03158959786038182, "auroc": 0.47376399477986125, "f1_bin": 0.09242664572823442}, "M3 · proyección al objetivo": {"auprc": 0.02972776304662008, "auroc": 0.43066831513153375, "f1_bin": 0.046816969756918965}, "M1+M3": {"auprc": 0.02972776304662008, "auroc": 0.43066831513153375, "f1_bin": 0.06694430518514047}, "M2+M3 (recorte z>2)": {"auprc": 0.03010860673552142, "auroc": 0.43531767291709594, "f1_bin": 0.060674355542782256}, "M1+M2 (recorte adaptativo, sin M3)": {"auprc": 0.0305977710409318, "auroc": 0.4553726217459991, "f1_bin": 0.0026315789473684197}, "M1+M2+M3 con recorte z>2": {"auprc": 0.03010860673552142, "auroc": 0.43531767291709594, "f1_bin": 0.06902029340563406}, "**Método final** (recorte adaptativo + proyección + IQR)": {"auprc": 0.03022954622912134, "auroc": 0.44189436087643374, "f1_bin": 0.06125237334522961}, "Error del LSTM (PD) + IQR": {"auprc": 0.029973419099554137, "auroc": 0.4333402019369462, "f1_bin": 0.06684418592032904}, "LOF + IQR": {"auprc": 0.051190687887674, "auroc": 0.5290576275843121, "f1_bin": 0.1190681622088006}, "IForest + IQR": {"auprc": 0.029406581068183724, "auroc": 0.4533759186757331, "f1_bin": 0.0}}, "T-3": {"Artículo · umbral estático": {"auprc": 0.027581544736950642, "auroc": 0.5315963966682449, "f1_bin": 0.04280395018564772}, "Artículo · umbral dinámico (usa etiquetas)": {"auprc": 0.027581544736950642, "auroc": 0.5315963966682449, "f1_bin": 0.15663052543786485}, "M1 · umbral IQR": {"auprc": 0.027581544736950642, "auroc": 0.5315963966682449, "f1_bin": 0.15683501683501683}, "M2 · reponderación suave": {"auprc": 0.031306885117785774, "auroc": 0.5275849196976239, "f1_bin": 0.061164644938676925}, "M2 · recorte z>2": {"auprc": 0.03757171681147882, "auroc": 0.5276264105592292, "f1_bin": 0.072181795276065}, "M3 · proyección al objetivo": {"auprc": 0.02400586272715772, "auroc": 0.5180489800271576, "f1_bin": 0.03903203076964086}, "M1+M3": {"auprc": 0.02400586272715772, "auroc": 0.5180489800271576, "f1_bin": 0.09099204249971307}, "M2+M3 (recorte z>2)": {"auprc": 0.03348927483053698, "auroc": 0.49435353595787956, "f1_bin": 0.050166810658535864}, "M1+M2 (recorte adaptativo, sin M3)": {"auprc": 0.03721255986927896, "auroc": 0.5255219524376857, "f1_bin": 0.09041694228427832}, "M1+M2+M3 con recorte z>2": {"auprc": 0.03348927483053698, "auroc": 0.49435353595787956, "f1_bin": 0.06910054544778645}, "**Método final** (recorte adaptativo + proyección + IQR)": {"auprc": 0.033435942083060624, "auroc": 0.49470789913168345, "f1_bin": 0.06885682480487419}, "Error del LSTM (PD) + IQR": {"auprc": 0.02898350340995424, "auroc": 0.5195842069392491, "f1_bin": 0.0699884248250919}, "LOF + IQR": {"auprc": 0.029064493772804102, "auroc": 0.5034857526364256, "f1_bin": 0.046203110704483}, "IForest + IQR": {"auprc": 0.018527082518122857, "auroc": 0.4389165422374369, "f1_bin": 0.0}}, "T-9": {"Artículo · umbral estático": {"auprc": 0.35297528939064937, "auroc": 0.853063651464814, "f1_bin": 0.4353883266000487}, "Artículo · umbral dinámico (usa etiquetas)": {"auprc": 0.35297528939064937, "auroc": 0.853063651464814, "f1_bin": 0.4790404511927111}, "M1 · umbral IQR": {"auprc": 0.35297528939064937, "auroc": 0.853063651464814, "f1_bin": 0.49198832841296075}, "M2 · reponderación suave": {"auprc": 0.32574205151957836, "auroc": 0.8153201449713077, "f1_bin": 0.42929259865430075}, "M2 · recorte z>2": {"auprc": 0.32114069283956825, "auroc": 0.7947221383267895, "f1_bin": 0.4391489361702128}, "M3 · proyección al objetivo": {"auprc": 0.3141086920140431, "auroc": 0.7839606614315915, "f1_bin": 0.44521157151762447}, "M1+M3": {"auprc": 0.3141086920140431, "auroc": 0.7839606614315915, "f1_bin": 0.49970857716615136}, "M2+M3 (recorte z>2)": {"auprc": 0.2476547383968996, "auroc": 0.6257003171247357, "f1_bin": 0.4388923766816143}, "M1+M2 (recorte adaptativo, sin M3)": {"auprc": 0.29412954561285554, "auroc": 0.7825600271821203, "f1_bin": 0.4637141717948503}, "M1+M2+M3 con recorte z>2": {"auprc": 0.2476547383968996, "auroc": 0.6257003171247357, "f1_bin": 0.43576767492904656}, "**Método final** (recorte adaptativo + proyección + IQR)": {"auprc": 0.2242475488490136, "auroc": 0.5503227876774389, "f1_bin": 0.4296625109501295}, "Error del LSTM (PD) + IQR": {"auprc": 0.2031353593970097, "auroc": 0.6425607822410149, "f1_bin": 0.3995481825631868}, "LOF + IQR": {"auprc": 0.2013764350270269, "auroc": 0.6661507097553608, "f1_bin": 0.4556962025316455}, "IForest + IQR": {"auprc": 0.3146550607184025, "auroc": 0.6142960963455149, "f1_bin": 0.20437350496421147}}}, "seeds": {"A-1": 5, "A-2": 5, "A-3": 5, "A-4": 5, "A-5": 5, "A-6": 5, "A-7": 5, "A-8": 5, "A-9": 5, "B-1": 5, "C-1": 5, "C-2": 5, "D-1": 5, "D-11": 5, "D-12": 5, "D-13": 5, "D-14": 5, "D-15": 5, "D-16": 5, "D-2": 5, "D-3": 5, "D-4": 5, "D-5": 5, "D-6": 5, "D-7": 5, "D-8": 5, "D-9": 5, "E-1": 5, "E-10": 5, "E-11": 5, "E-12": 5, "E-13": 5, "E-2": 5, "E-3": 5, "E-4": 5, "E-5": 5, "E-6": 5, "E-7": 5, "E-8": 5, "E-9": 5, "F-1": 5, "F-2": 5, "F-3": 5, "F-4": 5, "F-5": 5, "F-7": 5, "F-8": 5, "G-1": 5, "G-2": 5, "G-3": 5, "G-4": 5, "G-6": 5, "G-7": 5, "M-1": 5, "M-2": 5, "M-3": 5, "M-4": 5, "M-5": 5, "M-6": 5, "M-7": 5, "P-1": 5, "P-10": 5, "P-11": 5, "P-14": 5, "P-15": 5, "P-2": 5, "P-3": 5, "P-4": 5, "P-7": 5, "R-1": 5, "S-1": 5, "S-2": 5, "T-1": 5, "T-12": 5, "T-13": 5, "T-2": 5, "T-3": 5, "T-4": 5, "T-5": 5, "T-8": 5, "T-9": 5}}, "semi": {"n_units": 450, "n_runs": 450, "methods": {"Artículo · umbral estático": {"auprc": 0.2129513973254502, "auroc": 0.6328513209508395, "f1_bin": 0.19315273306029712, "f1_mac": 0.5380596207575371}, "Artículo · umbral dinámico (usa etiquetas)": {"auprc": 0.2129513973254502, "auroc": 0.6328513209508395, "f1_bin": 0.24443315026081947, "f1_mac": 0.5574241079736886}, "M1 · umbral IQR": {"auprc": 0.2129513973254502, "auroc": 0.6328513209508395, "f1_bin": 0.19064099812861598, "f1_mac": 0.5398921309014044}, "M2 · reponderación suave": {"auprc": 0.21351650051067, "auroc": 0.6350337405158415, "f1_bin": 0.19347197578673014, "f1_mac": 0.5383693008478514}, "M2 · recorte z>2": {"auprc": 0.20528745920414918, "auroc": 0.637346619054753, "f1_bin": 0.1947975388878123, "f1_mac": 0.5393296445336846}, "M3 · proyección al objetivo": {"auprc": 0.2608326466778444, "auroc": 0.6662718507607959, "f1_bin": 0.22984448947043767, "f1_mac": 0.5600146887151748}, "M1+M3": {"auprc": 0.2608326466778444, "auroc": 0.6662718507607959, "f1_bin": 0.23107935582592426, "f1_mac": 0.5624794166555395}, "M2+M3 (recorte z>2)": {"auprc": 0.25738813258679805, "auroc": 0.6771589960108683, "f1_bin": 0.23514152728046245, "f1_mac": 0.5633803916325819}, "M1+M2 (recorte adaptativo, sin M3)": {"auprc": 0.20100827662700266, "auroc": 0.6384175404681388, "f1_bin": 0.19306177941071215, "f1_mac": 0.5416402103128953}, "M1+M2+M3 con recorte z>2": {"auprc": 0.25738813258679805, "auroc": 0.6771589960108683, "f1_bin": 0.23740105950580617, "f1_mac": 0.5658604486022055}, "**Método final** (recorte adaptativo + proyección + IQR)": {"auprc": 0.2529740597446852, "auroc": 0.6795761516018676, "f1_bin": 0.23900075062700185, "f1_mac": 0.5644477418144671}, "Error del LSTM (PD) + IQR": {"auprc": 0.32350765583118385, "auroc": 0.7291536864412551, "f1_bin": 0.29203479981064207, "f1_mac": 0.6020328799138944}, "LOF + IQR": {"auprc": 0.49762158723204586, "auroc": 0.7828842138666224, "f1_bin": 0.4312694063933801, "f1_mac": 0.6850803636276646}, "IForest + IQR": {"auprc": 0.14214287844989265, "auroc": 0.5755466614749172, "f1_bin": 0.057329870951730776, "f1_mac": 0.486132344239043}}, "hyp": [{"h": "H1", "desc": "Final > Artículo", "metric": "auprc", "n": 450, "mean": 0.040022662419235015, "lo": 0.032836623990912764, "hi": 0.04739576103103032, "p": 2.3032019817097033e-18, "rb": 0.47555555555555556, "wins": 296, "losses": 154, "p_holm": 1.1516009908548516e-17}, {"h": "H2", "desc": "Final (sin etiquetas) ≥ Artículo dinámico (con etiquetas)", "metric": "f1_bin", "n": 450, "mean": -0.005432399633817589, "lo": -0.012014617227264192, "hi": 0.0009574894716889832, "p": 0.5568096499283784, "rb": -0.032001979707993074, "wins": 238, "losses": 211, "p_holm": 0.5568096499283784}, {"h": "H3a", "desc": "Final > M3 (mismo score sin recorte)", "metric": "auprc", "n": 450, "mean": -0.007858586933159214, "lo": -0.010071777994621129, "hi": -0.005669181497264723, "p": 1.945763363390844e-11, "rb": -0.36504557772850454, "wins": 159, "losses": 291, "p_holm": 7.783053453563376e-11}, {"h": "H3b", "desc": "Final > M1+M3 (todo menos el recorte)", "metric": "f1_bin", "n": 450, "mean": 0.007921394801077605, "lo": 0.006371878251863292, "hi": 0.009494572504026817, "p": 3.788028042502304e-21, "rb": 0.5146158129175946, "wins": 311, "losses": 137, "p_holm": 2.2728168255013826e-20}, {"h": "H5a", "desc": "M3 (sólo proyección al objetivo) > Artículo", "metric": "auroc", "n": 450, "mean": 0.033420529809956544, "lo": 0.029794177676107748, "hi": 0.0371613495858816, "p": 5.907617112760122e-44, "rb": 0.7564720374476472, "wins": 338, "losses": 112, "p_holm": 4.726093690208097e-43}, {"h": "H5b", "desc": "M3 (sólo proyección al objetivo) > Artículo", "metric": "auprc", "n": 450, "mean": 0.04788124935239422, "lo": 0.041311067558711306, "hi": 0.05459389049172174, "p": 6.30731730068059e-32, "rb": 0.6397536338999754, "wins": 308, "losses": 142, "p_holm": 4.415122110476413e-31}, {"h": "H4r", "desc": "Caída de AUROC 1 %→30 % menor en Final que en Artículo", "metric": "auroc_drop", "n": 90, "mean": 0.016537192531574943, "lo": 0.009779206851920515, "hi": 0.023259855057039338, "p": 1.1032487504976744e-05, "rb": 0.5335775335775336, "wins": 60, "losses": 30, "drop_paper": 0.013879404035366649, "drop_final": -0.0026577884962082935, "p_holm": 2.206497500995349e-05}, {"h": "H5c", "desc": "Caída de AUROC 1 %→30 % menor en M3 que en Artículo", "metric": "auroc_drop", "n": 90, "mean": 0.016056560490894156, "lo": 0.010171510966271512, "hi": 0.021992605366074537, "p": 1.3083206508620417e-06, "rb": 0.5873015873015873, "wins": 65, "losses": 25, "drop_paper": 0.013879404035366649, "drop_final": -0.0021771564555275067, "p_holm": 3.924961952586125e-06}], "curves": {"01": [{"Método": "Artículo · umbral estático", "1%": "0.631", "5%": "0.622", "10%": "0.617", "20%": "0.606", "30%": "0.599", "Caída": "0.031"}, {"Método": "M3 · proyección al objetivo", "1%": "0.695", "5%": "0.690", "10%": "0.693", "20%": "0.694", "30%": "0.700", "Caída": "-0.005"}, {"Método": "M2 · recorte z>2", "1%": "0.637", "5%": "0.628", "10%": "0.622", "20%": "0.610", "30%": "0.603", "Caída": "0.034"}, {"Método": "**Método final** (recorte adaptativo + proyección + IQR)", "1%": "0.708", "5%": "0.704", "10%": "0.707", "20%": "0.708", "30%": "0.714", "Caída": "-0.006"}, {"Método": "Error del LSTM (PD) + IQR", "1%": "0.754", "5%": "0.752", "10%": "0.754", "20%": "0.759", "30%": "0.766", "Caída": "-0.012"}, {"Método": "LOF + IQR", "1%": "0.797", "5%": "0.799", "10%": "0.800", "20%": "0.798", "30%": "0.761", "Caída": "0.036"}, {"Método": "IForest + IQR", "1%": "0.573", "5%": "0.577", "10%": "0.585", "20%": "0.598", "30%": "0.608", "Caída": "-0.036"}], "01234": [{"Método": "Artículo · umbral estático", "1%": "0.643", "5%": "0.637", "10%": "0.629", "20%": "0.632", "30%": "0.629", "Caída": "0.013"}, {"Método": "M3 · proyección al objetivo", "1%": "0.660", "5%": "0.661", "10%": "0.656", "20%": "0.664", "30%": "0.667", "Caída": "-0.007"}, {"Método": "M2 · recorte z>2", "1%": "0.648", "5%": "0.641", "10%": "0.633", "20%": "0.633", "30%": "0.631", "Caída": "0.017"}, {"Método": "**Método final** (recorte adaptativo + proyección + IQR)", "1%": "0.672", "5%": "0.674", "10%": "0.671", "20%": "0.677", "30%": "0.681", "Caída": "-0.009"}, {"Método": "Error del LSTM (PD) + IQR", "1%": "0.722", "5%": "0.722", "10%": "0.724", "20%": "0.728", "30%": "0.731", "Caída": "-0.009"}, {"Método": "LOF + IQR", "1%": "0.781", "5%": "0.787", "10%": "0.785", "20%": "0.787", "30%": "0.779", "Caída": "0.002"}, {"Método": "IForest + IQR", "1%": "0.561", "5%": "0.565", "10%": "0.573", "20%": "0.579", "30%": "0.592", "Caída": "-0.031"}], "234": [{"Método": "Artículo · umbral estático", "1%": "0.650", "5%": "0.646", "10%": "0.653", "20%": "0.645", "30%": "0.653", "Caída": "-0.003"}, {"Método": "M3 · proyección al objetivo", "1%": "0.649", "5%": "0.639", "10%": "0.645", "20%": "0.639", "30%": "0.643", "Caída": "0.005"}, {"Método": "M2 · recorte z>2", "1%": "0.660", "5%": "0.652", "10%": "0.658", "20%": "0.648", "30%": "0.656", "Caída": "0.004"}, {"Método": "**Método final** (recorte adaptativo + proyección + IQR)", "1%": "0.663", "5%": "0.653", "10%": "0.657", "20%": "0.650", "30%": "0.655", "Caída": "0.008"}, {"Método": "Error del LSTM (PD) + IQR", "1%": "0.707", "5%": "0.703", "10%": "0.707", "20%": "0.703", "30%": "0.704", "Caída": "0.003"}, {"Método": "LOF + IQR", "1%": "0.776", "5%": "0.776", "10%": "0.781", "20%": "0.771", "30%": "0.765", "Caída": "0.011"}, {"Método": "IForest + IQR", "1%": "0.569", "5%": "0.554", "10%": "0.563", "20%": "0.562", "30%": "0.573", "Caída": "-0.004"}]}, "anom_rate": [{"mix": "01", "ratio": 0.01, "anom_rate": 0.00994780164625571}, {"mix": "01", "ratio": 0.05, "anom_rate": 0.04884895937897338}, {"mix": "01", "ratio": 0.1, "anom_rate": 0.09532222445292104}, {"mix": "01", "ratio": 0.2, "anom_rate": 0.18117178612059154}, {"mix": "01", "ratio": 0.3, "anom_rate": 0.25883691360503247}, {"mix": "01234", "ratio": 0.01, "anom_rate": 0.00994110954962183}, {"mix": "01234", "ratio": 0.05, "anom_rate": 0.048942648731847636}, {"mix": "01234", "ratio": 0.1, "anom_rate": 0.09500434986281198}, {"mix": "01234", "ratio": 0.2, "anom_rate": 0.18111824934752055}, {"mix": "01234", "ratio": 0.3, "anom_rate": 0.2595529679448571}, {"mix": "234", "ratio": 0.01, "anom_rate": 0.009014254165830073}, {"mix": "234", "ratio": 0.05, "anom_rate": 0.0499765776617814}, {"mix": "234", "ratio": 0.1, "anom_rate": 0.09465301478953353}, {"mix": "234", "ratio": 0.2, "anom_rate": 0.18297865221173792}, {"mix": "234", "ratio": 0.3, "anom_rate": 0.25822793281134976}]}, "h6": {"d_auroc_m3": {"rho": 0.29916666473684383, "p": 0.0007886125378581167, "n": 109}, "d_auprc_m3": {"rho": 0.4064646614427122, "p": 5.783997036816196e-06, "n": 109}, "d_auroc_final": {"rho": 0.2895567824045914, "p": 0.001129391034165236, "n": 109}, "cuartiles": [{"anom_rate": "Q1 (menor)", "tasa_min": 0.0040821701343165, "tasa_max": 0.0143517969398647, "d_auroc_m3": -0.01968865409101735, "d_auprc_m3": -0.06269281554615956, "gana": 7, "n": 28}, {"anom_rate": "Q2", "tasa_min": 0.0147709458282529, "tasa_max": 0.0424733785091965, "d_auroc_m3": -0.008970975097313911, "d_auprc_m3": -0.024812108635007737, "gana": 8, "n": 27}, {"anom_rate": "Q3", "tasa_min": 0.0442612555485098, "tasa_max": 0.1255, "d_auroc_m3": -0.007115948593758733, "d_auprc_m3": -0.031287817114018214, "gana": 9, "n": 27}, {"anom_rate": "Q4 (mayor)", "tasa_min": 0.1328918322295805, "tasa_max": 0.5096025011165699, "d_auroc_m3": 0.0010765149403850493, "d_auprc_m3": 0.0012106898186806678, "gana": 14, "n": 27}]}}
EXTRA = {"halo": {"synth": [{"train": "adapt_r3", "proj": "tgt", "auroc_anom_vs_halo": 0.8858407441108294, "auroc_anom_vs_far": 0.9265505548147488, "topk_halo_share": 0.14062340532697906, "topk_anom_share": 0.7295949521264455, "halo_frac": 0.3005659741396617}, {"train": "adapt_r3", "proj": "win", "auroc_anom_vs_halo": 0.7716926752213122, "auroc_anom_vs_far": 0.9576455213502146, "topk_halo_share": 0.41189427778268284, "topk_anom_share": 0.5467850488179132, "halo_frac": 0.3005659741396617}, {"train": "base", "proj": "tgt", "auroc_anom_vs_halo": 0.8837714485733443, "auroc_anom_vs_far": 0.921704906874939, "topk_halo_share": 0.14114582837984466, "topk_anom_share": 0.7197829581348469, "halo_frac": 0.3005659741396617}, {"train": "base", "proj": "win", "auroc_anom_vs_halo": 0.7586098980805633, "auroc_anom_vs_far": 0.9571166816397481, "topk_halo_share": 0.42294649032236653, "topk_anom_share": 0.5481682438253852, "halo_frac": 0.3005659741396617}, {"train": "pd", "proj": "tgt", "auroc_anom_vs_halo": 0.8404408799355984, "auroc_anom_vs_far": 0.8725553853559497, "topk_halo_share": 0.13814046391886148, "topk_anom_share": 0.6459202662871951, "halo_frac": 0.3005659741396617}, {"train": "pd", "proj": "win", "auroc_anom_vs_halo": 0.7177344178536533, "auroc_anom_vs_far": 0.9465117681202497, "topk_halo_share": 0.45727008670737074, "topk_anom_share": 0.5180405467027804, "halo_frac": 0.3005659741396617}], "contam": [{"train": "adapt_r3", "proj": "tgt", "auroc_anom_vs_halo": 0.8901345618431756, "auroc_anom_vs_far": 0.934009159199102, "topk_halo_share": 0.14259808558394269, "topk_anom_share": 0.7402744509681661, "halo_frac": 0.3691926655699504}, {"train": "adapt_r3", "proj": "win", "auroc_anom_vs_halo": 0.7305847648725357, "auroc_anom_vs_far": 0.9658797426650932, "topk_halo_share": 0.4699898444670293, "topk_anom_share": 0.5038708261488759, "halo_frac": 0.3691926655699504}, {"train": "base", "proj": "tgt", "auroc_anom_vs_halo": 0.8661900131575048, "auroc_anom_vs_far": 0.9099017373721486, "topk_halo_share": 0.1486483734758504, "topk_anom_share": 0.69943163443027, "halo_frac": 0.3691926655699504}, {"train": "base", "proj": "win", "auroc_anom_vs_halo": 0.697720996414287, "auroc_anom_vs_far": 0.9512552075755492, "topk_halo_share": 0.4959149624370133, "topk_anom_share": 0.4928416517069235, "halo_frac": 0.3691926655699504}, {"train": "pd", "proj": "tgt", "auroc_anom_vs_halo": 0.8238816432956387, "auroc_anom_vs_far": 0.8572002701829479, "topk_halo_share": 0.14683062387841583, "topk_anom_share": 0.6313266799722169, "halo_frac": 0.3691926655699504}, {"train": "pd", "proj": "win", "auroc_anom_vs_halo": 0.6659582229621182, "auroc_anom_vs_far": 0.9338084956207974, "topk_halo_share": 0.5244516429885641, "topk_anom_share": 0.4585905687132444, "halo_frac": 0.3691926655699504}], "real": [{"train": "adapt_r3", "proj": "tgt", "auroc_anom_vs_halo": 0.6946638394565718, "auroc_anom_vs_far": 0.7816897538931232, "topk_halo_share": 0.13183383605417948, "topk_anom_share": 0.2811636439792875, "halo_frac": 0.15897878420772021}, {"train": "adapt_r3", "proj": "win", "auroc_anom_vs_halo": 0.5848369667890662, "auroc_anom_vs_far": 0.7575178705410098, "topk_halo_share": 0.2443792817658236, "topk_anom_share": 0.25471906934836774, "halo_frac": 0.15897878420772021}, {"train": "base", "proj": "tgt", "auroc_anom_vs_halo": 0.6642394548437832, "auroc_anom_vs_far": 0.7738576499234189, "topk_halo_share": 0.16527884046760424, "topk_anom_share": 0.27460116414853236, "halo_frac": 0.15897878420772021}, {"train": "base", "proj": "win", "auroc_anom_vs_halo": 0.5734316364577563, "auroc_anom_vs_far": 0.7662495441661898, "topk_halo_share": 0.2744570096724054, "topk_anom_share": 0.25175054991515405, "halo_frac": 0.15897878420772021}, {"train": "pd", "proj": "tgt", "auroc_anom_vs_halo": 0.6723267322767683, "auroc_anom_vs_far": 0.7548625428256954, "topk_halo_share": 0.13567118674440753, "topk_anom_share": 0.15572194999050676, "halo_frac": 0.15897878420772021}, {"train": "pd", "proj": "win", "auroc_anom_vs_halo": 0.5257922259238188, "auroc_anom_vs_far": 0.7236761571564201, "topk_halo_share": 0.16733181632776833, "topk_anom_share": 0.13039928634619127, "halo_frac": 0.15897878420772021}], "smd": [{"train": "adapt_r3", "proj": "tgt", "auroc_anom_vs_halo": 0.7122569468731298, "auroc_anom_vs_far": 0.8250590432399718, "topk_halo_share": 0.011755420135672778, "topk_anom_share": 0.400875876231569, "halo_frac": 0.004041848867930971}, {"train": "adapt_r3", "proj": "win", "auroc_anom_vs_halo": 0.6150319187944004, "auroc_anom_vs_far": 0.8605345542043986, "topk_halo_share": 0.03890658211500141, "topk_anom_share": 0.4805377992641818, "halo_frac": 0.004041848867930971}, {"train": "base", "proj": "tgt", "auroc_anom_vs_halo": 0.7115928528319679, "auroc_anom_vs_far": 0.8511399398369422, "topk_halo_share": 0.015019840457576638, "topk_anom_share": 0.40484418737928934, "halo_frac": 0.004041848867930971}, {"train": "base", "proj": "win", "auroc_anom_vs_halo": 0.5941549188390401, "auroc_anom_vs_far": 0.874514227704582, "topk_halo_share": 0.04211403464075579, "topk_anom_share": 0.45623828245433734, "halo_frac": 0.004041848867930971}, {"train": "pd", "proj": "tgt", "auroc_anom_vs_halo": 0.6560804237631873, "auroc_anom_vs_far": 0.7771627172834227, "topk_halo_share": 0.010644365388791738, "topk_anom_share": 0.2618764057990282, "halo_frac": 0.004041848867930971}, {"train": "pd", "proj": "win", "auroc_anom_vs_halo": 0.5426804619100676, "auroc_anom_vs_far": 0.81441738456199, "topk_halo_share": 0.03987799427230334, "topk_anom_share": 0.3429182581280502, "halo_frac": 0.004041848867930971}], "nasa": [{"train": "adapt_r3", "proj": "tgt", "auroc_anom_vs_halo": 0.61187488433037, "auroc_anom_vs_far": 0.6044710491245733, "topk_halo_share": 0.002891871247313034, "topk_anom_share": 0.21504641125981344, "halo_frac": 0.0025957644295768532}, {"train": "adapt_r3", "proj": "win", "auroc_anom_vs_halo": 0.6032090525161123, "auroc_anom_vs_far": 0.6064206611715681, "topk_halo_share": 0.004413945935172941, "topk_anom_share": 0.22404992930665663, "halo_frac": 0.0025957644295768532}, {"train": "base", "proj": "tgt", "auroc_anom_vs_halo": 0.5841999805765187, "auroc_anom_vs_far": 0.5756933647008116, "topk_halo_share": 0.003178901836327044, "topk_anom_share": 0.19029209508732667, "halo_frac": 0.0025957644295768532}, {"train": "base", "proj": "win", "auroc_anom_vs_halo": 0.5817279145833901, "auroc_anom_vs_far": 0.5797232278102045, "topk_halo_share": 0.0046507877304936265, "topk_anom_share": 0.2049527177195179, "halo_frac": 0.0025957644295768532}, {"train": "pd", "proj": "tgt", "auroc_anom_vs_halo": 0.5068210279400313, "auroc_anom_vs_far": 0.5138805343751666, "topk_halo_share": 0.0035375621854228535, "topk_anom_share": 0.1508721483142483, "halo_frac": 0.0025957644295768532}, {"train": "pd", "proj": "win", "auroc_anom_vs_halo": 0.5110611868408775, "auroc_anom_vs_far": 0.5230824161311809, "topk_halo_share": 0.004425253734550537, "topk_anom_share": 0.16328800256707085, "halo_frac": 0.0025957644295768532}], "semi": [{"train": "adapt_r3", "proj": "tgt", "auroc_anom_vs_halo": 0.6643877262235799, "auroc_anom_vs_far": 0.6860659777499316, "topk_halo_share": 0.22251892937302697, "topk_anom_share": 0.2747726183009687, "halo_frac": 0.3238494789840822}, {"train": "adapt_r3", "proj": "win", "auroc_anom_vs_halo": 0.5936263070838965, "auroc_anom_vs_far": 0.6696359546730573, "topk_halo_share": 0.3391615776372154, "topk_anom_share": 0.2305178068204673, "halo_frac": 0.3238494789840822}, {"train": "base", "proj": "tgt", "auroc_anom_vs_halo": 0.6441509019620613, "auroc_anom_vs_far": 0.6755147577056999, "topk_halo_share": 0.22941358296057363, "topk_anom_share": 0.2723274283761705, "halo_frac": 0.3238494789840822}, {"train": "base", "proj": "win", "auroc_anom_vs_halo": 0.5848488193460178, "auroc_anom_vs_far": 0.6646737126361525, "topk_halo_share": 0.3475059493671174, "topk_anom_share": 0.23028894739461037, "halo_frac": 0.3238494789840822}, {"train": "pd", "proj": "tgt", "auroc_anom_vs_halo": 0.7127031871854428, "auroc_anom_vs_far": 0.7350519522214631, "topk_halo_share": 0.1908591306113407, "topk_anom_share": 0.3323148901468777, "halo_frac": 0.3238494789840822}, {"train": "pd", "proj": "win", "auroc_anom_vs_halo": 0.6319718990881014, "auroc_anom_vs_far": 0.7338572793400862, "topk_halo_share": 0.37521859178132805, "topk_anom_share": 0.2668287545560858, "halo_frac": 0.3238494789840822}]}, "caidas": {"contam": [{"method": "Artículo", "drop": 0.16986344192422925}, {"method": "M2 (recorte z>2)", "drop": 0.1496836916790108}, {"method": "M1+M2", "drop": 0.1401941792400486}, {"method": "M3", "drop": 0.08256519707735809}, {"method": "M2+M3", "drop": 0.05442574787774757}, {"method": "Final", "drop": 0.04260694067136882}, {"method": "Error LSTM", "drop": 0.08222004564153126}, {"method": "LOF", "drop": 0.16584775716609995}, {"method": "IForest", "drop": 0.036395653750919105}], "semi": [{"method": "Artículo", "drop": 0.013879404035366649}, {"method": "M2 (recorte z>2)", "drop": 0.018482080644122826}, {"method": "M1+M2", "drop": 0.019526322982384815}, {"method": "M3", "drop": -0.0021771564555275067}, {"method": "M2+M3", "drop": -0.0007877739288337978}, {"method": "Final", "drop": -0.0026577884962082935}, {"method": "Error LSTM", "drop": -0.005901694721047404}, {"method": "LOF", "drop": 0.016708993042925287}, {"method": "IForest", "drop": -0.0237607763808302}]}, "real_iqr": {"cicids": {"anom": 0.0128089874436669, "iqr": 0.16756740876534287}, "creditcard": {"anom": 0.0017277161488785647, "iqr": 0.12844059571090954}, "gecco": {"anom": 0.012463623693883001, "iqr": 0.16432919564134224}, "swan_sf": {"anom": 0.23791700705223323, "iqr": 0.13910655040762906}}, "h6_series": [{"ds": "machine-1-1", "block": "smd", "anom_rate": 0.0947224077915685, "d_auroc_m3": -0.016681089462596277, "d_auprc_m3": -0.04817992925880965, "d_auroc_final": -0.07293556576992444}, {"ds": "machine-1-2", "block": "smd", "anom_rate": 0.0229117348664186, "d_auroc_m3": -0.018305897258159343, "d_auprc_m3": -0.025230634261929397, "d_auroc_final": -0.03633826657986228}, {"ds": "machine-1-3", "block": "smd", "anom_rate": 0.034523557997042, "d_auroc_m3": -0.030416038814546686, "d_auprc_m3": -0.06114158208911202, "d_auroc_final": -0.07040935101090628}, {"ds": "machine-1-4", "block": "smd", "anom_rate": 0.0304195361020744, "d_auroc_m3": -0.03336680731670705, "d_auprc_m3": -0.11172358494805673, "d_auroc_final": -0.07368162979747372}, {"ds": "machine-1-5", "block": "smd", "anom_rate": 0.0042251140780801, "d_auroc_m3": -0.05467404955872368, "d_auprc_m3": -0.14076085638853592, "d_auroc_final": -0.049866089613034625}, {"ds": "machine-1-6", "block": "smd", "anom_rate": 0.156779840175891, "d_auroc_m3": -0.010590660729357104, "d_auprc_m3": 0.03207879114577972, "d_auroc_final": -0.029395428805969792}, {"ds": "machine-1-7", "block": "smd", "anom_rate": 0.1013567775476562, "d_auroc_m3": -0.00938201876287248, "d_auprc_m3": -0.015459742098015539, "d_auroc_final": -0.09028095116656298}, {"ds": "machine-1-8", "block": "smd", "anom_rate": 0.0322471577701703, "d_auroc_m3": -0.02415883443207656, "d_auprc_m3": -0.05697483507803508, "d_auroc_final": -0.06398429206875289}, {"ds": "machine-2-1", "block": "smd", "anom_rate": 0.0494589110585052, "d_auroc_m3": -0.04921494171872187, "d_auprc_m3": -0.06635227963681756, "d_auroc_final": -0.0707607658630518}, {"ds": "machine-2-2", "block": "smd", "anom_rate": 0.11972783365734081, "d_auroc_m3": -0.054135989124424744, "d_auprc_m3": -0.05559475954054355, "d_auroc_final": -0.043457000651617284}, {"ds": "machine-2-3", "block": "smd", "anom_rate": 0.0113737262695023, "d_auroc_m3": -0.02405205414898326, "d_auprc_m3": -0.05227264348758398, "d_auroc_final": -0.033822779827141214}, {"ds": "machine-2-4", "block": "smd", "anom_rate": 0.0716248784406579, "d_auroc_m3": -0.02593021381976668, "d_auprc_m3": -0.040856179672440374, "d_auroc_final": -0.09469681510695571}, {"ds": "machine-2-5", "block": "smd", "anom_rate": 0.0414358800896368, "d_auroc_m3": -0.015018498864412755, "d_auprc_m3": -0.003969702356308336, "d_auroc_final": -0.04776208750007871}, {"ds": "machine-2-6", "block": "smd", "anom_rate": 0.0147709458282529, "d_auroc_m3": -0.01123975327124771, "d_auprc_m3": -0.0817343026613348, "d_auroc_final": -0.04359530666965561}, {"ds": "machine-2-7", "block": "smd", "anom_rate": 0.0176261729647476, "d_auroc_m3": 0.002023402576505906, "d_auprc_m3": 0.06278861344464015, "d_auroc_final": 0.0026753555204113155}, {"ds": "machine-2-8", "block": "smd", "anom_rate": 0.006803296006761, "d_auroc_m3": -0.00347190804578279, "d_auprc_m3": -0.02528507955421533, "d_auroc_final": -0.00419577056264242}, {"ds": "machine-2-9", "block": "smd", "anom_rate": 0.06118393529493791, "d_auroc_m3": -0.01664321679918257, "d_auprc_m3": -0.08370019498179948, "d_auroc_final": -0.05840945468187486}, {"ds": "machine-3-1", "block": "smd", "anom_rate": 0.01074593538483, "d_auroc_m3": -0.011108902383502262, "d_auprc_m3": -0.12791562371988444, "d_auroc_final": -0.030193923484993612}, {"ds": "machine-3-10", "block": "smd", "anom_rate": 0.0442612555485098, "d_auroc_m3": -0.015533377810493465, "d_auprc_m3": -0.025698943033401644, "d_auroc_final": -0.0025775080420518126}, {"ds": "machine-3-11", "block": "smd", "anom_rate": 0.006909065531439701, "d_auroc_m3": -0.016962740546718202, "d_auprc_m3": -0.09384379610843974, "d_auroc_final": -0.07021776088360776}, {"ds": "machine-3-2", "block": "smd", "anom_rate": 0.046862455102472, "d_auroc_m3": -0.016520961115354837, "d_auprc_m3": -0.005078869973986169, "d_auroc_final": -0.02119055732403352}, {"ds": "machine-3-3", "block": "smd", "anom_rate": 0.0267061060638073, "d_auroc_m3": -0.03521208150990851, "d_auprc_m3": -0.01967415926407748, "d_auroc_final": -0.08151903130730964}, {"ds": "machine-3-4", "block": "smd", "anom_rate": 0.0413125290709966, "d_auroc_m3": -0.02662651084325507, "d_auprc_m3": -0.01999432838929807, "d_auroc_final": -0.07016164478849818}, {"ds": "machine-3-5", "block": "smd", "anom_rate": 0.0180104003720458, "d_auroc_m3": -0.009979279820655496, "d_auprc_m3": -0.08609267014196659, "d_auroc_final": -0.030003723204599764}, {"ds": "machine-3-6", "block": "smd", "anom_rate": 0.0416201896263246, "d_auroc_m3": -0.01854607778593742, "d_auprc_m3": -0.059421426749930895, "d_auroc_final": -0.06118794732616539}, {"ds": "machine-3-7", "block": "smd", "anom_rate": 0.0151393588446645, "d_auroc_m3": -0.035419756695477256, "d_auprc_m3": -0.06222636420378985, "d_auroc_final": -0.05445847188985797}, {"ds": "machine-3-8", "block": "smd", "anom_rate": 0.0478266936440382, "d_auroc_m3": -0.039385451310070874, "d_auprc_m3": -0.0746936968356402, "d_auroc_final": -0.0648250755683003}, {"ds": "machine-3-9", "block": "smd", "anom_rate": 0.010566695727986, "d_auroc_m3": -0.012334710138150351, "d_auprc_m3": -0.15649640241294704, "d_auroc_final": 0.008246032554756977}, {"ds": "A-1", "block": "nasa", "anom_rate": 0.0098814229249011, "d_auroc_m3": -0.0216324444536532, "d_auprc_m3": -0.0014924261056618211, "d_auroc_final": -0.03553667751003187}, {"ds": "A-2", "block": "nasa", "anom_rate": 0.0140934484509903, "d_auroc_m3": 0.001978153298179075, "d_auprc_m3": 0.0007268119807774618, "d_auroc_final": 0.01889420650528184}, {"ds": "A-3", "block": "nasa", "anom_rate": 0.022774580629362, "d_auroc_m3": 0.043263705601879754, "d_auprc_m3": 0.0021155218577651597, "d_auroc_final": 0.037803156151774486}, {"ds": "A-4", "block": "nasa", "anom_rate": 0.013802536682417299, "d_auroc_m3": 0.08712794246774824, "d_auprc_m3": 0.002283685356974022, "d_auroc_final": 0.02332221264260098}, {"ds": "A-5", "block": "nasa", "anom_rate": 0.0109559613319011, "d_auroc_m3": -0.0344495834823938, "d_auprc_m3": -0.060658734884537546, "d_auroc_final": -0.060439771043082735}, {"ds": "A-6", "block": "nasa", "anom_rate": 0.009286523216308, "d_auroc_m3": -0.022992293708945177, "d_auprc_m3": -0.13731999616025847, "d_auroc_final": -0.02915621131519952}, {"ds": "A-7", "block": "nasa", "anom_rate": 0.2794134760851856, "d_auroc_m3": -0.01490849662123983, "d_auprc_m3": 0.02910657634342062, "d_auroc_final": -0.07161717044039578}, {"ds": "A-8", "block": "nasa", "anom_rate": 0.45424013434089006, "d_auroc_m3": 0.015226132198933873, "d_auprc_m3": 0.014252063158752826, "d_auroc_final": 0.11309253461088686}, {"ds": "A-9", "block": "nasa", "anom_rate": 0.45807527393997133, "d_auroc_m3": -0.00031293823181494096, "d_auprc_m3": 0.0005891471913139212, "d_auroc_final": 0.07350832890458481}, {"ds": "B-1", "block": "nasa", "anom_rate": 0.0088683487384461, "d_auroc_m3": -0.016085625282888316, "d_auprc_m3": -0.0013619302672033002, "d_auroc_final": 0.32730335383441145}, {"ds": "C-1", "block": "nasa", "anom_rate": 0.1401617250673854, "d_auroc_m3": -0.003775821879270169, "d_auprc_m3": -0.033452317999532244, "d_auroc_final": 0.015134099616858188}, {"ds": "C-2", "block": "nasa", "anom_rate": 0.0680576254346746, "d_auroc_m3": 0.01650817860644649, "d_auprc_m3": -0.022471411079012293, "d_auroc_final": 0.039205951473083034}, {"ds": "D-1", "block": "nasa", "anom_rate": 0.3824814071538189, "d_auroc_m3": 0.0025608433578547807, "d_auprc_m3": 0.0030513888387046495, "d_auroc_final": 0.5820374323536562}, {"ds": "D-11", "block": "nasa", "anom_rate": 0.0082510482889219, "d_auroc_m3": -0.0410663339683221, "d_auprc_m3": -0.0013364390972859415, "d_auroc_final": -0.10213839177913125}, {"ds": "D-12", "block": "nasa", "anom_rate": 0.3453045685279188, "d_auroc_m3": 3.7826873878099576e-05, "d_auprc_m3": 0.013685836089963654, "d_auroc_final": 0.3341986640346002}, {"ds": "D-13", "block": "nasa", "anom_rate": 0.0211147540983606, "d_auroc_m3": -0.007327261954690911, "d_auprc_m3": 0.007210166312592241, "d_auroc_final": -0.06020292850818498}, {"ds": "D-14", "block": "nasa", "anom_rate": 0.0858136838036335, "d_auroc_m3": 0.07807972877740321, "d_auprc_m3": 0.010797621589559228, "d_auroc_final": 0.061471154029293595}, {"ds": "D-15", "block": "nasa", "anom_rate": 0.3014150943396226, "d_auroc_m3": 0.015751739033495804, "d_auprc_m3": 0.005447961272035617, "d_auroc_final": 0.03155884817495269}, {"ds": "D-16", "block": "nasa", "anom_rate": 0.3023687877380399, "d_auroc_m3": -0.011452318567562791, "d_auprc_m3": -0.027359502907823463, "d_auroc_final": 0.004391993471070776}, {"ds": "D-2", "block": "nasa", "anom_rate": 0.49292976510459263, "d_auroc_m3": 0.040530219208910645, "d_auprc_m3": 0.04344093277961736, "d_auroc_final": 0.051138805136209387}, {"ds": "D-3", "block": "nasa", "anom_rate": 0.3808416647291328, "d_auroc_m3": 0.000629305083867715, "d_auprc_m3": 0.009205558836950123, "d_auroc_final": 0.1964200432187665}, {"ds": "D-4", "block": "nasa", "anom_rate": 0.382809721398933, "d_auroc_m3": 0.023287183107087328, "d_auprc_m3": 0.02406378593328906, "d_auroc_final": 0.1183529569652284}, {"ds": "D-5", "block": "nasa", "anom_rate": 0.0067193675889328, "d_auroc_m3": -0.04553316219709802, "d_auprc_m3": -0.03893028676436063, "d_auroc_final": 0.04781463188803836}, {"ds": "D-6", "block": "nasa", "anom_rate": 0.0103237318378791, "d_auroc_m3": 0.011085036528264758, "d_auprc_m3": -0.02026437789827551, "d_auroc_final": 0.08057984148561526}, {"ds": "D-7", "block": "nasa", "anom_rate": 0.3528406102051551, "d_auroc_m3": 0.030128933155788462, "d_auprc_m3": 0.02693859792669323, "d_auroc_final": 0.0728446161994627}, {"ds": "D-8", "block": "nasa", "anom_rate": 0.006508422664624801, "d_auroc_m3": -0.03674789376251464, "d_auprc_m3": -0.0031653178390447807, "d_auroc_final": -0.036831513594519305}, {"ds": "D-9", "block": "nasa", "anom_rate": 0.1543159609120521, "d_auroc_m3": 0.0025574174690708995, "d_auprc_m3": 0.02301475495813521, "d_auroc_final": 0.020102130706018406}, {"ds": "E-1", "block": "nasa", "anom_rate": 0.0599197924038688, "d_auroc_m3": 0.02237981505448583, "d_auprc_m3": -0.0004247959395729667, "d_auroc_final": 0.0364736363726178}, {"ds": "E-10", "block": "nasa", "anom_rate": 0.0380299988189441, "d_auroc_m3": 0.036228528724325026, "d_auprc_m3": 0.001725857899245481, "d_auroc_final": 0.034270882185847285}, {"ds": "E-11", "block": "nasa", "anom_rate": 0.0348041529023124, "d_auroc_m3": 0.038119992790239476, "d_auprc_m3": 0.020200599117972964, "d_auroc_final": 0.038891644343342024}, {"ds": "E-12", "block": "nasa", "anom_rate": 0.0687986783101251, "d_auroc_m3": 0.010906078162302824, "d_auprc_m3": -0.0019914677724680724, "d_auroc_final": 0.18340139547127232}, {"ds": "E-13", "block": "nasa", "anom_rate": 0.0306905370843989, "d_auroc_m3": -0.004327485698917777, "d_auprc_m3": -0.00040076025019704364, "d_auroc_final": -0.0014569114023419982}, {"ds": "E-2", "block": "nasa", "anom_rate": 0.1645867671297386, "d_auroc_m3": -0.007273446282577978, "d_auprc_m3": 5.3936724679098935e-05, "d_auroc_final": 0.036908853120821705}, {"ds": "E-3", "block": "nasa", "anom_rate": 0.3862619421937356, "d_auroc_m3": 0.017899842993790716, "d_auprc_m3": 0.00821255913369473, "d_auroc_final": 0.16615034963956432}, {"ds": "E-4", "block": "nasa", "anom_rate": 0.3381433381433381, "d_auroc_m3": 0.028507923372423916, "d_auprc_m3": 0.016227014467744938, "d_auroc_final": 0.02792139145150352}, {"ds": "E-5", "block": "nasa", "anom_rate": 0.0388808139534883, "d_auroc_m3": 0.006074000789121847, "d_auprc_m3": -0.020642116424549217, "d_auroc_final": 0.008465393471488447}, {"ds": "E-6", "block": "nasa", "anom_rate": 0.0079883805374001, "d_auroc_m3": 0.024405659819276226, "d_auprc_m3": -0.09642867998228036, "d_auroc_final": 0.029700186343670998}, {"ds": "E-7", "block": "nasa", "anom_rate": 0.0339700193423597, "d_auroc_m3": -0.02351992076495313, "d_auprc_m3": -0.01739908853034574, "d_auroc_final": 0.0118159619963919}, {"ds": "E-8", "block": "nasa", "anom_rate": 0.0733458912173298, "d_auroc_m3": 0.00532160543009641, "d_auprc_m3": -0.0010787424190385475, "d_auroc_final": 0.10516243772729328}, {"ds": "E-9", "block": "nasa", "anom_rate": 0.0424733785091965, "d_auroc_m3": -0.015474337551931394, "d_auprc_m3": -0.02145723286316112, "d_auroc_final": 0.13984722748781897}, {"ds": "F-1", "block": "nasa", "anom_rate": 0.0118183945705593, "d_auroc_m3": 0.035715667481490576, "d_auprc_m3": 0.0007900336841187589, "d_auroc_final": -0.01710661297035565}, {"ds": "F-2", "block": "nasa", "anom_rate": 0.3421052631578947, "d_auroc_m3": 0.058019054561227024, "d_auprc_m3": 0.01668676414543052, "d_auroc_final": 0.19006330837304303}, {"ds": "F-3", "block": "nasa", "anom_rate": 0.0049172463420484, "d_auroc_m3": -0.10914494513150497, "d_auprc_m3": -0.17948925047798434, "d_auroc_final": -0.1428509276053349}, {"ds": "F-4", "block": "nasa", "anom_rate": 0.0209810874704491, "d_auroc_m3": -0.010978518257143133, "d_auprc_m3": -0.016038157221912784, "d_auroc_final": 0.08848199368259058}, {"ds": "F-5", "block": "nasa", "anom_rate": 0.0388774459320288, "d_auroc_m3": 0.04785633059716188, "d_auprc_m3": 0.013850225998391669, "d_auroc_final": 0.08427218844634321}, {"ds": "F-7", "block": "nasa", "anom_rate": 0.0843301435406698, "d_auroc_m3": 0.1033147368361455, "d_auprc_m3": 0.022077017156985276, "d_auroc_final": 0.15250393882354635}, {"ds": "F-8", "block": "nasa", "anom_rate": 0.21151490404246634, "d_auroc_m3": 0.011646495204237384, "d_auprc_m3": 0.004829510907179951, "d_auroc_final": 0.0067947469552855355}, {"ds": "G-1", "block": "nasa", "anom_rate": 0.0143517969398647, "d_auroc_m3": 0.02048592256665771, "d_auprc_m3": 0.00106102746990068, "d_auroc_final": -0.06372885401438078}, {"ds": "G-2", "block": "nasa", "anom_rate": 0.0055987983067049, "d_auroc_m3": -0.012307996329070758, "d_auprc_m3": -0.0003449794604667789, "d_auroc_final": 0.0025271132964006515}, {"ds": "G-3", "block": "nasa", "anom_rate": 0.006481128478841, "d_auroc_m3": -0.010453001870996625, "d_auprc_m3": -0.0059306536748641, "d_auroc_final": 0.00033908677310778845}, {"ds": "G-4", "block": "nasa", "anom_rate": 0.0040821701343165, "d_auroc_m3": -0.08194776778288182, "d_auprc_m3": -0.1105167905190774, "d_auroc_final": -0.17343177523853404}, {"ds": "G-6", "block": "nasa", "anom_rate": 0.0117414554754708, "d_auroc_m3": -0.008523400275564641, "d_auprc_m3": -0.0009294723070650407, "d_auroc_final": 0.04991165861674973}, {"ds": "G-7", "block": "nasa", "anom_rate": 0.0335377299461894, "d_auroc_m3": -0.06826841127780758, "d_auprc_m3": -0.025905350333062682, "d_auroc_final": -0.0476858714326851}, {"ds": "M-1", "block": "nasa", "anom_rate": 0.5096025011165699, "d_auroc_m3": -0.043713851493193734, "d_auprc_m3": -0.026922433122017753, "d_auroc_final": -0.0672222142402168}, {"ds": "M-2", "block": "nasa", "anom_rate": 0.5096025011165699, "d_auroc_m3": 0.009129338818567367, "d_auprc_m3": -0.009554722242935965, "d_auroc_final": -0.10261195161627634}, {"ds": "M-3", "block": "nasa", "anom_rate": 0.1201531833413116, "d_auroc_m3": -0.005789898079065714, "d_auprc_m3": -0.019557284989539056, "d_auroc_final": -0.0027569807819862735}, {"ds": "M-4", "block": "nasa", "anom_rate": 0.1255, "d_auroc_m3": -0.0027396417759494307, "d_auprc_m3": -0.018547941524995323, "d_auroc_final": -0.02650575513839437}, {"ds": "M-5", "block": "nasa", "anom_rate": 0.1328918322295805, "d_auroc_m3": -0.003698296919298183, "d_auprc_m3": -0.029571981303338546, "d_auroc_final": 0.0030017051105953874}, {"ds": "M-6", "block": "nasa", "anom_rate": 0.0895077076081551, "d_auroc_m3": -0.03751866011287097, "d_auprc_m3": -0.1372817843488981, "d_auroc_final": -0.11474725408095143}, {"ds": "M-7", "block": "nasa", "anom_rate": 0.0476864966949952, "d_auroc_m3": -0.018524227236804247, "d_auprc_m3": -0.1091598734896922, "d_auroc_final": -0.01229990624248356}, {"ds": "P-1", "block": "nasa", "anom_rate": 0.0886972953820715, "d_auroc_m3": 0.03192679330617765, "d_auprc_m3": 0.0032405546715894434, "d_auroc_final": 0.016338402089075665}, {"ds": "P-10", "block": "nasa", "anom_rate": 0.0216100296931705, "d_auroc_m3": 0.011550901525301582, "d_auprc_m3": -0.044846952792776895, "d_auroc_final": -0.0030799486718123825}, {"ds": "P-11", "block": "nasa", "anom_rate": 0.0651987417786674, "d_auroc_m3": 0.001885334320812726, "d_auprc_m3": 0.015754265726614314, "d_auroc_final": 0.09713523637788246}, {"ds": "P-14", "block": "nasa", "anom_rate": 0.029858132629495205, "d_auroc_m3": 0.011227090518111993, "d_auprc_m3": -0.03799665071619304, "d_auroc_final": 0.034954592042357624}, {"ds": "P-15", "block": "nasa", "anom_rate": 0.007452093683463401, "d_auroc_m3": 0.0029385225666955783, "d_auprc_m3": 0.0002113424614073791, "d_auroc_final": 0.01883991351277725}, {"ds": "P-2", "block": "nasa", "anom_rate": 0.1500428344143923, "d_auroc_m3": -0.03183587662089804, "d_auprc_m3": -0.007073909461987882, "d_auroc_final": 0.07946185186098664}, {"ds": "P-3", "block": "nasa", "anom_rate": 0.1580130100532229, "d_auroc_m3": -0.012458416500332636, "d_auprc_m3": -0.0013193034134651727, "d_auroc_final": 0.04101841618070223}, {"ds": "P-4", "block": "nasa", "anom_rate": 0.057198192382182, "d_auroc_m3": -0.01715847663493042, "d_auprc_m3": -0.0022017713999907956, "d_auroc_final": -0.0171082723864886}, {"ds": "P-7", "block": "nasa", "anom_rate": 0.2055272002987676, "d_auroc_m3": -0.07447574103498611, "d_auprc_m3": -0.09772278309901838, "d_auroc_final": -0.05782869787661793}, {"ds": "R-1", "block": "nasa", "anom_rate": 0.011240632805995, "d_auroc_m3": -0.021521160927008953, "d_auprc_m3": -0.0018496298148910177, "d_auroc_final": -0.021446307125839392}, {"ds": "S-1", "block": "nasa", "anom_rate": 0.0614287673111202, "d_auroc_m3": 0.03814913127413122, "d_auprc_m3": 0.008536214418542479, "d_auroc_final": -0.06649274757382867}, {"ds": "S-2", "block": "nasa", "anom_rate": 0.0061486864169927, "d_auroc_m3": -0.11074240719910022, "d_auprc_m3": -0.33670686575244346, "d_auroc_final": -0.17719091931690367}, {"ds": "T-1", "block": "nasa", "anom_rate": 0.179146256123163, "d_auroc_m3": -0.010129636659917562, "d_auprc_m3": -0.0037282902436198417, "d_auroc_final": 0.18074033187399352}, {"ds": "T-12", "block": "nasa", "anom_rate": 0.05058528428093641, "d_auroc_m3": -0.01140357580852358, "d_auprc_m3": -0.0860282673157473, "d_auroc_final": -0.01756680531749588}, {"ds": "T-13", "block": "nasa", "anom_rate": 0.1053511705685618, "d_auroc_m3": -0.06586819462987681, "d_auprc_m3": -0.034854153571091034, "d_auroc_final": -0.09188251001335113}, {"ds": "T-2", "block": "nasa", "anom_rate": 0.2056597181786421, "d_auroc_m3": -0.0022208495082886026, "d_auprc_m3": -0.0014913109552679438, "d_auroc_final": 0.016336747666381712}, {"ds": "T-3", "block": "nasa", "anom_rate": 0.021543144830816, "d_auroc_m3": -0.013547416641087051, "d_auprc_m3": -0.0035756820097929236, "d_auroc_final": -0.036888497536561315}, {"ds": "T-4", "block": "nasa", "anom_rate": 0.0316659017898118, "d_auroc_m3": -0.036827391991208214, "d_auprc_m3": -0.0013723364899859217, "d_auroc_final": -0.025601346246308165}, {"ds": "T-5", "block": "nasa", "anom_rate": 0.0119266055045871, "d_auroc_m3": -0.03926683808299414, "d_auprc_m3": -0.16717150356833904, "d_auroc_final": -0.09764123991143503}, {"ds": "T-8", "block": "nasa", "anom_rate": 0.0688723835246455, "d_auroc_m3": -0.029069089564760064, "d_auprc_m3": -0.017098049383676414, "d_auroc_final": -0.03302193974036327}, {"ds": "T-9", "block": "nasa", "anom_rate": 0.10586011342155, "d_auroc_m3": -0.06910299003322262, "d_auprc_m3": -0.03886659737660619, "d_auroc_final": -0.3027408637873753}]}

In [ ]:
FINAL = "**Método final** (recorte adaptativo + proyección + IQR)"
ART_S, ART_D = "Artículo · umbral estático", "Artículo · umbral dinámico (usa etiquetas)"
M3N, M1M2N = "M3 · proyección al objetivo", "M1+M2 (recorte adaptativo, sin M3)"
CORTO = {ART_D: "Artículo (dinámico, con etiquetas)", ART_S: "Artículo (estático)", "M1 · umbral IQR": "M1",
         "M2 · reponderación suave": "M2 suave", "M2 · recorte z>2": "M2 recorte", M3N: "M3", "M1+M3": "M1+M3",
         "M2+M3 (recorte z>2)": "M2+M3", M1M2N: "M1+M2", "M1+M2+M3 con recorte z>2": "M1+M2+M3 (z>2)",
         FINAL: "MÉTODO FINAL (M1+M2+M3)", "Error del LSTM (PD) + IQR": "Error LSTM", "LOF + IQR": "LOF", "IForest + IQR": "IForest"}
BLOQUES = {"synth": "Sintéticos", "contam": "Contaminación", "real": "Reales del artículo", "smd": "SMD",
           "nasa": "SMAP+MSL", "semi": "Semisintético (fondo real)"}
BLQ = [b for b in BLOQUES if b in RESULTADOS]
if not BLQ:
    display(Markdown("> ⏳ **Pendiente.** La evaluación final todavía no termina; este cuaderno se regenera con las cifras."))
else:
    filas = []
    for b in BLQ:
        mm = RESULTADOS[b]["methods"]
        fila = {"bloque": BLOQUES[b]}
        for m, nom in ((ART_S, "artículo"), (M3N, "M3 sola"), (M1M2N, "M1+M2"), (FINAL, "FINAL")):
            fila[f"AUPRC {nom}"] = mm[m]["auprc"]
        fila["AUROC artículo"], fila["AUROC FINAL"] = mm[ART_S]["auroc"], mm[FINAL]["auroc"]
        fila["F1 artículo (con etiquetas)"], fila["F1 FINAL (sin etiquetas)"] = mm[ART_D]["f1_bin"], mm[FINAL]["f1_bin"]
        filas.append(fila)
    resumen = pd.DataFrame(filas).set_index("bloque")
    display(Markdown("### 9.1 Vista general: el método final frente al artículo en los 6 bloques"))
    display(resumen.round(3))
    fig, ax = plt.subplots(figsize=(12, 3.6))
    xs = np.arange(len(BLQ)); wd = 0.2
    for i, (col, c) in enumerate((("AUPRC artículo", "#7f7f7f"), ("AUPRC M3 sola", "#1f77b4"),
                                  ("AUPRC M1+M2", "#8c564b"), ("AUPRC FINAL", "#c0392b"))):
        ax.bar(xs + (i - 1.5) * wd, resumen[col], wd, color=c, label=col.replace("AUPRC ", ""))
    ax.set_xticks(xs); ax.set_xticklabels([BLOQUES[b] for b in BLQ], fontsize=8); ax.set_ylabel("AUPRC")
    ax.legend(fontsize=8, ncol=4); ax.set_title("AUPRC medio por bloque (más alto = mejor score)")
    plt.show()

**Cómo leer la vista general:**
* En **sintéticos y contaminación** el método final mejora muchísimo el AUPRC (de ~0.5 a ~0.8) y también el F1,
  **aun comparándolo con el artículo que usa etiquetas**.
* En el **semisintético** y en los **reales del artículo** mejora el AUPRC, con menor magnitud.
* En **SMD** queda **por debajo** del artículo, y en **SMAP+MSL** empata. Fíjate en la columna **M1+M2**: en SMD es
  la mejor variante. Lo que hace daño ahí es **M3** (anomalías largas, sección 6.5).
* El **F1 sin etiquetas** supera al del artículo con etiquetas en sintéticos y contaminación, empata en el
  semisintético y **pierde en los datos reales** (sección 4: el umbral IQR marca de más cuando las anomalías son raras).

In [ ]:
def veredicto(h):
    if h["p_holm"] < 0.05 and h["mean"] > 0:
        return "✅ sí (significativa)"
    if h["p_holm"] < 0.05:
        return "❌ no: es PEOR"
    return "➖ no significativa"

EXPLICA = {"H1": "¿El score del final es mejor que el del artículo? (AUPRC)",
           "H2": "¿El final SIN etiquetas detecta al menos tan bien como el artículo CON etiquetas? (F1)",
           "H3a": "¿El recorte (M2) aporta encima de M3? (AUPRC)",
           "H3b": "¿El recorte aporta encima de M1+M3? (F1)",
           "H4": "Con más anomalías, ¿el final se degrada menos que el artículo? (caída de AUROC)",
           "H4r": "Sobre fondo real, con más anomalías, ¿el final se degrada menos? (caída de AUROC)",
           "H5a": "¿M3 sola mejora al artículo? (AUROC)",
           "H5b": "¿M3 sola mejora al artículo? (AUPRC)",
           "H5c": "Sobre fondo real, con más anomalías, ¿M3 se degrada menos? (caída de AUROC)"}
if BLQ:
    filas = []
    for b in BLQ:
        nb = RESULTADOS[b]
        for h in nb["hyp"] + nb.get("hyp_h5", []):
            filas.append({"bloque": BLOQUES[b], "hip.": h["h"], "pregunta": EXPLICA.get(h["h"], h["h"]),
                          "diferencia media": f"{h['mean']:+.3f}", "IC 95 %": f"[{h['lo']:+.3f}, {h['hi']:+.3f}]",
                          "gana / total": f"{h['wins']} / {h['n']}",
                          "p (Holm)": f"{h['p_holm']:.1e}" if h["p_holm"] < 1e-3 else f"{h['p_holm']:.3f}",
                          "¿se sostiene?": veredicto(h)})
    display(Markdown("### 9.2 Todas las hipótesis, bloque por bloque"))
    display(pd.DataFrame(filas).set_index(["bloque", "hip."]))

**Cómo leer la tabla de hipótesis:**
* **diferencia media**: cuánto mejora (positivo) o empeora (negativo) en promedio.
* **IC 95 %**: si no incluye el 0, la diferencia es consistente.
* **gana / total**: en cuántas unidades ganó. En SMD y SMAP+MSL la unidad es la serie (promedio de 5 semillas);
  en los demás, cada (serie, semilla).
* **p (Holm)**: menor que 0.05 → significativa. Valores como `1e-60` quieren decir que es prácticamente imposible
  que sea suerte.

**Resumen honesto:**
* ✅ **Contaminación:** todas las hipótesis se sostienen con p < 10⁻¹⁰. Es el resultado más fuerte del proyecto.
* ✅ **Sintéticos:** todas se sostienen.
* ✅ **Semisintético:** H1, H3b, H4r, H5 y H5c se sostienen; H2 empata (sin etiquetas ≈ con etiquetas); H3a sale al revés.
* ⚖️ **Reales del artículo:** H1 sí (gana en las 40 corridas), H2 **no** (el umbral sin etiquetas pierde en F1).
* ❌ **SMD:** M3 pierde en 27 de 28 máquinas; el final no supera al artículo.
* ➖ **SMAP+MSL:** casi todo empata; el recorte sí aporta un poco (H3a).

In [ ]:
MIX = {"01234": "todas las anomalías", "01": "sólo puntuales", "234": "sólo de patrón"}
for bq in ("contam", "semi"):
    if not RESULTADOS.get(bq):
        continue
    curvas = RESULTADOS[bq]["curves"]
    display(Markdown(f"### 9.3 {BLOQUES[bq]}: AUROC al aumentar la proporción de anomalías"))
    fig, axes = plt.subplots(1, len(curvas), figsize=(15, 3.6), sharey=True)
    for ax, mix in zip(np.atleast_1d(axes), [m_ for m_ in ("01234", "01", "234") if m_ in curvas]):
        for f_ in curvas[mix]:
            niveles = [k for k in f_ if k.endswith("%")]
            xs = [float(k[:-1]) for k in niveles]
            es_final = f_["Método"] == FINAL
            ax.plot(xs, [float(f_[k]) for k in niveles], "o-" if es_final else ".-", lw=2.5 if es_final else 1,
                    color="#c0392b" if es_final else None, label=CORTO.get(f_["Método"], f_["Método"]))
        ax.set_title(MIX.get(mix, mix)); ax.set_xlabel("% de anomalías (nominal)")
    np.atleast_1d(axes)[0].set_ylabel("AUROC"); np.atleast_1d(axes)[-1].legend(fontsize=7, loc="lower left")
    plt.suptitle("Más plano = más robusto a la contaminación"); plt.tight_layout(); plt.show()
    caidas = pd.DataFrame(EXTRA.get("caidas", {}).get(bq, []))
    if len(caidas):
        caidas = caidas.rename(columns={"method": "método", "drop": "caída de AUROC 1 % → 30 %"}).set_index("método")
        display(caidas.round(3))

**Cómo leer las curvas de contaminación:**
* **Contaminación sintética, todas las anomalías:** el AUROC del artículo cae 0.170 de 1 % a 30 %; el del método
  final, sólo 0.043.
* **Sólo puntuales:** el método final queda **plano**. Es el caso ideal para M3: anomalías cortas y mucho halo.
* **Sólo de patrón:** M3 sola queda **por debajo** del artículo. El final (con el recorte) lo alcanza en AUROC y lo
  supera en AUPRC, pero no en F1.
* **Sobre fondo real (semisintético):** el mismo patrón, con diferencias más pequeñas, porque ahí el artículo casi
  no se degrada. Ojo: en ese bloque **LOF** y el **error del LSTM** son mejores que cualquier variante de TracIn.

In [ ]:
if RESULTADOS.get("real"):
    display(Markdown("### 9.4 Los 4 datasets reales del artículo (10 semillas cada uno)"))
    pdx = RESULTADOS["real"]["per_ds"]
    filas = []
    for ds, mm in pdx.items():
        filas.append({"dataset": ds, "AUPRC artículo": mm[ART_S]["auprc"], "AUPRC M3": mm[M3N]["auprc"],
                      "AUPRC M1+M2": mm[M1M2N]["auprc"], "AUPRC FINAL": mm[FINAL]["auprc"],
                      "F1 artículo (con etiquetas)": mm[ART_D]["f1_bin"], "F1 FINAL (sin etiquetas)": mm[FINAL]["f1_bin"],
                      "AUPRC error LSTM": mm["Error del LSTM (PD) + IQR"]["auprc"]})
    display(pd.DataFrame(filas).set_index("dataset").round(3))

**Cómo leer los reales del artículo:**
* El método final mejora el AUPRC en **los 4 datasets** (10 de 10 semillas en cada uno): mucho en Credit Card y
  GECCO, algo en SWAN-SF y casi nada en CICIDS, donde **ningún método funciona** (AUPRC ≈ 0.02, igual que el azar).
* **No todas las mejoras ayudan en todos lados:** M3 ayuda en Credit Card y SWAN-SF, pero perjudica en GECCO. Ahí
  la mejora viene del recorte (M1+M2).
* En **F1** el artículo con etiquetas gana, porque estas anomalías son muy raras (0.2–1.3 %) y el IQR marca de más
  (sección 4).

In [ ]:
filas = []
for bq in ("smd", "nasa"):
    if not RESULTADOS.get(bq):
        continue
    mm = RESULTADOS[bq]["methods"]
    for m in (ART_S, ART_D, "M1 · umbral IQR", "M2 · recorte z>2", M3N, M1M2N, FINAL, "Error del LSTM (PD) + IQR", "LOF + IQR", "IForest + IQR"):
        if m in mm:
            filas.append({"benchmark": f"{BLOQUES[bq]} ({RESULTADOS[bq]['n_units']} series)", "método": CORTO.get(m, m),
                          "AUROC": mm[m]["auroc"], "AUPRC": mm[m]["auprc"], "F1": mm[m]["f1_bin"]})
if filas:
    display(Markdown("### 9.5 Benchmarks reales adicionales: SMD y SMAP+MSL (promedio por serie)"))
    display(pd.DataFrame(filas).set_index(["benchmark", "método"]).round(3))

**Cómo leer SMD y SMAP+MSL (el resultado negativo más importante):**
* En **SMD**, M3 sola pierde frente al artículo en **27 de 28 máquinas** (AUROC −0.023, AUPRC −0.054). El método
  final hereda esa pérdida (AUPRC 0.392 frente a 0.454). En cambio **M1+M2** (recorte sin M3) es la mejor variante
  (AUPRC 0.472).
* En **SMAP+MSL** casi todo empata alrededor de AUROC 0.6 (cerca del azar). Aquí el mejor es Isolation Forest.
* **¿Por qué?** Las anomalías etiquetadas en estos benchmarks son **tramos larguísimos**: en SMD el 98 % de los
  instantes anómalos están en eventos de 10 pasos o más (la mitad en eventos de más de 400 pasos); en SMAP/MSL, el
  100 % (la mitad en eventos de más de 2 700 pasos). Es justo el caso de la sección 6.5, donde M3 pierde la
  evidencia acumulada.
* **Conclusión práctica:** si sabes que tus anomalías son tramos largos, usa **M1 + M2 sin M3**.

In [ ]:
if EXTRA.get("h6_series"):
    display(Markdown("### 9.6 H6: ¿M3 ayuda más en las series reales con más anomalías?"))
    h6 = pd.DataFrame(EXTRA["h6_series"])
    rho, p = spearmanr(h6["anom_rate"], h6["d_auprc_m3"])
    fig, axes = plt.subplots(1, 2, figsize=(13, 3.6))
    for ax, col, nom in ((axes[0], "d_auroc_m3", "ΔAUROC (M3 − artículo)"), (axes[1], "d_auprc_m3", "ΔAUPRC (M3 − artículo)")):
        for blk, c in (("smd", "#1f77b4"), ("nasa", "#d62728")):
            d_ = h6[h6.block == blk]
            ax.scatter(100 * d_.anom_rate, d_[col], s=14, color=c, alpha=0.7, label="SMD" if blk == "smd" else "SMAP+MSL")
        ax.axhline(0, color="k", lw=0.8); ax.set_xscale("log"); ax.set_xlabel("% de instantes anómalos de la serie (escala log)")
        ax.set_title(nom); ax.legend(fontsize=8)
    plt.tight_layout(); plt.show()
    print(f"Correlación de Spearman entre % de anomalías y ventaja de M3 (AUPRC): ρ = {rho:+.2f}, p = {p:.1e}, n = {len(h6)} series")
    display(pd.DataFrame(RESULTADOS["h6"]["cuartiles"]).rename(columns={
        "anom_rate": "cuartil de % anomalías", "tasa_min": "desde", "tasa_max": "hasta", "d_auroc_m3": "ΔAUROC M3",
        "d_auprc_m3": "ΔAUPRC M3", "gana": "series donde gana M3", "n": "series"}).round(3))

**Cómo leer H6:** cada punto es una serie real. Con pocas anomalías (izquierda), M3 suele perder; con muchas
(derecha), se acerca a cero o gana. La correlación es positiva y muy significativa. Esto **confirma en datos
reales** el mecanismo del halo: cuantas más anomalías, más halo, y más útil es M3. Pero ojo: en estos benchmarks
la contaminación alta **sólo neutraliza** la desventaja de M3 con eventos largos; no la convierte en ventaja.

In [ ]:
if EXTRA.get("halo"):
    display(Markdown("### 9.7 El mecanismo medido en todos los bloques (método del artículo vs M3, mismo entrenamiento)"))
    filas = []
    for b, rows in EXTRA["halo"].items():
        for r_ in rows:
            if r_["train"] == "base":
                filas.append({"bloque": BLOQUES.get(b, b), "proyección": "Ec. 4 (artículo)" if r_["proj"] == "win" else "M3 (objetivo)",
                              "AUROC anom. vs halo": r_["auroc_anom_vs_halo"], "AUROC anom. vs lejanas": r_["auroc_anom_vs_far"],
                              "% del top-k que es halo": 100 * r_["topk_halo_share"]})
    display(pd.DataFrame(filas).set_index(["bloque", "proyección"]).round(3))

**Cómo leer la tabla del mecanismo:** el "top-k" son los *k* instantes con mayor score (*k* = número de anomalías).
Si una buena parte de ellos son normales del halo, son falsas alarmas causadas por la proyección.
* En todos los bloques, **M3 sube el AUROC contra el halo** y **baja el % del top-k que es halo**: el halo existe y
  M3 lo quita.
* Pero en bloques con anomalías largas (SMD, SMAP+MSL) el AUROC **contra lejanas** baja un poco con M3, y el halo
  es una fracción muy pequeña de las normales, así que pesa más lo segundo.

### 9.8 En una frase: qué funciona y qué no
| ✅ Se sostiene | ❌ No se sostiene / límites |
|---|---|
| La combinación mejora mucho el score con contaminación (AUPRC +0.33, 402 de 420) | En SMD el final queda por debajo del artículo (M3 perjudica con anomalías largas) |
| Con contaminación se degrada 4 veces menos que el artículo (0.043 vs 0.170) | En SMAP+MSL no hay diferencia; nadie funciona bien (AUROC ≈ 0.6) |
| Sin etiquetas supera en F1 al artículo con etiquetas (sintéticos y contaminación) | Con anomalías muy raras (< 2 %), el umbral IQR pierde en F1 frente al umbral con etiquetas |
| Mejora el AUPRC en los 4 reales del artículo (40 de 40) | M3 sola empeora con anomalías de patrón; necesita al recorte |
| El mecanismo (halo) está identificado y medido, y H6 lo confirma en 109 series reales | En el semisintético, LOF y el error del LSTM superan a todas las variantes de TracIn |

# PARTE E · Úsalo tú

## 10. Pruébalo en datos reales

### 10.1 Una máquina del SMD, descargada en vivo
El **SMD** está publicado en GitHub (repositorio *OmniAnomaly*). La celda siguiente descarga los datos de prueba de
una máquina con sus etiquetas, corre **las cuatro variantes** y las compara:
* **Artículo**: Ec. 4, umbral dinámico **con** etiquetas;
* **M3 sola**: proyección al objetivo, umbral IQR;
* **M1+M2**: recorte adaptativo con la Ec. 4, umbral IQR (la recomendada para anomalías largas);
* **Final**: M1+M2+M3.

Truco: las cuatro salen de **sólo dos entrenamientos** (el normal y el de 3 rondas de recorte), porque M3 y la
Ec. 4 son sólo dos formas de proyectar el mismo score.

> Para que corra rápido en Colab usamos el gradiente de la **última capa**, batch 256 y máximo 30 épocas. En la
> evaluación grande usamos el gradiente de todos los parámetros y 50 épocas, así que los números pueden variar un
> poco, pero la tendencia es la misma.

Puedes cambiar `MAQUINA` (de `machine-1-1` a `machine-1-8`, `machine-2-1` a `machine-2-9`, `machine-3-1` a
`machine-3-11`).

In [ ]:
URL_SMD = "https://raw.githubusercontent.com/NetManAIOps/OmniAnomaly/master/ServerMachineDataset"
CFG_RAPIDA = Cfg(batch=256, max_epochs=30)

def bajar_smd(maquina, parte="test"):
    leer = lambda ruta: np.loadtxt(io.StringIO(urllib.request.urlopen(f"{URL_SMD}/{ruta}/{maquina}.txt", timeout=60).read().decode()), delimiter=",")
    X = leer(parte).astype(np.float32)
    y = leer("test_label").astype(np.int8) if parte == "test" else None
    return X, y

def cuatro_variantes(X, y, semilla=0, cfg=None, titulo=""):
    cfg = cfg or CFG_RAPIDA
    n = len(y); m = valid_mask(n); yb = np.asarray(y)[m].astype(bool)
    t = time.time()
    s0 = run(X, L, P, seed=semilla, cfg=cfg)["si"]; s = s0
    for k in range(3):
        s = run(X, L, P, seed=semilla, cfg=cfg, weights=adapt_weights(s))["si"]
        print(f"  ronda de recorte {k + 1}/3 lista ({time.time() - t:.0f} s)")
    sc = {"Artículo (con etiquetas)": proyectar(s0, n, False), "M3 sola": proyectar(s0, n, True),
          "M1+M2 (sin M3)": proyectar(s, n, False), "FINAL (M1+M2+M3)": proyectar(s, n, True)}
    filas = {}
    for k, v in sc.items():
        thr = umbral_dinamico(yb, v[m]) if k.startswith("Artículo") else thr_iqr(v[m])
        filas[k] = evaluar(y, v, thr)
    fig, axes = plt.subplots(3, 1, figsize=(13, 7), sharex=True)
    dibujar_serie(X, y, titulo, axes[0])
    for ax, k in zip(axes[1:], ("Artículo (con etiquetas)", "FINAL (M1+M2+M3)")):
        v = sc[k]; thr = umbral_dinamico(yb, v[m]) if k.startswith("Artículo") else thr_iqr(v[m])
        dibujar_deteccion(y, v, thr, k, ax)
    plt.tight_layout(); plt.show()
    tabla = pd.DataFrame(filas).T
    display(tabla.round(3))
    largos = [b - a for a, b in intervalos(y)]
    print(f"{len(largos)} eventos anómalos; largo mediano {np.median(largos):.0f} pasos; {np.mean(y):.1%} de instantes anómalos.")
    return tabla

MAQUINA = "machine-1-1"
try:
    Xs_, ys_ = bajar_smd(MAQUINA)
    print(f"{MAQUINA}: {Xs_.shape[0]} instantes × {Xs_.shape[1]} métricas")
    _ = cuatro_variantes(Xs_, ys_, titulo=f"SMD {MAQUINA} (primeras 6 métricas)")
except Exception as e:
    print("No se pudo descargar (¿sin internet?):", e)

**Qué esperar:** en la mayoría de las máquinas del SMD, el **artículo** o **M1+M2** quedan arriba en AUPRC y el
final un poco abajo, igual que en la evaluación grande. Fíjate en la última línea: los eventos anómalos duran
decenas o cientos de pasos. **Éste es el caso donde no conviene M3.**

### 10.2 Semisintético en vivo: fondo real + anomalías cortas inyectadas
Ahora tomamos 10 000 pasos de la parte de **entrenamiento** de la misma máquina (que no trae anomalías
etiquetadas) e **inyectamos** anomalías **puntuales** con nuestra función `inject` (de `src/data_bench.py`):
* **tipo 0:** un valor fuera del rango del canal;
* **tipo 1:** un valor lejos de su vecindad, pero dentro del rango.

Cada anomalía cae en un canal elegido al azar. Así tenemos un fondo real con anomalías cortas, el caso donde la
combinación sí debería ayudar.

In [ ]:
try:
    Xbg, _ = bajar_smd(MAQUINA, parte="train")
    bg = standardize(Xbg[:10_000])
    Xi, yi = inject(bg, "01", 0.20, seed=123)          # 20 % de anomalías puntuales inyectadas
    print(f"Fondo real {bg.shape}, {yi.mean():.1%} de instantes con anomalía inyectada")
    _ = cuatro_variantes(Xi, yi, titulo=f"Fondo real ({MAQUINA}, entrenamiento) + 20 % de anomalías puntuales")
except Exception as e:
    print("No se pudo descargar (¿sin internet?):", e)

**Qué esperar:** aquí M3 y el método final deberían quedar **por encima** del artículo en AUROC y AUPRC, igual que
en el bloque semisintético de la evaluación grande (mezcla de anomalías puntuales).

### 10.3 Con tu propio archivo CSV
Prepara un CSV donde **cada fila es un instante** y **cada columna una medición** (todas numéricas). Si tienes
etiquetas, ponlas en una columna (por ejemplo `label`, con 1 = anómalo y 0 = normal); si no, el cuaderno sólo te
dará el score y los instantes marcados.

1. Ejecuta la celda siguiente y elige tu archivo.
2. Cambia `COLUMNA_ETIQUETA` si tu columna se llama distinto (o pon `None` si no tienes etiquetas).
3. Cambia `USAR_M3` a `False` si tus anomalías son tramos largos (decenas de pasos o más).

In [ ]:
# @title Subir y probar tu CSV (opcional)
COLUMNA_ETIQUETA = "label"     # None si no hay etiquetas
USAR_M3 = True                 # False si tus anomalías son tramos largos

def probar_csv(ruta, col_etiqueta=COLUMNA_ETIQUETA, usar_m3=USAR_M3, semilla=0):
    df = pd.read_csv(ruta)
    y = None
    if col_etiqueta and col_etiqueta in df:
        y = df.pop(col_etiqueta).astype(int).values
    X = df.select_dtypes("number").ffill().fillna(0).values.astype(np.float32)
    print(f"{len(X)} instantes × {X.shape[1]} columnas numéricas")
    s, thr, s_art = metodo_final(X, semilla, M1=True, M2=True, M3=usar_m3, cfg=CFG_RAPIDA)
    marcados = np.flatnonzero(s >= thr)
    print(f"El método marcó {len(marcados)} instantes ({len(marcados) / len(X):.1%}). Primeros: {marcados[:20].tolist()}")
    fig, ax = plt.subplots(figsize=(12, 3))
    if y is not None:
        dibujar_deteccion(y, s, thr, "Tu serie: score del método final", ax)
        plt.show()
        m = valid_mask(len(y))
        display(pd.DataFrame([evaluar(y, s_art, umbral_dinamico(np.asarray(y)[m].astype(bool), s_art[m])), evaluar(y, s, thr)],
                             index=["Artículo (con etiquetas)", "Final (sin etiquetas)"]).round(3))
    else:
        ax.plot(s, lw=0.8); ax.axhline(thr, color="k", ls="--"); ax.set_title("Tu serie: score y umbral IQR"); plt.show()
    return s, thr

try:
    from google.colab import files
    subido = files.upload()
    for nombre in subido:
        _ = probar_csv(nombre)
except ImportError:
    print("Fuera de Colab: llama a probar_csv('ruta/a/tu_archivo.csv').")
except Exception as e:
    print("No se subió ningún archivo o hubo un problema:", e)

## 11. Cómo está organizado el código del proyecto

Todo vive en la carpeta `experimentos/`:

| Archivo / carpeta | Qué contiene |
|---|---|
| `PROTOCOLO_FINAL.md` | el protocolo escrito **antes** de correr: hipótesis, métodos, datos; y las desviaciones anotadas |
| `src/synthetic.py` | el generador de series sintéticas (Lai et al., 2021) |
| `src/replica/lab.py` | la réplica del artículo: el LSTM, el entrenamiento y TracIn (`run`) |
| `src/common.py` | proyecciones (`project_sum` = Ec. 4, `project_targets_mean` = M3), umbral IQR (`thr_iqr`) |
| `src/forecaster.py` | z robusto y pesos para M2 suave / recorte (`weights_from_influence`) |
| `src/final/core.py` | recorte adaptativo (`adapt_weights`) y `all_variants`, que corre todas las variantes de una vez |
| `src/data_bench.py` | descarga y preparación de SMD y SMAP/MSL; `inject` para el semisintético |
| `src/baselines.py` | LOF e Isolation Forest |
| `src/final/run_synth.py`, `run_contam.py`, `run_real.py`, `run_bench.py` | corren cada bloque y guardan los scores en `results/final/<bloque>/*.npz` |
| `src/final/evaluate.py` | calcula todas las métricas de cada corrida → `results/final/<bloque>_eval.csv` |
| `src/final/stats.py` | pruebas de hipótesis → `results/final/final_numbers.json` y `results/final/tables/*.md` |
| `src/final/paper.py` | tablas LaTeX, figuras, Friedman/Nemenyi y el análisis del halo → `results/final/paper/` |
| `src/final/macros.py` | pasa las cifras a macros LaTeX, para que el texto cite números calculados y no copiados a mano |
| `results/final/paper/resultados.pdf` | la sección de resultados lista para el artículo |
| `colab/build_paso_a_paso.py` | genera **este** cuaderno |

### Cómo se reproduce todo (en una máquina con GPU)
```bash
cd experimentos/src/final
python run_synth.py --seeds 20 --workers 18
python run_contam.py --seeds 20 --workers 18
python run_real.py --seeds 10 --claim          # se pueden lanzar varios en paralelo
python ../data_bench.py                         # prepara SMD, SMAP/MSL y fondos (datos en data/raw/)
python run_bench.py --blocks smd,nasa,semi      # GPU; con --device cpu también funciona
for b in synth contam real smd nasa semi; do python evaluate.py $b --workers 20; done
python stats.py && python paper.py --workers 20 && python macros.py
cd ../../results/final/paper && pdflatex resultados.tex
```
Cada corrida está fijada por su semilla: repetirla da exactamente el mismo resultado.

## 12. Resumen, glosario y preguntas que te pueden hacer

### El proyecto en cinco frases
1. El artículo detecta anomalías midiendo cuánto le costó a un LSTM aprender cada pedazo de la serie (TracIn).
2. **Mejora 1:** cambiamos el umbral que se calibra con etiquetas por la regla IQR, que no las necesita.
3. **Mejora 2:** con muchas anomalías el modelo las memoriza; quitamos del entrenamiento lo sospechoso (con el mismo
   umbral IQR) y reentrenamos tres veces.
4. **Mejora 3:** el score de cada ventana va sólo al instante que predijo. Así desaparece el "halo" de falsas
   alarmas que la fórmula del artículo crea alrededor de cada anomalía, y que crece con la contaminación.
5. **Método final:** las tres juntas, sin etiquetas. Funciona muy bien con contaminación y anomalías cortas; con
   anomalías largas (SMD, SMAP/MSL) conviene quitar la Mejora 3.

### Cómo explicarlo en dos minutos
> "El artículo usa TracIn: entrena un LSTM a predecir la serie y mide cuánto le cuesta aprender cada ventana. Le
> encontramos tres problemas: su mejor umbral usa etiquetas, con muchas anomalías el modelo se las aprende, y su
> forma de pasar de ventanas a instantes contagia el score de una anomalía a sus vecinos normales. Lo arreglamos con
> un umbral IQR, con rondas de recorte de las ventanas sospechosas y proyectando el score sólo al instante predicho.
> Con un protocolo escrito antes de correr, en 1 755 corridas, la combinación mejora el AUPRC hasta en 0.33 y se
> degrada cuatro veces menos con la contaminación. Mostramos que la causa principal de la degradación del artículo
> es ese contagio, al que llamamos halo. Pero en benchmarks con anomalías muy largas la proyección al objetivo no
> ayuda, y ahí conviene usar sólo el umbral y el recorte."

### Glosario
| Término | En simple |
|---|---|
| Serie de tiempo | Mediciones tomadas una tras otra en el tiempo |
| Canal | Cada una de las cosas que se miden a la vez (cada columna de `X`) |
| Anomalía puntual / de patrón | Un solo instante raro / un tramo cuya forma es rara |
| No supervisado | El algoritmo nunca ve las etiquetas de qué es anómalo |
| Contaminación | Proporción de anomalías en los datos |
| LSTM | Red neuronal para secuencias; aquí predice el siguiente instante |
| Gradiente | Hacia dónde y cuánto mover los parámetros para reducir el error de un ejemplo |
| Época | Una pasada completa por todos los datos de entrenamiento |
| Tasa de aprendizaje (η) | Tamaño de cada paso de ajuste |
| Self-influence (TracIn) | Suma, sobre las épocas, de η · ‖gradiente‖² de cada ventana: alto = le costó aprenderla |
| Ventana | Pedazo de L + P instantes consecutivos (aquí 4 de entrada + 1 a predecir) |
| Objetivo | El instante que una ventana predice |
| Proyección | Cómo pasar de un score por ventana a un score por instante |
| Ec. 4 | La proyección del artículo: cada instante suma el score de todas las ventanas que lo contienen |
| Halo | Normales a 1–4 pasos de una anomalía, que con la Ec. 4 heredan su score alto |
| Umbral | Valor a partir del cual se marca "anómalo" |
| IQR | Rango intercuartil, $Q_3 - Q_1$; la regla $Q_3 + 1.5\,IQR$ es la del diagrama de caja |
| z robusto | Cuántas "desviaciones" está algo de la mediana, usando mediana y MAD |
| Recorte / reponderación | Quitar / bajar el peso de ventanas sospechosas al reentrenar |
| AUROC | Probabilidad de que una anomalía tenga más score que un normal (0.5 = azar) |
| AUPRC | Qué tan arriba quedan las anomalías al ordenar por score (sensible a pocas anomalías) |
| F1 | Equilibrio entre precisión y exhaustividad con un umbral dado |
| Semilla | Número que fija el azar para poder repetir un experimento exactamente |
| Wilcoxon pareado | Prueba que compara dos métodos sobre las mismas series |
| Holm | Corrección del p-valor cuando se hacen varias pruebas a la vez |
| IC 95 % bootstrap | Rango probable de la mejora real, obtenido remuestreando |
| Preregistro | Escribir hipótesis y análisis antes de ver los resultados |

### Preguntas que te pueden hacer (y cómo contestarlas)
* **"¿La proyección al objetivo es invento suyo?"** No: es la práctica estándar en detectores basados en predicción.
  Lo nuestro es mostrar que la Ec. 4 del artículo crea un halo de falsos positivos, medir que ese halo explica la
  caída con la contaminación y combinarla con las otras dos mejoras.
* **"¿Por qué IQR con 1.5?"** Es el valor estándar del diagrama de caja. Lo fijamos antes de ver resultados y no lo
  ajustamos.
* **"¿El recorte no quita anomalías que luego ya no detectan?"** No: se quitan del *entrenamiento*, pero se les sigue
  calculando el score. Justamente por no haberlas aprendido, su score queda más alto.
* **"¿Funciona en datos reales?"** Depende del tipo de anomalía. En los 4 datasets del artículo mejora el AUPRC en las
  40 corridas. En SMD y SMAP/MSL, con anomalías de cientos de pasos, no: ahí conviene quitar M3. Y en 109 series
  reales, la ventaja de M3 crece con la proporción de anomalías (H6).
* **"¿Por qué el umbral sin etiquetas pierde en F1 en reales?"** Porque las anomalías son muy raras (0.2–1.3 %) y
  la regla IQR marca 10–17 % de los instantes. El umbral del artículo gana porque mira las respuestas.
* **"¿Cómo saben que no es suerte?"** Pruebas de Wilcoxon pareadas sobre cientos de series, corrección de Holm,
  intervalos de confianza y un protocolo escrito antes de correr.
* **"¿Le ganan a los métodos clásicos?"** En contaminación sintética sí (salvo Isolation Forest, que cae poco pero
  parte de un AUROC mucho más bajo). En el semisintético, LOF y el error del LSTM son mejores que todas las
  variantes de TracIn, incluida la del artículo. Lo decimos tal cual.
* **"¿Qué harían después?"** Una proyección híbrida que use M3 en anomalías cortas y la suma en eventos largos, y un
  umbral sin etiquetas que funcione mejor con anomalías muy raras.